# MatrixDB on Google Colab

GPU-accelerated database engine: page-ownership point ops + a resident-column `u32x4` scan kernel, behind a lock-free SPSC ring + dual-trigger batcher.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

This notebook writes its own source files (cells below), runs the CPU coverage test, builds with `nvcc`, and runs. No uploads needed. Success ends with `Scan result sum: 83886070 (oracle 83886070)` — asserts fire on any mismatch.

## 1. Confirm a GPU is attached

In [ ]:
!nvcc --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Write the source files

In [ ]:
%%writefile types.hpp
#pragma once

#include <cstdint>
#include <cstddef> // ponytail: spec wrote <csize_t> (does not exist); size_t lives in <cstddef>

#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    // Apple Silicon L1 cache lines are typically 128 bytes
    constexpr size_t MATRIX_CACHE_LINE = 128;
#else
    // Standard Intel/AMD x86_64 cache lines are 64 bytes
    constexpr size_t MATRIX_CACHE_LINE = 64;
#endif

/**
 * @brief Represents a single transaction query record.
 * Designed with descending structural member alignment to prevent internal compiler padding.
 * Aligned to target CPU cache line boundaries to eliminate false sharing.
 */
struct alignas(MATRIX_CACHE_LINE) DatabaseQuery {
    uint64_t query_id;
    uint64_t timestamp_us;
    uint64_t transaction_id;
    uint32_t opcode;
    uint32_t payload_size;
    uint8_t payload[32];

    static constexpr size_t data_footprint =
        sizeof(uint64_t) * 3 +
        sizeof(uint32_t) * 2 +
        sizeof(uint8_t) * 32;

    static constexpr size_t padding_needed =
        (MATRIX_CACHE_LINE > data_footprint) ? (MATRIX_CACHE_LINE - data_footprint) : 0;

    // Explicit structural padding to align with target L1 cache line size
    uint8_t padding[padding_needed > 0 ? padding_needed : 1];
};

static_assert((sizeof(DatabaseQuery) % MATRIX_CACHE_LINE) == 0,
    "DatabaseQuery structure violates cache-line alignment constraints.");

// Operational instruction carried in DatabaseQuery::opcode (spec: Read, Write, Scan).
enum MatrixOpcode : uint32_t {
    OP_READ  = 1,
    OP_WRITE = 2,
    OP_SCAN  = 3,
    OP_TXN_WRITE = 4,   // a transactional put: buffered on WAL replay until a commit marker
};

// Aggregate reduction carried by OP_SCAN (payload offset 16). AGG_COUNT==0 is the default, so a
// scan with no agg op set counts matches (the original behavior). SUM/MIN/MAX reduce the values
// matching the predicate (value > threshold).
enum MatrixAggOp : uint32_t {
    AGG_COUNT = 0,
    AGG_SUM   = 1,
    AGG_MIN   = 2,
    AGG_MAX   = 3,
};

// Columnar store + page-ownership layout (shard-per-thread).
// The keyspace (STORE_SLOTS) is split into PAGE_COUNT contiguous pages. Exactly one
// owner (one GPU thread) reads/writes a page's slots, so the same key always routes
// to the same owner: writes to a key serialize by ownership. No cross-thread conflict
// => no store atomics, no OCC, no delta log. Each page has a single owner; pages are
// independent of one another (shared-nothing).
constexpr size_t MATRIX_STORE_SLOTS = 4096;                       // power of two
constexpr size_t MATRIX_STORE_MASK  = MATRIX_STORE_SLOTS - 1;
constexpr size_t MATRIX_PAGE_COUNT  = 1024;                       // owner threads; the tuning knob
constexpr size_t MATRIX_PAGE_SIZE   = MATRIX_STORE_SLOTS / MATRIX_PAGE_COUNT; // slots per page
constexpr size_t MATRIX_BATCH_MAX   = 65536;                      // sweep ceiling / scratch buffer size

// Resident analytical column (the GPU-DB's actual data): a uint32 column that lives
// in VRAM/RAM and is scanned in place by OP_SCAN — never shipped per query. 16M values
// = 64MB, well past CPU cache so the GPU bandwidth advantage is real. Filled value[i]=i
// so a threshold T yields a known count (SCAN_SIZE-1-T) for oracle verification.
constexpr size_t MATRIX_SCAN_COLUMN_SIZE = 1u << 24;             // 16,777,216 values (divisible by 4 for uint4)

static_assert(MATRIX_STORE_SLOTS % MATRIX_PAGE_COUNT == 0,
    "store slots must divide evenly into pages so each page owns a contiguous slice");


In [ ]:
%%writefile ring_buffer.hpp
#pragma once

#include "types.hpp"
#include <atomic>
#include <array>
#include <utility>

/**
 * @brief Ultra-low latency, lock-free, zero-allocation SPSC Ring Buffer.
 * Enforces power-of-two capacities to replace expensive modulo operations with bitwise AND masking.
 */
template <typename T, size_t Capacity>
class SPSCRingBuffer {
    static_assert((Capacity & (Capacity - 1)) == 0, "SPSCRingBuffer Capacity must be a power of two.");
    static_assert(Capacity >= 2, "SPSCRingBuffer Capacity must be at least two elements.");

public:
    SPSCRingBuffer()
        : head_(0)
        , tail_(0)
        , cached_head_(0)
        , cached_tail_(0) {}

    ~SPSCRingBuffer() = default;

    SPSCRingBuffer(const SPSCRingBuffer&) = delete;
    SPSCRingBuffer& operator=(const SPSCRingBuffer&) = delete;
    SPSCRingBuffer(SPSCRingBuffer&&) = delete;
    SPSCRingBuffer& operator=(SPSCRingBuffer&&) = delete;

    /**
     * @brief Emplaces a new item at the tail of the buffer. Only the Producer may call this method.
     */
    template <typename... Args>
    bool try_emplace(Args&&... args) {
        const size_t current_tail = tail_.load(std::memory_order_relaxed);

        // Check local cached head before loading the atomic head_ cursor from shared memory
        if (current_tail - cached_head_ >= Capacity) {
            cached_head_ = head_.load(std::memory_order_acquire);
            if (current_tail - cached_head_ >= Capacity) {
                return false; // Queue is genuinely full
            }
        }

        const size_t index = current_tail & MASK;
        buffer_[index] = T{std::forward<Args>(args)...};

        tail_.store(current_tail + 1, std::memory_order_release);
        return true;
    }

    /**
     * @brief Pops an item from the head of the buffer. Only the Consumer may call this method.
     */
    bool try_pop(T& value) {
        const size_t current_head = head_.load(std::memory_order_relaxed);

        // Check local cached tail before loading the atomic tail_ cursor from shared memory
        if (current_head >= cached_tail_) {
            cached_tail_ = tail_.load(std::memory_order_acquire);
            if (current_head >= cached_tail_) {
                return false; // Queue is genuinely empty
            }
        }

        const size_t index = current_head & MASK;
        value = std::move(buffer_[index]);

        head_.store(current_head + 1, std::memory_order_release);
        return true;
    }

    [[nodiscard]] bool empty() const noexcept {
        return head_.load(std::memory_order_relaxed) == tail_.load(std::memory_order_relaxed);
    }

    [[nodiscard]] size_t size() const noexcept {
        const size_t t = tail_.load(std::memory_order_relaxed);
        const size_t h = head_.load(std::memory_order_relaxed);
        return (t >= h) ? (t - h) : 0;
    }

private:
    static constexpr size_t MASK = Capacity - 1;

    // Isolate variables on separate cache lines to eliminate false sharing
    alignas(MATRIX_CACHE_LINE) std::array<T, Capacity> buffer_{};
    alignas(MATRIX_CACHE_LINE) std::atomic<size_t> head_;
    alignas(MATRIX_CACHE_LINE) std::atomic<size_t> tail_;

    // Thread-local shadow indices
    alignas(MATRIX_CACHE_LINE) size_t cached_head_;
    alignas(MATRIX_CACHE_LINE) size_t cached_tail_;
};


In [ ]:
%%writefile compute.hpp
#pragma once

#include "types.hpp"
#include <cstddef>
#include <cstdint>
#include <cstring>
#include <limits>

// memcpy-based bit_cast (the well-defined type-pun): identical to std::bit_cast but works under C++17
// too. The engine packs typed results (double/int64) into a uint64 out vector via this; std::bit_cast
// would force C++20 on every build, but the GPU/CPU pipeline (main.cpp) compiles under nvcc -std=c++17.
template <typename To, typename From>
inline To matrix_bit_cast(const From& src) {
    static_assert(sizeof(To) == sizeof(From), "matrix_bit_cast requires equal sizes");
    To dst;
    std::memcpy(&dst, &src, sizeof(To));
    return dst;
}

/**
 * @brief Pure virtual interface defining the compute engine's entry point.
 * Enables zero-copy, raw pointer processing over batch slices.
 *
 * Two implementations satisfy this contract and are selected at compile time:
 *   - CPUMockEngine (compute_mock.cpp)  — default, runs anywhere, no GPU needed
 *   - CUDAGPUEngine (compute_cuda.cuh)  — built with nvcc when MATRIX_USE_CUDA is set
 *
 * Both use the page-ownership model: a key always routes to one owning page, so the
 * two backends produce byte-identical store contents (verifiable in main.cpp).
 */
class ComputeInterface {
public:
    virtual ~ComputeInterface() = default;
    virtual void execute_batch(DatabaseQuery* batch_array, size_t count) = 0;

    // Run a single OP_SCAN over the resident column (sets q.transaction_id to the count).
    // Separate from execute_batch so scans run on their own thread/queue and stream and
    // don't head-of-line-block point ops. Touches only scan-path state (disjoint from
    // the point-op store/counters), so the two may run concurrently.
    virtual void execute_scan(DatabaseQuery& q) = 0;

    virtual uint64_t reads() const = 0;
    virtual uint64_t writes() const = 0;
    virtual uint64_t commits() const = 0; // writes actually applied to the store

    // OP_SCAN result accounting: how many scan queries ran, and the running sum of their
    // filter-count results. main asserts the sum against a known oracle to prove scans
    // flowed ring -> batcher -> execute_batch -> (GPU) scan kernel correctly.
    virtual uint64_t scans() const = 0;
    virtual uint64_t scan_result_sum() const = 0;

    // Wall time the engine spent inside OP_SCAN work. Lets the pipeline report point-op
    // and scan throughput separately — a single 64MB scan costs ~1000x a point op, so
    // one combined "ops/sec" number is meaningless once scans are mixed in.
    virtual double scan_time_s() const = 0;

    // Pure kernel time for OP_SCAN work (GPU: cudaEvent-measured, excludes launch/sync/
    // copy and host jitter; CPU: same as the compute loop). scan_time_s() - this =
    // per-scan overhead. The split attributes the integrated-vs-standalone gap.
    virtual double scan_kernel_time_s() const = 0;

    // Order-independent fingerprint of the whole store (sum of slot values). Lets
    // main.cpp assert CPU and GPU reach byte-identical state — the real test of the
    // ownership model, not just matching counts.
    virtual uint64_t store_checksum() const = 0;

    // Scan benchmark: allocate n resident values (value[i]=i), then time a single
    // filter-count scan (count of value[i] > threshold) over that resident data.
    // alloc+fill are NOT timed — the point is "data lives here, query scans it".
    // This is the GPU's home turf (bandwidth over data too big for CPU cache),
    // the opposite of the point-op path. Returns seconds; sets out_count.
    // out_count is deterministic (value[i]=i) so it must match across CPU and GPU.
    virtual double benchmark_scan(size_t n, uint64_t threshold, uint64_t& out_count) = 0;

    // Same scan over a uint32 column. Scan is bandwidth-bound, so halving bytes/value
    // should ~double values/sec at the same GB/s — the columnar "narrowest type" win.
    // If GB/s holds vs the uint64 scan, we are at the bandwidth wall (vectorized loads
    // won't help); if it drops, narrower loads underfill the bus and vectorizing is next.
    virtual double benchmark_scan_u32(size_t n, uint32_t threshold, uint64_t& out_count) = 0;

    // Vectorized uint32 scan (4 values per load via uint4). Tests the memory-level
    // parallelism theory: if this beats the scalar u32 GB/s, narrow scalar loads were
    // underfilling the bus (MLP-bound) and wider loads are the fix. If it doesn't, the
    // kernel is ALU-bound and we stop optimizing. CPU may treat this same as scalar.
    virtual double benchmark_scan_u32x4(size_t n, uint32_t threshold, uint64_t& out_count) = 0;

    // Items-per-thread scan (register blocking, ITEMS independent loads/thread before
    // comparing). CUB's BlockReduce lever. Tests whether deeper memory-level parallelism
    // per thread beats both scalar and uint4. CPU delegates (compiler auto-vectorizes).
    virtual double benchmark_scan_ipt(size_t n, uint32_t threshold, uint64_t& out_count) = 0;
};

// Which page owns a query's key. Single source of truth for both engines and the
// kernel: slot = key & STORE_MASK, page = slot / PAGE_SIZE.
inline size_t matrix_page_of(uint64_t key) {
    return (key & MATRIX_STORE_MASK) / MATRIX_PAGE_SIZE;
}

// OP_SCAN carries its filter threshold in the query payload, and may target a specific
// catalog column: threshold at payload[0] (u32), column id at payload[8] (u64). Column
// id 0 == the legacy fixed scan column. payload begins 8-byte aligned (DatabaseQuery
// starts with u64 fields, payload is at offset 32), so the u64 at payload+8 is aligned.
// One codec used by both engines so they decode identically.
inline void matrix_set_scan_target(DatabaseQuery& q, uint32_t threshold, uint64_t column_id) {
    q.opcode = OP_SCAN;
    *reinterpret_cast<uint32_t*>(q.payload) = threshold;
    *reinterpret_cast<uint64_t*>(q.payload + 8) = column_id;
}
inline uint64_t matrix_get_scan_column_id(const DatabaseQuery& q) {
    return *reinterpret_cast<const uint64_t*>(q.payload + 8);
}
inline void matrix_set_scan_threshold(DatabaseQuery& q, uint32_t threshold) {
    matrix_set_scan_target(q, threshold, 0); // legacy: target the fixed column (id 0)
}
inline uint32_t matrix_get_scan_threshold(const DatabaseQuery& q) {
    return *reinterpret_cast<const uint32_t*>(q.payload);
}

// OP_SCAN's aggregate op lives at payload offset 16 (u32). Default 0 == AGG_COUNT (the original
// count semantics), so a query built with only set_scan_target/set_scan_threshold reduces by COUNT.
inline void matrix_set_scan_agg_op(DatabaseQuery& q, MatrixAggOp op) {
    *reinterpret_cast<uint32_t*>(q.payload + 16) = static_cast<uint32_t>(op);
}
inline MatrixAggOp matrix_get_scan_agg_op(const DatabaseQuery& q) {
    return static_cast<MatrixAggOp>(*reinterpret_cast<const uint32_t*>(q.payload + 16));
}

// Filtered reduction over a uint32 column: reduce the values matching the predicate
// (value > threshold) by `op`. One tight loop per op (dispatch OUTSIDE the loop, so the COUNT
// loop stays exactly as fast as the original scan). Empty match-set sentinels: COUNT/SUM -> 0,
// MIN -> UINT64_MAX, MAX -> 0 (matched values are > threshold >= 0, i.e. >= 1, so 0 / UINT64_MAX
// unambiguously signal "no match"). SUM accumulates in u64 (no overflow for the engine's columns).
inline uint64_t matrix_cpu_reduce(const uint32_t* v, size_t n, uint32_t threshold, MatrixAggOp op) {
    switch (op) {
        case AGG_SUM: { uint64_t s = 0; for (size_t i = 0; i < n; ++i) if (v[i] > threshold) s += v[i]; return s; }
        case AGG_MIN: { uint64_t m = UINT64_MAX; for (size_t i = 0; i < n; ++i) if (v[i] > threshold && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { uint64_t m = 0; for (size_t i = 0; i < n; ++i) if (v[i] > threshold && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { uint64_t c = 0; for (size_t i = 0; i < n; ++i) c += (v[i] > threshold); return c; }
    }
}

// Unfiltered scalar reduction (aggregate ALL values; the no-WHERE companion to matrix_cpu_reduce).
// COUNT -> n; SUM -> Σv (u64); MIN/MAX over all (empty sentinels MIN UINT64_MAX / MAX 0 reachable
// only when n==0). Separate from matrix_cpu_reduce (which always filters value>threshold) — clearer
// than threading if constexpr through its four per-op loops.
inline uint64_t matrix_cpu_reduce_all(const uint32_t* v, size_t n, MatrixAggOp op) {
    switch (op) {
        case AGG_SUM: { uint64_t s = 0; for (size_t i = 0; i < n; ++i) s += v[i]; return s; }
        case AGG_MIN: { uint64_t m = UINT64_MAX; for (size_t i = 0; i < n; ++i) if (v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { uint64_t m = 0; for (size_t i = 0; i < n; ++i) if (v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      return n;
    }
}

// Null-aware unfiltered scalar reduce over a uint32 column (DM-3j): rows where nulls[i] != 0 are SKIPPED
// (SQL NULL semantics — COUNT counts non-null, SUM/MIN/MAX ignore nulls). Empty/all-null sentinels match
// matrix_cpu_reduce (COUNT/SUM 0, MIN UINT64_MAX, MAX 0). `nulls` has one byte per row.
inline uint64_t matrix_cpu_reduce_all_nullable(const uint32_t* v, size_t n, MatrixAggOp op, const uint8_t* nulls) {
    switch (op) {
        case AGG_SUM: { uint64_t s = 0; for (size_t i = 0; i < n; ++i) if (!nulls[i]) s += v[i]; return s; }
        case AGG_MIN: { uint64_t m = UINT64_MAX; for (size_t i = 0; i < n; ++i) if (!nulls[i] && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { uint64_t m = 0; for (size_t i = 0; i < n; ++i) if (!nulls[i] && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { uint64_t c = 0; for (size_t i = 0; i < n; ++i) c += (nulls[i] == 0); return c; }
    }
}

// Per-column element type. U32 == 0 is the default (an untagged column is uint32). int64 columns
// (DM-3a) hold signed 64-bit values — negatives and values beyond UINT32_MAX, which uint32 cannot.
enum class MatrixType : uint32_t { U32 = 0, I64, F64 };   // F64 == 2 (DM-3e)

// Signed unfiltered scalar reduce over an int64 column (the int64 sibling of matrix_cpu_reduce_all).
// COUNT -> n; SUM -> Σv (int64; see the overflow note in the design); MIN/MAX over all (empty
// sentinels MIN INT64_MAX / MAX INT64_MIN, reachable only when n == 0).
inline int64_t matrix_cpu_reduce_all_i64(const int64_t* v, size_t n, MatrixAggOp op) {
    switch (op) {
        case AGG_SUM: { int64_t s = 0; for (size_t i = 0; i < n; ++i) s += v[i]; return s; }
        case AGG_MIN: { int64_t m = INT64_MAX; for (size_t i = 0; i < n; ++i) if (v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { int64_t m = INT64_MIN; for (size_t i = 0; i < n; ++i) if (v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      return static_cast<int64_t>(n);
    }
}

// Null-aware int64 reduce (DM-3j across types): skip rows where nulls[i] != 0. Sentinels match
// matrix_cpu_reduce_all_i64 (SUM/COUNT 0, MIN INT64_MAX, MAX INT64_MIN).
inline int64_t matrix_cpu_reduce_all_i64_nullable(const int64_t* v, size_t n, MatrixAggOp op, const uint8_t* nulls) {
    switch (op) {
        case AGG_SUM: { int64_t s = 0; for (size_t i = 0; i < n; ++i) if (!nulls[i]) s += v[i]; return s; }
        case AGG_MIN: { int64_t m = INT64_MAX; for (size_t i = 0; i < n; ++i) if (!nulls[i] && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { int64_t m = INT64_MIN; for (size_t i = 0; i < n; ++i) if (!nulls[i] && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { int64_t c = 0; for (size_t i = 0; i < n; ++i) c += (nulls[i] == 0); return c; }
    }
}

enum class MatrixCmp : uint32_t { GT = 0, GE, LT, LE, EQ, NE, BETWEEN }; // GT == 0 == the default

// A value predicate for WHERE. `a` = primary bound (threshold; inclusive lower for BETWEEN);
// `b` = inclusive upper for BETWEEN (ignored otherwise). uint32 to match the column element type.
struct MatrixPredicate { MatrixCmp cmp = MatrixCmp::GT; uint32_t a = 0; uint32_t b = 0; };

inline bool matrix_pred_match(uint32_t v, const MatrixPredicate& p) {
    switch (p.cmp) {
        case MatrixCmp::GE:      return v >= p.a;
        case MatrixCmp::LT:      return v <  p.a;
        case MatrixCmp::LE:      return v <= p.a;
        case MatrixCmp::EQ:      return v == p.a;
        case MatrixCmp::NE:      return v != p.a;
        case MatrixCmp::BETWEEN: return v >= p.a && v <= p.b;   // inclusive [a, b]
        case MatrixCmp::GT:
        default:                 return v >  p.a;
    }
}

// Predicate-aware filtered scalar reduce — the general sibling of matrix_cpu_reduce. Same empty-set
// sentinels (COUNT/SUM 0, MIN UINT64_MAX, MAX 0; see the MAX caveat in the design). matrix_cpu_reduce
// (the GT fast path) is intentionally left untouched so the id-0 oracle scan is byte-identical.
inline uint64_t matrix_cpu_reduce_pred(const uint32_t* v, size_t n, const MatrixPredicate& p, MatrixAggOp op,
                                       const uint8_t* nulls = nullptr) {
    switch (op) {
        case AGG_SUM: { uint64_t s = 0; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match(v[i], p)) s += v[i]; return s; }
        case AGG_MIN: { uint64_t m = UINT64_MAX; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match(v[i], p) && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { uint64_t m = 0; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match(v[i], p) && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { uint64_t c = 0; for (size_t i = 0; i < n; ++i) c += ((!nulls || !nulls[i]) && matrix_pred_match(v[i], p)); return c; }
    }
}

// int64 value predicate for WHERE (the signed sibling of MatrixPredicate). a = primary bound / BETWEEN
// lower (inclusive); b = BETWEEN upper (inclusive). Signed comparisons so negatives order correctly.
struct MatrixPredicateI64 { MatrixCmp cmp = MatrixCmp::GT; int64_t a = 0; int64_t b = 0; };

inline bool matrix_pred_match_i64(int64_t v, const MatrixPredicateI64& p) {
    switch (p.cmp) {
        case MatrixCmp::GE:      return v >= p.a;
        case MatrixCmp::LT:      return v <  p.a;
        case MatrixCmp::LE:      return v <= p.a;
        case MatrixCmp::EQ:      return v == p.a;
        case MatrixCmp::NE:      return v != p.a;
        case MatrixCmp::BETWEEN: return v >= p.a && v <= p.b;
        case MatrixCmp::GT:
        default:                 return v >  p.a;
    }
}

// Filtered signed scalar reduce — the int64 sibling of matrix_cpu_reduce_pred. Same empty sentinels as
// matrix_cpu_reduce_all_i64 (SUM/COUNT 0, MIN INT64_MAX, MAX INT64_MIN; the MAX sentinel is ambiguous
// only for a column literally containing INT64_MIN — read COUNT).
inline int64_t matrix_cpu_reduce_pred_i64(const int64_t* v, size_t n, const MatrixPredicateI64& p, MatrixAggOp op,
                                          const uint8_t* nulls = nullptr) {
    switch (op) {
        case AGG_SUM: { int64_t s = 0; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match_i64(v[i], p)) s += v[i]; return s; }
        case AGG_MIN: { int64_t m = INT64_MAX; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match_i64(v[i], p) && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { int64_t m = INT64_MIN; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match_i64(v[i], p) && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { int64_t c = 0; for (size_t i = 0; i < n; ++i) c += ((!nulls || !nulls[i]) && matrix_pred_match_i64(v[i], p)); return c; }
    }
}

// Unfiltered scalar reduce over a double column. COUNT (double)n; SUM Σv; MIN/MAX over all (empty
// sentinels MIN +inf, MAX -inf). IEEE: NaN values are skipped by MIN/MAX (NaN compares false) and
// poison SUM (NaN propagates) — documented, not special-cased.
inline double matrix_cpu_reduce_all_f64(const double* v, size_t n, MatrixAggOp op) {
    switch (op) {
        case AGG_SUM: { double s = 0.0; for (size_t i = 0; i < n; ++i) s += v[i]; return s; }
        case AGG_MIN: { double m = std::numeric_limits<double>::infinity();  for (size_t i = 0; i < n; ++i) if (v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { double m = -std::numeric_limits<double>::infinity(); for (size_t i = 0; i < n; ++i) if (v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      return static_cast<double>(n);
    }
}

// Null-aware double reduce (DM-3j across types): skip rows where nulls[i] != 0. Sentinels match
// matrix_cpu_reduce_all_f64 (SUM/COUNT 0, MIN +inf, MAX -inf). COUNT returns the non-null count.
inline double matrix_cpu_reduce_all_f64_nullable(const double* v, size_t n, MatrixAggOp op, const uint8_t* nulls) {
    switch (op) {
        case AGG_SUM: { double s = 0.0; for (size_t i = 0; i < n; ++i) if (!nulls[i]) s += v[i]; return s; }
        case AGG_MIN: { double m = std::numeric_limits<double>::infinity();  for (size_t i = 0; i < n; ++i) if (!nulls[i] && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { double m = -std::numeric_limits<double>::infinity(); for (size_t i = 0; i < n; ++i) if (!nulls[i] && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { double c = 0.0; for (size_t i = 0; i < n; ++i) c += (nulls[i] == 0) ? 1.0 : 0.0; return c; }
    }
}

struct MatrixPredicateF64 { MatrixCmp cmp = MatrixCmp::GT; double a = 0.0; double b = 0.0; };
inline bool matrix_pred_match_f64(double v, const MatrixPredicateF64& p) {
    switch (p.cmp) {
        case MatrixCmp::GE:      return v >= p.a;
        case MatrixCmp::LT:      return v <  p.a;
        case MatrixCmp::LE:      return v <= p.a;
        case MatrixCmp::EQ:      return v == p.a;
        case MatrixCmp::NE:      return v != p.a;
        case MatrixCmp::BETWEEN: return v >= p.a && v <= p.b;
        case MatrixCmp::GT:
        default:                 return v >  p.a;
    }
}
inline double matrix_cpu_reduce_pred_f64(const double* v, size_t n, const MatrixPredicateF64& p, MatrixAggOp op,
                                         const uint8_t* nulls = nullptr) {
    switch (op) {
        case AGG_SUM: { double s = 0.0; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match_f64(v[i], p)) s += v[i]; return s; }
        case AGG_MIN: { double m = std::numeric_limits<double>::infinity();  for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match_f64(v[i], p) && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { double m = -std::numeric_limits<double>::infinity(); for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match_f64(v[i], p) && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { double c = 0.0; for (size_t i = 0; i < n; ++i) c += ((!nulls || !nulls[i]) && matrix_pred_match_f64(v[i], p)) ? 1.0 : 0.0; return c; }
    }
}

// A structured analytical query over catalog columns (value_col / key_col > 0). has_filter applies
// WHERE value <cmp> threshold (BETWEEN uses [threshold, upper]); grouped applies GROUP BY key_col into num_groups dense buckets.
struct MatrixQuery {
    uint64_t    value_col  = 0;
    MatrixAggOp agg        = AGG_COUNT;
    bool        has_filter = false;
    uint32_t    threshold  = 0;
    MatrixCmp   cmp        = MatrixCmp::GT;   // comparison op for the filter (default keeps value>threshold)
    uint32_t    upper      = 0;               // BETWEEN's inclusive upper bound (ignored for other ops)
    int64_t     lo_i64     = 0;   // int64 filter primary bound / BETWEEN lower (I64 columns; not wire-serialized)
    int64_t     hi_i64     = 0;   // int64 filter BETWEEN upper (I64 columns)
    double      lo_f64     = 0.0; // double filter primary bound / BETWEEN lower (F64 columns; not wire-serialized)
    double      hi_f64     = 0.0; // double filter BETWEEN upper (F64 columns)
    bool        grouped    = false;
    uint64_t    key_col    = 0;
    uint32_t    num_groups = 0;
    uint64_t    limit      = 0;   // ORDER BY agg DESC LIMIT n -> top-N groups; 0 = no limit (parse-side; not wire-serialized)
};

// Result of CPUMockEngine::execute_query — OK or a specific input-validation rejection (the query
// boundary never crashes on caller input; on any ERR the out vector is left empty).
enum class MatrixQueryStatus { OK, ERR_UNKNOWN_COLUMN, ERR_INVALID_GROUP, ERR_TOO_MANY_GROUPS, ERR_UNSUPPORTED_TYPE, ERR_PARSE };

// Grouped reduction core (GROUP BY key). Filtered==true applies WHERE matrix_pred_match(value, pred) (compiled
// out when false via if constexpr, so the unfiltered path is byte-identical to the original). Dense
// groups [0, num_groups); keys >= num_groups ignored; out initialized per op (empty-group sentinels
// match matrix_cpu_reduce: COUNT/SUM/MAX -> 0, MIN -> UINT64_MAX). SUM accumulates in u64. One pass;
// the op branch is inside the loop because grouped reduction is scatter-bound (random out[k] writes),
// not branch-bound.
template <bool Filtered>
inline void matrix_group_reduce_impl(const uint32_t* keys, const uint32_t* values, size_t n,
                                     uint32_t num_groups, MatrixAggOp op, const MatrixPredicate& pred, uint64_t* out,
                                     const uint8_t* nulls = nullptr) {
    const uint64_t init = (op == AGG_MIN) ? UINT64_MAX : 0;
    for (uint32_t g = 0; g < num_groups; ++g) out[g] = init;
    for (size_t i = 0; i < n; ++i) {
        if (nulls && nulls[i]) continue;                     // NULL row: excluded from every group aggregate
        const uint32_t k = keys[i];
        if (k >= num_groups) continue;                       // out-of-range key: ignored
        const uint32_t v = values[i];
        if constexpr (Filtered) { if (!matrix_pred_match(v, pred)) continue; }  // WHERE predicate
        switch (op) {
            case AGG_SUM:   out[k] += v; break;
            case AGG_MIN:   if (v < out[k]) out[k] = v; break;
            case AGG_MAX:   if (v > out[k]) out[k] = v; break;
            case AGG_COUNT:
            default:        out[k] += 1; break;
        }
    }
}
// GROUP BY key (all rows). Unchanged signature from GBY-1 — now a thin wrapper. `nulls` (default none) skips NULL rows.
inline void matrix_cpu_group_reduce(const uint32_t* keys, const uint32_t* values, size_t n,
                                    uint32_t num_groups, MatrixAggOp op, uint64_t* out, const uint8_t* nulls = nullptr) {
    matrix_group_reduce_impl<false>(keys, values, n, num_groups, op, MatrixPredicate{}, out, nulls);
}
// GROUP BY key WHERE value > threshold.
inline void matrix_cpu_group_reduce_where(const uint32_t* keys, const uint32_t* values, size_t n,
                                          uint32_t num_groups, MatrixAggOp op, uint32_t threshold, uint64_t* out) {
    matrix_group_reduce_impl<true>(keys, values, n, num_groups, op, MatrixPredicate{MatrixCmp::GT, threshold, 0}, out);
}
inline void matrix_cpu_group_reduce_pred(const uint32_t* keys, const uint32_t* values, size_t n,
                                         uint32_t num_groups, MatrixAggOp op, const MatrixPredicate& pred, uint64_t* out,
                                         const uint8_t* nulls = nullptr) {
    matrix_group_reduce_impl<true>(keys, values, n, num_groups, op, pred, out, nulls);
}

// Grouped reduction over an int64 value column keyed by a uint32 key column (dense group ids). Same
// semantics as matrix_group_reduce_impl but values + accumulators are signed int64. Empty-group
// sentinels: COUNT/SUM 0, MIN INT64_MAX, MAX INT64_MIN. NOTE: MAX inits to INT64_MIN (not 0) because
// int64 values may be negative. Filtered applies the int64 predicate.
template <bool Filtered>
inline void matrix_group_reduce_i64_impl(const uint32_t* keys, const int64_t* values, size_t n,
                                         uint32_t num_groups, MatrixAggOp op, const MatrixPredicateI64& pred, int64_t* out,
                                         const uint8_t* nulls = nullptr) {
    for (uint32_t g = 0; g < num_groups; ++g)
        out[g] = (op == AGG_MIN) ? INT64_MAX : (op == AGG_MAX ? INT64_MIN : 0);
    for (size_t i = 0; i < n; ++i) {
        if (nulls && nulls[i]) continue;                     // NULL row: excluded
        const uint32_t k = keys[i];
        if (k >= num_groups) continue;
        const int64_t v = values[i];
        if constexpr (Filtered) { if (!matrix_pred_match_i64(v, pred)) continue; }
        switch (op) {
            case AGG_SUM:   out[k] += v; break;
            case AGG_MIN:   if (v < out[k]) out[k] = v; break;
            case AGG_MAX:   if (v > out[k]) out[k] = v; break;
            case AGG_COUNT:
            default:        out[k] += 1; break;
        }
    }
}
inline void matrix_cpu_group_reduce_i64(const uint32_t* keys, const int64_t* values, size_t n,
                                        uint32_t num_groups, MatrixAggOp op, int64_t* out, const uint8_t* nulls = nullptr) {
    matrix_group_reduce_i64_impl<false>(keys, values, n, num_groups, op, MatrixPredicateI64{}, out, nulls);
}
inline void matrix_cpu_group_reduce_i64_pred(const uint32_t* keys, const int64_t* values, size_t n,
                                             uint32_t num_groups, MatrixAggOp op, const MatrixPredicateI64& pred, int64_t* out,
                                             const uint8_t* nulls = nullptr) {
    matrix_group_reduce_i64_impl<true>(keys, values, n, num_groups, op, pred, out, nulls);
}

// Grouped reduction over a double value column keyed by a uint32 key column (dense group ids). Double
// accumulators; empty-group sentinels COUNT/SUM 0.0, MIN +inf, MAX -inf (MAX inits -inf so a negative
// group yields the right max). Filtered applies the double predicate.
template <bool Filtered>
inline void matrix_group_reduce_f64_impl(const uint32_t* keys, const double* values, size_t n,
                                         uint32_t num_groups, MatrixAggOp op, const MatrixPredicateF64& pred, double* out,
                                         const uint8_t* nulls = nullptr) {
    for (uint32_t g = 0; g < num_groups; ++g)
        out[g] = (op == AGG_MIN) ? std::numeric_limits<double>::infinity()
               : (op == AGG_MAX ? -std::numeric_limits<double>::infinity() : 0.0);
    for (size_t i = 0; i < n; ++i) {
        if (nulls && nulls[i]) continue;                     // NULL row: excluded
        const uint32_t k = keys[i];
        if (k >= num_groups) continue;
        const double v = values[i];
        if constexpr (Filtered) { if (!matrix_pred_match_f64(v, pred)) continue; }
        switch (op) {
            case AGG_SUM:   out[k] += v; break;
            case AGG_MIN:   if (v < out[k]) out[k] = v; break;
            case AGG_MAX:   if (v > out[k]) out[k] = v; break;
            case AGG_COUNT:
            default:        out[k] += 1.0; break;
        }
    }
}
inline void matrix_cpu_group_reduce_f64(const uint32_t* keys, const double* values, size_t n,
                                        uint32_t num_groups, MatrixAggOp op, double* out, const uint8_t* nulls = nullptr) {
    matrix_group_reduce_f64_impl<false>(keys, values, n, num_groups, op, MatrixPredicateF64{}, out, nulls);
}
inline void matrix_cpu_group_reduce_f64_pred(const uint32_t* keys, const double* values, size_t n,
                                             uint32_t num_groups, MatrixAggOp op, const MatrixPredicateF64& pred, double* out,
                                             const uint8_t* nulls = nullptr) {
    matrix_group_reduce_f64_impl<true>(keys, values, n, num_groups, op, pred, out, nulls);
}

/**
 * @brief Stable counting-sort of a batch by owning page (the "bin in the batcher" step).
 * Fills `binned` with the queries grouped by page — order within a page preserved, so
 * same-slot writes keep batch order and last-writer-wins is deterministic. `offsets`
 * is a CSR index: page p's queries are binned[offsets[p] .. offsets[p+1]).
 * Caller owns the buffers (binned >= count, offsets >= PAGE_COUNT+1) — zero alloc here.
 */
inline void matrix_bin_by_page(const DatabaseQuery* batch, size_t count,
                               DatabaseQuery* binned, uint32_t* offsets) {
    uint32_t counts[MATRIX_PAGE_COUNT] = {0};
    for (size_t i = 0; i < count; ++i) {
        counts[matrix_page_of(batch[i].query_id)]++;
    }
    offsets[0] = 0;
    for (size_t p = 0; p < MATRIX_PAGE_COUNT; ++p) {
        offsets[p + 1] = offsets[p] + counts[p];
    }
    uint32_t cursor[MATRIX_PAGE_COUNT];
    for (size_t p = 0; p < MATRIX_PAGE_COUNT; ++p) cursor[p] = offsets[p];
    for (size_t i = 0; i < count; ++i) {
        const size_t p = matrix_page_of(batch[i].query_id);
        binned[cursor[p]++] = batch[i];
    }
}


In [ ]:
%%writefile memory_model.hpp
#pragma once

// Where a dataset physically lives / which processor addresses it.
enum class MemorySpace {
    HOST,    // CPU RAM
    DEVICE,  // GPU VRAM (discrete)
    COLD,    // SSD — cold columns + durability log (append-only, high latency)
    UNIFIED, // shared CPU+GPU pool (DGX Spark / Grace-Hopper) — placement is zero-copy
};

// Boot-time description of the machine's memory architecture. The CostModel reads this
// so the data-transfer term is included (discrete) or zero (unified).
//   DISCRETE: HOST and DEVICE are distinct; placing data in DEVICE costs a transfer.
//   UNIFIED : one physical pool; "placement" only chooses the executor, never moves data.
struct MemoryModel {
    enum Kind { DISCRETE, UNIFIED } kind = DISCRETE;

    // ponytail: unified-memory hardware isn't available to test on, so detection
    // defaults to DISCRETE. When a unified box (DGX Spark / GH) is in hand, set this
    // from cudaDeviceGetAttribute(cudaDevAttrPageableMemoryAccess / Managed) and
    // implement the UNIFIED cost branch. Seam only for now.
    static MemoryModel detect(bool gpu_available) {
        MemoryModel m;
        m.kind = DISCRETE; // discrete-only until validated on real unified hardware
        (void)gpu_available;
        return m;
    }

    bool is_unified() const { return kind == UNIFIED; }
};


In [ ]:
%%writefile tier_model.hpp
#pragma once

#include "memory_model.hpp"
#include <cstddef>

// The storage tiers, ordered fastest-scan to coldest. DEVICE=VRAM, HOST=RAM, COLD=SSD.
// UNIFIED is the existing seam from memory_model.hpp (CPU+GPU share one pool); when the
// machine is unified, DEVICE and HOST collapse and migration between them is a no-op.
//
// NOTE: MemorySpace is DEFINED in memory_model.hpp as { HOST, DEVICE, COLD, UNIFIED };
// this header layers the per-tier physics on top of that enum.

// Physics of one tier: what it's good at and what it costs. bandwidth in bytes/microsecond
// (so it composes with CostModel's existing *_BPus units); latency in microseconds to first
// byte; capacity_bytes is the usable budget (0 == "treat as unbounded for now").
struct TierPhysics {
    double   scan_bytes_per_us;  // sequential scan bandwidth
    double   access_latency_us;  // latency to first byte (random-access cost proxy)
    size_t   capacity_bytes;     // usable capacity budget; 0 = unbounded (not yet enforced)
    const char* concern;         // the tier's defining risk, for docs/telemetry
};

// First-estimate physics per tier (CALIBRATION TARGETS, like CostModel's constants).
// VRAM/RAM scan numbers are measured; SSD and latencies are estimates to refine on the box.
inline TierPhysics tier_physics(MemorySpace s) {
    switch (s) {
        case MemorySpace::DEVICE: // VRAM
            return TierPhysics{240'000.0, 5.0, 16ull*1024*1024*1024, "scarce capacity; PCIe to reach"};
        case MemorySpace::HOST:   // RAM
            return TierPhysics{10'000.0, 0.1, 256ull*1024*1024*1024, "medium capacity"};
        case MemorySpace::COLD:   // SSD
            return TierPhysics{3'000.0, 100.0, 0 /*unbounded*/, "high latency; write wear; append-only"};
        case MemorySpace::UNIFIED:
            return TierPhysics{240'000.0, 0.1, 0, "unified pool (future hardware)"};
    }
    return TierPhysics{1.0, 1.0, 0, "unknown"};
}


In [ ]:
%%writefile cost_model.hpp
#pragma once

#include "memory_model.hpp"
#include "tier_model.hpp"
#include <cstddef>

// Cost-based placement: predict microseconds for a workload on each processor, choose
// the cheaper home. Constants are MEASURED on the target machine (values below are first
// estimates from Tesla T4 runs — see the spec's calibration note). The structure is the
// deliverable; the constants get one calibration pass before the boundary is trusted.
class CostModel {
public:
    explicit CostModel(MemoryModel mm, bool gpu_available = true)
        : mm_(mm), gpu_available_(gpu_available) {}

    // Point ops: CPU always wins (PCIe slower than CPU cache, ~1 memory op/query).
    MemorySpace place_point() const { return MemorySpace::HOST; }

    // Scan of `bytes`: place where the predicted scan time is lower.
    MemorySpace place_scan(size_t bytes) const {
        if (!gpu_available_) return MemorySpace::HOST;
        return device_scan_us(bytes) < host_scan_us(bytes)
                   ? MemorySpace::DEVICE
                   : MemorySpace::HOST;
    }

    // Predicted microseconds (exposed for tests / future tuning).
    double host_scan_us(size_t bytes) const {
        return static_cast<double>(bytes) / CPU_SCAN_BPus;
    }
    double device_scan_us(size_t bytes) const {
        // Transfer is amortized to 0 today (a column is placed once and scanned many).
        // Both branches are 0.0 ON PURPOSE: this is the seam where the discrete one-time
        // transfer term will go; UNIFIED stays 0. Deliberately inert until calibrated.
        const double transfer = mm_.is_unified() ? 0.0 : 0.0;
        return LAUNCH_US + transfer + static_cast<double>(bytes) / GPU_SCAN_BPus;
    }

    // Predicted scan time (us) for `bytes` resident in tier `s`. Uses each tier's measured
    // bandwidth from tier_model. This generalizes host_scan_us/device_scan_us to all tiers
    // (the existing two stay for the router's HOST/DEVICE fast path and back-compat).
    double scan_us(MemorySpace s, size_t bytes) const {
        const TierPhysics p = tier_physics(s);
        const double launch = (s == MemorySpace::DEVICE) ? LAUNCH_US : 0.0;
        return launch + static_cast<double>(bytes) / p.scan_bytes_per_us;
    }

    // Predicted one-time cost (us) to move `bytes` from tier `from` to tier `to`. Zero if
    // same tier (no move) or if memory is unified (zero-copy). Otherwise bounded by the
    // slower of read-from-source and write-to-dest bandwidth.
    double migration_us(MemorySpace from, MemorySpace to, size_t bytes) const {
        if (from == to) return 0.0;
        if (mm_.is_unified()) return 0.0; // unified pool: placement is zero-copy
        const double read_bw  = tier_physics(from).scan_bytes_per_us;
        const double write_bw = tier_physics(to).scan_bytes_per_us;
        const double slower = (read_bw < write_bw) ? read_bw : write_bw;
        return static_cast<double>(bytes) / slower;
    }

private:
    // --- measured constants (bytes per microsecond) — CALIBRATION TARGETS ---
    static constexpr double CPU_SCAN_BPus = 10'000.0;   // ~10 GB/s  (measured)
    static constexpr double GPU_SCAN_BPus = 240'000.0;  // ~240 GB/s (measured at 64 MB)
    static constexpr double LAUNCH_US     = 30.0;       // per-scan GPU launch floor (measured)

    MemoryModel mm_;
    bool gpu_available_;
};


In [ ]:
%%writefile router.hpp
#pragma once

#include "compute.hpp"
#include "cost_model.hpp"
#include "memory_model.hpp"
#include <cassert>
#include <cstdint>
#include <vector>

// Routing policy over two live engines. Holds no data and runs no kernels: it decides
// (via CostModel) where each dataset lives, records it in a placement map, and dispatches
// each query to the engine that owns its data. gpu may be null (no-GPU build) — then the
// CostModel places everything HOST and all routing goes to the CPU engine.
class Router {
public:
    Router(ComputeInterface* cpu, ComputeInterface* gpu, CostModel cm)
        : cpu_(cpu), gpu_(gpu), cm_(cm) {}

    // Point ops run where the cost model places them — always HOST today (page-ownership
    // KV store lives in CPU RAM; GPU loses point ops). Routed via the model so the
    // invariant lives in one place.
    void route_batch(DatabaseQuery* batch, size_t count) {
        ComputeInterface* eng =
            (cm_.place_point() == MemorySpace::DEVICE && gpu_) ? gpu_ : cpu_;
        eng->execute_batch(batch, count);
    }

    // Register a scan column of `bytes`, deciding its home now. Returns a dataset id used
    // by route_scan. One home per dataset — recorded once, never duplicated.
    uint64_t place_scan_column(size_t bytes) {
        const MemorySpace home =
            (gpu_ != nullptr) ? cm_.place_scan(bytes) : MemorySpace::HOST;
        placement_.push_back(home);
        return static_cast<uint64_t>(placement_.size() - 1);
    }

    // Dispatch a scan to the engine that owns the column's data.
    void route_scan(DatabaseQuery& q, uint64_t dataset_id) {
        assert(dataset_id < placement_.size() && "route_scan: unknown dataset_id");
        const MemorySpace home = placement_[dataset_id];
        ComputeInterface* eng = (home == MemorySpace::DEVICE && gpu_) ? gpu_ : cpu_;
        eng->execute_scan(q);
    }

    MemorySpace home_of(uint64_t dataset_id) const {
        assert(dataset_id < placement_.size() && "home_of: unknown dataset_id");
        return placement_[dataset_id];
    }

private:
    ComputeInterface* cpu_;
    ComputeInterface* gpu_;             // may be null
    CostModel cm_;
    std::vector<MemorySpace> placement_; // dataset id -> home (the entire coherence story)
};


In [ ]:
%%writefile kv_store.hpp
#pragma once

#include <cstdint>
#include <cstddef>
#include <vector>

// Open-addressing hash table (linear probing), key -> value, fixed capacity.
// This is the point-op store (gap DM-1). The prototype used store[key & MASK], which
// SILENTLY OVERWROTE colliding keys — a data-loss bug. Here, distinct keys probe to
// distinct slots and never overwrite each other; a full table is an explicit error
// (put returns false), never silent corruption.
//
// Single-owner: the engine's point-op path is single-threaded per the page-ownership
// model, so no internal locking. capacity is rounded up to a power of two so the slot
// index is a mask, not a modulo. SSD-spill (gap DM-9 / Inc 3) is a future seam: when
// full we return false today; later that path demotes cold entries to the cold tier.
class KVStore {
public:
    explicit KVStore(size_t capacity)
        : mask_(round_up_pow2(capacity) - 1),
          slots_(round_up_pow2(capacity)) {}

    // Insert or update. Returns false ONLY if the table is full and the key is new
    // (explicit failure, never overwrite of a different key).
    bool put(uint64_t key, uint64_t value) {
        size_t i = key & mask_;
        for (size_t probe = 0; probe <= mask_; ++probe) {
            Slot& s = slots_[i];
            if (!s.occupied) {            // empty slot: new insert
                s.key = key; s.value = value; s.occupied = true;
                ++size_;
                return true;
            }
            if (s.key == key) {           // same key: update in place
                s.value = value;
                return true;
            }
            i = (i + 1) & mask_;          // collision with a DIFFERENT key: probe on
        }
        return false;                     // table full, key not present: explicit error
    }

    // Look up. Returns true and sets out if present; false if absent.
    bool get(uint64_t key, uint64_t& out) const {
        size_t i = key & mask_;
        for (size_t probe = 0; probe <= mask_; ++probe) {
            const Slot& s = slots_[i];
            if (!s.occupied) return false;     // empty slot ends the probe chain: miss
            if (s.key == key) { out = s.value; return true; }
            i = (i + 1) & mask_;
        }
        return false;
    }

    size_t size() const { return size_; }
    size_t capacity() const { return mask_ + 1; }

    // Order-independent fingerprint: sum of stored values (matches the engine's old
    // store_checksum semantics so cross-checks stay meaningful).
    uint64_t checksum() const {
        uint64_t sum = 0;
        for (const Slot& s : slots_) if (s.occupied) sum += s.value;
        return sum;
    }

    // Visit every live entry (snapshot / iteration). Allocation-free; same walk as checksum().
    template <typename F>
    void for_each(F&& f) const { for (const Slot& s : slots_) if (s.occupied) f(s.key, s.value); }

private:
    struct Slot { uint64_t key = 0; uint64_t value = 0; bool occupied = false; };

    static size_t round_up_pow2(size_t n) {
        size_t p = 1;
        while (p < n) p <<= 1;
        return p < 2 ? 2 : p; // minimum 2 slots
    }

    size_t mask_;
    size_t size_ = 0;
    std::vector<Slot> slots_;
};


In [ ]:
%%writefile cold_store.hpp
#pragma once

#include "types.hpp"   // OP_WRITE
#include <cstdio>
#include <cstdint>
#include <cstddef>
#include <cstring>
#include <cstdlib>
#include <string>
#include <vector>
#include <utility>
#include <unistd.h>    // fsync, fileno

// SSD-backed write-ahead log (gap DU-1/2/3 / three-tier cold tier). Append-only: a record
// is [u32 length][u32 crc32(payload)][payload]. Synchronous (fsync per policy) so a
// returned append_put is durable. replay() rebuilds state front-to-back and stops at the
// first torn or corrupt record — never replays corruption. "SSD" is a file here; the
// interface is identical on real flash.

enum class SyncPolicy {
    SYNC_EACH,  // fsync after every append — committed write survives a crash (default)
    SYNC_OFF,   // no fsync — tests / throughput; crash loses unflushed tail
};

// On-disk payload layout (serialized explicitly, NOT a C struct — avoids padding/ABI
// dependence): key (8 bytes) + value (8 bytes) + opcode (4 bytes) = 20 bytes.
constexpr size_t MATRIX_WAL_PAYLOAD_BYTES = 20;
constexpr uint32_t MATRIX_WAL_MAX_RECORD = 4096;    // sane upper bound for the length field
constexpr size_t   MATRIX_WAL_COMMIT_BYTES = 4;          // commit-marker record length
constexpr uint32_t MATRIX_WAL_COMMIT       = 0x434F4D4Du; // 'COMM' — commit-marker payload

// Standard CRC32 (reflected, poly 0xEDB88320). Inline, no dependency.
inline uint32_t matrix_crc32(const unsigned char* data, size_t n) {
    uint32_t crc = 0xFFFFFFFFu;
    for (size_t i = 0; i < n; ++i) {
        crc ^= data[i];
        for (int k = 0; k < 8; ++k)
            crc = (crc & 1u) ? (crc >> 1) ^ 0xEDB88320u : (crc >> 1);
    }
    return crc ^ 0xFFFFFFFFu;
}

class ColdStore {
public:
    explicit ColdStore(std::string path, SyncPolicy policy = SyncPolicy::SYNC_EACH)
        : path_(std::move(path)), policy_(policy) {
        // "ab" creates if missing and positions writes at end (append).
        fp_ = std::fopen(path_.c_str(), "ab");
        if (!fp_) {
            // A WAL that can't open its file cannot guarantee durability — fail loudly
            // rather than null-deref on the first append.
            std::fprintf(stderr, "ColdStore: cannot open WAL '%s'\n", path_.c_str());
            std::abort();
        }
    }

    ~ColdStore() {
        if (fp_) std::fclose(fp_);
    }

    ColdStore(const ColdStore&) = delete;
    ColdStore& operator=(const ColdStore&) = delete;

    // Append one auto-commit put durably (applied immediately on replay). Behavior unchanged.
    void append_put(uint64_t key, uint64_t value) { append_record(OP_WRITE, key, value); }

    // Append a transactional put — buffered on replay until a commit marker; discarded if a crash
    // leaves it without one. Written as part of the engine's commit() group.
    void append_txn_put(uint64_t key, uint64_t value) { append_record(OP_TXN_WRITE, key, value); }

    // Append a commit marker durably — replay applies all txn-puts buffered since the last commit.
    void append_commit() {
        const uint32_t magic = MATRIX_WAL_COMMIT;
        const uint32_t length = static_cast<uint32_t>(MATRIX_WAL_COMMIT_BYTES);
        const uint32_t crc = matrix_crc32(reinterpret_cast<const unsigned char*>(&magic), MATRIX_WAL_COMMIT_BYTES);
        std::fwrite(&length, sizeof(length), 1, fp_);
        std::fwrite(&crc,    sizeof(crc),    1, fp_);
        std::fwrite(&magic,  1, MATRIX_WAL_COMMIT_BYTES, fp_);
        std::fflush(fp_);
        if (policy_ == SyncPolicy::SYNC_EACH) ::fsync(::fileno(fp_));
        ++records_written_;
    }

    // Empty the log after its contents are captured in a checkpoint. Durable per SyncPolicy.
    void truncate() {
        std::fclose(fp_);
        fp_ = std::fopen(path_.c_str(), "wb");   // "wb" truncates to zero length
        if (!fp_) { std::fprintf(stderr, "ColdStore::truncate: reopen failed %s\n", path_.c_str()); std::abort(); }
        if (policy_ == SyncPolicy::SYNC_EACH) ::fsync(::fileno(fp_));
        std::fclose(fp_);
        fp_ = std::fopen(path_.c_str(), "ab");   // back to append mode
        if (!fp_) { std::fprintf(stderr, "ColdStore::truncate: reopen-append failed %s\n", path_.c_str()); std::abort(); }
        records_written_ = 0;
    }

    // Replay every intact record in append order, calling apply(key, value). Stops at the
    // first short read or CRC mismatch (torn/corrupt tail). Missing/empty file → nothing.
    template <typename Apply>
    void replay(Apply&& apply) const {
        FILE* r = std::fopen(path_.c_str(), "rb");
        if (!r) return;
        std::vector<std::pair<uint64_t, uint64_t>> txn; // txn-puts buffered since the last commit
        for (;;) {
            uint32_t length = 0;
            if (std::fread(&length, sizeof(length), 1, r) != 1) break;     // clean EOF
            if (length == 0 || length > MATRIX_WAL_MAX_RECORD) break;      // torn tail
            uint32_t crc = 0;
            if (std::fread(&crc, sizeof(crc), 1, r) != 1) break;           // torn tail
            unsigned char buf[MATRIX_WAL_MAX_RECORD];
            if (std::fread(buf, 1, length, r) != length) break;            // torn tail
            if (matrix_crc32(buf, length) != crc) break;                   // corruption
            if (length == MATRIX_WAL_PAYLOAD_BYTES) {                      // 20-byte put record
                uint32_t opcode = 0; std::memcpy(&opcode, buf + 16, 4);
                uint64_t key = 0, value = 0;
                std::memcpy(&key,   buf + 0, 8);
                std::memcpy(&value, buf + 8, 8);
                if (opcode == OP_WRITE)          apply(key, value);        // auto-commit (unchanged)
                else if (opcode == OP_TXN_WRITE) txn.emplace_back(key, value); // buffer until commit
                // other opcode at length 20: skip (forward-compat)
            } else if (length == MATRIX_WAL_COMMIT_BYTES) {               // 4-byte marker
                uint32_t magic = 0; std::memcpy(&magic, buf, 4);
                if (magic == MATRIX_WAL_COMMIT) { for (auto& kv : txn) apply(kv.first, kv.second); txn.clear(); }
                // other 4-byte record: skip (forward-compat)
            }
            // other lengths: skip (forward-compat)
        }
        std::fclose(r);   // EOF: any still-buffered txn was uncommitted -> discarded
    }

    uint64_t records_written() const { return records_written_; }
    SyncPolicy policy() const { return policy_; }   // the durability level in force (DU-5)

private:
    // Write one length-20 [key8][value8][opcode4] record durably (the put/txn-put wire form).
    void append_record(uint32_t opcode, uint64_t key, uint64_t value) {
        unsigned char payload[MATRIX_WAL_PAYLOAD_BYTES];
        std::memcpy(payload + 0,  &key,    8);
        std::memcpy(payload + 8,  &value,  8);
        std::memcpy(payload + 16, &opcode, 4);
        const uint32_t length = MATRIX_WAL_PAYLOAD_BYTES;
        const uint32_t crc = matrix_crc32(payload, MATRIX_WAL_PAYLOAD_BYTES);
        std::fwrite(&length, sizeof(length), 1, fp_);
        std::fwrite(&crc,    sizeof(crc),    1, fp_);
        std::fwrite(payload, 1, MATRIX_WAL_PAYLOAD_BYTES, fp_);
        std::fflush(fp_);
        if (policy_ == SyncPolicy::SYNC_EACH) ::fsync(::fileno(fp_));
        ++records_written_;
    }

    std::string path_;
    SyncPolicy policy_;
    FILE* fp_ = nullptr;
    uint64_t records_written_ = 0;
};


In [ ]:
%%writefile tier_manager.hpp
#pragma once

#include "cost_model.hpp"
#include "memory_model.hpp"
#include "tier_model.hpp"
#include <cstdint>
#include <cstddef>
#include <vector>
#include <unordered_map>
#include <algorithm>

// A migration the brain decided on (the future executor will move the bytes).
struct MigrationDecision {
    uint64_t column_id;
    MemorySpace from;
    MemorySpace to;
};

// The auto-tiering brain. Tracks per-column access heat and, on rebalance(), returns the
// cost-benefit migrations that lower total scan cost. Decides only — moves no bytes, does
// no I/O. Pure logic over the Increment-1 CostModel + tier_model.
class TierManager {
public:
    TierManager(CostModel cm, size_t device_capacity_bytes, size_t host_capacity_bytes)
        : cm_(cm), device_cap_(device_capacity_bytes), host_cap_(host_capacity_bytes) {}

    void register_column(uint64_t id, size_t bytes, MemorySpace initial_tier) {
        cols_[id] = Column{id, bytes, initial_tier, /*heat*/0.0, /*recent*/0,
                           /*arrived_tick*/tick_};
    }

    // TierManager (public): update a column's byte size in place after an append. Preserves tier/heat/
    // recent_bytes (unlike register_column, which resets them). No-op if id is unknown.
    void update_bytes(uint64_t id, size_t bytes) {
        auto it = cols_.find(id);
        if (it != cols_.end()) it->second.bytes = bytes;
    }

    // --- tunables (calibration targets) ---
    static constexpr double HEAT_ALPHA = 0.5;          // EWMA weight on recent accesses
    static constexpr double HYSTERESIS = 1.5;          // promote only if benefit > 1.5x cost
    static constexpr int    SCAN_HORIZON = 8;          // cap on est. future scans
    static constexpr uint64_t MIN_RESIDENCY_TICKS = 2; // anti-thrash: min ticks before evict
    static constexpr double SWAP_MARGIN = 1.5;         // swap-on-promote: candidate must be > 1.5x
                                                       // the victim's keep-value (anti-thrash band)

    // Record that `bytes` of column `id` were scanned. O(1); accumulates until rebalance.
    void record_access(uint64_t id, size_t bytes) {
        auto it = cols_.find(id);
        if (it != cols_.end()) it->second.recent_bytes += bytes;
    }

    // Global pass. (This task: age heat only. Later tasks add promotion + eviction.)
    std::vector<MigrationDecision> rebalance() {
        ++tick_;
        for (auto& kv : cols_) {
            Column& c = kv.second;
            c.heat = HEAT_ALPHA * static_cast<double>(c.recent_bytes)
                     + (1.0 - HEAT_ALPHA) * c.heat;
            c.recent_bytes = 0;
        }

        std::vector<MigrationDecision> decisions;

        // Promotion: move qualifying columns one tier toward DEVICE — but capacity-gated,
        // so the emitted plan never over-subscribes a bounded tier (the executor in Inc 4
        // could not honor an over-capacity plan). Approve best-first (highest net benefit)
        // and only if the column fits its target tier given current + already-approved
        // same-tick promotions (resident_bytes reflects approved moves as we apply them).
        std::vector<uint64_t> candidates;
        for (auto& kv : cols_) if (should_promote(kv.second)) candidates.push_back(kv.first);
        std::sort(candidates.begin(), candidates.end(),
                  [this](uint64_t a, uint64_t b) {
                      const double na = promote_net_benefit(cols_.at(a));
                      const double nb = promote_net_benefit(cols_.at(b));
                      if (na != nb) return na > nb;     // best benefit first
                      return a < b;                      // tie-break by id (determinism)
                  });
        for (uint64_t id : candidates) {
            Column& c = cols_.at(id);
            const MemorySpace to = faster_tier(c.tier);
            const size_t cap = capacity_of(to);
            const bool fits = (cap == 0) || (resident_bytes(to) + c.bytes <= cap);
            if (!fits) {
                // `to` is full. Swap-on-promote: displace a colder resident if cand is worth it.
                // (Free-space promotion is the common path above; this is the contended path.)
                try_swap_promote(c, to, cap, decisions); // does nothing if no worthwhile victim
                continue;
            }
            const MemorySpace from = c.tier;
            decisions.push_back(MigrationDecision{c.id, from, to});
            c.tier = to;
            c.arrived_tick = tick_;
        }

        // Capacity eviction: for each bounded tier over capacity, demote the lowest
        // keep_score residents (cost-benefit, not pure LRU) until it fits. Respect
        // MIN_RESIDENCY_TICKS so a freshly-arrived column isn't immediately thrashed out.
        for (MemorySpace tier : {MemorySpace::DEVICE, MemorySpace::HOST}) {
            const size_t cap = capacity_of(tier);
            if (cap == 0) continue;
            for (;;) {
                if (resident_bytes(tier) <= cap) break;
                Column* victim = nullptr;
                double worst = 1e301;
                for (auto& kv : cols_) {
                    Column& c = kv.second;
                    if (c.tier != tier) continue;
                    if (tick_ - c.arrived_tick < MIN_RESIDENCY_TICKS) continue; // anti-thrash
                    const double s = keep_score(c);
                    if (s < worst) { worst = s; victim = &c; }
                }
                if (!victim) break; // nothing evictable (all within min residency) this pass
                const MemorySpace from = victim->tier;
                const MemorySpace to = slower_tier(from);
                decisions.push_back(MigrationDecision{victim->id, from, to});
                victim->tier = to;
                victim->arrived_tick = tick_;
            }
        }

        return decisions;
    }

    MemorySpace tier_of(uint64_t id) const { return cols_.at(id).tier; }
    double heat_of(uint64_t id) const { return cols_.at(id).heat; }

    size_t resident_bytes(MemorySpace tier) const {
        size_t sum = 0;
        for (const auto& kv : cols_) if (kv.second.tier == tier) sum += kv.second.bytes;
        return sum;
    }

private:
    struct Column {
        uint64_t id;
        size_t   bytes;
        MemorySpace tier;
        double   heat;
        size_t   recent_bytes;   // accesses since last rebalance
        uint64_t arrived_tick;   // when it last landed on its current tier
    };

    static MemorySpace faster_tier(MemorySpace t) {
        if (t == MemorySpace::COLD) return MemorySpace::HOST;
        if (t == MemorySpace::HOST) return MemorySpace::DEVICE;
        return t; // DEVICE already fastest
    }

    // Heat-derived estimate of upcoming scans for a column, clamped to the horizon.
    int est_future_scans(const Column& c) const {
        if (c.bytes == 0) return 0;
        const double scans = c.heat / static_cast<double>(c.bytes); // ~scans-per-tick
        int n = static_cast<int>(scans + 0.5);
        if (n < 0) n = 0;
        if (n > SCAN_HORIZON) n = SCAN_HORIZON;
        return n;
    }

    // Benefit (scan-time saved over the horizon) and cost (one-time migration) of promoting
    // a column one tier toward DEVICE. Single source of the promotion arithmetic so
    // should_promote and promote_net_benefit can never drift. benefit==cost==0 if already
    // on the fastest tier.
    struct PromoteEval { double benefit; double cost; };
    PromoteEval promote_eval(const Column& c) const {
        const MemorySpace faster = faster_tier(c.tier);
        if (faster == c.tier) return PromoteEval{0.0, 0.0};
        const double benefit = (cm_.scan_us(c.tier, c.bytes) - cm_.scan_us(faster, c.bytes))
                               * static_cast<double>(est_future_scans(c));
        const double cost = cm_.migration_us(c.tier, faster, c.bytes);
        return PromoteEval{benefit, cost};
    }

    // Should this column be promoted one tier now? (cost-benefit + hysteresis)
    bool should_promote(const Column& c) const {
        if (faster_tier(c.tier) == c.tier) return false; // already fastest
        const PromoteEval e = promote_eval(c);
        return e.benefit > HYSTERESIS * e.cost;
    }

    // Net benefit (benefit - cost) of promoting one tier — used to rank candidates so the
    // best columns win scarce capacity first. Only meaningful when should_promote is true.
    double promote_net_benefit(const Column& c) const {
        const PromoteEval e = promote_eval(c);
        return e.benefit - e.cost;
    }

    static MemorySpace slower_tier(MemorySpace t) {
        if (t == MemorySpace::DEVICE) return MemorySpace::HOST;
        if (t == MemorySpace::HOST) return MemorySpace::COLD;
        return t; // COLD already slowest
    }

    // Usable capacity of a tier; 0 means unbounded (COLD/SSD).
    size_t capacity_of(MemorySpace t) const {
        if (t == MemorySpace::DEVICE) return device_cap_;
        if (t == MemorySpace::HOST)   return host_cap_;
        return 0; // COLD unbounded
    }

    // Cost-benefit score of keeping a column on its tier (higher = more worth keeping).
    // Lower-scored residents are evicted first when a tier is over capacity.
    double keep_score(const Column& c) const {
        const MemorySpace slower = slower_tier(c.tier);
        if (slower == c.tier) return 1e300; // COLD: never "evict" further (infinite keep)
        const double penalty = (cm_.scan_us(slower, c.bytes) - cm_.scan_us(c.tier, c.bytes))
                               * static_cast<double>(est_future_scans(c));
        return penalty; // time/heat that would be lost by demoting
    }

    // Swap-on-promote: `cand` wants to move up into tier `to` but `to` is full. If the lowest-
    // keep_score resident of `to` (a) is past MIN_RESIDENCY_TICKS and (b) is decisively colder
    // than cand (promote_eval(cand).benefit > SWAP_MARGIN * keep_score(victim)), and evicting that
    // one victim makes room, demote the victim one tier down and promote cand. Emits both
    // decisions, updates tiers/arrival ticks, returns true. Single victim by design (see spec §2).
    //
    // The comparison is gross-benefit vs gross-keep (both are horizon scan-µs deltas, so it's
    // dimensionally sound). cand's one-time migration cost is omitted here, but cand only reaches
    // this path after should_promote (benefit > HYSTERESIS*cost), so net benefit is already
    // positive; SWAP_MARGIN then demands cand be 1.5x the incumbent's keep value to displace it.
    // ponytail: single-pass + best-first. On a true 3-tier system a HOST column could be both a
    // DEVICE-promotion candidate and a swap victim this tick — cols_.at(id) re-reads the live tier
    // so there is no corruption, only a transient sub-optimal order that self-corrects next tick.
    // (Moot on the CPU build: DEVICE is inert via device_cap=1.)
    bool try_swap_promote(Column& cand, MemorySpace to, size_t cap,
                          std::vector<MigrationDecision>& decisions) {
        if (cap == 0) return false; // unbounded tier never needs to evict to fit
        Column* victim = nullptr;
        double worst = 1e301;
        for (auto& kv : cols_) {
            Column& v = kv.second;
            if (v.tier != to) continue;
            if (tick_ - v.arrived_tick < MIN_RESIDENCY_TICKS) continue; // don't evict a fresh arrival
            const double s = keep_score(v);
            if (s < worst) { worst = s; victim = &v; }
        }
        if (!victim) return false;                                   // nothing evictable
        // Margin gate: cand must be decisively more valuable than the incumbent to displace it.
        // Note by design this does NOT protect a stone-cold victim: when keep_score(victim)==0
        // (idle, heat decayed out), SWAP_MARGIN*0==0 and any positive-benefit candidate wins —
        // a worthless resident SHOULD yield. The margin only damps swaps between close-heat columns.
        if (promote_eval(cand).benefit <= SWAP_MARGIN * keep_score(*victim)) return false; // not worth it
        if (resident_bytes(to) - victim->bytes + cand.bytes > cap) return false; // one eviction won't fit cand
        const MemorySpace v_to = slower_tier(to);
        decisions.push_back(MigrationDecision{victim->id, victim->tier, v_to});
        victim->tier = v_to;
        victim->arrived_tick = tick_;
        decisions.push_back(MigrationDecision{cand.id, cand.tier, to});
        cand.tier = to;
        cand.arrived_tick = tick_;
        return true;
    }

    CostModel cm_;
    size_t device_cap_;
    size_t host_cap_;
    uint64_t tick_ = 0;
    std::unordered_map<uint64_t, Column> cols_;
};


In [ ]:
%%writefile tiered_column.hpp
#pragma once

#include "memory_model.hpp"
#include <cstdint>
#include <cstddef>
#include <cstdio>
#include <cstdlib>
#include <string>
#include <vector>
#include <atomic>
#include <unistd.h>   // getpid — per-process COLD-file namespace
#if defined(MATRIX_USE_CUDA)
#include <cuda_runtime.h>
#endif

// A column's bytes resident in exactly ONE tier (HOST/DEVICE/COLD). migrate_to() moves
// them by always staging through HOST: read the bytes to a host buffer, free the source
// tier, push to the destination. This collapses the 3x3 transition matrix into two halves
// and routes DEVICE<->COLD via HOST for free. The integrity invariant: checksum() is the
// same regardless of which tier holds the bytes. DEVICE requires a CUDA build.
class TieredColumn {
public:
    TieredColumn(uint64_t id, const unsigned char* bytes, size_t n)
        : id_(id), size_(n), tier_(MemorySpace::HOST), host_(bytes, bytes + n) {}

    ~TieredColumn() { free_current(); }

    TieredColumn(const TieredColumn&) = delete;
    TieredColumn& operator=(const TieredColumn&) = delete;

    MemorySpace tier() const { return tier_; }
    size_t size_bytes() const { return size_; }
    uint64_t id() const { return id_; }

    // Pointer to the resident HOST bytes for in-place reads (e.g. a scan). Valid only while
    // tier()==HOST; nullptr otherwise (the bytes live on SSD/VRAM — migrate to HOST first).
    // Zero-copy: returns the live buffer, no allocation.
    const unsigned char* host_ptr() const {
        return tier_ == MemorySpace::HOST ? host_.data() : nullptr;
    }

    // Move the column's bytes to `dest` (no-op if already there). Always stages through HOST.
    void migrate_to(MemorySpace dest) {
        if (dest == tier_) return;
        std::vector<unsigned char> staged = read_to_host(); // pull bytes from wherever we are
        free_current();                                     // release the source tier's resource
        tier_ = MemorySpace::HOST;                          // logically in HOST now (staged holds bytes)
        host_ = std::move(staged);
        if (dest == MemorySpace::HOST) return;
        if (dest == MemorySpace::COLD) {
            write_cold(host_);
            host_.clear(); host_.shrink_to_fit();
            tier_ = MemorySpace::COLD;
            return;
        }
        if (dest == MemorySpace::DEVICE) {
#if defined(MATRIX_USE_CUDA)
            push_device(host_);
            host_.clear(); host_.shrink_to_fit();
            tier_ = MemorySpace::DEVICE;
            return;
#else
            std::fprintf(stderr, "TieredColumn: DEVICE tier requires a CUDA build\n");
            std::abort();
#endif
        }
        std::fprintf(stderr, "TieredColumn: unsupported destination tier\n");
        std::abort();
    }

    // TieredColumn (public): append bytes to a HOST-resident column (grow in place). Caller ensures the
    // column is HOST (the engine borrows COLD->HOST first). size_ grows so checksum()/host_ptr() span it.
    void append_bytes(const unsigned char* b, size_t n) {
        if (tier_ != MemorySpace::HOST) {
            std::fprintf(stderr, "TieredColumn::append_bytes requires HOST tier\n"); std::abort();
        }
        host_.insert(host_.end(), b, b + n);
        size_ += n;
    }

    // Byte checksum wherever the column lives (DEVICE copies back to host first).
    uint64_t checksum() const {
        const std::vector<unsigned char> b = read_to_host();
        uint64_t sum = 0;
        for (unsigned char c : b) sum += c;
        return sum;
    }

#if defined(MATRIX_USE_CUDA)
    const void* device_ptr() const { return device_; } // valid only while tier()==DEVICE
#endif

private:
    // Process-unique serial so two columns (even with the same logical id, even in two engine
    // instances) never share a COLD file. Assigned once at construction, stable for the object's
    // life (so HOST<->COLD round-trips hit the same path).
    static uint64_t next_serial() {
        static std::atomic<uint64_t> counter{0};
        return counter.fetch_add(1, std::memory_order_relaxed);
    }

    std::string cold_path() const {
        // pid + per-object serial: unique within and across engine instances/processes.
        return std::string("/tmp/matrixdb_tcol_")
             + std::to_string(static_cast<long long>(getpid())) + "_"
             + std::to_string(serial_) + ".bin";
    }

    std::vector<unsigned char> read_to_host() const {
        if (tier_ == MemorySpace::HOST) return host_;
        if (tier_ == MemorySpace::COLD) {
            std::vector<unsigned char> b(size_);
            FILE* f = std::fopen(cold_path().c_str(), "rb");
            if (!f) { std::fprintf(stderr, "TieredColumn: cold read failed %s\n", cold_path().c_str()); std::abort(); }
            const size_t got = std::fread(b.data(), 1, size_, f);
            std::fclose(f);
            if (got != size_) { std::fprintf(stderr, "TieredColumn: short cold read\n"); std::abort(); }
            return b;
        }
#if defined(MATRIX_USE_CUDA)
        if (tier_ == MemorySpace::DEVICE) {
            std::vector<unsigned char> b(size_);
            cudaMemcpy(b.data(), device_, size_, cudaMemcpyDeviceToHost);
            return b;
        }
#endif
        std::fprintf(stderr, "TieredColumn: read from unsupported tier\n");
        std::abort();
    }

    void write_cold(const std::vector<unsigned char>& b) {
        FILE* f = std::fopen(cold_path().c_str(), "wb");
        if (!f) { std::fprintf(stderr, "TieredColumn: cold write failed %s\n", cold_path().c_str()); std::abort(); }
        const size_t wrote = std::fwrite(b.data(), 1, b.size(), f);
        std::fclose(f);
        // Fail loud on a short write (e.g. disk full) rather than leave a truncated cold
        // copy that only surfaces as corruption on read-back — symmetric with read_to_host.
        if (wrote != b.size()) {
            std::fprintf(stderr, "TieredColumn: short cold write (%zu/%zu)\n", wrote, b.size());
            std::abort();
        }
    }

    void free_current() {
        if (tier_ == MemorySpace::COLD) std::remove(cold_path().c_str());
#if defined(MATRIX_USE_CUDA)
        if (tier_ == MemorySpace::DEVICE && device_) { cudaFree(device_); device_ = nullptr; }
#endif
        // HOST: host_ vector frees itself / is overwritten by the caller.
    }

#if defined(MATRIX_USE_CUDA)
    void push_device(const std::vector<unsigned char>& b) {
        cudaMalloc(&device_, size_);
        cudaMemcpy(device_, b.data(), size_, cudaMemcpyHostToDevice);
    }
    void* device_ = nullptr;
#endif

    uint64_t id_;
    size_t size_;
    MemorySpace tier_;
    const uint64_t serial_ = next_serial();
    std::vector<unsigned char> host_;
};


In [ ]:
%%writefile migration_executor.hpp
#pragma once

#include "tier_manager.hpp"   // MigrationDecision
#include "tiered_column.hpp"
#include <cstdint>
#include <cstddef>
#include <vector>
#include <unordered_map>
#include <cstdio>

// Turns a TierManager migration plan into physical byte movement. The brain decides
// (which column goes where); the executor actuates (calls migrate_to on the real bytes).
class MigrationExecutor {
public:
    // Apply each decision by migrating the named column to its target tier. A decision whose
    // column_id is not in the registry is skipped (logged). Returns the number applied.
    size_t apply(const std::vector<MigrationDecision>& plan,
                 std::unordered_map<uint64_t, TieredColumn*>& columns) {
        size_t applied = 0;
        for (const MigrationDecision& d : plan) {
            auto it = columns.find(d.column_id);
            if (it == columns.end()) {
                std::fprintf(stderr, "MigrationExecutor: no column %llu — skipped\n",
                             static_cast<unsigned long long>(d.column_id));
                continue;
            }
            it->second->migrate_to(d.to);
            ++applied;
        }
        return applied;
    }
};


In [ ]:
%%writefile column_io.hpp
#pragma once
#include <cstdint>
#include <cstddef>
#include <cstdio>
#include <cstdlib>
#include <string>
#include <vector>

// Binary column file format v0: [u32 magic][u64 count][count × u32 data].
// ponytail: host-endian, raw little-endian-native writes — a same-machine persistence/cache, NOT
// portable across architectures (a big-endian reader would see a magic mismatch and abort, which is
// the safe failure). A byte-swapped portable encoding is the v1 upgrade if cross-machine files matter.
inline constexpr uint32_t MATRIX_COLUMN_MAGIC = 0x4D43'4F4Cu; // 'MCOL' — MatrixDB column file v0

// Write a uint32 column to `path` as [magic][u64 count][count×u32]. Fail-loud (abort) on
// open/short-write — never leave a partially/wrongly written file mistaken for valid.
inline void matrix_write_column(const std::string& path, const uint32_t* data, size_t n) {
    FILE* f = std::fopen(path.c_str(), "wb");
    if (!f) { std::fprintf(stderr, "matrix_write_column: open failed %s\n", path.c_str()); std::abort(); }
    const uint32_t magic = MATRIX_COLUMN_MAGIC;
    const uint64_t count = n;
    const bool ok = std::fwrite(&magic, sizeof magic, 1, f) == 1
                 && std::fwrite(&count, sizeof count, 1, f) == 1
                 && (n == 0 || std::fwrite(data, sizeof(uint32_t), n, f) == n);
    std::fclose(f);
    if (!ok) { std::fprintf(stderr, "matrix_write_column: short write %s\n", path.c_str()); std::abort(); }
}

// Read a column written by matrix_write_column. Fail-loud on open / bad magic / short read.
inline void matrix_read_column(const std::string& path, std::vector<uint32_t>& out) {
    FILE* f = std::fopen(path.c_str(), "rb");
    if (!f) { std::fprintf(stderr, "matrix_read_column: open failed %s\n", path.c_str()); std::abort(); }
    uint32_t magic = 0; uint64_t count = 0;
    bool ok = std::fread(&magic, sizeof magic, 1, f) == 1 && magic == MATRIX_COLUMN_MAGIC
           && std::fread(&count, sizeof count, 1, f) == 1;
    if (ok) { out.resize(static_cast<size_t>(count));
              ok = (count == 0 || std::fread(out.data(), sizeof(uint32_t), out.size(), f) == out.size()); }
    std::fclose(f);
    if (!ok) { std::fprintf(stderr, "matrix_read_column: bad/short file %s\n", path.c_str()); std::abort(); }
}

// Typed single-column file v1: [u32 magic][u32 type][u64 byte_count][byte_count raw bytes]. Carries the
// element type (0=U32, 1=I64, 2=F64) so int64/double columns persist to a single file (the v0 functions
// above stay for the u32-only path). Fail-loud, host-endian — same contract as matrix_write/read_column.
inline constexpr uint32_t MATRIX_COLUMN_MAGIC_TYPED = 0x314F'434Du; // 'MCO1' — typed single-column file v1
inline void matrix_write_column_typed(const std::string& path, const void* bytes, size_t byte_count, uint32_t type) {
    FILE* f = std::fopen(path.c_str(), "wb");
    if (!f) { std::fprintf(stderr, "matrix_write_column_typed: open failed %s\n", path.c_str()); std::abort(); }
    const uint32_t magic = MATRIX_COLUMN_MAGIC_TYPED;
    const uint64_t bc = byte_count;
    const bool ok = std::fwrite(&magic, sizeof magic, 1, f) == 1
                 && std::fwrite(&type,  sizeof type,  1, f) == 1
                 && std::fwrite(&bc,    sizeof bc,    1, f) == 1
                 && (byte_count == 0 || std::fwrite(bytes, 1, byte_count, f) == byte_count);
    std::fclose(f);
    if (!ok) { std::fprintf(stderr, "matrix_write_column_typed: short write %s\n", path.c_str()); std::abort(); }
}
// Read a typed column file; fills `out` with the raw bytes and `out_type` with the element type.
// Fail-loud on open / bad magic / short read (same as matrix_read_column).
inline void matrix_read_column_typed(const std::string& path, std::vector<unsigned char>& out, uint32_t& out_type) {
    FILE* f = std::fopen(path.c_str(), "rb");
    if (!f) { std::fprintf(stderr, "matrix_read_column_typed: open failed %s\n", path.c_str()); std::abort(); }
    uint32_t magic = 0, type = 0; uint64_t bc = 0;
    bool ok = std::fread(&magic, sizeof magic, 1, f) == 1 && magic == MATRIX_COLUMN_MAGIC_TYPED
           && std::fread(&type, sizeof type, 1, f) == 1 && std::fread(&bc, sizeof bc, 1, f) == 1;
    if (ok) { out.resize(static_cast<size_t>(bc));
              ok = (bc == 0 || std::fread(out.data(), 1, out.size(), f) == out.size()); }
    std::fclose(f);
    if (!ok) { std::fprintf(stderr, "matrix_read_column_typed: bad/short file %s\n", path.c_str()); std::abort(); }
    out_type = type;
}


In [ ]:
%%writefile csv_ingest.hpp
#pragma once
#include <cstdint>
#include <charconv>      // std::from_chars — locale-free, no-throw integer parse
#include <cstdlib>       // std::strtod  (DM-3g double CSV parse — portable across libc++ versions)
#include <cerrno>        // errno, ERANGE (DM-3g double overflow detection)
#include <fstream>
#include <string>
#include <vector>

// Parse one uint32 column (0-based col_index) out of a simple CSV file. has_header skips line 1.
// Returns true + fills `out` on success; false + empty `out` on any error (open fail / short row /
// non-integer / overflow). CSV is untrusted input, so malformed data is a graceful false, NOT abort
// (cf. column_io.hpp, which aborts on corruption of our OWN binary format). See VAL-1.
// ponytail: no quoted-field / escape handling — simple unquoted integer CSV only; quote-aware split is
// the upgrade path if real files need it.
inline bool matrix_read_csv_column(const std::string& path, size_t col_index, bool has_header,
                                   char delim, std::vector<uint32_t>& out) {
    out.clear();
    std::ifstream in(path);
    if (!in) return false;
    std::string line;
    bool first = true;
    while (std::getline(in, line)) {
        if (!line.empty() && line.back() == '\r') line.pop_back();   // tolerate CRLF
        if (has_header && first) { first = false; continue; }
        first = false;
        // Walk to the col_index-th field without allocating a vector of all fields.
        size_t start = 0, field = 0;
        while (field < col_index) {
            size_t comma = line.find(delim, start);
            if (comma == std::string::npos) { out.clear(); return false; }   // short row
            start = comma + 1;
            ++field;
        }
        size_t end = line.find(delim, start);
        if (end == std::string::npos) end = line.size();
        const char* b = line.data() + start;
        const char* e = line.data() + end;
        uint32_t value = 0;
        auto [ptr, ec] = std::from_chars(b, e, value);
        if (ec != std::errc{} || ptr != e) { out.clear(); return false; }    // non-integer, overflow, or trailing junk
        out.push_back(value);
    }
    return true;
}

// int64 sibling of matrix_read_csv_column (DM-3g). Parses the col_index-th field as a signed 64-bit
// integer via std::from_chars (rejects trailing junk / overflow). Graceful false on any error.
inline bool matrix_read_csv_column_i64(const std::string& path, size_t col_index, bool has_header,
                                       char delim, std::vector<int64_t>& out) {
    out.clear();
    std::ifstream in(path);
    if (!in) return false;
    std::string line; bool first = true;
    while (std::getline(in, line)) {
        if (!line.empty() && line.back() == '\r') line.pop_back();
        if (has_header && first) { first = false; continue; }
        first = false;
        size_t start = 0, field = 0;
        while (field < col_index) {
            size_t comma = line.find(delim, start);
            if (comma == std::string::npos) { out.clear(); return false; }
            start = comma + 1; ++field;
        }
        size_t end = line.find(delim, start);
        if (end == std::string::npos) end = line.size();
        int64_t value = 0;
        auto [ptr, ec] = std::from_chars(line.data() + start, line.data() + end, value);
        if (ec != std::errc{} || ptr != line.data() + end) { out.clear(); return false; }
        out.push_back(value);
    }
    return true;
}

// double sibling (DM-3g). Parses via std::strtod (portable across libc++ versions), requiring the parse
// to consume exactly the field (no trailing junk) and not overflow. Graceful false on any error.
// ponytail: strtod skips leading whitespace and is C-locale '.'-decimal — fine for simple numeric CSV.
inline bool matrix_read_csv_column_f64(const std::string& path, size_t col_index, bool has_header,
                                       char delim, std::vector<double>& out) {
    out.clear();
    std::ifstream in(path);
    if (!in) return false;
    std::string line; bool first = true;
    while (std::getline(in, line)) {
        if (!line.empty() && line.back() == '\r') line.pop_back();
        if (has_header && first) { first = false; continue; }
        first = false;
        size_t start = 0, field = 0;
        while (field < col_index) {
            size_t comma = line.find(delim, start);
            if (comma == std::string::npos) { out.clear(); return false; }
            start = comma + 1; ++field;
        }
        size_t end = line.find(delim, start);
        if (end == std::string::npos) end = line.size();
        if (start == end) { out.clear(); return false; }     // empty field
        errno = 0;
        char* endptr = nullptr;
        const double value = std::strtod(line.c_str() + start, &endptr);
        if (errno == ERANGE || endptr != line.c_str() + end) { out.clear(); return false; }  // overflow / trailing junk / not a number
        out.push_back(value);
    }
    return true;
}


In [ ]:
%%writefile server.hpp
#pragma once
// Server core: a serializable request/response protocol + a stateless dispatcher over the engine.
// Transport-agnostic — a TCP/Unix-socket accept-loop is a thin adapter that shuttles these byte
// buffers (deferred; not buildable in this sandbox). Wire form is host-endian (same-machine /
// trusted-transport assumption, like column_io); a cross-arch encoding is a future upgrade.
#include "compute_mock.cpp"   // CPUMockEngine + compute.hpp (MatrixQuery/MatrixQueryStatus/MatrixAggOp)
#include <cstdint>
#include <cstring>
#include <unordered_set>
#include <vector>

enum class ReqKind : uint32_t { GET = 1, PUT = 2, QUERY = 3, HEALTH = 4, STATS = 5 };
enum class ServerStatus : uint32_t { OK = 0, ERR_BADREQUEST = 1000, ERR_FORBIDDEN = 1001, ERR_UNAUTHENTICATED = 1002 };

struct MatrixRequest {
    ReqKind  kind  = ReqKind::GET;
    uint64_t key   = 0;
    uint64_t value = 0;
    MatrixQuery query{};
};
struct MatrixResponse {
    uint32_t status = 0;
    std::vector<uint64_t> results;
};

// One audited request: principal, kind (ReqKind value, or 0 for a malformed request), the response
// status, and the target (GET/PUT key; QUERY value_col). The irrelevant target field is 0.
struct AuditEntry { uint64_t principal; uint32_t kind; uint32_t status; uint64_t key; uint64_t column; };

// Append-only audit trail of served requests (allowed, denied, and malformed alike).
class AuditLog {
public:
    void record(const AuditEntry& e) { entries_.push_back(e); }
    const std::vector<AuditEntry>& entries() const { return entries_; }
    size_t size() const { return entries_.size(); }
private:
    std::vector<AuditEntry> entries_;
};

// Per-principal authorization. Grants are additive; a principal with no grants can do nothing.
// permissive() allows everything (the no-auth / backward-compat default).
class AccessPolicy {
public:
    static AccessPolicy permissive() { AccessPolicy p; p.allow_all_ = true; return p; }
    void allow_column(uint64_t principal, uint64_t col) { cols_[principal].insert(col); } // QUERY access
    void allow_read(uint64_t principal)  { read_.insert(principal); }                      // GET
    void allow_write(uint64_t principal) { write_.insert(principal); }                     // PUT
    bool can_query(uint64_t principal, uint64_t col) const {
        if (allow_all_) return true;
        auto it = cols_.find(principal); return it != cols_.end() && it->second.count(col) != 0;
    }
    bool can_read(uint64_t principal)  const { return allow_all_ || read_.count(principal)  != 0; }
    bool can_write(uint64_t principal) const { return allow_all_ || write_.count(principal) != 0; }
private:
    bool allow_all_ = false;
    std::unordered_map<uint64_t, std::unordered_set<uint64_t>> cols_;
    std::unordered_set<uint64_t> read_, write_;
};

// SE-1 authentication: validate a bearer credential (token) -> principal id. Distinct from AccessPolicy
// (authz): authn establishes WHO a caller is; authz decides what that principal may do. The transport
// extracts the token from the connection and passes it here; an unknown/empty token authenticates to
// nobody. (Tokens are opaque strings — a real deployment stores salted hashes, not plaintext; this is the
// mechanism, not a credential store.)
class Authenticator {
public:
    void add_credential(const std::string& token, uint64_t principal) { tokens_[token] = principal; }
    // True + principal if the token is known; false (principal untouched) otherwise. Empty token -> false.
    bool authenticate(const std::string& token, uint64_t& principal) const {
        if (token.empty()) return false;
        auto it = tokens_.find(token);
        if (it == tokens_.end()) return false;
        principal = it->second;
        return true;
    }
private:
    std::unordered_map<std::string, uint64_t> tokens_;
};

namespace matrixsrv_detail {
    inline void put_u32(std::vector<uint8_t>& b, uint32_t v) { const uint8_t* p = reinterpret_cast<const uint8_t*>(&v); b.insert(b.end(), p, p + 4); }
    inline void put_u64(std::vector<uint8_t>& b, uint64_t v) { const uint8_t* p = reinterpret_cast<const uint8_t*>(&v); b.insert(b.end(), p, p + 8); }
    struct Reader {
        const uint8_t* p; const uint8_t* end; bool ok = true;
        bool u8(uint8_t& o)   { if (end - p < 1) { ok = false; return false; } o = *p++; return true; }
        bool u32(uint32_t& o) { if (end - p < 4) { ok = false; return false; } std::memcpy(&o, p, 4); p += 4; return true; }
        bool u64(uint64_t& o) { if (end - p < 8) { ok = false; return false; } std::memcpy(&o, p, 8); p += 8; return true; }
        bool done() const { return p == end; }
    };
}

inline std::vector<uint8_t> matrix_serialize_request(const MatrixRequest& r) {
    using namespace matrixsrv_detail;
    std::vector<uint8_t> b;
    put_u32(b, static_cast<uint32_t>(r.kind));
    put_u64(b, r.key); put_u64(b, r.value);
    put_u64(b, r.query.value_col);
    put_u32(b, static_cast<uint32_t>(r.query.agg));
    b.push_back(r.query.has_filter ? 1 : 0);
    put_u32(b, r.query.threshold);
    b.push_back(r.query.grouped ? 1 : 0);
    put_u64(b, r.query.key_col);
    put_u32(b, r.query.num_groups);
    return b;
}
inline bool matrix_deserialize_request(const std::vector<uint8_t>& b, MatrixRequest& out) {
    using namespace matrixsrv_detail;
    Reader r{ b.data(), b.data() + b.size() };
    uint32_t kind = 0, agg = 0; uint8_t hf = 0, gr = 0;
    r.u32(kind); r.u64(out.key); r.u64(out.value);
    r.u64(out.query.value_col); r.u32(agg); r.u8(hf); r.u32(out.query.threshold);
    r.u8(gr); r.u64(out.query.key_col); r.u32(out.query.num_groups);
    if (!r.ok || !r.done()) return false;
    if (kind < 1 || kind > 5) return false;
    out.kind = static_cast<ReqKind>(kind);
    out.query.agg = static_cast<MatrixAggOp>(agg);
    out.query.has_filter = (hf != 0);
    out.query.grouped = (gr != 0);
    return true;
}
inline std::vector<uint8_t> matrix_serialize_response(const MatrixResponse& r) {
    using namespace matrixsrv_detail;
    std::vector<uint8_t> b;
    put_u32(b, r.status);
    put_u64(b, static_cast<uint64_t>(r.results.size()));
    for (uint64_t x : r.results) put_u64(b, x);
    return b;
}
inline bool matrix_deserialize_response(const std::vector<uint8_t>& b, MatrixResponse& out) {
    using namespace matrixsrv_detail;
    Reader r{ b.data(), b.data() + b.size() };
    uint64_t count = 0;
    if (!r.u32(out.status) || !r.u64(count)) return false;
    if (count > (1ull << 28)) return false;
    if (static_cast<uint64_t>(r.end - r.p) != count * 8) return false;
    out.results.resize(static_cast<size_t>(count));
    for (uint64_t i = 0; i < count; ++i) r.u64(out.results[i]);
    return r.ok && r.done();
}

namespace matrixsrv_detail {
// The full serve pipeline (deserialize -> authorize -> dispatch), also filling `out` with the
// audit record for this request. Returns the serialized response. Shared by all matrix_serve overloads.
inline std::vector<uint8_t> serve_core(CPUMockEngine& eng, const AccessPolicy& policy,
                                       uint64_t principal, const std::vector<uint8_t>& req_bytes,
                                       AuditEntry& out) {
    MatrixRequest req; MatrixResponse resp;
    out.principal = principal; out.kind = 0; out.key = 0; out.column = 0;
    if (!matrix_deserialize_request(req_bytes, req)) {
        resp.status = static_cast<uint32_t>(ServerStatus::ERR_BADREQUEST);
        out.status = resp.status;
        return matrix_serialize_response(resp);
    }
    out.kind = static_cast<uint32_t>(req.kind);
    out.key = req.key;
    out.column = req.query.value_col;
    bool authorized = false;
    switch (req.kind) {
        case ReqKind::GET:   authorized = policy.can_read(principal);  break;
        case ReqKind::PUT:   authorized = policy.can_write(principal); break;
        case ReqKind::QUERY: authorized = policy.can_query(principal, req.query.value_col)
                                 && (!req.query.grouped || policy.can_query(principal, req.query.key_col)); break;
        case ReqKind::HEALTH: authorized = true; break;   // operational status (no data) — always probeable
        case ReqKind::STATS:  authorized = true; break;   // operational metrics (no row data) — always readable
    }
    if (!authorized) {
        resp.status = static_cast<uint32_t>(ServerStatus::ERR_FORBIDDEN);
        out.status = resp.status;
        return matrix_serialize_response(resp);
    }
    switch (req.kind) {
        case ReqKind::GET: {
            uint64_t v = 0; if (eng.kv_get(req.key, v)) resp.results.push_back(v);
            resp.status = static_cast<uint32_t>(ServerStatus::OK);
            break;
        }
        case ReqKind::PUT: {
            eng.begin(); eng.txn_put(req.key, req.value); eng.commit();
            resp.status = static_cast<uint32_t>(ServerStatus::OK);
            break;
        }
        case ReqKind::QUERY: {
            std::vector<uint64_t> o;
            const MatrixQueryStatus st = eng.execute_query(req.query, o);
            resp.status = static_cast<uint32_t>(st);
            resp.results = std::move(o);
            break;
        }
        case ReqKind::HEALTH: {
            // Pack the health snapshot into the result vector (fixed order — the client reads by index):
            // [ready, durable, catalog_columns, host_resident_bytes, wal_records_pending, dropped_writes].
            const HealthStatus h = eng.health();
            resp.results = { h.ready ? 1u : 0u, h.durable ? 1u : 0u,
                             static_cast<uint64_t>(h.catalog_columns), static_cast<uint64_t>(h.host_resident_bytes),
                             h.wal_records_pending, h.dropped_writes };
            resp.status = static_cast<uint32_t>(ServerStatus::OK);
            break;
        }
        case ReqKind::STATS: {
            // Operational metrics over the wire (OB-5/OB-2). Fixed order — the client reads by index:
            // [cold_borrows, rebalances, migrations, catalog_columns, host_resident_bytes,
            //  cold_resident_bytes, query_count, total_query_ns, max_query_ns, p50_ns, p99_ns, version_u64].
            const EngineStats s = eng.stats();
            resp.results = { s.cold_borrows, s.rebalances, s.migrations,
                             static_cast<uint64_t>(s.catalog_columns), static_cast<uint64_t>(s.host_resident_bytes),
                             static_cast<uint64_t>(s.cold_resident_bytes), s.query_count, s.total_query_ns,
                             s.max_query_ns, eng.query_latency_percentile_ns(0.50), eng.query_latency_percentile_ns(0.99),
                             eng.version_u64() };
            resp.status = static_cast<uint32_t>(ServerStatus::OK);
            break;
        }
    }
    out.status = resp.status;
    return matrix_serialize_response(resp);
}
} // namespace matrixsrv_detail

// Authorizing dispatch (principal supplied by the authenticated caller; denied -> ERR_FORBIDDEN
// with no engine side effects; bad request -> ERR_BADREQUEST). See SE-2.
inline std::vector<uint8_t> matrix_serve(CPUMockEngine& eng, const AccessPolicy& policy,
                                         uint64_t principal, const std::vector<uint8_t>& req_bytes) {
    AuditEntry ignored;
    return matrixsrv_detail::serve_core(eng, policy, principal, req_bytes, ignored);
}
// Same, but records the request (allowed / denied / malformed) to `audit`.
inline std::vector<uint8_t> matrix_serve(CPUMockEngine& eng, const AccessPolicy& policy,
                                         uint64_t principal, const std::vector<uint8_t>& req_bytes,
                                         AuditLog& audit) {
    AuditEntry entry;
    auto resp = matrixsrv_detail::serve_core(eng, policy, principal, req_bytes, entry);
    audit.record(entry);
    return resp;
}
// No-auth convenience (backward-compat): permissive policy, anonymous principal.
inline std::vector<uint8_t> matrix_serve(CPUMockEngine& eng, const std::vector<uint8_t>& req_bytes) {
    static const AccessPolicy permissive = AccessPolicy::permissive();
    return matrix_serve(eng, permissive, /*principal=*/0, req_bytes);
}

// SE-1 authenticating dispatch: validate the bearer `token` -> principal FIRST; an unknown/empty token is
// rejected with ERR_UNAUTHENTICATED and NO engine call (authn precedes authz precedes any work). On a valid
// token, serve as that principal (so AccessPolicy still gates what it may do). The transport supplies the
// token from the connection; it is never read from the request payload (no spoofing).
inline std::vector<uint8_t> matrix_serve(CPUMockEngine& eng, const AccessPolicy& policy,
                                         const Authenticator& auth, const std::string& token,
                                         const std::vector<uint8_t>& req_bytes) {
    uint64_t principal = 0;
    if (!auth.authenticate(token, principal)) {
        MatrixResponse resp; resp.status = static_cast<uint32_t>(ServerStatus::ERR_UNAUTHENTICATED);
        return matrix_serialize_response(resp);
    }
    return matrix_serve(eng, policy, principal, req_bytes);
}


In [ ]:
%%writefile server_tcp.hpp
#pragma once
// TCP transport adapter for the serve LOGIC (NW-1). Length-prefixed wire framing:
//   request  on the wire: [u32 len][len request bytes]   (the matrix_serialize_request blob)
//   response on the wire: [u32 len][len response bytes]   (the matrix_serialize_response blob)
// matrix_serve_conn() (serve one framed request on a connected fd) is the wire protocol — runtime-tested
// over a socketpair (no bind needed). matrix_serve_tcp() is the bind/listen/accept loop — HOST-ONLY:
// loopback bind() is blocked in the build sandbox (proven), so its accept loop is compile-verified here
// and runtime-verified only on a non-sandboxed host. One connection served at a time (single-owner per
// the page-ownership model; concurrent serving is NW-2, which is contra the lock-free thesis).
#include "server.hpp"          // matrix_serve, AccessPolicy, CPUMockEngine
#include <sys/socket.h>
#include <sys/time.h>          // struct timeval (SO_RCVTIMEO)
#include <netinet/in.h>
#include <unistd.h>
#include <cstdint>
#include <vector>

// NW-5 connection management: bound how long a recv may block, so a client that connects but never sends
// (slowloris-style) can't hang the single-owner serve loop forever. After the timeout, recv fails →
// recv_all returns false → matrix_serve_conn returns false → the loop drops the stuck connection and moves
// on. ms == 0 clears the timeout (block forever, the default). Returns true on success.
inline bool matrix_set_recv_timeout(int fd, unsigned ms) {
    struct timeval tv;
    tv.tv_sec  = static_cast<long>(ms / 1000);
    tv.tv_usec = static_cast<long>((ms % 1000) * 1000);
    return ::setsockopt(fd, SOL_SOCKET, SO_RCVTIMEO, &tv, sizeof tv) == 0;
}
// Symmetric send timeout (SO_SNDTIMEO): bound how long a send may block, so a client that connects but
// never READS the response (slow-reader, the other half of the DoS) can't wedge the loop's send_all once
// the socket buffers fill. Same kernel mechanism as the recv timeout (verified there) — send returns
// EAGAIN past the deadline → send_all returns false → serve_conn returns false → the loop drops it.
inline bool matrix_set_send_timeout(int fd, unsigned ms) {
    struct timeval tv;
    tv.tv_sec  = static_cast<long>(ms / 1000);
    tv.tv_usec = static_cast<long>((ms % 1000) * 1000);
    return ::setsockopt(fd, SOL_SOCKET, SO_SNDTIMEO, &tv, sizeof tv) == 0;
}

namespace matrixsrv_detail {
// Read/write EXACTLY n bytes over a stream socket, looping over partial transfers; false on EOF/error.
inline bool recv_all(int fd, void* buf, size_t n) {
    auto* p = static_cast<unsigned char*>(buf); size_t got = 0;
    while (got < n) { const ssize_t r = ::recv(fd, p + got, n - got, 0); if (r <= 0) return false; got += static_cast<size_t>(r); }
    return true;
}
inline bool send_all(int fd, const void* buf, size_t n) {
    // MSG_NOSIGNAL (Linux): writing to a peer that closed returns EPIPE instead of raising SIGPIPE (which
    // would kill the process). macOS/BSD lacks the flag — there the caller ignores SIGPIPE / sets
    // SO_NOSIGPIPE (matrix_serve_tcp/clients should `signal(SIGPIPE, SIG_IGN)`); EPIPE then surfaces as w<0.
#ifdef MSG_NOSIGNAL
    const int flags = MSG_NOSIGNAL;
#else
    const int flags = 0;
#endif
    const auto* p = static_cast<const unsigned char*>(buf); size_t sent = 0;
    while (sent < n) { const ssize_t w = ::send(fd, p + sent, n - sent, flags); if (w <= 0) return false; sent += static_cast<size_t>(w); }
    return true;
}
} // namespace matrixsrv_detail

// Serve ONE framed request on a connected socket `fd`: read [u32 len][req], dispatch via matrix_serve
// (authorizing), write [u32 len][resp]. Returns false if the peer closed or the framing was short.
inline bool matrix_serve_conn(CPUMockEngine& eng, const AccessPolicy& policy, uint64_t principal, int fd) {
    uint32_t len = 0;
    if (!matrixsrv_detail::recv_all(fd, &len, sizeof len)) return false;
    if (len > (1u << 28)) return false;                       // sane cap — never alloc on a bogus length
    std::vector<uint8_t> req(len);
    if (len != 0 && !matrixsrv_detail::recv_all(fd, req.data(), len)) return false;
    const std::vector<uint8_t> resp = matrix_serve(eng, policy, principal, req);
    const uint32_t rlen = static_cast<uint32_t>(resp.size());
    return matrixsrv_detail::send_all(fd, &rlen, sizeof rlen)
        && (rlen == 0 || matrixsrv_detail::send_all(fd, resp.data(), rlen));
}

// SE-1 over the transport: a connection authenticates ONCE with a leading token frame, then serves that
// principal's requests for the life of the connection. Reads [u32 len][token]; authenticates → principal
// (on failure: serve nothing, return false so the caller closes the connection); then runs the normal
// matrix_serve_conn loop as that principal (AccessPolicy still gates each request). This is the realistic
// model — authenticate per connection, not per request — and the inverse of MatrixClient::authenticate.
inline bool matrix_serve_conn_auth(CPUMockEngine& eng, const AccessPolicy& policy,
                                   const Authenticator& auth, int fd) {
    uint32_t tlen = 0;
    if (!matrixsrv_detail::recv_all(fd, &tlen, sizeof tlen)) return false;
    if (tlen > 4096) return false;                            // sane token cap — never alloc on a bogus length
    std::string token(tlen, '\0');
    if (tlen != 0 && !matrixsrv_detail::recv_all(fd, token.data(), tlen)) return false;
    uint64_t principal = 0;
    if (!auth.authenticate(token, principal)) return false;   // unauthenticated: serve nothing, close
    while (matrix_serve_conn(eng, policy, principal, fd)) { /* serve until the peer closes */ }
    return true;
}

// TCP accept-loop on `port`: serve each connection's requests (one at a time) until the peer closes.
// HOST-ONLY (see file header — bind is blocked in the build sandbox). Returns -1 on a setup error.
// principal=0 here; a real deployment derives the principal from the authenticated connection (SE-1/3).
// Each accepted connection gets a recv timeout (NW-5) so one stuck client can't hang the loop forever.
inline int matrix_serve_tcp(CPUMockEngine& eng, const AccessPolicy& policy, uint16_t port,
                            unsigned recv_timeout_ms = 30000) {
    const int srv = ::socket(AF_INET, SOCK_STREAM, 0);
    if (srv < 0) return -1;
    int yes = 1; ::setsockopt(srv, SOL_SOCKET, SO_REUSEADDR, &yes, sizeof yes);
    sockaddr_in addr{}; addr.sin_family = AF_INET; addr.sin_addr.s_addr = INADDR_ANY; addr.sin_port = htons(port);
    if (::bind(srv, reinterpret_cast<sockaddr*>(&addr), sizeof addr) != 0) { ::close(srv); return -1; }
    if (::listen(srv, 16) != 0) { ::close(srv); return -1; }
    for (;;) {
        const int c = ::accept(srv, nullptr, nullptr);
        if (c < 0) continue;
        if (recv_timeout_ms) matrix_set_recv_timeout(c, recv_timeout_ms);   // NW-5: drop a stuck reader/writer
        if (recv_timeout_ms) matrix_set_send_timeout(c, recv_timeout_ms);   // both directions, same deadline
        while (matrix_serve_conn(eng, policy, /*principal=*/0, c)) { /* serve until the peer closes */ }
        ::close(c);
    }
}


In [ ]:
%%writefile client.hpp
#pragma once
// NW-4 client driver — the app-side of the wire protocol. Wraps a connected stream socket `fd` with
// typed calls that frame a request ([u32 len][bytes]), send it, read the framed response, and
// deserialize. The inverse of matrix_serve_conn; reuses the same length-prefixed framing + recv_all/
// send_all, so it interoperates with matrix_serve_tcp on a real host. One request in flight at a time
// (matches the single-owner serving model). A transport failure (peer closed / short frame) -> false.
#include "server.hpp"
#include "server_tcp.hpp"   // matrixsrv_detail::recv_all / send_all + the framing contract
#include <cstdint>
#include <vector>

class MatrixClient {
public:
    explicit MatrixClient(int fd) : fd_(fd) {}

    // SE-1 handshake: send the leading token frame ([u32 len][token]) that authenticates this connection.
    // Call ONCE before any op when the server uses matrix_serve_conn_auth. Returns false on a transport
    // error. (The server validates the token; an invalid one makes it close the connection, so subsequent
    // ops here return false.)
    bool authenticate(const std::string& token) {
        const uint32_t len = static_cast<uint32_t>(token.size());
        if (!matrixsrv_detail::send_all(fd_, &len, sizeof len)) return false;
        return len == 0 || matrixsrv_detail::send_all(fd_, token.data(), len);
    }

    // GET key -> value. Returns false on a transport error OR a miss (no such key); true + out on a hit.
    bool get(uint64_t key, uint64_t& out) {
        MatrixRequest r; r.kind = ReqKind::GET; r.key = key;
        MatrixResponse resp;
        if (!round_trip(r, resp) || resp.status != 0 || resp.results.empty()) return false;
        out = resp.results[0];
        return true;
    }

    // PUT key=value (durably committed server-side). Returns true on OK.
    bool put(uint64_t key, uint64_t value) {
        MatrixRequest r; r.kind = ReqKind::PUT; r.key = key; r.value = value;
        MatrixResponse resp;
        return round_trip(r, resp) && resp.status == 0;
    }

    // Run an analytical query. Returns false on a transport error; otherwise true with the query's
    // MatrixQueryStatus in `st` and the result vector in `out` (so a wire failure is distinct from a
    // query-level error like ERR_UNKNOWN_COLUMN).
    bool query(const MatrixQuery& q, MatrixQueryStatus& st, std::vector<uint64_t>& out) {
        MatrixRequest r; r.kind = ReqKind::QUERY; r.query = q;
        MatrixResponse resp;
        if (!round_trip(r, resp)) return false;
        st = static_cast<MatrixQueryStatus>(resp.status);
        out = resp.results;
        return true;
    }

    // HEALTH probe -> the server's readiness snapshot. False on transport error / unexpected shape.
    bool health(HealthStatus& out) {
        MatrixRequest r; r.kind = ReqKind::HEALTH;
        MatrixResponse resp;
        if (!round_trip(r, resp) || resp.status != 0 || resp.results.size() != 6) return false;
        const auto& f = resp.results;
        out = HealthStatus{ f[0] != 0, f[1] != 0, static_cast<size_t>(f[2]),
                            static_cast<size_t>(f[3]), f[4], f[5] };
        return true;
    }

    // STATS -> the raw 12-field metrics vector (the STATS wire layout; see serve_core). False on error.
    bool stats(std::vector<uint64_t>& out) {
        MatrixRequest r; r.kind = ReqKind::STATS;
        MatrixResponse resp;
        if (!round_trip(r, resp) || resp.status != 0 || resp.results.size() != 12) return false;
        out = resp.results;
        return true;
    }

    // The server's build version (packed major<<32|minor<<16|patch), read from the STATS snapshot. 0 on error.
    uint64_t server_version() {
        std::vector<uint64_t> s;
        return stats(s) ? s[11] : 0;
    }

private:
    int fd_;
    // Frame + send the request, read + deframe the response. False on any short transfer / EOF / bad frame.
    bool round_trip(const MatrixRequest& req, MatrixResponse& resp) {
        const std::vector<uint8_t> rb = matrix_serialize_request(req);
        const uint32_t len = static_cast<uint32_t>(rb.size());
        if (!matrixsrv_detail::send_all(fd_, &len, sizeof len)) return false;
        if (len != 0 && !matrixsrv_detail::send_all(fd_, rb.data(), len)) return false;
        uint32_t rlen = 0;
        if (!matrixsrv_detail::recv_all(fd_, &rlen, sizeof rlen)) return false;
        if (rlen > (1u << 28)) return false;                       // sane cap — never alloc on a bogus length
        std::vector<uint8_t> respb(rlen);
        if (rlen != 0 && !matrixsrv_detail::recv_all(fd_, respb.data(), rlen)) return false;
        return matrix_deserialize_response(respb, resp);
    }
};


In [ ]:
%%writefile version.hpp
#pragma once
// BP-3 versioning: the build's version, so a running instance can report which build it is (ops correlate
// behavior ↔ build). Semantic versioning; bump on release and pair with a git tag (the release process).
// Pre-1.0 (0.x): the analytical CPU engine + tiering + durability + server protocol are feature-complete
// and verified locally; 1.0 is gated on the GPU backend (Colab) and a real network deployment.
#include <cstdint>

#define MATRIXDB_VERSION_MAJOR 0
#define MATRIXDB_VERSION_MINOR 1
#define MATRIXDB_VERSION_PATCH 0

inline constexpr const char* MATRIXDB_VERSION = "0.1.0";
inline const char* matrixdb_version() { return MATRIXDB_VERSION; }

// Packed numeric form for the wire / a version comparison: major<<32 | minor<<16 | patch. Lets a client
// read and compare the server's version without parsing a string (fits a uint64 result field).
inline constexpr uint64_t matrixdb_version_u64() {
    return (static_cast<uint64_t>(MATRIXDB_VERSION_MAJOR) << 32)
         | (static_cast<uint64_t>(MATRIXDB_VERSION_MINOR) << 16)
         |  static_cast<uint64_t>(MATRIXDB_VERSION_PATCH);
}


In [ ]:
%%writefile logging.hpp
#pragma once
// OB-1 structured logging: a tiny leveled logger with a runtime-settable threshold, so operators can
// filter/route diagnostics instead of grepping unconditional `cout`. Levels DEBUG<INFO<WARN<ERROR;
// default WARN (quiet — only problems surface). Writes "[LEVEL] msg\n" to stderr (logs are not data — keeps
// stdout clean). One global threshold (the engine is single-owner; a real deployment swaps the sink for a
// structured/JSON appender + per-subsystem levels).
#include <iostream>
#include <string>

enum class LogLevel : int { DEBUG = 0, INFO = 1, WARN = 2, ERROR = 3 };

class Log {
public:
    static void set_level(LogLevel l) { level() = l; }
    static LogLevel get_level() { return level(); }
    // True if a message at `l` passes the current threshold (would be emitted). Lets callers skip building
    // an expensive message — and lets tests assert the filter without capturing stderr.
    static bool enabled(LogLevel l) { return static_cast<int>(l) >= static_cast<int>(level()); }
    static void emit(LogLevel l, const std::string& msg) {
        if (!enabled(l)) return;
        std::cerr << prefix(l) << msg << '\n';
    }
private:
    static LogLevel& level() { static LogLevel lv = LogLevel::WARN; return lv; }
    static const char* prefix(LogLevel l) {
        switch (l) {
            case LogLevel::DEBUG: return "[DEBUG] ";
            case LogLevel::INFO:  return "[INFO] ";
            case LogLevel::WARN:  return "[WARN] ";
            default:              return "[ERROR] ";
        }
    }
};


In [ ]:
%%writefile compute_mock.cpp
#include "compute.hpp"
#include "kv_store.hpp"
#include "cold_store.hpp"
#include "migration_executor.hpp"  // MigrationExecutor + TierManager + TieredColumn + CostModel
#include "memory_model.hpp"        // MemorySpace, MemoryModel
#include "column_io.hpp"           // matrix_write_column / matrix_read_column (binary column persistence)
#include "csv_ingest.hpp"          // matrix_read_csv_column (CSV column ingest, graceful on bad input)
#include "version.hpp"             // MATRIXDB_VERSION (BP-3)
#include "logging.hpp"             // Log / LogLevel (OB-1 structured logging)
#include <unordered_map>
#include <unordered_set>
#include <map>
#include <algorithm>
#include <limits>
#include <memory>
#include <string>
#include <vector>
#include <array>
#include <bit>
#include <cassert>
#include <cctype>
#include <chrono>
#include <iostream>

/**
 * @brief CPU Mock Engine — the no-GPU fallback (Component 5: Local Sandbox).
 * Page-ownership model: bins the batch by owning page, then processes each page's
 * queries against its own slice of the store. One owner per page ⇒ no atomics, no
 * delta log, deterministic last-writer-wins. The point-op store is a real KVStore
 * (DM-1 fix: distinct colliding keys never overwrite). The CUDA engine's point-op
 * store is still the flat key&MASK array (replaced in a later increment), so CPU/GPU
 * store parity holds only while keys don't collide; CPU is the DM-1-correct path.
 */
// Engine observability snapshot: tiering activity counters + current resident-bytes gauges.
struct EngineStats {
    uint64_t cold_borrows;        // COLD->HOST borrows performed for a scan/aggregate
    uint64_t rebalances;          // rebalance() passes run (scan-driven)
    uint64_t migrations;          // migration decisions actually executed
    size_t   catalog_columns;     // # columns in the tiered catalog
    size_t   host_resident_bytes; // catalog bytes currently in RAM (TierManager's view)
    size_t   cold_resident_bytes; // catalog bytes currently on SSD
    uint64_t query_count;         // execute_query calls served (OK and ERR)
    uint64_t total_query_ns;      // summed execute_query wall-time (ns)
    uint64_t max_query_ns;        // slowest single execute_query (ns)
};

// One catalog column's metadata (DM-2 introspection): id, optional name (""), element type, row count, tier.
struct ColumnInfo { uint64_t id; std::string name; MatrixType type; size_t rows; MemorySpace tier; };

// OB-3 health/readiness snapshot for an orchestrator probe: a ready verdict + the gauges that justify it.
// `ready` is false when the engine is degraded (point-op writes dropped — the KVStore filled), so a
// liveness/readiness probe can pull the instance out of rotation and page someone.
struct HealthStatus {
    bool     ready;                 // false if degraded (dropped_writes > 0)
    bool     durable;               // a WAL is attached (committed point-ops survive restart)
    size_t   catalog_columns;       // analytical columns registered
    size_t   host_resident_bytes;   // catalog bytes currently in RAM
    uint64_t wal_records_pending;   // un-checkpointed WAL records (a restart-recovery-cost proxy)
    uint64_t dropped_writes;        // point-op writes lost to a full KVStore (the degradation signal)
};

// Tokenize a query string: identifiers/keywords/numbers, the comparison operators (> >= < <= = !=),
// and parens. Whitespace-separated; operators and parens need no surrounding spaces. (free helper)
inline std::vector<std::string> matrixparse_tokenize(const std::string& s) {
    std::vector<std::string> t;
    const size_t n = s.size();
    for (size_t i = 0; i < n; ) {
        const char c = s[i];
        if (std::isspace(static_cast<unsigned char>(c))) { ++i; continue; }
        if (c == '(' || c == ')') { t.emplace_back(1, c); ++i; continue; }
        if (c == '>' || c == '<' || c == '=' || c == '!') {
            if (i + 1 < n && s[i + 1] == '=') { t.push_back(s.substr(i, 2)); i += 2; }
            else { t.emplace_back(1, c); ++i; }
            continue;
        }
        size_t j = i;   // a run up to the next space / paren / operator char (covers names and signed numbers)
        while (j < n && !std::isspace(static_cast<unsigned char>(s[j])) && s[j] != '(' && s[j] != ')'
               && s[j] != '>' && s[j] != '<' && s[j] != '=' && s[j] != '!') ++j;
        t.push_back(s.substr(i, j - i));
        i = j;
    }
    return t;
}

#if defined(MATRIX_USE_CUDA)
// GPU-3: defined in compute_cuda.cuh (included after this file in the nvcc TU). Reduce a DEVICE/VRAM-
// resident column's bytes with the cross-checked GPU kernels; the return matches the CPU reducers'
// encoding so execute_query's DEVICE path is bit-identical to its CPU path.
inline uint64_t matrix_gpu_reduce_dev_u32(const void* d, size_t n, MatrixPredicate pred, MatrixAggOp op, bool has_filter);
inline int64_t  matrix_gpu_reduce_dev_i64(const void* d, size_t n, MatrixPredicateI64 pred, MatrixAggOp op, bool has_filter);
inline double   matrix_gpu_reduce_dev_f64(const void* d, size_t n, MatrixPredicateF64 pred, MatrixAggOp op, bool has_filter);
inline void     matrix_gpu_group_reduce_dev(const void* keys, const void* vals, size_t n, uint32_t num_groups, MatrixAggOp op, MatrixPredicate pred, bool has_filter, uint64_t* out_host);
inline void     matrix_gpu_group_reduce_dev_i64(const void* keys, const void* vals, size_t n, uint32_t num_groups, MatrixAggOp op, MatrixPredicateI64 pred, bool has_filter, uint64_t* out_host);
inline void     matrix_gpu_group_reduce_dev_f64(const void* keys, const void* vals, size_t n, uint32_t num_groups, MatrixAggOp op, MatrixPredicateF64 pred, bool has_filter, uint64_t* out_host);
#endif

class CPUMockEngine : public ComputeInterface {
public:
    // host_cap is the RAM budget (bytes) for the tiered catalog; default unbounded so the
    // existing pipeline (empty catalog) is unaffected. device_cap=1 makes the DEVICE tier
    // inert on the CPU build: scan_us() ignores gpu_available, so the brain would otherwise
    // emit HOST->DEVICE promotions the CPU executor's migrate_to(DEVICE) aborts on; a 1-byte
    // cap means no real column ever fits, so no DEVICE decision is emitted (cap==0 == unbounded).
    explicit CPUMockEngine(size_t /*worker_count*/ = 0, std::string wal_path = "",
                           size_t host_cap = SIZE_MAX, SyncPolicy sync = SyncPolicy::SYNC_EACH)
#if defined(MATRIX_USE_CUDA)
        // GPU-3: a real device budget + gpu=true cost model makes the DEVICE tier live, so hot analytical
        // columns promote to VRAM and scan_tiered_column* reduces them in place on the GPU (1 GiB << T4's 16 GB).
        : tier_mgr_(CostModel(MemoryModel::detect(true), true), /*device_cap=*/(size_t)1 << 30, host_cap)
#else
        : tier_mgr_(CostModel(MemoryModel::detect(false), false), /*device_cap=*/1, host_cap)
#endif
        , binned_(MATRIX_BATCH_MAX)
        , scan_column_(MATRIX_SCAN_COLUMN_SIZE) {
        for (size_t i = 0; i < MATRIX_SCAN_COLUMN_SIZE; ++i)
            scan_column_[i] = static_cast<uint32_t>(i); // resident analytical column
        // Durability is opt-in: with a WAL path, recover the point-op store by replaying
        // the log into kv_ (a write committed before a crash is restored here). DU-5: `sync` picks the
        // durability level — SYNC_EACH (default) fsyncs each append so a committed write survives power
        // loss; SYNC_OFF trades that for throughput (a crash may lose the unflushed tail).
        if (!wal_path.empty()) {
            checkpoint_path_ = wal_path + ".ckpt";
            load_checkpoint(checkpoint_path_);                                    // restore the last compaction (no-op if none)
            cold_store_ = std::make_unique<ColdStore>(wal_path, sync);
            cold_store_->replay([this](uint64_t k, uint64_t v){ kv_.put(k, v); }); // post-checkpoint records on top
            kv_.for_each([this](uint64_t k, uint64_t v){ key_index_[k] = v; });     // rebuild the ordered index from recovered state
        }
        std::cout << "CPUMockEngine initialized (page-ownership, "
                  << MATRIX_PAGE_COUNT << " pages, "
                  << MATRIX_SCAN_COLUMN_SIZE << "-value scan column"
                  << (cold_store_ ? ", WAL durability ON" : "") << ")." << std::endl;
    }

    // RM-2 admission control: cap the groups a single grouped query may allocate (the out vector is
    // num_groups * 8 bytes, so this bounds one query's result memory). Default is MAX_QUERY_GROUPS (2^28,
    // ~2 GB); an operator can tighten it so one runaway GROUP BY can't OOM the box. A query above the cap
    // returns ERR_TOO_MANY_GROUPS (no allocation). Runtime-settable (a step toward OB-4 runtime config).
    void set_max_query_groups(uint32_t n) { max_query_groups_ = n; }
    uint32_t max_query_groups() const { return max_query_groups_; }

    // OB-4 runtime config: tune the heat-rebalance cadence (run the brain + executor every N tiered scans)
    // without recompiling. Default REBALANCE_EVERY (4). A smaller N re-tiers more aggressively (more
    // responsive, more migration work); a large N relaxes it. Clamped to ≥ 1 (0 → 1, rebalance every scan).
    void set_rebalance_interval(uint64_t n) { rebalance_every_ = n ? n : 1; }
    uint64_t rebalance_interval() const { return rebalance_every_; }

    // BP-3: the build version this instance is running (semver string + packed numeric form for the wire).
    const char* version() const { return matrixdb_version(); }
    uint64_t version_u64() const { return matrixdb_version_u64(); }

    // OB-1/OB-4: set the diagnostic log threshold at runtime (DEBUG<INFO<WARN<ERROR; default WARN). Global
    // (the logger is process-wide); exposed here for API discoverability alongside the other tuning knobs.
    void set_log_level(LogLevel l) { Log::set_level(l); }
    LogLevel log_level() const { return Log::get_level(); }

    // Register a uint32 analytical column into the tiered catalog (born resident in HOST).
    // id must be > 0 (0 is reserved for the legacy fixed scan column).
    void load_scan_column(uint64_t id, const uint32_t* data, size_t n) {
        assert(id != 0 && "column id 0 is reserved for the legacy fixed scan column");
        // Register once: re-registering an id would desync the catalog from the brain (and
        // could orphan a demoted column's COLD file). Callers assign distinct ids.
        assert(catalog_.find(id) == catalog_.end() && "column id already registered");
        const size_t bytes = n * sizeof(uint32_t);
        catalog_[id] = std::make_unique<TieredColumn>(
            id, reinterpret_cast<const unsigned char*>(data), bytes);
        tier_mgr_.register_column(id, bytes, MemorySpace::HOST);
    }

    // Register a signed int64 analytical column (born HOST-resident, like load_scan_column). DM-3a.
    void load_scan_column_i64(uint64_t id, const int64_t* data, size_t n) {
        assert(id != 0 && "column id 0 is reserved for the legacy fixed scan column");
        assert(catalog_.find(id) == catalog_.end() && "column id already registered");
        const size_t bytes = n * sizeof(int64_t);
        catalog_[id] = std::make_unique<TieredColumn>(id, reinterpret_cast<const unsigned char*>(data), bytes);
        tier_mgr_.register_column(id, bytes, MemorySpace::HOST);
        col_types_[id] = MatrixType::I64;
    }

    // Register a double (float64) analytical column (born HOST-resident, like load_scan_column). DM-3e.
    void load_scan_column_f64(uint64_t id, const double* data, size_t n) {
        assert(id != 0 && "column id 0 is reserved for the legacy fixed scan column");
        assert(catalog_.find(id) == catalog_.end() && "column id already registered");
        const size_t bytes = n * sizeof(double);
        catalog_[id] = std::make_unique<TieredColumn>(id, reinterpret_cast<const unsigned char*>(data), bytes);
        tier_mgr_.register_column(id, bytes, MemorySpace::HOST);
        col_types_[id] = MatrixType::F64;
    }

    // --- Minimal variable-length string columns (DM-3i) ---
    // A SELF-CONTAINED store, separate from the byte catalog_ (whose columns are fixed-width TieredColumns).
    // Supports load + row count + equality-filter count + element access — the meaningful string ops
    // (SUM/MIN/MAX don't apply numerically). ponytail: a plain id->vector<string> map — not tiered, and
    // NOT in catalog_columns()/execute_query (those stay fixed-width-typed); full integration needs the
    // catalog generalized beyond TieredColumn (XL — the upgrade path).
    void load_string_column(uint64_t id, const std::vector<std::string>& data) {
        assert(id != 0 && "column id 0 is reserved");
        string_columns_[id] = data;
    }
    size_t string_column_rows(uint64_t id) const {
        auto it = string_columns_.find(id);
        return it == string_columns_.end() ? 0 : it->second.size();
    }
    // COUNT of rows whose string equals `value` (a string WHERE col = 'literal' count).
    uint64_t string_count_where_eq(uint64_t id, const std::string& value) const {
        auto it = string_columns_.find(id);
        if (it == string_columns_.end()) return 0;
        uint64_t c = 0;
        for (const std::string& s : it->second) if (s == value) ++c;
        return c;
    }
    std::string string_column_at(uint64_t id, size_t row) const {
        auto it = string_columns_.find(id);
        assert(it != string_columns_.end() && row < it->second.size() && "string_column_at: bad id/row");
        return it->second[row];
    }

    // --- Nullable columns (DM-3j) ---
    // Mark which rows of a U32 catalog column are NULL (1=null), so scalar aggregates skip them (SQL NULL
    // semantics). is_null must have one byte per row. ponytail: U32 unfiltered scalar only for this slice —
    // int64/double/filtered/grouped null-awareness is the follow-up (the maskless path is byte-identical).
    void set_column_nulls(uint64_t id, const std::vector<uint8_t>& is_null) {
        assert(catalog_has(id) && "set_column_nulls: unknown catalog column");   // any byte-catalog column (U32/I64/F64)
        assert(is_null.size() == column_rows(id) && "null mask must have one entry per row");
        null_masks_[id] = is_null;
    }

    // Append `n` more rows to an existing catalog column, growing it (DM-9). The store is no longer
    // load-once. Asserts the column exists and the element type matches; works across the COLD tier
    // (append_raw borrows COLD->HOST, grows, returns the borrow). Appended rows are immediately queryable.
    void append_to_column(uint64_t id, const uint32_t* data, size_t n) {
        assert(catalog_has(id) && column_type(id) == MatrixType::U32 && "append type must match column (u32)");
        append_raw(id, reinterpret_cast<const unsigned char*>(data), n * sizeof(uint32_t));
    }
    void append_to_column_i64(uint64_t id, const int64_t* data, size_t n) {
        assert(catalog_has(id) && column_type(id) == MatrixType::I64 && "append type must match column (int64)");
        append_raw(id, reinterpret_cast<const unsigned char*>(data), n * sizeof(int64_t));
    }
    void append_to_column_f64(uint64_t id, const double* data, size_t n) {
        assert(catalog_has(id) && column_type(id) == MatrixType::F64 && "append type must match column (double)");
        append_raw(id, reinterpret_cast<const unsigned char*>(data), n * sizeof(double));
    }

    // Inspection (tests): where the bytes actually live vs where the brain believes, the
    // HOST bytes the brain is accounting for, and a column's integrity checksum.
    MemorySpace column_tier(uint64_t id) const { return catalog_.at(id)->tier(); }
    MemorySpace manager_tier(uint64_t id) const { return tier_mgr_.tier_of(id); }
    size_t host_resident_bytes() const { return tier_mgr_.resident_bytes(MemorySpace::HOST); }
    uint64_t column_checksum(uint64_t id) const { return catalog_.at(id)->checksum(); }
    // A column's element type (DM-3a). Absent from col_types_ ⇒ U32 (every legacy/untagged column).
    MatrixType column_type(uint64_t id) const { auto it = col_types_.find(id); return it == col_types_.end() ? MatrixType::U32 : it->second; }

    // Type-aware row count of a catalog column (U32 = 4 bytes/row, I64 and F64 = 8).
    size_t column_rows(uint64_t id) const {
        const size_t w = (column_type(id) == MatrixType::I64 || column_type(id) == MatrixType::F64) ? 8 : sizeof(uint32_t);
        return catalog_.at(id)->size_bytes() / w;
    }

    // Attach (or overwrite) a name for an existing catalog column. Duplicate names: last wins for column_id.
#if defined(MATRIX_USE_CUDA)
    // GPU-3: pin a catalog column to DEVICE/VRAM now (deterministic promotion for tests/ops; the heat
    // brain also promotes hot columns under the device budget). Returns false if the id is unknown.
    bool pin_device(uint64_t col_id) {
        auto it = catalog_.find(col_id);
        if (it == catalog_.end()) return false;
        it->second->migrate_to(MemorySpace::DEVICE);
        return true;
    }
#endif

    void name_column(uint64_t id, const std::string& name) {
        assert(catalog_has(id) && "name_column: unknown column id");
        column_names_[id] = name;
        name_to_id_[name] = id;
    }
    // Resolve a column name to its id; 0 (the reserved legacy id) if no such name.
    uint64_t column_id(const std::string& name) const {
        auto it = name_to_id_.find(name);
        return it == name_to_id_.end() ? 0 : it->second;
    }
    // A column's name, or "" if unnamed.
    std::string column_name(uint64_t id) const {
        auto it = column_names_.find(id);
        return it == column_names_.end() ? std::string{} : it->second;
    }
    // List every catalog column with its metadata (id, name, type, row count, tier). Order unspecified.
    std::vector<ColumnInfo> catalog_columns() const {
        std::vector<ColumnInfo> out;
        out.reserve(catalog_.size());
        for (const auto& kv : catalog_)
            out.push_back(ColumnInfo{ kv.first, column_name(kv.first), column_type(kv.first),
                                      column_rows(kv.first), kv.second->tier() });
        return out;
    }

    // Group existing, equal-length columns into a named table (a row-aligned unit) — DM-2c. Returns false
    // (no table created) if a column id is unknown or the columns differ in row count (the table invariant).
    // Organizational schema over the named columns; queries still run per-column (a table-scoped query
    // planner is the upgrade). ponytail: stores column ids; a dropped column would dangle (no drop exists).
    bool create_table(const std::string& name, const std::vector<uint64_t>& col_ids) {
        if (col_ids.empty()) return false;
        size_t rows = 0; bool first = true;
        for (uint64_t id : col_ids) {
            if (!catalog_has(id)) return false;
            const size_t r = column_rows(id);
            if (first) { rows = r; first = false; } else if (r != rows) return false;   // row-aligned invariant
        }
        tables_[name] = col_ids;
        return true;
    }
    // The columns of a named table (in declared order), as ColumnInfo; empty if no such table.
    std::vector<ColumnInfo> table_columns(const std::string& name) const {
        std::vector<ColumnInfo> out;
        auto it = tables_.find(name);
        if (it == tables_.end()) return out;
        for (uint64_t id : it->second)
            if (catalog_has(id))
                out.push_back(ColumnInfo{ id, column_name(id), column_type(id), column_rows(id), catalog_.at(id)->tier() });
        return out;
    }
    // Names of all registered tables (order unspecified).
    std::vector<std::string> tables() const {
        std::vector<std::string> out; out.reserve(tables_.size());
        for (const auto& kv : tables_) out.push_back(kv.first);
        return out;
    }

    // Point-op read accessor (tests): true + sets out if present. Mirrors execute_batch's OP_READ.
    bool kv_get(uint64_t key, uint64_t& out) const { return kv_.get(key, out); }

    // Range scan over the point-op store: every (key, value) with lo <= key <= hi (inclusive). Order
    // unspecified (hash order — sort if needed). ponytail: O(capacity) full scan via KVStore::for_each;
    // a sorted secondary index for log-time selective ranges is the deferred upgrade (the "index" half
    // of DM-7).
    std::vector<std::pair<uint64_t, uint64_t>> kv_range(uint64_t lo, uint64_t hi) const {
        std::vector<std::pair<uint64_t, uint64_t>> out;
        kv_.for_each([&](uint64_t k, uint64_t v) { if (k >= lo && k <= hi) out.emplace_back(k, v); });
        return out;
    }

    // Sorted secondary index range scan (DM-7): every (key, value) with lo <= key <= hi, in ASCENDING
    // key order, via the ordered index — O(log n) to locate `lo` + O(result) to walk the range (not the
    // O(n) full scan of kv_range). The result is sorted by key (kv_range's is unordered).
    std::vector<std::pair<uint64_t, uint64_t>> kv_range_sorted(uint64_t lo, uint64_t hi) const {
        std::vector<std::pair<uint64_t, uint64_t>> out;
        for (auto it = key_index_.lower_bound(lo); it != key_index_.end() && it->first <= hi; ++it)
            out.emplace_back(it->first, it->second);
        return out;
    }

    // --- Atomic transactions (WAL group commit) ---
    void begin() { assert(!in_txn_ && "transaction already open"); txn_buf_.clear(); in_txn_ = true; }
    void txn_put(uint64_t key, uint64_t value) { assert(in_txn_ && "txn_put outside a transaction"); txn_buf_.emplace_back(key, value); }
    // Durably commit the buffered writes as one all-or-nothing group, then apply them.
    void commit() {
        assert(in_txn_ && "commit without begin");
        if (cold_store_) {
            for (auto& kv : txn_buf_) cold_store_->append_txn_put(kv.first, kv.second);
            cold_store_->append_commit();   // the durable, atomic commit point
        }
        for (auto& kv : txn_buf_) apply_committed_write(kv.first, kv.second);
        in_txn_ = false; ++transactions_committed_; txn_buf_.clear();
    }
    void rollback() { assert(in_txn_ && "rollback without begin"); in_txn_ = false; ++transactions_rolled_back_; txn_buf_.clear(); }
    uint64_t transactions_committed() const { return transactions_committed_; }
    uint64_t transactions_rolled_back() const { return transactions_rolled_back_; }

    // Compact the WAL: snapshot the point-op store, then truncate the log (DU-4). No-op if durability off.
    void checkpoint() {
        assert(!in_txn_ && "checkpoint inside a transaction");
        if (!cold_store_) return;
        save_checkpoint(checkpoint_path_);
        cold_store_->truncate();
        ++checkpoints_;
    }

    uint64_t checkpoints() const { return checkpoints_; }
    uint64_t wal_records() const { return cold_store_ ? cold_store_->records_written() : 0; }
    // DU-5: the durability level in force. SYNC_EACH = a committed write is fsync'd (survives power loss);
    // SYNC_OFF = buffered (faster, a crash may lose the tail). SYNC_OFF when no WAL is attached (nothing to sync).
    SyncPolicy durability_level() const { return cold_store_ ? cold_store_->policy() : SyncPolicy::SYNC_OFF; }

    // RM-4 graceful shutdown: stop cleanly and bound restart-recovery time. Rolls back any open
    // (uncommitted) transaction — its writes are correctly discarded on a clean stop — then, if a WAL is
    // attached, checkpoints (snapshot kv_ + truncate the log) so a restart replays an ~empty WAL.
    // Idempotent and safe to call before destruction; a no-op without a WAL.
    void shutdown() {
        if (in_txn_) rollback();
        if (cold_store_) checkpoint();
    }

    // Observability snapshot (counters since construction + current resident-bytes gauges).
    EngineStats stats() const {
        return EngineStats{ cold_borrows_, rebalances_, migrations_, catalog_.size(),
                            tier_mgr_.resident_bytes(MemorySpace::HOST),
                            tier_mgr_.resident_bytes(MemorySpace::COLD),
                            query_count_, total_query_ns_, max_query_ns_ };
    }

    // OB-3 health/readiness probe: a ready verdict + the gauges behind it. `ready` is false when any
    // point-op write has been dropped (KVStore full = data loss in progress) — a real degradation signal
    // an orchestrator can act on. Cheap + const, so it's safe to poll on a liveness interval.
    HealthStatus health() const {
        return HealthStatus{ store_overflows_ == 0, cold_store_ != nullptr, catalog_.size(),
                             tier_mgr_.resident_bytes(MemorySpace::HOST), wal_records(), store_overflows_ };
    }

    // OB-2b: the per-query latency histogram — log2 buckets, bucket b counts queries with latency in
    // ~[2^(b-1), 2^b) ns. Sums to stats().query_count.
    std::array<uint64_t, 40> query_latency_histogram() const { return latency_hist_; }
    // Estimate the p-th percentile (0..1) query latency in ns from the histogram (bucket-granular —
    // returns ~the bucket's upper bound). p50/p99 are the real ops latency metrics (vs mean/max).
    uint64_t query_latency_percentile_ns(double p) const {
        if (query_count_ == 0) return 0;
        const uint64_t target = static_cast<uint64_t>(p * static_cast<double>(query_count_));
        uint64_t cum = 0;
        for (int b = 0; b < LAT_BUCKETS; ++b) {
            cum += latency_hist_[static_cast<size_t>(b)];
            if (cum >= target) return (b == 0) ? 0ull : (1ull << b);
        }
        return max_query_ns_;
    }

    // Persist a catalog column (any type) to `path` via the typed file format (borrows to HOST, returns it).
    void save_column(uint64_t id, const std::string& path) {
        TieredColumn& col = *catalog_.at(id);
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        matrix_write_column_typed(path, col.host_ptr(), col.size_bytes(), static_cast<uint32_t>(column_type(id)));
        if (home != MemorySpace::HOST) col.migrate_to(home);
    }
    // Load a typed column file into the catalog under `id` (born HOST-resident; dispatched by element type).
    void load_column_from_file(uint64_t id, const std::string& path) {
        std::vector<unsigned char> bytes; uint32_t type = 0;
        matrix_read_column_typed(path, bytes, type);
        if (type == static_cast<uint32_t>(MatrixType::I64))
            load_scan_column_i64(id, reinterpret_cast<const int64_t*>(bytes.data()), bytes.size() / sizeof(int64_t));
        else if (type == static_cast<uint32_t>(MatrixType::F64))
            load_scan_column_f64(id, reinterpret_cast<const double*>(bytes.data()), bytes.size() / sizeof(double));
        else
            load_scan_column(id, reinterpret_cast<const uint32_t*>(bytes.data()), bytes.size() / sizeof(uint32_t));
    }

    // Ingest one uint32 column from a CSV file into the catalog under `id` (born HOST-resident, like
    // load_column_from_file). Returns false (no catalog entry created) if the CSV is malformed — CSV is
    // untrusted input, so a bad file is reported, never a crash. See DM-5b / VAL-1.
    bool load_column_from_csv(uint64_t id, const std::string& path, size_t col_index,
                              bool has_header = false, char delim = ',') {
        std::vector<uint32_t> data;
        if (!matrix_read_csv_column(path, col_index, has_header, delim, data)) return false;
        load_scan_column(id, data.data(), data.size());
        return true;
    }

    // int64 / double siblings of load_column_from_csv (DM-3g). Ingest a signed-64-bit or floating-point
    // column straight from CSV. Same graceful contract: malformed CSV → false, no catalog entry, no crash.
    bool load_column_from_csv_i64(uint64_t id, const std::string& path, size_t col_index,
                                  bool has_header = false, char delim = ',') {
        std::vector<int64_t> data;
        if (!matrix_read_csv_column_i64(path, col_index, has_header, delim, data)) return false;
        load_scan_column_i64(id, data.data(), data.size());
        return true;
    }
    bool load_column_from_csv_f64(uint64_t id, const std::string& path, size_t col_index,
                                  bool has_header = false, char delim = ',') {
        std::vector<double> data;
        if (!matrix_read_csv_column_f64(path, col_index, has_header, delim, data)) return false;
        load_scan_column_f64(id, data.data(), data.size());
        return true;
    }

    // Snapshot every catalog column to `path`: [u32 magic][u64 num_cols] then per column
    // [u64 id][u64 count][count×u32]. Borrows COLD columns to HOST to read, returns them.
    // Fail-loud on I/O error (never leave a half-written snapshot mistaken for valid).
    void save_catalog(const std::string& path) {
        FILE* f = std::fopen(path.c_str(), "wb");
        if (!f) { std::fprintf(stderr, "save_catalog: open failed %s\n", path.c_str()); std::abort(); }
        const uint32_t magic = MATRIX_CATALOG_MAGIC;
        const uint64_t ncols = catalog_.size();
        bool ok = std::fwrite(&magic, sizeof magic, 1, f) == 1
               && std::fwrite(&ncols, sizeof ncols, 1, f) == 1;
        for (auto& kv : catalog_) {
            if (!ok) break;
            TieredColumn& col = *kv.second;
            const MemorySpace home = col.tier();
            if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
            const uint64_t id = kv.first;
            const uint32_t type = static_cast<uint32_t>(column_type(id));   // 0=U32, 1=I64, 2=F64
            const uint64_t count = column_rows(id);                          // type-aware row count
            const std::string nm = column_name(id);                          // optional name ("" if unnamed)
            const uint32_t namelen = static_cast<uint32_t>(nm.size());
            ok = std::fwrite(&id,      sizeof id,      1, f) == 1
              && std::fwrite(&type,    sizeof type,    1, f) == 1
              && std::fwrite(&count,   sizeof count,   1, f) == 1
              && std::fwrite(&namelen, sizeof namelen, 1, f) == 1
              && (namelen == 0 || std::fwrite(nm.data(), 1, namelen, f) == namelen)   // the name
              && (col.size_bytes() == 0
                  || std::fwrite(col.host_ptr(), 1, col.size_bytes(), f) == col.size_bytes());  // raw bytes
            if (home != MemorySpace::HOST) col.migrate_to(home);
        }
        std::fclose(f);
        if (!ok) { std::fprintf(stderr, "save_catalog: short write %s\n", path.c_str()); std::abort(); }
    }
    // Restore a snapshot into the catalog (columns land in HOST; the TierManager re-tiers under load).
    void load_catalog(const std::string& path) {
        FILE* f = std::fopen(path.c_str(), "rb");
        if (!f) { std::fprintf(stderr, "load_catalog: open failed %s\n", path.c_str()); std::abort(); }
        uint32_t magic = 0; uint64_t ncols = 0;
        bool ok = std::fread(&magic, sizeof magic, 1, f) == 1 && magic == MATRIX_CATALOG_MAGIC
               && std::fread(&ncols, sizeof ncols, 1, f) == 1;
        for (uint64_t c = 0; ok && c < ncols; ++c) {
            uint64_t id = 0, count = 0; uint32_t type = 0, namelen = 0;
            ok = std::fread(&id, sizeof id, 1, f) == 1 && std::fread(&type, sizeof type, 1, f) == 1
              && std::fread(&count, sizeof count, 1, f) == 1 && std::fread(&namelen, sizeof namelen, 1, f) == 1;
            if (!ok) break;
            if (namelen > 4096) { ok = false; break; }   // sane bound — corrupt snapshot guard
            std::string nm(static_cast<size_t>(namelen), '\0');
            if (namelen > 0) { ok = std::fread(nm.data(), 1, namelen, f) == namelen; if (!ok) break; }
            if (type == static_cast<uint32_t>(MatrixType::I64)) {
                std::vector<int64_t> d(static_cast<size_t>(count));
                ok = (count == 0 || std::fread(d.data(), sizeof(int64_t), d.size(), f) == d.size());
                if (ok) load_scan_column_i64(id, d.data(), d.size());
            } else if (type == static_cast<uint32_t>(MatrixType::F64)) {
                std::vector<double> d(static_cast<size_t>(count));
                ok = (count == 0 || std::fread(d.data(), sizeof(double), d.size(), f) == d.size());
                if (ok) load_scan_column_f64(id, d.data(), d.size());
            } else if (type == static_cast<uint32_t>(MatrixType::U32)) {
                std::vector<uint32_t> d(static_cast<size_t>(count));
                ok = (count == 0 || std::fread(d.data(), sizeof(uint32_t), d.size(), f) == d.size());
                if (ok) load_scan_column(id, d.data(), d.size());
            } else {
                ok = false;   // unknown element type -> bad/corrupt snapshot, fail loud below
            }
            if (ok && namelen > 0) name_column(id, nm);   // restore the column's name (DM-2b)
        }
        std::fclose(f);
        if (!ok) { std::fprintf(stderr, "load_catalog: bad/short snapshot %s\n", path.c_str()); std::abort(); }
    }

    // Back up the whole durable state under one path prefix: <prefix>.catalog (tiered analytical columns,
    // all types) + <prefix>.kv (the point-op store). A point-in-time snapshot; reuses the existing fail-loud
    // writers. Works with or without an attached WAL (save_checkpoint snapshots the in-memory kv_).
    void backup(const std::string& prefix) {
        save_catalog(prefix + ".catalog");
        save_checkpoint(prefix + ".kv");
    }

    // Restore a backup written by backup() into THIS engine (intended for a fresh engine — loading an
    // already-registered column id asserts). Catalog-only backups restore the analytical store; a missing
    // <prefix>.kv leaves the point-op store empty (load_checkpoint returns false).
    void restore(const std::string& prefix) {
        load_catalog(prefix + ".catalog");
        load_checkpoint(prefix + ".kv");
    }

    // Snapshot kv_ atomically to `path`: write temp -> fsync -> rename (POSIX-atomic replace). Fail-loud.
    // ponytail: file data is fsync'd; the rename's own power-loss durability would need a directory fsync
    // (a pre-existing gap shared with the WAL) — the upgrade path if rename-durability matters.
    void save_checkpoint(const std::string& path) {
        const std::string tmp = path + ".tmp";
        FILE* f = std::fopen(tmp.c_str(), "wb");
        if (!f) { std::fprintf(stderr, "save_checkpoint: open failed %s\n", tmp.c_str()); std::abort(); }
        const uint32_t magic = MATRIX_CKPT_MAGIC;
        const uint64_t count = kv_.size();
        bool ok = std::fwrite(&magic, sizeof magic, 1, f) == 1
               && std::fwrite(&count, sizeof count, 1, f) == 1;
        kv_.for_each([&](uint64_t k, uint64_t v) {
            ok = ok && std::fwrite(&k, sizeof k, 1, f) == 1 && std::fwrite(&v, sizeof v, 1, f) == 1;
        });
        std::fflush(f);
        ::fsync(::fileno(f));                       // checkpoint durable BEFORE it replaces the old one
        std::fclose(f);
        if (!ok) { std::fprintf(stderr, "save_checkpoint: short write %s\n", tmp.c_str()); std::abort(); }
        if (std::rename(tmp.c_str(), path.c_str()) != 0) { std::fprintf(stderr, "save_checkpoint: rename failed\n"); std::abort(); }
    }

    // Load a checkpoint into kv_. Missing file -> false (none taken yet). Bad/short -> abort (our format).
    bool load_checkpoint(const std::string& path) {
        FILE* f = std::fopen(path.c_str(), "rb");
        if (!f) return false;
        uint32_t magic = 0; uint64_t count = 0;
        bool ok = std::fread(&magic, sizeof magic, 1, f) == 1 && magic == MATRIX_CKPT_MAGIC
               && std::fread(&count, sizeof count, 1, f) == 1;
        for (uint64_t i = 0; ok && i < count; ++i) {
            uint64_t k = 0, v = 0;
            ok = std::fread(&k, sizeof k, 1, f) == 1 && std::fread(&v, sizeof v, 1, f) == 1;
            if (ok) kv_.put(k, v);
        }
        std::fclose(f);
        if (!ok) { std::fprintf(stderr, "load_checkpoint: bad/short %s\n", path.c_str()); std::abort(); }
        return true;
    }

    // Null-mask pointer for a value column, but only if it has one AND it covers all n rows; else nullptr
    // (no masking). ponytail: a mask whose length != n is treated as absent (fail-safe to "no nulls")
    // rather than risk an out-of-bounds read in the reducer.
    const uint8_t* value_nulls(uint64_t value_id, size_t n) const {
        auto it = null_masks_.find(value_id);
        return (it != null_masks_.end() && it->second.size() == n) ? it->second.data() : nullptr;
    }

    // Distinct non-NULL value count over a typed array (helper for count_distinct). Exact hash set.
    template <typename T>
    static uint64_t distinct_count(const T* v, size_t n, const uint8_t* nulls) {
        std::unordered_set<T> seen;
        for (size_t i = 0; i < n; ++i) if (!nulls || !nulls[i]) seen.insert(v[i]);
        return seen.size();
    }

    // uint64 comparison for HAVING (a grouped aggregate can exceed u32, so this is the wide sibling of
    // matrix_pred_match). BETWEEN is inclusive [a, b].
    static bool cmp_u64(MatrixCmp c, uint64_t v, uint64_t a, uint64_t b) {
        switch (c) {
            case MatrixCmp::GE:      return v >= a;
            case MatrixCmp::LT:      return v <  a;
            case MatrixCmp::LE:      return v <= a;
            case MatrixCmp::EQ:      return v == a;
            case MatrixCmp::NE:      return v != a;
            case MatrixCmp::BETWEEN: return v >= a && v <= b;
            case MatrixCmp::GT:
            default:                 return v >  a;
        }
    }

    // GROUP BY: out[g] = aggregate (by op) of value-column rows whose key-column value == g, for
    // g in [0, num_groups). key_id and value_id are distinct catalog columns of equal length
    // (uint32). Borrows both to HOST for the reduction, then returns each to its home tier (so
    // residency stays in lockstep with the TierManager). Records access heat on both columns;
    // migration stays scan-driven (GROUP BY does not itself rebalance — see spec).
    void grouped_aggregate(uint64_t key_id, uint64_t value_id, uint32_t num_groups,
                           MatrixAggOp op, std::vector<uint64_t>& out) {
        assert(key_id != value_id && "group-by key and value must be distinct columns");
        TieredColumn& kc = *catalog_.at(key_id);
        TieredColumn& vc = *catalog_.at(value_id);
        assert(kc.size_bytes() == vc.size_bytes() && "key and value columns must be the same length");
        tier_mgr_.record_access(key_id, kc.size_bytes());
        tier_mgr_.record_access(value_id, vc.size_bytes());

#if defined(MATRIX_USE_CUDA)
        {   // GPU-3g: both columns VRAM-resident + no null mask -> GROUP BY in VRAM, no borrow-down
            const size_t gn = kc.size_bytes() / sizeof(uint32_t);
            if (kc.tier() == MemorySpace::DEVICE && vc.tier() == MemorySpace::DEVICE && value_nulls(value_id, gn) == nullptr) {
                out.resize(num_groups);
                matrix_gpu_group_reduce_dev(kc.device_ptr(), vc.device_ptr(), gn, num_groups, op, MatrixPredicate{}, false, out.data());
                return;
            }
        }
#endif
        const MemorySpace kh = kc.tier(); if (kh != MemorySpace::HOST) { ++cold_borrows_; kc.migrate_to(MemorySpace::HOST); }
        const MemorySpace vh = vc.tier(); if (vh != MemorySpace::HOST) { ++cold_borrows_; vc.migrate_to(MemorySpace::HOST); }
        const uint32_t* keys = reinterpret_cast<const uint32_t*>(kc.host_ptr());
        const uint32_t* vals = reinterpret_cast<const uint32_t*>(vc.host_ptr());
        const size_t n = kc.size_bytes() / sizeof(uint32_t);
        out.resize(num_groups);   // matrix_cpu_group_reduce initializes every slot per op (MIN sentinel ≠ 0)
        matrix_cpu_group_reduce(keys, vals, n, num_groups, op, out.data(), value_nulls(value_id, n));
        if (vh != MemorySpace::HOST) vc.migrate_to(vh);       // return borrows
        if (kh != MemorySpace::HOST) kc.migrate_to(kh);
    }

    // Inner hash equi-join on two uint32 key columns: every (left_row, right_row) with left_key[left_row]
    // == right_key[right_row]. Build a value->left-rows hash, probe with the right. Borrow-and-return both
    // columns (like grouped_aggregate). Result order unspecified; cardinality = result.size(). DM-8.
    // ponytail: builds on the LEFT side unconditionally + materializes all pairs in RAM — a planner would
    // build on the smaller side, and a huge result would need spilling; both are deferred.
    std::vector<std::pair<uint64_t, uint64_t>> hash_join(uint64_t left_key_id, uint64_t right_key_id) {
        assert(catalog_has(left_key_id) && catalog_has(right_key_id) && "hash_join: unknown column id");
        assert(column_type(left_key_id) == MatrixType::U32 && column_type(right_key_id) == MatrixType::U32
               && "hash_join: keys must be uint32 (typed-key join is deferred)");
        TieredColumn& lc = *catalog_.at(left_key_id);
        TieredColumn& rc = *catalog_.at(right_key_id);
        tier_mgr_.record_access(left_key_id, lc.size_bytes());
        tier_mgr_.record_access(right_key_id, rc.size_bytes());
        const MemorySpace lh = lc.tier(); if (lh != MemorySpace::HOST) { ++cold_borrows_; lc.migrate_to(MemorySpace::HOST); }
        const MemorySpace rh = rc.tier(); if (rh != MemorySpace::HOST) { ++cold_borrows_; rc.migrate_to(MemorySpace::HOST); }
        const uint32_t* lk = reinterpret_cast<const uint32_t*>(lc.host_ptr());
        const uint32_t* rk = reinterpret_cast<const uint32_t*>(rc.host_ptr());
        const size_t ln = lc.size_bytes() / sizeof(uint32_t);
        const size_t rn = rc.size_bytes() / sizeof(uint32_t);
        std::unordered_map<uint32_t, std::vector<uint64_t>> build;     // left value -> left rows
        for (size_t i = 0; i < ln; ++i) build[lk[i]].push_back(static_cast<uint64_t>(i));
        std::vector<std::pair<uint64_t, uint64_t>> out;
        for (size_t j = 0; j < rn; ++j) {
            auto it = build.find(rk[j]);
            if (it != build.end()) for (uint64_t i : it->second) out.emplace_back(i, static_cast<uint64_t>(j));
        }
        if (rh != MemorySpace::HOST) rc.migrate_to(rh);                // return borrows
        if (lh != MemorySpace::HOST) lc.migrate_to(lh);
        return out;
    }

    // Count an equi-join's matching pairs WITHOUT materializing them — resource-safe for huge joins
    // (addresses hash_join's materialize-all ceiling). Builds a value->count map (O(distinct) memory,
    // not O(left rows)) and sums match counts on probe. Equals hash_join(left,right).size(). DM-8/§6.
    uint64_t hash_join_count(uint64_t left_key_id, uint64_t right_key_id) {
        assert(catalog_has(left_key_id) && catalog_has(right_key_id) && "hash_join_count: unknown column id");
        assert(column_type(left_key_id) == MatrixType::U32 && column_type(right_key_id) == MatrixType::U32
               && "hash_join_count: keys must be uint32");
        TieredColumn& lc = *catalog_.at(left_key_id);
        TieredColumn& rc = *catalog_.at(right_key_id);
        tier_mgr_.record_access(left_key_id, lc.size_bytes());
        tier_mgr_.record_access(right_key_id, rc.size_bytes());
        const MemorySpace lh = lc.tier(); if (lh != MemorySpace::HOST) { ++cold_borrows_; lc.migrate_to(MemorySpace::HOST); }
        const MemorySpace rh = rc.tier(); if (rh != MemorySpace::HOST) { ++cold_borrows_; rc.migrate_to(MemorySpace::HOST); }
        const uint32_t* lk = reinterpret_cast<const uint32_t*>(lc.host_ptr());
        const uint32_t* rk = reinterpret_cast<const uint32_t*>(rc.host_ptr());
        const size_t ln = lc.size_bytes() / sizeof(uint32_t);
        const size_t rn = rc.size_bytes() / sizeof(uint32_t);
        std::unordered_map<uint32_t, uint64_t> build;                 // left value -> # of left rows
        for (size_t i = 0; i < ln; ++i) ++build[lk[i]];
        uint64_t pairs = 0;
        for (size_t j = 0; j < rn; ++j) { auto it = build.find(rk[j]); if (it != build.end()) pairs += it->second; }
        if (rh != MemorySpace::HOST) rc.migrate_to(rh);
        if (lh != MemorySpace::HOST) lc.migrate_to(lh);
        return pairs;
    }

    // Inner hash equi-join on two int64 key columns (the typed sibling of hash_join — DM-8c). Same
    // build-left / probe-right shape over int64 values; returns matching (left_row, right_row) pairs.
    // (double-key joins are intentionally unsupported — exact float equality is semantically fraught.)
    std::vector<std::pair<uint64_t, uint64_t>> hash_join_i64(uint64_t left_key_id, uint64_t right_key_id) {
        assert(catalog_has(left_key_id) && catalog_has(right_key_id) && "hash_join_i64: unknown column id");
        assert(column_type(left_key_id) == MatrixType::I64 && column_type(right_key_id) == MatrixType::I64
               && "hash_join_i64: keys must be int64");
        TieredColumn& lc = *catalog_.at(left_key_id);
        TieredColumn& rc = *catalog_.at(right_key_id);
        tier_mgr_.record_access(left_key_id, lc.size_bytes());
        tier_mgr_.record_access(right_key_id, rc.size_bytes());
        const MemorySpace lh = lc.tier(); if (lh != MemorySpace::HOST) { ++cold_borrows_; lc.migrate_to(MemorySpace::HOST); }
        const MemorySpace rh = rc.tier(); if (rh != MemorySpace::HOST) { ++cold_borrows_; rc.migrate_to(MemorySpace::HOST); }
        const int64_t* lk = reinterpret_cast<const int64_t*>(lc.host_ptr());
        const int64_t* rk = reinterpret_cast<const int64_t*>(rc.host_ptr());
        const size_t ln = lc.size_bytes() / sizeof(int64_t);
        const size_t rn = rc.size_bytes() / sizeof(int64_t);
        std::unordered_map<int64_t, std::vector<uint64_t>> build;     // left value -> left rows
        for (size_t i = 0; i < ln; ++i) build[lk[i]].push_back(static_cast<uint64_t>(i));
        std::vector<std::pair<uint64_t, uint64_t>> out;
        for (size_t j = 0; j < rn; ++j) {
            auto it = build.find(rk[j]);
            if (it != build.end()) for (uint64_t i : it->second) out.emplace_back(i, static_cast<uint64_t>(j));
        }
        if (rh != MemorySpace::HOST) rc.migrate_to(rh);
        if (lh != MemorySpace::HOST) lc.migrate_to(lh);
        return out;
    }

    // Gather a column's values at the given row indices (the "project" step — e.g. over a join's matched
    // rows). Returns one value per index, in index order, as a uint64: a U32 value zero-extended; an
    // I64/F64 value as its bit pattern (caller reads via static_cast<int64_t> / std::bit_cast<double>,
    // per column_type(col_id) — same convention as execute_query). Borrows COLD->HOST and returns it.
    std::vector<uint64_t> gather(uint64_t col_id, const std::vector<uint64_t>& rows) {
        assert(catalog_has(col_id) && "gather: unknown column id");
        TieredColumn& col = *catalog_.at(col_id);
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        const bool wide = (column_type(col_id) != MatrixType::U32);   // I64/F64 are 8 bytes, U32 is 4
        const size_t width = wide ? 8 : 4;
        const size_t n = col.size_bytes() / width;
        const unsigned char* base = col.host_ptr();
        std::vector<uint64_t> out; out.reserve(rows.size());
        for (uint64_t r : rows) {
            assert(r < n && "gather: row index out of range");
            uint64_t v = 0;
            if (wide) std::memcpy(&v, base + r * 8, 8);
            else { uint32_t u = 0; std::memcpy(&u, base + r * 4, 4); v = u; }
            out.push_back(v);
        }
        if (home != MemorySpace::HOST) col.migrate_to(home);
        return out;
    }
    // GROUP BY key WHERE <predicate> — same double borrow-and-return as grouped_aggregate_where.
    void grouped_aggregate_pred(uint64_t key_id, uint64_t value_id, uint32_t num_groups,
                                MatrixAggOp op, const MatrixPredicate& pred, std::vector<uint64_t>& out) {
        assert(key_id != value_id && "group-by key and value must be distinct columns");
        TieredColumn& kc = *catalog_.at(key_id);
        TieredColumn& vc = *catalog_.at(value_id);
        assert(kc.size_bytes() == vc.size_bytes() && "key and value columns must be the same length");
        tier_mgr_.record_access(key_id, kc.size_bytes());
        tier_mgr_.record_access(value_id, vc.size_bytes());
#if defined(MATRIX_USE_CUDA)
        {   // GPU-3g: both columns VRAM-resident + no null mask -> filtered GROUP BY in VRAM, no borrow-down
            const size_t gn = kc.size_bytes() / sizeof(uint32_t);
            if (kc.tier() == MemorySpace::DEVICE && vc.tier() == MemorySpace::DEVICE && value_nulls(value_id, gn) == nullptr) {
                out.resize(num_groups);
                matrix_gpu_group_reduce_dev(kc.device_ptr(), vc.device_ptr(), gn, num_groups, op, pred, true, out.data());
                return;
            }
        }
#endif
        const MemorySpace kh = kc.tier(); if (kh != MemorySpace::HOST) { ++cold_borrows_; kc.migrate_to(MemorySpace::HOST); }
        const MemorySpace vh = vc.tier(); if (vh != MemorySpace::HOST) { ++cold_borrows_; vc.migrate_to(MemorySpace::HOST); }
        const uint32_t* keys = reinterpret_cast<const uint32_t*>(kc.host_ptr());
        const uint32_t* vals = reinterpret_cast<const uint32_t*>(vc.host_ptr());
        const size_t n = kc.size_bytes() / sizeof(uint32_t);
        out.resize(num_groups);
        matrix_cpu_group_reduce_pred(keys, vals, n, num_groups, op, pred, out.data(), value_nulls(value_id, n));
        if (vh != MemorySpace::HOST) vc.migrate_to(vh);
        if (kh != MemorySpace::HOST) kc.migrate_to(kh);
    }

    // GROUP BY key WHERE value > threshold (filtered grouped aggregate). Same contract and double
    // borrow-and-return as grouped_aggregate; only rows with value > threshold contribute.
    void grouped_aggregate_where(uint64_t key_id, uint64_t value_id, uint32_t num_groups,
                                 MatrixAggOp op, uint32_t threshold, std::vector<uint64_t>& out) {
        grouped_aggregate_pred(key_id, value_id, num_groups, op, MatrixPredicate{MatrixCmp::GT, threshold, 0}, out);
    }

    // GROUP BY a uint32 key over an int64 value column (DM-3c). Double borrow-and-return like
    // grouped_aggregate; out holds int64 group aggregates as uint64 bit-patterns. No rebalance (GROUP BY
    // is not scan-driven, matching grouped_aggregate).
    void grouped_aggregate_i64(uint64_t key_id, uint64_t value_id, uint32_t num_groups, MatrixAggOp op,
                               const MatrixPredicateI64& pred, bool has_filter, std::vector<uint64_t>& out) {
        TieredColumn& kc = *catalog_.at(key_id);
        TieredColumn& vc = *catalog_.at(value_id);
        tier_mgr_.record_access(key_id, kc.size_bytes());
        tier_mgr_.record_access(value_id, vc.size_bytes());
#if defined(MATRIX_USE_CUDA)
        {   // GPU-3g: u32 key + i64 value both VRAM-resident + no null mask -> GROUP BY in VRAM
            const size_t gn = vc.size_bytes() / sizeof(int64_t);
            if (kc.tier() == MemorySpace::DEVICE && vc.tier() == MemorySpace::DEVICE && value_nulls(value_id, gn) == nullptr) {
                out.resize(num_groups);
                matrix_gpu_group_reduce_dev_i64(kc.device_ptr(), vc.device_ptr(), gn, num_groups, op, pred, has_filter, out.data());
                return;
            }
        }
#endif
        const MemorySpace kh = kc.tier(); if (kh != MemorySpace::HOST) { ++cold_borrows_; kc.migrate_to(MemorySpace::HOST); }
        const MemorySpace vh = vc.tier(); if (vh != MemorySpace::HOST) { ++cold_borrows_; vc.migrate_to(MemorySpace::HOST); }
        const uint32_t* keys = reinterpret_cast<const uint32_t*>(kc.host_ptr());
        const int64_t*  vals = reinterpret_cast<const int64_t*>(vc.host_ptr());
        const size_t n = vc.size_bytes() / sizeof(int64_t);
        std::vector<int64_t> tmp(num_groups);
        const uint8_t* nulls = value_nulls(value_id, n);
        if (has_filter) matrix_cpu_group_reduce_i64_pred(keys, vals, n, num_groups, op, pred, tmp.data(), nulls);
        else            matrix_cpu_group_reduce_i64(keys, vals, n, num_groups, op, tmp.data(), nulls);
        out.resize(num_groups);
        for (uint32_t g = 0; g < num_groups; ++g) out[g] = static_cast<uint64_t>(tmp[g]);
        if (vh != MemorySpace::HOST) vc.migrate_to(vh);
        if (kh != MemorySpace::HOST) kc.migrate_to(kh);
    }

    // GROUP BY a uint32 key over a double value column (DM-3f). Double borrow-and-return like
    // grouped_aggregate; out holds double group aggregates as uint64 bit-patterns. No rebalance (GROUP BY
    // is not scan-driven, matching grouped_aggregate).
    void grouped_aggregate_f64(uint64_t key_id, uint64_t value_id, uint32_t num_groups, MatrixAggOp op,
                               const MatrixPredicateF64& pred, bool has_filter, std::vector<uint64_t>& out) {
        TieredColumn& kc = *catalog_.at(key_id);
        TieredColumn& vc = *catalog_.at(value_id);
        tier_mgr_.record_access(key_id, kc.size_bytes());
        tier_mgr_.record_access(value_id, vc.size_bytes());
        tier_mgr_.record_access(key_id, kc.size_bytes());
        tier_mgr_.record_access(value_id, vc.size_bytes());
#if defined(MATRIX_USE_CUDA)
        {   // GPU-3g: u32 key + f64 value both VRAM-resident + no null mask -> GROUP BY in VRAM
            const size_t gn = vc.size_bytes() / sizeof(double);
            if (kc.tier() == MemorySpace::DEVICE && vc.tier() == MemorySpace::DEVICE && value_nulls(value_id, gn) == nullptr) {
                out.resize(num_groups);
                matrix_gpu_group_reduce_dev_f64(kc.device_ptr(), vc.device_ptr(), gn, num_groups, op, pred, has_filter, out.data());
                return;
            }
        }
#endif
        const MemorySpace kh = kc.tier(); if (kh != MemorySpace::HOST) { ++cold_borrows_; kc.migrate_to(MemorySpace::HOST); }
        const MemorySpace vh = vc.tier(); if (vh != MemorySpace::HOST) { ++cold_borrows_; vc.migrate_to(MemorySpace::HOST); }
        const uint32_t* keys = reinterpret_cast<const uint32_t*>(kc.host_ptr());
        const double*   vals = reinterpret_cast<const double*>(vc.host_ptr());
        const size_t n = vc.size_bytes() / sizeof(double);
        std::vector<double> tmp(num_groups);
        const uint8_t* nulls = value_nulls(value_id, n);
        if (has_filter) matrix_cpu_group_reduce_f64_pred(keys, vals, n, num_groups, op, pred, tmp.data(), nulls);
        else            matrix_cpu_group_reduce_f64(keys, vals, n, num_groups, op, tmp.data(), nulls);
        out.resize(num_groups);
        for (uint32_t g = 0; g < num_groups; ++g) out[g] = matrix_bit_cast<uint64_t>(tmp[g]);
        if (vh != MemorySpace::HOST) vc.migrate_to(vh);
        if (kh != MemorySpace::HOST) kc.migrate_to(kh);
    }

    // Unified analytical query over catalog columns. Validates input at the boundary and returns a
    // status (never crashes on caller input); on any ERR, out is cleared. Scalar -> out[0];
    // grouped -> out[0..num_groups). Catalog columns only (the legacy id-0 fixed column is the
    // benchmark fixture, not a query target). Public API: times every call (OB-2) — see the
    // execute_query_impl below for the body; this thin wrapper records latency on all return paths.
    MatrixQueryStatus execute_query(const MatrixQuery& q, std::vector<uint64_t>& out) {
        const auto t0 = std::chrono::steady_clock::now();
        const MatrixQueryStatus st = execute_query_impl(q, out);
        const uint64_t ns = static_cast<uint64_t>(
            std::chrono::duration_cast<std::chrono::nanoseconds>(std::chrono::steady_clock::now() - t0).count());
        ++query_count_; total_query_ns_ += ns; if (ns > max_query_ns_) max_query_ns_ = ns;
        // OB-2b: bucket by floor(log2(ns+1)) (log-scale latency histogram) for percentile estimation.
        { uint64_t x = ns + 1, b = 0; while (x > 1 && b < LAT_BUCKETS - 1) { x >>= 1; ++b; } ++latency_hist_[b]; }
        return st;
    }

    // Top-N groups by aggregate value (ORDER BY agg DESC LIMIT k): run a grouped query, then return the k
    // (group_id, value) pairs with the largest aggregate. The staple "top 10 X by Y" analytical query.
    // ponytail: sorts by the RAW uint64 aggregate — exact for U32-valued groups and COUNT (non-negative);
    // for int64/double SUM/MIN/MAX the result bits aren't value-order, so use this for U32/COUNT grouping.
    std::vector<std::pair<uint64_t, uint64_t>> top_groups(const MatrixQuery& q, size_t k) {
        std::vector<uint64_t> out;
        if (!q.grouped || execute_query(q, out) != MatrixQueryStatus::OK) return {};
        std::vector<std::pair<uint64_t, uint64_t>> pairs;
        pairs.reserve(out.size());
        for (uint64_t g = 0; g < out.size(); ++g) pairs.emplace_back(g, out[g]);
        const size_t kk = std::min(k, pairs.size());
        std::partial_sort(pairs.begin(), pairs.begin() + static_cast<std::ptrdiff_t>(kk), pairs.end(),
                          [](const auto& a, const auto& b) { return a.second > b.second; });   // value desc
        pairs.resize(kk);
        return pairs;
    }

    // HAVING: the groups whose aggregate satisfies `cmp` against threshold (BETWEEN uses [threshold, upper]) —
    // e.g. "regions where SUM(amount) > 1000". Runs the grouped query, then filters the (group, value) pairs.
    // ponytail: compares the RAW uint64 aggregate (a uint64 comparator, since a grouped SUM can exceed u32),
    // so it's exact for U32-valued groups and COUNT; I64/F64 bit-patterns aren't value-order (use for U32/COUNT),
    // matching top_groups.
    std::vector<std::pair<uint64_t, uint64_t>> having(const MatrixQuery& q, MatrixCmp cmp,
                                                      uint64_t threshold, uint64_t upper = 0) {
        std::vector<uint64_t> out;
        if (!q.grouped || execute_query(q, out) != MatrixQueryStatus::OK) return {};
        std::vector<std::pair<uint64_t, uint64_t>> pairs;
        for (uint64_t g = 0; g < out.size(); ++g)
            if (cmp_u64(cmp, out[g], threshold, upper)) pairs.emplace_back(g, out[g]);
        return pairs;
    }

    // AVG(value_col) = SUM/COUNT as double(s), derived from the two existing aggregates — so it inherits
    // their per-type handling (U32/I64/F64) and NULL-skipping for free, with no new reducer op threaded
    // through every typed/grouped/filtered/nullable path. A scalar query yields one element; a grouped
    // query yields one per group. A zero-count set/group yields NaN (the average of nothing). NULL-aware on
    // both paths: SUM and COUNT both skip NULLs (scalar always; grouped since the grouped reducers take the mask).
    std::vector<double> average(const MatrixQuery& q) {
        MatrixQuery sq = q; sq.agg = AGG_SUM;   std::vector<uint64_t> sv;
        MatrixQuery cq = q; cq.agg = AGG_COUNT; std::vector<uint64_t> cv;
        if (execute_query(sq, sv) != MatrixQueryStatus::OK || execute_query(cq, cv) != MatrixQueryStatus::OK
            || sv.size() != cv.size())
            return {};
        const MatrixType ty = column_type(q.value_col);
        std::vector<double> out(sv.size());
        for (size_t i = 0; i < sv.size(); ++i) {
            const double sum = (ty == MatrixType::F64) ? matrix_bit_cast<double>(sv[i])
                             : (ty == MatrixType::I64) ? static_cast<double>(static_cast<int64_t>(sv[i]))
                                                       : static_cast<double>(sv[i]);            // U32
            // COUNT is encoded like the column: F64 columns return it as double-bits (execute_query bit_casts
            // the whole F64 result), U32/I64 return a plain integer count.
            const double count = (ty == MatrixType::F64) ? matrix_bit_cast<double>(cv[i])
                                                         : static_cast<double>(cv[i]);
            out[i] = (count != 0.0) ? sum / count : std::numeric_limits<double>::quiet_NaN();
        }
        return out;
    }

    // COUNT(DISTINCT col): number of distinct non-NULL values in a column. Borrow-and-return like the
    // scalar aggregates; null-aware (skips masked rows, via value_nulls). Typed over U32/I64/F64.
    // ponytail: an exact hash set over every value — O(distinct) memory. A HyperLogLog sketch is the
    // upgrade path when the column is huge and an estimate is acceptable. F64 NaN edge: each NaN counts as
    // distinct (NaN != NaN), since the set is over double values — documented, not special-cased.
    uint64_t count_distinct(uint64_t col_id) {
        TieredColumn& col = *catalog_.at(col_id);
        tier_mgr_.record_access(col_id, col.size_bytes());
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        const MatrixType ty = column_type(col_id);
        const size_t elem = (ty == MatrixType::U32) ? sizeof(uint32_t) : sizeof(uint64_t);
        const size_t n = col.size_bytes() / elem;
        const uint8_t* nulls = value_nulls(col_id, n);
        uint64_t result;
        if (ty == MatrixType::I64)
            result = distinct_count(reinterpret_cast<const int64_t*>(col.host_ptr()), n, nulls);
        else if (ty == MatrixType::F64)
            result = distinct_count(reinterpret_cast<const double*>(col.host_ptr()), n, nulls);
        else
            result = distinct_count(reinterpret_cast<const uint32_t*>(col.host_ptr()), n, nulls);
        if (home != MemorySpace::HOST) col.migrate_to(home);
        maybe_rebalance();
        return result;
    }

    // Parse + run an AVG query string ("SELECT AVG(col) [WHERE ...] [GROUP BY k]") -> the average(s) as
    // double(s). AVG isn't a reducer op (see average()), so rather than teach the parser a phantom agg we
    // rewrite the AVG token to SUM, reuse the full parser (WHERE/GROUP BY/etc.), then derive SUM/COUNT.
    // The tokenizer round-trips (space-join re-tokenizes identically), so this needs no parser change.
    // Empty on parse error or a non-AVG query.
    std::vector<double> avg_query(const std::string& sql) {
        std::vector<std::string> tk = matrixparse_tokenize(sql);
        if (tk.size() < 2) return {};
        std::string a = tk[1];
        for (char& c : a) c = static_cast<char>(std::toupper(static_cast<unsigned char>(c)));
        if (a != "AVG") return {};                                   // only the AVG string form lives here
        tk[1] = "SUM";                                               // rewrite the aggregate keyword
        std::string rewritten;
        for (size_t i = 0; i < tk.size(); ++i) { if (i) rewritten += ' '; rewritten += tk[i]; }
        MatrixQuery q;
        if (parse_query(rewritten, q) != MatrixQueryStatus::OK) return {};
        return average(q);
    }

    // Parse + run a top-N grouped query string ("SELECT SUM(x) GROUP BY k ORDER BY SUM DESC LIMIT n").
    // Returns the (group, value) pairs by value desc; empty on parse error or a query without GROUP BY + LIMIT.
    std::vector<std::pair<uint64_t, uint64_t>> top_query(const std::string& sql) {
        MatrixQuery q;
        if (parse_query(sql, q) != MatrixQueryStatus::OK || !q.grouped || q.limit == 0) return {};
        return top_groups(q, q.limit);
    }

    // Parse + run a HAVING query string ("SELECT SUM(x) GROUP BY k HAVING SUM <cmp> v") -> the (group,value)
    // pairs whose aggregate satisfies the comparison. Splits the HAVING tail off, parses the head (the
    // grouped query) via the full parser — the tokenizer round-trips, so the space-joined head re-parses
    // identically — then runs having(). Empty on parse error or a query without GROUP BY + HAVING.
    // ponytail: the string form takes a single comparison (GT/GE/LT/LE/EQ/NE); BETWEEN-in-HAVING is
    // reachable only via the having() method (rare in practice).
    std::vector<std::pair<uint64_t, uint64_t>> having_query(const std::string& sql) {
        std::vector<std::string> tk = matrixparse_tokenize(sql);
        auto up = [](std::string s){ for (char& c : s) c = static_cast<char>(std::toupper(static_cast<unsigned char>(c))); return s; };
        size_t hi = tk.size();
        for (size_t i = 0; i < tk.size(); ++i) if (up(tk[i]) == "HAVING") { hi = i; break; }
        if (hi == tk.size() || hi + 4 != tk.size()) return {};            // need exactly: HAVING <key> <cmp> <value>
        std::string head;
        for (size_t i = 0; i < hi; ++i) { if (i) head += ' '; head += tk[i]; }
        MatrixQuery q;
        if (parse_query(head, q) != MatrixQueryStatus::OK || !q.grouped) return {};
        // sort key must name the SELECT aggregate or the value column (same rule as ORDER BY)
        static const char* AGGN[] = { "COUNT", "SUM", "MIN", "MAX" };
        if (up(tk[hi + 1]) != AGGN[q.agg] && column_id(tk[hi + 1]) != q.value_col) return {};
        MatrixCmp cmp;
        const std::string op = tk[hi + 2];
        if      (op == ">")  cmp = MatrixCmp::GT;  else if (op == ">=") cmp = MatrixCmp::GE;
        else if (op == "<")  cmp = MatrixCmp::LT;  else if (op == "<=") cmp = MatrixCmp::LE;
        else if (op == "=")  cmp = MatrixCmp::EQ;  else if (op == "!=") cmp = MatrixCmp::NE;
        else return {};
        const std::string& v = tk[hi + 3];
        uint64_t threshold = 0; auto [p, ec] = std::from_chars(v.data(), v.data() + v.size(), threshold);
        if (ec != std::errc{} || p != v.data() + v.size()) return {};
        return having(q, cmp, threshold);
    }

    // Parse + run a COUNT(DISTINCT col) query string -> the distinct non-NULL value count. Completes the
    // aggregate-string surface (every supported aggregate is now expressible as SQL). Returns false on a
    // malformed / non-distinct query or an unknown column (out untouched), true with the count otherwise.
    bool distinct_query(const std::string& sql, uint64_t& out) {
        std::vector<std::string> tk = matrixparse_tokenize(sql);
        auto up = [](std::string s){ for (char& c : s) c = static_cast<char>(std::toupper(static_cast<unsigned char>(c))); return s; };
        if (tk.size() != 6) return false;                                 // SELECT COUNT ( DISTINCT col )
        if (up(tk[0]) != "SELECT" || up(tk[1]) != "COUNT" || tk[2] != "(" || up(tk[3]) != "DISTINCT" || tk[5] != ")")
            return false;
        const uint64_t cid = column_id(tk[4]);
        if (cid == 0) return false;                                       // unknown column
        out = count_distinct(cid);
        return true;
    }

    // Parse a scalar query string into `out` (see DM-4 grammar). Returns OK, ERR_UNKNOWN_COLUMN (bad name),
    // or ERR_PARSE (any malformed form). Untrusted input — never crashes; on ANY non-OK status `out` is
    // reset to a default MatrixQuery (so a caller that ignores the status can't execute partial garbage).
    MatrixQueryStatus parse_query(const std::string& sql, MatrixQuery& out) {
        const MatrixQueryStatus st = parse_query_impl(sql, out);
        if (st != MatrixQueryStatus::OK) out = MatrixQuery{};   // no partial state survives a parse error
        return st;
    }

    // The parse pipeline (see grammar). `out` is reset at entry; the public parse_query wrapper above also
    // clears it on any error exit, so partial fields set before a mid-parse failure never leak to a caller.
    MatrixQueryStatus parse_query_impl(const std::string& sql, MatrixQuery& out) {
        out = MatrixQuery{};
        const std::vector<std::string> tk = matrixparse_tokenize(sql);
        size_t k = 0;
        auto up   = [](std::string s){ for (char& c : s) c = static_cast<char>(std::toupper(static_cast<unsigned char>(c))); return s; };
        auto next = [&]{ return k < tk.size() ? tk[k++] : std::string{}; };
        if (up(next()) != "SELECT") return MatrixQueryStatus::ERR_PARSE;
        const std::string aggs = up(next());
        MatrixAggOp agg;
        if (aggs == "COUNT") agg = AGG_COUNT; else if (aggs == "SUM") agg = AGG_SUM;
        else if (aggs == "MIN") agg = AGG_MIN; else if (aggs == "MAX") agg = AGG_MAX;
        else return MatrixQueryStatus::ERR_PARSE;
        if (next() != "(") return MatrixQueryStatus::ERR_PARSE;
        const std::string col = next();
        if (next() != ")") return MatrixQueryStatus::ERR_PARSE;
        const uint64_t vid = column_id(col);
        if (vid == 0) return MatrixQueryStatus::ERR_UNKNOWN_COLUMN;
        out.value_col = vid; out.agg = agg;
        auto peek = [&]{ return k < tk.size() ? up(tk[k]) : std::string{}; };   // uppercased lookahead, no consume
        // optional WHERE <valuecol> <op> <val> [AND <val>]
        if (peek() == "WHERE") {
            next();                                                            // consume WHERE
            if (column_id(next()) != vid) return MatrixQueryStatus::ERR_PARSE; // filter col must == select col
            const std::string ops = up(next());
            const MatrixType ty = column_type(vid);
            out.has_filter = true;
            if (ops == "BETWEEN") {
                out.cmp = MatrixCmp::BETWEEN;
                if (!set_bound(ty, out, true, next())) return MatrixQueryStatus::ERR_PARSE;
                if (up(next()) != "AND")               return MatrixQueryStatus::ERR_PARSE;
                if (!set_bound(ty, out, false, next())) return MatrixQueryStatus::ERR_PARSE;
            } else {
                if      (ops == ">")  out.cmp = MatrixCmp::GT;  else if (ops == ">=") out.cmp = MatrixCmp::GE;
                else if (ops == "<")  out.cmp = MatrixCmp::LT;  else if (ops == "<=") out.cmp = MatrixCmp::LE;
                else if (ops == "=")  out.cmp = MatrixCmp::EQ;  else if (ops == "!=") out.cmp = MatrixCmp::NE;
                else return MatrixQueryStatus::ERR_PARSE;
                if (!set_bound(ty, out, true, next())) return MatrixQueryStatus::ERR_PARSE;
            }
        }
        // optional GROUP BY <keycol> (uint32 key). num_groups is derived from the key column (max+1).
        if (peek() == "GROUP") {
            next();                                                            // consume GROUP
            if (up(next()) != "BY") return MatrixQueryStatus::ERR_PARSE;
            const uint64_t kid = column_id(next());
            if (kid == 0) return MatrixQueryStatus::ERR_UNKNOWN_COLUMN;
            if (column_type(kid) != MatrixType::U32) return MatrixQueryStatus::ERR_PARSE;  // grouped key must be u32
            out.grouped = true; out.key_col = kid; out.num_groups = derive_num_groups_u32(kid);
        }
        // optional ORDER BY <agg|valuecol> DESC LIMIT <n> -> top-N groups. DESC only (top_groups is descending)
        // and requires GROUP BY (top-N is over groups). Sets out.limit; top_query applies it.
        if (peek() == "ORDER") {
            next();                                                            // consume ORDER
            if (up(next()) != "BY") return MatrixQueryStatus::ERR_PARSE;
            if (!out.grouped)       return MatrixQueryStatus::ERR_PARSE;       // top-N is meaningless without groups
            const std::string sortraw = next();                               // sort key: the agg name or the value col
            if (up(sortraw) != aggs && column_id(sortraw) != vid) return MatrixQueryStatus::ERR_PARSE;
            if (up(next()) != "DESC")  return MatrixQueryStatus::ERR_PARSE;    // only DESC (top_groups sorts desc)
            if (up(next()) != "LIMIT") return MatrixQueryStatus::ERR_PARSE;
            const std::string lim = next();
            uint64_t n = 0; auto [p, ec] = std::from_chars(lim.data(), lim.data() + lim.size(), n);
            if (ec != std::errc{} || p != lim.data() + lim.size() || n == 0) return MatrixQueryStatus::ERR_PARSE;
            out.limit = n;
        }
        if (k != tk.size()) return MatrixQueryStatus::ERR_PARSE;     // trailing junk
        return MatrixQueryStatus::OK;
    }

private:
    // Parse the numeric literal `v` into the bound field matching the column type (lo=true -> primary/lower).
    // Returns false on junk / overflow / empty. int64 via from_chars; double via strtod; u32 via from_chars.
    bool set_bound(MatrixType ty, MatrixQuery& q, bool lo, const std::string& v) {
        if (v.empty()) return false;
        if (ty == MatrixType::F64) {
            errno = 0; char* e = nullptr; const double d = std::strtod(v.c_str(), &e);
            if (e != v.c_str() + v.size() || errno == ERANGE) return false;
            (lo ? q.lo_f64 : q.hi_f64) = d; return true;
        }
        if (ty == MatrixType::I64) {
            int64_t x = 0; auto [p, ec] = std::from_chars(v.data(), v.data() + v.size(), x);
            if (ec != std::errc{} || p != v.data() + v.size()) return false;
            (lo ? q.lo_i64 : q.hi_i64) = x; return true;
        }
        uint32_t x = 0; auto [p, ec] = std::from_chars(v.data(), v.data() + v.size(), x);
        if (ec != std::errc{} || p != v.data() + v.size()) return false;
        (lo ? q.threshold : q.upper) = x; return true;
    }

    // num_groups for a parsed GROUP BY = (max key in the u32 key column) + 1 — the dense-group count.
    // ponytail: this reads/borrows the key column's DATA at PARSE time (a max-scan), unlike pure-metadata
    // parsing — an explicit `INTO n` or a planner stats-lookup would avoid it. A sparse/huge key makes
    // num_groups large and execute_query then rejects it (ERR_TOO_MANY_GROUPS); empty -> 0 (ERR_INVALID_GROUP).
    uint32_t derive_num_groups_u32(uint64_t kid) {
        TieredColumn& col = *catalog_.at(kid);
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        const uint32_t* keys = reinterpret_cast<const uint32_t*>(col.host_ptr());
        const size_t n = col.size_bytes() / sizeof(uint32_t);
        const uint64_t mx = matrix_cpu_reduce_all(keys, n, AGG_MAX);   // max key (0 if n==0)
        if (home != MemorySpace::HOST) col.migrate_to(home);
        return (n == 0) ? 0u : static_cast<uint32_t>(mx + 1);
    }

    MatrixQueryStatus execute_query_impl(const MatrixQuery& q, std::vector<uint64_t>& out) {
        out.clear();
        if (!catalog_has(q.value_col)) return MatrixQueryStatus::ERR_UNKNOWN_COLUMN;
        if (column_type(q.value_col) == MatrixType::I64) {
            if (q.grouped) {
                // Key-type check FIRST: an int64 GROUP BY key is unsupported (DM-3d) regardless of whether
                // it aliases the value column — i64val+i64key must report ERR_UNSUPPORTED_TYPE (spec §1),
                // not the ERR_INVALID_GROUP that the key==value/row-count guard below would otherwise give.
                if (catalog_has(q.key_col) && column_type(q.key_col) != MatrixType::U32)
                    return MatrixQueryStatus::ERR_UNSUPPORTED_TYPE; // int64 key = DM-3d
                if (!catalog_has(q.key_col) || q.key_col == q.value_col || q.num_groups == 0
                    || column_rows(q.key_col) != column_rows(q.value_col))
                    return MatrixQueryStatus::ERR_INVALID_GROUP;
                if (q.num_groups > max_query_groups_) return MatrixQueryStatus::ERR_TOO_MANY_GROUPS;
                grouped_aggregate_i64(q.key_col, q.value_col, q.num_groups, q.agg,
                                      MatrixPredicateI64{q.cmp, q.lo_i64, q.hi_i64}, q.has_filter, out);
                return MatrixQueryStatus::OK;
            }
            if (null_masks_.count(q.value_col)) { out.assign(1, scalar_aggregate_nullable(q)); return MatrixQueryStatus::OK; }
            out.assign(1, static_cast<uint64_t>(
                scan_tiered_column_i64(q.value_col, MatrixPredicateI64{q.cmp, q.lo_i64, q.hi_i64}, q.agg, q.has_filter)));
            return MatrixQueryStatus::OK;
        }
        if (column_type(q.value_col) == MatrixType::F64) {
            if (q.grouped) {
                if (catalog_has(q.key_col) && column_type(q.key_col) != MatrixType::U32)
                    return MatrixQueryStatus::ERR_UNSUPPORTED_TYPE;                  // double key = later
                if (!catalog_has(q.key_col) || q.key_col == q.value_col || q.num_groups == 0
                    || column_rows(q.key_col) != column_rows(q.value_col))
                    return MatrixQueryStatus::ERR_INVALID_GROUP;
                if (q.num_groups > max_query_groups_) return MatrixQueryStatus::ERR_TOO_MANY_GROUPS;
                grouped_aggregate_f64(q.key_col, q.value_col, q.num_groups, q.agg,
                                      MatrixPredicateF64{q.cmp, q.lo_f64, q.hi_f64}, q.has_filter, out);
                return MatrixQueryStatus::OK;
            }
            if (null_masks_.count(q.value_col)) { out.assign(1, scalar_aggregate_nullable(q)); return MatrixQueryStatus::OK; }
            out.assign(1, matrix_bit_cast<uint64_t>(
                scan_tiered_column_f64(q.value_col, MatrixPredicateF64{q.cmp, q.lo_f64, q.hi_f64}, q.agg, q.has_filter)));
            return MatrixQueryStatus::OK;
        }
        if (q.grouped) {
            if (!catalog_has(q.key_col) || q.key_col == q.value_col || q.num_groups == 0
                || catalog_.at(q.key_col)->size_bytes() != catalog_.at(q.value_col)->size_bytes())
                return MatrixQueryStatus::ERR_INVALID_GROUP;
            // A typed (int64) GROUP BY key would be reinterpreted as uint32 by grouped_aggregate; an
            // 8N-byte int64 key of N rows even passes the byte-length guard above (== a 2N-row u32 value).
            // Reject it — typed-key grouping lands in DM-3b. (value_col is already known U32 here.)
            if (column_type(q.key_col) != MatrixType::U32) return MatrixQueryStatus::ERR_UNSUPPORTED_TYPE;
            if (q.num_groups > max_query_groups_) return MatrixQueryStatus::ERR_TOO_MANY_GROUPS;
            if (q.has_filter) grouped_aggregate_pred(q.key_col, q.value_col, q.num_groups, q.agg, MatrixPredicate{q.cmp, q.threshold, q.upper}, out);
            else              grouped_aggregate(q.key_col, q.value_col, q.num_groups, q.agg, out);
        } else {
            // Null-aware path: a column with a null mask skips NULL rows; the filter (if any) is applied too (DM-3j).
            if (null_masks_.count(q.value_col))
                out.assign(1, scalar_aggregate_nullable(q));
            else
                out.assign(1, scan_tiered_column(q.value_col, MatrixPredicate{q.cmp, q.threshold, q.upper}, q.agg, q.has_filter));
        }
        return MatrixQueryStatus::OK;
    }

public:
    ~CPUMockEngine() override {
        // Make the fixed-capacity overflow seam loud (not silent) even in release builds:
        // if any write was dropped because the KVStore filled, report it. Inc 3's SSD
        // spill removes the drop entirely; until then this is the visible failure signal.
        if (store_overflows_ > 0) {
            // Dropped writes = data loss → an ERROR-level diagnostic (prints at the default WARN threshold).
            Log::emit(LogLevel::ERROR, std::to_string(store_overflows_)
                      + " point-op writes dropped — KVStore full (Inc 3 adds SSD spill).");
        }
        std::cout << "CPUMockEngine shutdown cleanly." << std::endl;
    }

    void execute_batch(DatabaseQuery* batch_array, size_t count) override {
        if (count == 0) return;
        if (count > MATRIX_BATCH_MAX) count = MATRIX_BATCH_MAX;

        // Bin by page (the step the dual-trigger batcher will eventually fold in).
        matrix_bin_by_page(batch_array, count, binned_.data(), offsets_.data());

        // One logical owner per page: process only this page's queries against its
        // contiguous store slice. Pages are independent — this is the parallel unit.
        // Scans arrive on a separate path (execute_scan), so this sees only point ops.
        for (size_t page = 0; page < MATRIX_PAGE_COUNT; ++page) {
            const uint32_t begin = offsets_[page];
            const uint32_t end = offsets_[page + 1];
            for (uint32_t j = begin; j < end; ++j) {
                DatabaseQuery& q = binned_[j];
                if (q.opcode == OP_READ) {
                    uint64_t v = 0;
                    kv_.get(q.query_id, v); // miss leaves v=0 (matches old zero-init store)
                    q.transaction_id = v;
                    ++reads_;
                } else if (q.opcode == OP_WRITE) {
                    // Durability invariant: append to the WAL FIRST (fsync per policy) so a
                    // write is only counted committed once it is durable. The in-memory kv_
                    // is volatile and rebuilt from the WAL on recovery.
                    if (cold_store_) cold_store_->append_put(q.query_id, q.query_id);
                    apply_committed_write(q.query_id, q.query_id);
                }
            }
        }

        // ponytail: read results land in binned_ (reordered), not scattered back to
        // batch_array. Callers here assert on counters + store contents, not on each
        // query's transaction_id, so the scatter-back is YAGNI. Add it if a caller
        // needs per-query read results in original order.
    }

    void execute_scan(DatabaseQuery& q) override {
        // id==0 -> the legacy fixed resident column; id>0 -> a tiered catalog column. The agg op
        // (default AGG_COUNT) selects the reduction; AGG_COUNT preserves the original count result.
        const uint32_t threshold = matrix_get_scan_threshold(q);
        const uint64_t col_id = matrix_get_scan_column_id(q);
        const MatrixAggOp op = matrix_get_scan_agg_op(q);
        const auto st0 = std::chrono::steady_clock::now();
        uint64_t c = 0;
        if (col_id == 0) {
            c = matrix_cpu_reduce(scan_column_.data(), MATRIX_SCAN_COLUMN_SIZE, threshold, op);
        } else {
            c = scan_tiered_column(col_id, MatrixPredicate{MatrixCmp::GT, threshold, 0}, op);
        }
        scan_time_s_ += std::chrono::duration<double>(
            std::chrono::steady_clock::now() - st0).count();
        q.transaction_id = c;
        ++scans_;
        scan_result_sum_ += c;
    }

    uint64_t reads() const override { return reads_; }
    uint64_t writes() const override { return writes_; }
    uint64_t commits() const override { return commits_; }
    uint64_t scans() const override { return scans_; }
    uint64_t scan_result_sum() const override { return scan_result_sum_; }
    double scan_time_s() const override { return scan_time_s_; }
    // CPU has no launch/sync layer, so kernel time == the timed scan loop (zero overhead).
    double scan_kernel_time_s() const override { return scan_time_s_; }

    uint64_t store_checksum() const override {
        return kv_.checksum();
    }

    double benchmark_scan(size_t n, uint64_t threshold, uint64_t& out_count) override {
        std::vector<uint64_t> data(n);
        for (size_t i = 0; i < n; ++i) data[i] = i; // resident; fill not timed
        const auto t0 = std::chrono::steady_clock::now();
        uint64_t count = 0;
        for (size_t i = 0; i < n; ++i) count += (data[i] > threshold);
        const auto t1 = std::chrono::steady_clock::now();
        out_count = count;
        return std::chrono::duration<double>(t1 - t0).count();
    }

    double benchmark_scan_u32(size_t n, uint32_t threshold, uint64_t& out_count) override {
        std::vector<uint32_t> data(n);
        for (size_t i = 0; i < n; ++i) data[i] = static_cast<uint32_t>(i);
        const auto t0 = std::chrono::steady_clock::now();
        uint64_t count = 0;
        for (size_t i = 0; i < n; ++i) count += (data[i] > threshold);
        const auto t1 = std::chrono::steady_clock::now();
        out_count = count;
        return std::chrono::duration<double>(t1 - t0).count();
    }

    double benchmark_scan_u32x4(size_t n, uint32_t threshold, uint64_t& out_count) override {
        // ponytail: uint4 vectorized loads are a GPU concept; the CPU compiler already
        // auto-vectorizes the scalar loop, so this just delegates. Keeps the interface
        // uniform without faking a "CPU vectorized" path.
        return benchmark_scan_u32(n, threshold, out_count);
    }

    double benchmark_scan_ipt(size_t n, uint32_t threshold, uint64_t& out_count) override {
        // ponytail: register-blocking is a GPU latency-hiding lever; on CPU it's the same
        // auto-vectorized loop. Delegate.
        return benchmark_scan_u32(n, threshold, out_count);
    }

private:
    // True iff `id` names a real catalog column (id 0 is the legacy fixed column, never a query target).
    bool catalog_has(uint64_t id) const { return id != 0 && catalog_.count(id) != 0; }

    // Append `n` elements of raw bytes to catalog column `id`, growing it (borrow COLD->HOST, append,
    // return the borrow, update the brain's accounting). Private — typed wrappers enforce the element type.
    void append_raw(uint64_t id, const unsigned char* bytes, size_t byte_count) {
        TieredColumn& col = *catalog_.at(id);
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        col.append_bytes(bytes, byte_count);
        if (home != MemorySpace::HOST) col.migrate_to(home);     // return the borrow (grown buffer)
        tier_mgr_.update_bytes(id, col.size_bytes());
    }

    // Scan one catalog column for value>threshold. A cold column is borrowed to HOST for the
    // scan then returned to its home tier, so the engine's residency always matches the brain's
    // accounting (no side-channel migration the budget can't see). Every REBALANCE_EVERY scans,
    // run the brain + executor: promote hot columns (DEVICE inert here), demote the coldest
    // HOST columns to SSD under the budget.
    // Null-aware scalar aggregate over a U32/I64/F64 column (DM-3j): borrow-and-return like
    // scan_tiered_column, but skip NULL rows via the column's null mask. When q.has_filter, the WHERE
    // predicate is applied too (the pred reducers null-check), so a NULL row is excluded whether or not it
    // would match. Returns the result as a uint64 (U32 zero-extended; I64/F64 as the bit pattern).
    uint64_t scalar_aggregate_nullable(const MatrixQuery& q) {
        const uint64_t col_id = q.value_col; const MatrixAggOp op = q.agg;
        TieredColumn& col = *catalog_.at(col_id);
        tier_mgr_.record_access(col_id, col.size_bytes());
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        const uint8_t* nulls = null_masks_.at(col_id).data();
        const MatrixType ty = column_type(col_id);
        uint64_t result;
        if (ty == MatrixType::I64) {
            const auto* v = reinterpret_cast<const int64_t*>(col.host_ptr());
            const size_t nn = col.size_bytes() / sizeof(int64_t);
            const int64_t r = q.has_filter
                ? matrix_cpu_reduce_pred_i64(v, nn, MatrixPredicateI64{q.cmp, q.lo_i64, q.hi_i64}, op, nulls)
                : matrix_cpu_reduce_all_i64_nullable(v, nn, op, nulls);
            result = static_cast<uint64_t>(r);
        } else if (ty == MatrixType::F64) {
            const auto* v = reinterpret_cast<const double*>(col.host_ptr());
            const size_t nn = col.size_bytes() / sizeof(double);
            const double r = q.has_filter
                ? matrix_cpu_reduce_pred_f64(v, nn, MatrixPredicateF64{q.cmp, q.lo_f64, q.hi_f64}, op, nulls)
                : matrix_cpu_reduce_all_f64_nullable(v, nn, op, nulls);
            result = matrix_bit_cast<uint64_t>(r);
        } else {
            const auto* v = reinterpret_cast<const uint32_t*>(col.host_ptr());
            const size_t nn = col.size_bytes() / sizeof(uint32_t);
            result = q.has_filter
                ? matrix_cpu_reduce_pred(v, nn, MatrixPredicate{q.cmp, q.threshold, q.upper}, op, nulls)
                : matrix_cpu_reduce_all_nullable(v, nn, op, nulls);
        }
        if (home != MemorySpace::HOST) col.migrate_to(home);
        maybe_rebalance();
        return result;
    }

    uint64_t scan_tiered_column(uint64_t col_id, MatrixPredicate pred, MatrixAggOp op, bool has_filter = true) {
        auto it = catalog_.find(col_id);
        if (it == catalog_.end()) {
            assert(false && "scan of unregistered column id"); // debug: catch the caller bug
            std::cerr << "CPUMockEngine: scan of unregistered column id " << col_id
                      << " — empty result\n";                 // release: diagnosable, no null-deref
            return 0;
        }
        TieredColumn& col = *it->second;
        tier_mgr_.record_access(col_id, col.size_bytes());          // heat signal
#if defined(MATRIX_USE_CUDA)
        if (col.tier() == MemorySpace::DEVICE) {                    // GPU-3: reduce in VRAM, no borrow-down
            const size_t nvals = col.size_bytes() / sizeof(uint32_t);
            const uint64_t result = matrix_gpu_reduce_dev_u32(col.device_ptr(), nvals, pred, op, has_filter);
            maybe_rebalance();
            return result;
        }
#endif

        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); } // pull SSD->RAM to scan
        const uint32_t* vals = reinterpret_cast<const uint32_t*>(col.host_ptr());
        const size_t nvals = col.size_bytes() / sizeof(uint32_t);
        const uint64_t result = has_filter ? matrix_cpu_reduce_pred(vals, nvals, pred, op)
                                           : matrix_cpu_reduce_all(vals, nvals, op);
        // ponytail: returning the borrow rewrites the COLD file each cold scan; skip-if-unchanged
        // (or a TierManager note_residency) is the upgrade path if cold-scan churn ever matters.
        if (home != MemorySpace::HOST) col.migrate_to(home);        // return the borrow

        maybe_rebalance();
        return result;
    }

    // Every REBALANCE_EVERY scans, run the brain + executor: promote hot (DEVICE inert here), demote the
    // coldest HOST columns to SSD under the budget. Shared by the u32 and int64 scan paths.
    void maybe_rebalance() {
        if (++scans_since_rebalance_ >= rebalance_every_) {
            std::unordered_map<uint64_t, TieredColumn*> ptrs;
            for (auto& kv : catalog_) ptrs[kv.first] = kv.second.get();
            migrations_ += executor_.apply(tier_mgr_.rebalance(), ptrs);
            ++rebalances_;
            scans_since_rebalance_ = 0;
        }
    }

    // int64 scalar scan over a catalog column (DM-3a unfiltered; DM-3b adds the filtered path). Same
    // borrow-to-HOST-and-return as scan_tiered_column: record heat, pull a COLD column to RAM, reinterpret
    // the raw bytes as int64, reduce by op (filtered when has_filter, else over all), return the borrow to
    // its home tier, and drive the shared rebalance cadence (int64 scans now count too — DM-3a follow-up).
    int64_t scan_tiered_column_i64(uint64_t col_id, MatrixPredicateI64 pred, MatrixAggOp op, bool has_filter = false) {
        TieredColumn& col = *catalog_.at(col_id);
        tier_mgr_.record_access(col_id, col.size_bytes());
#if defined(MATRIX_USE_CUDA)
        if (col.tier() == MemorySpace::DEVICE) {                    // GPU-3: reduce in VRAM, no borrow-down
            const size_t nvals = col.size_bytes() / sizeof(int64_t);
            const int64_t result = matrix_gpu_reduce_dev_i64(col.device_ptr(), nvals, pred, op, has_filter);
            maybe_rebalance();
            return result;
        }
#endif
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        const int64_t* vals = reinterpret_cast<const int64_t*>(col.host_ptr());
        const size_t nvals = col.size_bytes() / sizeof(int64_t);
        const int64_t result = has_filter ? matrix_cpu_reduce_pred_i64(vals, nvals, pred, op)
                                          : matrix_cpu_reduce_all_i64(vals, nvals, op);
        if (home != MemorySpace::HOST) col.migrate_to(home);
        maybe_rebalance();
        return result;
    }

    // double scalar scan over a catalog column (DM-3e). Same borrow-to-HOST-and-return as
    // scan_tiered_column_i64: record heat, pull a COLD column to RAM, reinterpret the raw bytes as
    // double, reduce by op (filtered when has_filter, else over all), return the borrow to its home
    // tier, and drive the shared rebalance cadence.
    double scan_tiered_column_f64(uint64_t col_id, MatrixPredicateF64 pred, MatrixAggOp op, bool has_filter = false) {
        TieredColumn& col = *catalog_.at(col_id);
        tier_mgr_.record_access(col_id, col.size_bytes());
#if defined(MATRIX_USE_CUDA)
        if (col.tier() == MemorySpace::DEVICE) {                    // GPU-3: reduce in VRAM, no borrow-down
            const size_t nvals = col.size_bytes() / sizeof(double);
            const double result = matrix_gpu_reduce_dev_f64(col.device_ptr(), nvals, pred, op, has_filter);
            maybe_rebalance();
            return result;
        }
#endif
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        const double* vals = reinterpret_cast<const double*>(col.host_ptr());
        const size_t nvals = col.size_bytes() / sizeof(double);
        const double result = has_filter ? matrix_cpu_reduce_pred_f64(vals, nvals, pred, op)
                                         : matrix_cpu_reduce_all_f64(vals, nvals, op);
        if (home != MemorySpace::HOST) col.migrate_to(home);
        maybe_rebalance();
        return result;
    }

    // Point-op store: a real hash table (gap DM-1). Distinct keys never overwrite; a full
    // table is an explicit error, not silent loss. Sized with headroom over the demo's
    // distinct write-keys; real capacity / SSD-spill is gap DM-9 / Inc 3 (the seam).
    KVStore kv_{1u << 16}; // 65536 slots
    // Ordered secondary index (DM-7): key -> value, mirrors kv_ (maintained on commit, rebuilt on recovery).
    // Enables log-time range scans (kv_range_sorted) vs kv_range's O(n) full scan.
    // ponytail: a std::map mirror — doubles point-op key memory + a map insert per commit; a packed
    // B-tree/ART would be denser, and the index is rebuilt from kv_ on restart (not separately persisted).
    std::map<uint64_t, uint64_t> key_index_;
    std::unique_ptr<ColdStore> cold_store_; // null = durability off (default); set via WAL path
    std::string checkpoint_path_;     // <wal_path>.ckpt — last point-op compaction snapshot
    uint64_t checkpoints_ = 0;        // DU-4: number of WAL compactions performed
    // --- live tiering (INT-1): a catalog of analytical columns the brain auto-tiers ---
    static constexpr uint64_t REBALANCE_EVERY = 4;     // default: rebalance every N tiered scans
    uint64_t rebalance_every_ = REBALANCE_EVERY;       // OB-4: runtime-tunable rebalance cadence (default = the constant)
    static constexpr uint32_t MATRIX_CATALOG_MAGIC = 0x4D434132u; // 'MCA2' — typed+named catalog snapshot v2 (DM-2b)
    static constexpr uint32_t MATRIX_CKPT_MAGIC = 0x4D434B50u; // 'MCKP' — point-op checkpoint file
    static constexpr uint32_t MAX_QUERY_GROUPS = 1u << 28; // default grouped-query num_groups ceiling (out alloc guard)
    uint32_t max_query_groups_ = MAX_QUERY_GROUPS;     // RM-2: runtime-tunable admission cap (default = the constant)
    TierManager tier_mgr_;                              // decides migrations (heat-driven)
    MigrationExecutor executor_;                        // moves the bytes per decision
    std::unordered_map<uint64_t, std::unique_ptr<TieredColumn>> catalog_; // id -> column
    std::unordered_map<uint64_t, MatrixType> col_types_; // id -> element type (absent ⇒ U32); DM-3a
    std::unordered_map<uint64_t, std::vector<std::string>> string_columns_; // DM-3i: separate string-column store
    std::unordered_map<uint64_t, std::vector<uint8_t>> null_masks_;          // DM-3j: id -> per-row NULL flag (1=null)
    std::unordered_map<uint64_t, std::string> column_names_;   // id -> optional name
    std::unordered_map<std::string, uint64_t> name_to_id_;     // name -> id (resolve)
    std::unordered_map<std::string, std::vector<uint64_t>> tables_;   // table name -> ordered column ids (DM-2c)
    uint64_t scans_since_rebalance_ = 0;
    uint64_t cold_borrows_ = 0;    // observability: COLD->HOST borrows
    uint64_t rebalances_ = 0;      // observability: rebalance passes
    uint64_t migrations_ = 0;      // observability: migration decisions executed
    uint64_t query_count_ = 0;     // observability: execute_query calls served (OK and ERR)
    uint64_t total_query_ns_ = 0;  // observability: summed execute_query wall-time (ns)
    uint64_t max_query_ns_ = 0;    // observability: slowest single execute_query (ns)
    static constexpr int LAT_BUCKETS = 40;                  // log2 latency buckets (OB-2b); 2^39 ns ≈ 9 min
    std::array<uint64_t, LAT_BUCKETS> latency_hist_{};      // per-query latency histogram (bucket = floor(log2(ns+1)))
    std::vector<DatabaseQuery> binned_;                // scratch: batch sorted by page
    std::array<uint32_t, MATRIX_PAGE_COUNT + 1> offsets_{}; // CSR page offsets
    std::vector<uint32_t> scan_column_;                // resident analytical column
    uint64_t reads_ = 0;
    uint64_t writes_ = 0;
    uint64_t commits_ = 0;
    uint64_t scans_ = 0;
    uint64_t scan_result_sum_ = 0;
    uint64_t store_overflows_ = 0; // writes dropped because the fixed-capacity KVStore was full (Inc 3 adds SSD spill)
    double scan_time_s_ = 0.0;

    // Apply one durable write to the point-op store with the standard overflow accounting.
    // mock projection: value == key. Fixed-capacity seam: a full table is counted as an
    // overflow (always live, even under NDEBUG) so a dropped write is never silent. Inc 3
    // replaces this with SSD spill; the assert makes it fail loud in debug builds too.
    void apply_committed_write(uint64_t key, uint64_t value) {
        ++writes_;
        if (kv_.put(key, value)) { ++commits_; key_index_[key] = value; }   // mirror into the ordered secondary index
        else { ++store_overflows_; assert(false && "KVStore full — point-op store capacity exceeded (Inc 3 adds spill)"); }
    }

    // --- Atomic transactions (WAL group commit) ---
    std::vector<std::pair<uint64_t, uint64_t>> txn_buf_; // pending writes in the open transaction
    bool in_txn_ = false;
    uint64_t transactions_committed_ = 0;
    uint64_t transactions_rolled_back_ = 0;
};


In [ ]:
%%writefile compute_cuda.cuh
#pragma once

// CUDA backend — page-ownership model (Component 4: Parallel Engine).
// Compile on a GPU host (Google Colab) with:
//     nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA main.cpp -o matrixdb_proto
//
// One CUDA BLOCK owns one page. The batch is binned by page on the host (CSR offsets),
// so block p processes only page p's contiguous queries against page p's store slice.
// Different blocks touch disjoint store slots ⇒ no cross-block conflict, no store
// atomics, no delta log. Each page has a single owner; pages are independent
// (shared-nothing). Threads within a block stride over the page's queries.

#include "compute.hpp"
#include <cuda_runtime.h>
#include <vector>
#include <chrono>
#include <cstdio>
#include <cstdlib>
#include <limits>
#include <cstdint>

#define CUDA_CHECK(call)                                                            \
    do {                                                                           \
        cudaError_t _err = (call);                                                 \
        if (_err != cudaSuccess) {                                                 \
            std::fprintf(stderr, "CUDA error %s at %s:%d -> %s\n",                 \
                         #call, __FILE__, __LINE__, cudaGetErrorString(_err));     \
            std::abort();                                                          \
        }                                                                          \
    } while (0)

// One block per page. blockIdx.x == page id; the block's threads cooperatively process
// that page's queries [offsets[page], offsets[page+1]). Writes to a slot within a page
// are owned by this block alone, so no atomics on the store are needed. Same-slot writes
// within the page race between threads -> last-writer-wins, matching the CPU mock's
// deterministic in-order result only when keys are unique (true for our benchmark).
__global__ void matrix_page_kernel(const DatabaseQuery* binned, const uint32_t* offsets,
                                   uint64_t* store,
                                   unsigned long long* reads,
                                   unsigned long long* writes) {
    const size_t page = blockIdx.x;
    if (page >= MATRIX_PAGE_COUNT) return;

    const uint32_t begin = offsets[page];
    const uint32_t end = offsets[page + 1];

    unsigned r = 0, w = 0;
    for (uint32_t j = begin + threadIdx.x; j < end; j += blockDim.x) {
        const DatabaseQuery q = binned[j];
        const size_t slot = q.query_id & MATRIX_STORE_MASK;
        if (q.opcode == OP_READ) {
            volatile uint64_t v = store[slot]; (void)v; // touch the store (read path)
            ++r;
        } else if (q.opcode == OP_WRITE) {
            store[slot] = q.query_id; // mock projection: value == key
            ++w;
        }
    }
    if (r) atomicAdd(reads, (unsigned long long)r);   // counters only — not on the store
    if (w) atomicAdd(writes, (unsigned long long)w);
}

// Filter-count scan: count values > threshold. Grid-stride over resident VRAM data,
// block-local reduction, one atomicAdd per block. This is the GPU's home turf —
// streaming bandwidth over data too large for CPU cache.
__global__ void matrix_scan_kernel(const uint64_t* data, size_t n,
                                   uint64_t threshold, unsigned long long* count) {
    __shared__ unsigned long long block_count;
    if (threadIdx.x == 0) block_count = 0;
    __syncthreads();

    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        if (data[i] > threshold) ++local;
    }
    atomicAdd(&block_count, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(count, block_count);
}

// uint32 column variant — half the bytes/value. Same grid-stride filter-count.
__global__ void matrix_scan_kernel_u32(const uint32_t* data, size_t n,
                                       uint32_t threshold, unsigned long long* count) {
    __shared__ unsigned long long block_count;
    if (threadIdx.x == 0) block_count = 0;
    __syncthreads();

    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        if (data[i] > threshold) ++local;
    }
    atomicAdd(&block_count, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(count, block_count);
}

// AGG-2 (GPU-1): SUM / MIN / MAX of values matching `value > threshold`, same grid-stride +
// block-local + one-atomic-per-block shape as the COUNT kernel, differing only in accumulator + atomic.
// Each is correct iff its result equals matrix_cpu_reduce(host, n, threshold, op) over the same bytes
// (the cross-backend invariant — test_gpu_agg.cu is the merge gate). Empty-match sentinels match the CPU:
// SUM 0, MIN UINT64_MAX, MAX 0. `out` must be pre-initialized to the sentinel before launch.
__global__ void matrix_sum_kernel_u32(const uint32_t* data, size_t n,
                                      uint32_t threshold, unsigned long long* out) {
    __shared__ unsigned long long blk;
    if (threadIdx.x == 0) blk = 0;
    __syncthreads();
    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (data[i] > threshold) local += data[i];
    atomicAdd(&blk, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(out, blk);
}
__global__ void matrix_min_kernel_u32(const uint32_t* data, size_t n,
                                      uint32_t threshold, unsigned long long* out) {
    __shared__ unsigned long long blk;
    if (threadIdx.x == 0) blk = 0xFFFFFFFFFFFFFFFFull;   // UINT64_MAX (CPU empty-MIN sentinel)
    __syncthreads();
    unsigned long long local = 0xFFFFFFFFFFFFFFFFull;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (data[i] > threshold && data[i] < local) local = data[i];
    atomicMin(&blk, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicMin(out, blk);
}
__global__ void matrix_max_kernel_u32(const uint32_t* data, size_t n,
                                      uint32_t threshold, unsigned long long* out) {
    __shared__ unsigned long long blk;
    if (threadIdx.x == 0) blk = 0;                       // 0 (CPU empty-MAX sentinel)
    __syncthreads();
    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (data[i] > threshold && data[i] > local) local = data[i];
    atomicMax(&blk, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicMax(out, blk);
}

// === GPU-4 (predicates) + GPU-2 (grouped) — ADDITIVE; the GPU-1 threshold kernels above are untouched. ===
// Device predicate mirroring matrix_pred_match (compute.hpp) so a GPU WHERE matches the CPU's exactly.
__device__ __forceinline__ bool matrix_pred_match_dev(uint32_t v, MatrixCmp c, uint32_t a, uint32_t b) {
    switch (c) {
        case MatrixCmp::GE:      return v >= a;
        case MatrixCmp::LT:      return v <  a;
        case MatrixCmp::LE:      return v <= a;
        case MatrixCmp::EQ:      return v == a;
        case MatrixCmp::NE:      return v != a;
        case MatrixCmp::BETWEEN: return v >= a && v <= b;
        case MatrixCmp::GT:
        default:                 return v >  a;
    }
}
// GPU-4: predicate-filtered scalar reductions — the general sibling of the GPU-1 threshold kernels.
// Each == matrix_cpu_reduce_pred(host, n, {cmp,a,b}, op). Sentinels: COUNT/SUM 0, MIN UINT64_MAX, MAX 0
// (caller pre-inits `out` to the same: cudaMemset 0x00 for COUNT/SUM/MAX, 0xFF for MIN).
__global__ void matrix_count_kernel_pred_u32(const uint32_t* data, size_t n, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    __shared__ unsigned long long blk; if (threadIdx.x == 0) blk = 0; __syncthreads();
    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_dev(data[i], c, a, b)) ++local;
    atomicAdd(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicAdd(out, blk);
}
__global__ void matrix_sum_kernel_pred_u32(const uint32_t* data, size_t n, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    __shared__ unsigned long long blk; if (threadIdx.x == 0) blk = 0; __syncthreads();
    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_dev(data[i], c, a, b)) local += data[i];
    atomicAdd(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicAdd(out, blk);
}
__global__ void matrix_min_kernel_pred_u32(const uint32_t* data, size_t n, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    __shared__ unsigned long long blk; if (threadIdx.x == 0) blk = 0xFFFFFFFFFFFFFFFFull; __syncthreads();
    unsigned long long local = 0xFFFFFFFFFFFFFFFFull;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_dev(data[i], c, a, b) && data[i] < local) local = data[i];
    atomicMin(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicMin(out, blk);
}
__global__ void matrix_max_kernel_pred_u32(const uint32_t* data, size_t n, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    __shared__ unsigned long long blk; if (threadIdx.x == 0) blk = 0; __syncthreads();
    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_dev(data[i], c, a, b) && data[i] > local) local = data[i];
    atomicMax(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicMax(out, blk);
}
// GPU-2: grouped reduction — one atomic per row into its dense group slot out[k] (k = keys[i], k < num_groups;
// out-of-range keys ignored, matching the CPU). Each == matrix_cpu_group_reduce(keys, vals, n, num_groups, op).
// Caller pre-inits out[num_groups] per op via cudaMemset: COUNT/SUM/MAX -> 0x00, MIN -> 0xFF (UINT64_MAX).
// ponytail: simple global-atomic per row (correct first); a per-block shared-memory privatized histogram is
// the contention-reduction follow-up for high group counts.
__global__ void matrix_group_count_kernel(const uint32_t* keys, size_t n, uint32_t num_groups, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups) atomicAdd(&out[k], 1ull);
    }
}
__global__ void matrix_group_sum_kernel(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups) atomicAdd(&out[k], (unsigned long long)vals[i]);
    }
}
__global__ void matrix_group_min_kernel(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups) atomicMin(&out[k], (unsigned long long)vals[i]);
    }
}
__global__ void matrix_group_max_kernel(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups) atomicMax(&out[k], (unsigned long long)vals[i]);
    }
}

// Filtered grouped kernels (GROUP BY ... WHERE pred(value)) — same one-atomic-per-row shape as the
// unfiltered ones above, gated by the verified matrix_pred_match_dev on the VALUE (matches the CPU's
// matrix_cpu_group_reduce_pred). COUNT reads vals too (it must apply the predicate). Sentinels identical
// to the unfiltered path (COUNT/SUM/MAX 0, MIN ULLONG_MAX).
__global__ void matrix_group_count_kernel_pred(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_dev(vals[i], c, a, b)) atomicAdd(&out[k], 1ull);
    }
}
__global__ void matrix_group_sum_kernel_pred(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_dev(vals[i], c, a, b)) atomicAdd(&out[k], (unsigned long long)vals[i]);
    }
}
__global__ void matrix_group_min_kernel_pred(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_dev(vals[i], c, a, b)) atomicMin(&out[k], (unsigned long long)vals[i]);
    }
}
__global__ void matrix_group_max_kernel_pred(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_dev(vals[i], c, a, b)) atomicMax(&out[k], (unsigned long long)vals[i]);
    }
}

// === GPU-5 (typed int64 + double) — ADDITIVE; the u32 kernels above are untouched. ===
// Device predicates mirroring matrix_pred_match_i64 / matrix_pred_match_f64 (compute.hpp).
__device__ __forceinline__ bool matrix_pred_match_i64_dev(long long v, MatrixCmp c, long long a, long long b) {
    switch (c) {
        case MatrixCmp::GE:      return v >= a;
        case MatrixCmp::LT:      return v <  a;
        case MatrixCmp::LE:      return v <= a;
        case MatrixCmp::EQ:      return v == a;
        case MatrixCmp::NE:      return v != a;
        case MatrixCmp::BETWEEN: return v >= a && v <= b;
        case MatrixCmp::GT:
        default:                 return v >  a;
    }
}
__device__ __forceinline__ bool matrix_pred_match_f64_dev(double v, MatrixCmp c, double a, double b) {
    switch (c) {
        case MatrixCmp::GE:      return v >= a;
        case MatrixCmp::LT:      return v <  a;
        case MatrixCmp::LE:      return v <= a;
        case MatrixCmp::EQ:      return v == a;
        case MatrixCmp::NE:      return v != a;
        case MatrixCmp::BETWEEN: return v >= a && v <= b;
        case MatrixCmp::GT:
        default:                 return v >  a;
    }
}
// double has no native atomicMin/Max — CAS loop on the bit pattern (the standard idiom). Works on
// shared or global. NaN: matches the CPU's IEEE behavior (a NaN val never updates, since cur<=NaN is false).
__device__ __forceinline__ double atomicMinDouble(double* addr, double val) {
    unsigned long long* p = (unsigned long long*)addr;
    unsigned long long old = *p, assumed;
    do { assumed = old;
         if (__longlong_as_double(assumed) <= val) break;
         old = atomicCAS(p, assumed, __double_as_longlong(val));
    } while (assumed != old);
    return __longlong_as_double(old);
}
__device__ __forceinline__ double atomicMaxDouble(double* addr, double val) {
    unsigned long long* p = (unsigned long long*)addr;
    unsigned long long old = *p, assumed;
    do { assumed = old;
         if (__longlong_as_double(assumed) >= val) break;
         old = atomicCAS(p, assumed, __double_as_longlong(val));
    } while (assumed != old);
    return __longlong_as_double(old);
}
// Native atomicAdd(double*) exists on arch >= 6.0 and in the host pass; nvcc without an explicit -arch
// can default the device pass below 6.0, where the overload is absent. Provide the CAS fallback ONLY in
// that device pass (defined(__CUDA_ARCH__) && < 600) — NOT the host pass, where the native one is already
// declared (defining it there is the redefinition Colab caught).
#if defined(__CUDA_ARCH__) && __CUDA_ARCH__ < 600
__device__ __forceinline__ double atomicAdd(double* address, double val) {
    unsigned long long* p = (unsigned long long*)address;
    unsigned long long old = *p, assumed;
    do { assumed = old;
         old = atomicCAS(p, assumed, __double_as_longlong(val + __longlong_as_double(assumed)));
    } while (assumed != old);
    return __longlong_as_double(old);
}
#endif
#define MATRIX_DEV_POSINF __longlong_as_double((long long)0x7FF0000000000000ULL)
#define MATRIX_DEV_NEGINF __longlong_as_double((long long)0xFFF0000000000000ULL)

// int64 predicate-filtered reductions. out is `long long*`; sentinels match matrix_cpu_reduce_pred_i64
// (COUNT/SUM 0, MIN INT64_MAX, MAX INT64_MIN — caller cudaMemcpy's the init). SUM accumulates as unsigned
// (two's-complement add is bit-identical to signed; same overflow-wrap as the CPU).
__global__ void matrix_count_kernel_pred_i64(const long long* d, size_t n, MatrixCmp c, long long a, long long b, long long* out) {
    __shared__ unsigned long long blk; if (threadIdx.x == 0) blk = 0; __syncthreads();
    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_i64_dev(d[i], c, a, b)) ++local;
    atomicAdd(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicAdd((unsigned long long*)out, blk);
}
__global__ void matrix_sum_kernel_pred_i64(const long long* d, size_t n, MatrixCmp c, long long a, long long b, long long* out) {
    __shared__ unsigned long long blk; if (threadIdx.x == 0) blk = 0; __syncthreads();
    long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_i64_dev(d[i], c, a, b)) local += d[i];
    atomicAdd(&blk, (unsigned long long)local); __syncthreads();
    if (threadIdx.x == 0) atomicAdd((unsigned long long*)out, blk);
}
__global__ void matrix_min_kernel_pred_i64(const long long* d, size_t n, MatrixCmp c, long long a, long long b, long long* out) {
    __shared__ long long blk; if (threadIdx.x == 0) blk = INT64_MAX; __syncthreads();
    long long local = INT64_MAX;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_i64_dev(d[i], c, a, b) && d[i] < local) local = d[i];
    atomicMin(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicMin(out, blk);
}
__global__ void matrix_max_kernel_pred_i64(const long long* d, size_t n, MatrixCmp c, long long a, long long b, long long* out) {
    __shared__ long long blk; if (threadIdx.x == 0) blk = INT64_MIN; __syncthreads();
    long long local = INT64_MIN;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_i64_dev(d[i], c, a, b) && d[i] > local) local = d[i];
    atomicMax(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicMax(out, blk);
}
// double predicate-filtered reductions. out is `double*`; sentinels match matrix_cpu_reduce_pred_f64
// (COUNT/SUM 0.0, MIN +inf, MAX -inf — caller cudaMemcpy's the init). atomicAdd(double*) needs arch>=6.0
// (T4 is 7.5); MIN/MAX use the CAS-loop helpers above.
__global__ void matrix_count_kernel_pred_f64(const double* d, size_t n, MatrixCmp c, double a, double b, double* out) {
    __shared__ double blk; if (threadIdx.x == 0) blk = 0.0; __syncthreads();
    double local = 0.0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_f64_dev(d[i], c, a, b)) local += 1.0;
    atomicAdd(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicAdd(out, blk);
}
__global__ void matrix_sum_kernel_pred_f64(const double* d, size_t n, MatrixCmp c, double a, double b, double* out) {
    __shared__ double blk; if (threadIdx.x == 0) blk = 0.0; __syncthreads();
    double local = 0.0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_f64_dev(d[i], c, a, b)) local += d[i];
    atomicAdd(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicAdd(out, blk);
}
__global__ void matrix_min_kernel_pred_f64(const double* d, size_t n, MatrixCmp c, double a, double b, double* out) {
    __shared__ double blk; if (threadIdx.x == 0) blk = MATRIX_DEV_POSINF; __syncthreads();
    double local = MATRIX_DEV_POSINF;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_f64_dev(d[i], c, a, b) && d[i] < local) local = d[i];
    atomicMinDouble(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicMinDouble(out, blk);
}
__global__ void matrix_max_kernel_pred_f64(const double* d, size_t n, MatrixCmp c, double a, double b, double* out) {
    __shared__ double blk; if (threadIdx.x == 0) blk = MATRIX_DEV_NEGINF; __syncthreads();
    double local = MATRIX_DEV_NEGINF;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_f64_dev(d[i], c, a, b) && d[i] > local) local = d[i];
    atomicMaxDouble(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicMaxDouble(out, blk);
}
// Vectorized uint32 scan: each thread loads 4 values per instruction via uint4
// (LDG.128 = 16 bytes/load). This is the standard memory-bound fix across DL/DB/HPC
// kernels — more bytes in flight per thread => deeper memory-level parallelism =>
// closer to peak DRAM bandwidth than the scalar 4-byte load. Requires n % 4 == 0 and
// 16-byte-aligned data (cudaMalloc guarantees alignment; n is a power of two here).
__global__ void matrix_scan_kernel_u32x4(const uint4* data4, size_t n4,
                                         uint32_t threshold, unsigned long long* count) {
    __shared__ unsigned long long block_count;
    if (threadIdx.x == 0) block_count = 0;
    __syncthreads();

    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n4; i += stride) {
        const uint4 v = data4[i]; // one 16-byte load -> 4 comparisons
        local += (v.x > threshold) + (v.y > threshold)
               + (v.z > threshold) + (v.w > threshold);
    }
    atomicAdd(&block_count, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(count, block_count);
}

// Items-per-thread uint32 scan (register blocking) — the lever CUB's BlockReduce uses
// (128 threads x 4 items each). Each thread issues ITEMS independent loads into a
// register array BEFORE comparing any, so multiple loads are in flight per thread =>
// DRAM latency is hidden => bandwidth saturates. Unlike uint4 this needs no special
// type/alignment and doesn't inflate per-load register width. Strided access keeps the
// warp's ITEMS loads coalesced (lane L reads base+L, base+L+blockDim, ...).
template <int ITEMS>
__global__ void matrix_scan_kernel_ipt(const uint32_t* data, size_t n,
                                       uint32_t threshold, unsigned long long* count) {
    __shared__ unsigned long long block_count;
    if (threadIdx.x == 0) block_count = 0;
    __syncthreads();

    unsigned long long local = 0;
    const size_t base_stride = (size_t)gridDim.x * blockDim.x;
    const size_t chunk = base_stride * ITEMS;
    // Striped layout: thread's global id is the base; item k is base + k*base_stride.
    // (base init must NOT include *ITEMS — that mixes the blocked convention with the
    // striped strides below and leaves most indices unvisited. Verified by
    // test_scan_coverage.cpp.)
    for (size_t base = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
         base < n; base += chunk) {
        uint32_t reg[ITEMS];
        #pragma unroll
        for (int k = 0; k < ITEMS; ++k) {
            const size_t idx = base + (size_t)k * base_stride;
            reg[k] = (idx < n) ? data[idx] : 0u; // load all ITEMS first (in flight)
        }
        #pragma unroll
        for (int k = 0; k < ITEMS; ++k) {
            const size_t idx = base + (size_t)k * base_stride;
            if (idx < n) local += (reg[k] > threshold); // then compare
        }
    }
    atomicAdd(&block_count, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(count, block_count);
}

/**
 * @brief GPU-3 device-resident scalar reduce. Run the (already cross-checked) predicate kernels over a
 * catalog column's VRAM bytes and return the SAME bit-encoding as matrix_cpu_reduce_pred / _all, so
 * CPUMockEngine::execute_query's DEVICE path == its CPU path. Forward-declared in compute_mock.cpp (which
 * is included before this header in the nvcc TU) and called from scan_tiered_column* when tier()==DEVICE.
 * Unfiltered (!has_filter) synthesizes an always-true predicate (GE min) to mirror matrix_cpu_reduce_all.
 * ponytail: an always-true pred drops NaN rows that reduce_all would count — fine for real numeric data
 * (no NaN in the cross-check); add dedicated unfiltered kernels only if NaN COUNT ever matters.
 */
inline uint64_t matrix_gpu_reduce_dev_u32(const void* d_data, size_t n,
                                          MatrixPredicate pred, MatrixAggOp op, bool has_filter) {
    const MatrixCmp c = has_filter ? pred.cmp : MatrixCmp::GE;
    const uint32_t  a = has_filter ? pred.a   : 0u;            // GE 0 == every u32
    const uint32_t  b = has_filter ? pred.b   : 0u;
    const uint32_t* d = reinterpret_cast<const uint32_t*>(d_data);
    unsigned long long* d_out = nullptr; CUDA_CHECK(cudaMalloc(&d_out, sizeof(unsigned long long)));
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      { CUDA_CHECK(cudaMemset(d_out, 0x00, sizeof(*d_out))); matrix_sum_kernel_pred_u32<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out); }
    else if (op == AGG_MIN) { CUDA_CHECK(cudaMemset(d_out, 0xFF, sizeof(*d_out))); matrix_min_kernel_pred_u32<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out); }
    else if (op == AGG_MAX) { CUDA_CHECK(cudaMemset(d_out, 0x00, sizeof(*d_out))); matrix_max_kernel_pred_u32<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out); }
    else                    { CUDA_CHECK(cudaMemset(d_out, 0x00, sizeof(*d_out))); matrix_count_kernel_pred_u32<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out); }
    CUDA_CHECK(cudaGetLastError());
    unsigned long long h = 0; CUDA_CHECK(cudaMemcpy(&h, d_out, sizeof(h), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
    return h;
}
inline int64_t matrix_gpu_reduce_dev_i64(const void* d_data, size_t n,
                                         MatrixPredicateI64 pred, MatrixAggOp op, bool has_filter) {
    const MatrixCmp  c = has_filter ? pred.cmp : MatrixCmp::GE;
    const long long  a = has_filter ? pred.a   : INT64_MIN;    // GE INT64_MIN == every i64
    const long long  b = has_filter ? pred.b   : 0;
    const long long* d = reinterpret_cast<const long long*>(d_data);
    long long* d_out = nullptr; CUDA_CHECK(cudaMalloc(&d_out, sizeof(long long)));
    long long init = (op == AGG_MIN) ? INT64_MAX : (op == AGG_MAX ? INT64_MIN : 0);   // sentinels aren't memset-able
    CUDA_CHECK(cudaMemcpy(d_out, &init, sizeof(init), cudaMemcpyHostToDevice));
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      matrix_sum_kernel_pred_i64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else if (op == AGG_MIN) matrix_min_kernel_pred_i64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else if (op == AGG_MAX) matrix_max_kernel_pred_i64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else                    matrix_count_kernel_pred_i64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    CUDA_CHECK(cudaGetLastError());
    long long h = 0; CUDA_CHECK(cudaMemcpy(&h, d_out, sizeof(h), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
    return static_cast<int64_t>(h);
}
inline double matrix_gpu_reduce_dev_f64(const void* d_data, size_t n,
                                        MatrixPredicateF64 pred, MatrixAggOp op, bool has_filter) {
    const double inf = std::numeric_limits<double>::infinity();
    const MatrixCmp c = has_filter ? pred.cmp : MatrixCmp::GE;
    const double    a = has_filter ? pred.a   : -inf;          // GE -inf == every non-NaN f64
    const double    b = has_filter ? pred.b   : 0.0;
    const double*   d = reinterpret_cast<const double*>(d_data);
    double* d_out = nullptr; CUDA_CHECK(cudaMalloc(&d_out, sizeof(double)));
    double init = (op == AGG_MIN) ? inf : (op == AGG_MAX ? -inf : 0.0);
    CUDA_CHECK(cudaMemcpy(d_out, &init, sizeof(init), cudaMemcpyHostToDevice));
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      matrix_sum_kernel_pred_f64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else if (op == AGG_MIN) matrix_min_kernel_pred_f64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else if (op == AGG_MAX) matrix_max_kernel_pred_f64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else                    matrix_count_kernel_pred_f64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    CUDA_CHECK(cudaGetLastError());
    double h = 0; CUDA_CHECK(cudaMemcpy(&h, d_out, sizeof(h), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
    return h;
}

/**
 * @brief GPU-3 (grouped) device-resident GROUP BY. Run the grouped kernels over key+value VRAM bytes and
 * write num_groups results into out_host with the SAME per-group encoding/sentinels as
 * matrix_cpu_group_reduce(_pred), so CPUMockEngine's grouped DEVICE path is bit-identical to its CPU path.
 * Forward-declared in compute_mock.cpp; u32 key+value only (matches the verified kernels), no null mask.
 */
inline void matrix_gpu_group_reduce_dev(const void* keys_dev, const void* vals_dev, size_t n,
                                        uint32_t num_groups, MatrixAggOp op, MatrixPredicate pred,
                                        bool has_filter, uint64_t* out_host) {
    const uint32_t* dk = reinterpret_cast<const uint32_t*>(keys_dev);
    const uint32_t* dv = reinterpret_cast<const uint32_t*>(vals_dev);
    unsigned long long* d_out = nullptr;
    CUDA_CHECK(cudaMalloc(&d_out, (size_t)num_groups * sizeof(unsigned long long)));
    CUDA_CHECK(cudaMemset(d_out, (op == AGG_MIN) ? 0xFF : 0x00, (size_t)num_groups * sizeof(unsigned long long))); // MIN -> ULLONG_MAX
    constexpr int TPB = 256, BLOCKS = 1024;
    if (has_filter) {
        const MatrixCmp c = pred.cmp; const uint32_t a = pred.a, b = pred.b;
        if (op == AGG_SUM)      matrix_group_sum_kernel_pred<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
        else if (op == AGG_MIN) matrix_group_min_kernel_pred<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
        else if (op == AGG_MAX) matrix_group_max_kernel_pred<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
        else                    matrix_group_count_kernel_pred<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    } else {
        if (op == AGG_SUM)      matrix_group_sum_kernel<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, d_out);
        else if (op == AGG_MIN) matrix_group_min_kernel<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, d_out);
        else if (op == AGG_MAX) matrix_group_max_kernel<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, d_out);
        else                    matrix_group_count_kernel<<<BLOCKS, TPB>>>(dk, n, num_groups, d_out);   // unfiltered COUNT: keys only
    }
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(out_host, d_out, (size_t)num_groups * sizeof(unsigned long long), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
}

// === Typed (int64/double) grouped kernels — GROUP BY a u32 key over an i64/f64 value, predicate-gated on
// the VALUE (unfiltered = an all-match predicate synthesized by the wrapper). Per-group atomics reuse the
// verified GPU-5 typed atomics; out[] carries the same int64/double bit-pattern per group as the CPU
// matrix_cpu_group_reduce_{i64,f64}. ===
__global__ void matrix_group_count_kernel_i64(const uint32_t* keys, const long long* vals, size_t n, uint32_t num_groups, MatrixCmp c, long long a, long long b, long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_i64_dev(vals[i], c, a, b)) atomicAdd((unsigned long long*)&out[k], 1ull);
    }
}
__global__ void matrix_group_sum_kernel_i64(const uint32_t* keys, const long long* vals, size_t n, uint32_t num_groups, MatrixCmp c, long long a, long long b, long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_i64_dev(vals[i], c, a, b)) atomicAdd((unsigned long long*)&out[k], (unsigned long long)vals[i]);
    }
}
__global__ void matrix_group_min_kernel_i64(const uint32_t* keys, const long long* vals, size_t n, uint32_t num_groups, MatrixCmp c, long long a, long long b, long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_i64_dev(vals[i], c, a, b)) atomicMin(&out[k], vals[i]);
    }
}
__global__ void matrix_group_max_kernel_i64(const uint32_t* keys, const long long* vals, size_t n, uint32_t num_groups, MatrixCmp c, long long a, long long b, long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_i64_dev(vals[i], c, a, b)) atomicMax(&out[k], vals[i]);
    }
}
__global__ void matrix_group_count_kernel_f64(const uint32_t* keys, const double* vals, size_t n, uint32_t num_groups, MatrixCmp c, double a, double b, double* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_f64_dev(vals[i], c, a, b)) atomicAdd(&out[k], 1.0);
    }
}
__global__ void matrix_group_sum_kernel_f64(const uint32_t* keys, const double* vals, size_t n, uint32_t num_groups, MatrixCmp c, double a, double b, double* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_f64_dev(vals[i], c, a, b)) atomicAdd(&out[k], vals[i]);
    }
}
__global__ void matrix_group_min_kernel_f64(const uint32_t* keys, const double* vals, size_t n, uint32_t num_groups, MatrixCmp c, double a, double b, double* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_f64_dev(vals[i], c, a, b)) atomicMinDouble(&out[k], vals[i]);
    }
}
__global__ void matrix_group_max_kernel_f64(const uint32_t* keys, const double* vals, size_t n, uint32_t num_groups, MatrixCmp c, double a, double b, double* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_f64_dev(vals[i], c, a, b)) atomicMaxDouble(&out[k], vals[i]);
    }
}

// Typed grouped wrappers: same per-group encoding as matrix_cpu_group_reduce_{i64,f64} (int64/double bits).
// Sentinels (MIN INT64_MAX/+inf, MAX INT64_MIN/-inf, SUM/COUNT 0) aren't byte-fillable, so init via memcpy.
// Unfiltered synthesizes an all-match predicate (GE min) to mirror the unfiltered CPU reducer.
inline void matrix_gpu_group_reduce_dev_i64(const void* keys_dev, const void* vals_dev, size_t n,
                                            uint32_t num_groups, MatrixAggOp op, MatrixPredicateI64 pred,
                                            bool has_filter, uint64_t* out_host) {
    const uint32_t*  dk = reinterpret_cast<const uint32_t*>(keys_dev);
    const long long* dv = reinterpret_cast<const long long*>(vals_dev);
    long long* d_out = nullptr; CUDA_CHECK(cudaMalloc(&d_out, (size_t)num_groups * sizeof(long long)));
    std::vector<long long> init(num_groups, (op == AGG_MIN) ? INT64_MAX : (op == AGG_MAX ? INT64_MIN : 0));
    CUDA_CHECK(cudaMemcpy(d_out, init.data(), (size_t)num_groups * sizeof(long long), cudaMemcpyHostToDevice));
    const MatrixCmp  c = has_filter ? pred.cmp : MatrixCmp::GE;
    const long long  a = has_filter ? pred.a   : INT64_MIN;
    const long long  b = has_filter ? pred.b   : 0;
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      matrix_group_sum_kernel_i64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    else if (op == AGG_MIN) matrix_group_min_kernel_i64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    else if (op == AGG_MAX) matrix_group_max_kernel_i64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    else                    matrix_group_count_kernel_i64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(out_host, d_out, (size_t)num_groups * sizeof(long long), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
}
inline void matrix_gpu_group_reduce_dev_f64(const void* keys_dev, const void* vals_dev, size_t n,
                                            uint32_t num_groups, MatrixAggOp op, MatrixPredicateF64 pred,
                                            bool has_filter, uint64_t* out_host) {
    const uint32_t* dk = reinterpret_cast<const uint32_t*>(keys_dev);
    const double*   dv = reinterpret_cast<const double*>(vals_dev);
    const double inf = std::numeric_limits<double>::infinity();
    double* d_out = nullptr; CUDA_CHECK(cudaMalloc(&d_out, (size_t)num_groups * sizeof(double)));
    std::vector<double> init(num_groups, (op == AGG_MIN) ? inf : (op == AGG_MAX ? -inf : 0.0));
    CUDA_CHECK(cudaMemcpy(d_out, init.data(), (size_t)num_groups * sizeof(double), cudaMemcpyHostToDevice));
    const MatrixCmp c = has_filter ? pred.cmp : MatrixCmp::GE;
    const double    a = has_filter ? pred.a   : -inf;
    const double    b = has_filter ? pred.b   : 0.0;
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      matrix_group_sum_kernel_f64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    else if (op == AGG_MIN) matrix_group_min_kernel_f64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    else if (op == AGG_MAX) matrix_group_max_kernel_f64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    else                    matrix_group_count_kernel_f64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(out_host, d_out, (size_t)num_groups * sizeof(double), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
}

/**
 * @brief Real CUDA GPU engine, page-ownership model. Device-resident store persists
 * across batches (it is the database). Same ComputeInterface + correctness contract
 * as CPUMockEngine.
 */
class CUDAGPUEngine : public ComputeInterface {
public:
    explicit CUDAGPUEngine(size_t /*worker_count*/ = 0)
        : h_binned_(MATRIX_BATCH_MAX) {
        int device_count = 0;
        CUDA_CHECK(cudaGetDeviceCount(&device_count));
        if (device_count == 0) {
            std::fprintf(stderr, "CUDAGPUEngine: no CUDA device found.\n");
            std::abort();
        }
        cudaDeviceProp prop{};
        CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));

        CUDA_CHECK(cudaMalloc(&d_store_, MATRIX_STORE_SLOTS * sizeof(uint64_t)));
        CUDA_CHECK(cudaMemset(d_store_, 0, MATRIX_STORE_SLOTS * sizeof(uint64_t)));
        CUDA_CHECK(cudaMalloc(&d_binned_, MATRIX_BATCH_MAX * sizeof(DatabaseQuery)));
        CUDA_CHECK(cudaMalloc(&d_offsets_, (MATRIX_PAGE_COUNT + 1) * sizeof(uint32_t)));
        CUDA_CHECK(cudaMalloc(&d_reads_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMalloc(&d_writes_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_reads_, 0, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_writes_, 0, sizeof(unsigned long long)));

        // Resident analytical column (value[i]=i), filled once, scanned in place by
        // OP_SCAN. This is the GPU-DB's actual data — never shipped per query.
        CUDA_CHECK(cudaMalloc(&d_scan_col_, MATRIX_SCAN_COLUMN_SIZE * sizeof(uint32_t)));
        {
            std::vector<uint32_t> h(MATRIX_SCAN_COLUMN_SIZE);
            for (size_t i = 0; i < MATRIX_SCAN_COLUMN_SIZE; ++i) h[i] = static_cast<uint32_t>(i);
            CUDA_CHECK(cudaMemcpy(d_scan_col_, h.data(),
                                  MATRIX_SCAN_COLUMN_SIZE * sizeof(uint32_t),
                                  cudaMemcpyHostToDevice));
        }
        CUDA_CHECK(cudaMalloc(&d_scan_count_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_scan_count_, 0, sizeof(unsigned long long)));

        offsets_.resize(MATRIX_PAGE_COUNT + 1);

        // ponytail: single stream. Hyper-Q multi-stream is the throughput upgrade.
        CUDA_CHECK(cudaStreamCreate(&stream_));
        // Dedicated stream for scans so they overlap point-op work and don't serialize
        // behind it (the HTAP separation, GPU side).
        CUDA_CHECK(cudaStreamCreate(&scan_stream_));
        // Reused events to time just the scan kernel (per-scan create/destroy would
        // itself add overhead and pollute the measurement).
        CUDA_CHECK(cudaEventCreate(&scan_k0_));
        CUDA_CHECK(cudaEventCreate(&scan_k1_));
        std::printf("CUDAGPUEngine initialized on '%s' (%d SMs, page-ownership, %zu pages).\n",
                    prop.name, prop.multiProcessorCount, MATRIX_PAGE_COUNT);
    }

    ~CUDAGPUEngine() override {
        cudaFree(d_store_);
        cudaFree(d_binned_);
        cudaFree(d_offsets_);
        cudaFree(d_reads_);
        cudaFree(d_writes_);
        cudaFree(d_scan_col_);
        cudaFree(d_scan_count_);
        cudaEventDestroy(scan_k0_);
        cudaEventDestroy(scan_k1_);
        cudaStreamDestroy(stream_);
        cudaStreamDestroy(scan_stream_);
        std::printf("CUDAGPUEngine released device resources.\n");
    }

    void execute_batch(DatabaseQuery* batch_array, size_t count) override {
        if (count == 0) return;
        if (count > MATRIX_BATCH_MAX) count = MATRIX_BATCH_MAX;

        // Bin by page on the host (folds into the dual-trigger batcher later).
        matrix_bin_by_page(batch_array, count, h_binned_.data(), offsets_.data());

        CUDA_CHECK(cudaMemcpyAsync(d_binned_, h_binned_.data(), count * sizeof(DatabaseQuery),
                                   cudaMemcpyHostToDevice, stream_));
        CUDA_CHECK(cudaMemcpyAsync(d_offsets_, offsets_.data(),
                                   (MATRIX_PAGE_COUNT + 1) * sizeof(uint32_t),
                                   cudaMemcpyHostToDevice, stream_));

        // One block per page; 128 threads/block stride over the page's queries.
        // Point ops only — scans arrive via execute_scan on their own stream.
        constexpr int TPB = 128;
        matrix_page_kernel<<<MATRIX_PAGE_COUNT, TPB, 0, stream_>>>(
            d_binned_, d_offsets_, d_store_, d_reads_, d_writes_);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaStreamSynchronize(stream_));
    }

    void execute_scan(DatabaseQuery& q) override {
        // u32x4 filter-count over the resident column, on the dedicated scan stream so
        // it overlaps point-op submission on stream_ and never head-of-line-blocks them.
        // AGG-2 (GPU-1): dispatch on matrix_get_scan_agg_op(q). COUNT keeps the byte-identical u32x4
        // path (the 83886070 oracle); SUM/MIN/MAX init `out` to the matching CPU sentinel then launch the
        // scalar reduction kernel. Each is verified GPU==matrix_cpu_reduce by test_gpu_agg.cu.
        const auto st0 = std::chrono::steady_clock::now();
        const uint32_t threshold = matrix_get_scan_threshold(q);
        const MatrixAggOp op = matrix_get_scan_agg_op(q);
        constexpr int SCAN_TPB = 256;
        const int blocks = 1024;
        CUDA_CHECK(cudaEventRecord(scan_k0_, scan_stream_));
        if (op == AGG_SUM) {
            CUDA_CHECK(cudaMemsetAsync(d_scan_count_, 0, sizeof(unsigned long long), scan_stream_));  // SUM sentinel 0
            matrix_sum_kernel_u32<<<blocks, SCAN_TPB, 0, scan_stream_>>>(d_scan_col_, MATRIX_SCAN_COLUMN_SIZE, threshold, d_scan_count_);
        } else if (op == AGG_MIN) {
            CUDA_CHECK(cudaMemsetAsync(d_scan_count_, 0xFF, sizeof(unsigned long long), scan_stream_)); // MIN sentinel UINT64_MAX
            matrix_min_kernel_u32<<<blocks, SCAN_TPB, 0, scan_stream_>>>(d_scan_col_, MATRIX_SCAN_COLUMN_SIZE, threshold, d_scan_count_);
        } else if (op == AGG_MAX) {
            CUDA_CHECK(cudaMemsetAsync(d_scan_count_, 0, sizeof(unsigned long long), scan_stream_));    // MAX sentinel 0
            matrix_max_kernel_u32<<<blocks, SCAN_TPB, 0, scan_stream_>>>(d_scan_col_, MATRIX_SCAN_COLUMN_SIZE, threshold, d_scan_count_);
        } else {                                                                                       // AGG_COUNT (default) — unchanged
            CUDA_CHECK(cudaMemsetAsync(d_scan_count_, 0, sizeof(unsigned long long), scan_stream_));
            const uint4* col4 = reinterpret_cast<const uint4*>(d_scan_col_);
            matrix_scan_kernel_u32x4<<<blocks, SCAN_TPB, 0, scan_stream_>>>(
                col4, MATRIX_SCAN_COLUMN_SIZE / 4, threshold, d_scan_count_);
        }
        CUDA_CHECK(cudaEventRecord(scan_k1_, scan_stream_));
        CUDA_CHECK(cudaGetLastError());
        unsigned long long c = 0;
        CUDA_CHECK(cudaMemcpyAsync(&c, d_scan_count_, sizeof(c),
                                   cudaMemcpyDeviceToHost, scan_stream_));
        CUDA_CHECK(cudaStreamSynchronize(scan_stream_));
        scan_time_s_ += std::chrono::duration<double>(
            std::chrono::steady_clock::now() - st0).count();
        float k_ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&k_ms, scan_k0_, scan_k1_));
        scan_kernel_time_s_ += k_ms / 1e3;
        q.transaction_id = c;
        ++scans_;
        scan_result_sum_ += c;
    }

    uint64_t reads() const override { return read_counter(d_reads_); }
    uint64_t writes() const override { return read_counter(d_writes_); }
    uint64_t commits() const override { return read_counter(d_writes_); } // every write commits (owned slot)
    uint64_t scans() const override { return scans_; }
    uint64_t scan_result_sum() const override { return scan_result_sum_; }
    double scan_time_s() const override { return scan_time_s_; }
    double scan_kernel_time_s() const override { return scan_kernel_time_s_; }

    uint64_t store_checksum() const override {
        // ponytail: copy the whole store back (32KB) and sum on host. Once, off the
        // hot path — a device reduction would be premature for a correctness check.
        std::vector<uint64_t> h(MATRIX_STORE_SLOTS);
        CUDA_CHECK(cudaMemcpy(h.data(), d_store_, MATRIX_STORE_SLOTS * sizeof(uint64_t),
                              cudaMemcpyDeviceToHost));
        uint64_t sum = 0;
        for (uint64_t v : h) sum += v;
        return sum;
    }

    double benchmark_scan(size_t n, uint64_t threshold, uint64_t& out_count) override {
        return timed_scan<uint64_t>(n, threshold, out_count,
            [](const uint64_t* d, size_t nn, uint64_t thr, unsigned long long* c,
               int blocks, int tpb, cudaStream_t s) {
                matrix_scan_kernel<<<blocks, tpb, 0, s>>>(d, nn, thr, c);
            });
    }

    double benchmark_scan_u32(size_t n, uint32_t threshold, uint64_t& out_count) override {
        return timed_scan<uint32_t>(n, threshold, out_count,
            [](const uint32_t* d, size_t nn, uint32_t thr, unsigned long long* c,
               int blocks, int tpb, cudaStream_t s) {
                matrix_scan_kernel_u32<<<blocks, tpb, 0, s>>>(d, nn, thr, c);
            });
    }

    double benchmark_scan_u32x4(size_t n, uint32_t threshold, uint64_t& out_count) override {
        // Same resident-column setup, but the kernel reads 4 u32 per uint4 load.
        // n is a power of two (>= 65536) so n % 4 == 0; cudaMalloc is 256-byte aligned.
        return timed_scan<uint32_t>(n, threshold, out_count,
            [](const uint32_t* d, size_t nn, uint32_t thr, unsigned long long* c,
               int blocks, int tpb, cudaStream_t s) {
                const uint4* d4 = reinterpret_cast<const uint4*>(d);
                matrix_scan_kernel_u32x4<<<blocks, tpb, 0, s>>>(d4, nn / 4, thr, c);
            });
    }

    double benchmark_scan_ipt(size_t n, uint32_t threshold, uint64_t& out_count) override {
        return timed_scan<uint32_t>(n, threshold, out_count,
            [](const uint32_t* d, size_t nn, uint32_t thr, unsigned long long* c,
               int blocks, int tpb, cudaStream_t s) {
                matrix_scan_kernel_ipt<8><<<blocks, tpb, 0, s>>>(d, nn, thr, c);
            });
    }

private:
    // Shared scan harness: alloc resident column, fill (untimed), time kernel via CUDA
    // events, return seconds. Templated on column type so u32/u64 share one code path.
    template <typename T, typename Launch>
    double timed_scan(size_t n, T threshold, uint64_t& out_count, Launch launch) {
        T* d_data = nullptr;
        unsigned long long* d_count = nullptr;
        CUDA_CHECK(cudaMalloc(&d_data, n * sizeof(T)));
        CUDA_CHECK(cudaMalloc(&d_count, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(unsigned long long)));
        {
            std::vector<T> h(n);
            for (size_t i = 0; i < n; ++i) h[i] = static_cast<T>(i);
            CUDA_CHECK(cudaMemcpy(d_data, h.data(), n * sizeof(T), cudaMemcpyHostToDevice));
        }

        constexpr int TPB = 256;
        const int blocks = 1024; // saturate the 40 SMs; grid-stride handles any n
        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));
        CUDA_CHECK(cudaEventRecord(start, stream_));
        launch(d_data, n, threshold, d_count, blocks, TPB, stream_);
        CUDA_CHECK(cudaEventRecord(stop, stream_));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));

        unsigned long long h_count = 0;
        CUDA_CHECK(cudaMemcpy(&h_count, d_count, sizeof(h_count), cudaMemcpyDeviceToHost));
        out_count = h_count;

        cudaEventDestroy(start);
        cudaEventDestroy(stop);
        cudaFree(d_data);
        cudaFree(d_count);
        return ms / 1e3; // seconds
    }

    static uint64_t read_counter(const unsigned long long* d_ptr) {
        unsigned long long h = 0;
        CUDA_CHECK(cudaMemcpy(&h, d_ptr, sizeof(h), cudaMemcpyDeviceToHost));
        return static_cast<uint64_t>(h);
    }

    uint64_t* d_store_ = nullptr;
    DatabaseQuery* d_binned_ = nullptr;
    uint32_t* d_offsets_ = nullptr;
    unsigned long long* d_reads_ = nullptr;
    unsigned long long* d_writes_ = nullptr;
    uint32_t* d_scan_col_ = nullptr;            // resident analytical column
    unsigned long long* d_scan_count_ = nullptr; // scratch for one scan's count
    uint64_t scans_ = 0;                          // host-side scan accounting
    uint64_t scan_result_sum_ = 0;
    double scan_time_s_ = 0.0;
    double scan_kernel_time_s_ = 0.0;            // cudaEvent-measured pure kernel time
    cudaEvent_t scan_k0_{}, scan_k1_{};          // reused scan-kernel timing events
    std::vector<DatabaseQuery> h_binned_;
    std::vector<uint32_t> offsets_;
    cudaStream_t stream_{};
    cudaStream_t scan_stream_{};                  // dedicated scan path (HTAP)
};


In [ ]:
%%writefile main.cpp
#include "types.hpp"
#include "ring_buffer.hpp"
#include "compute_mock.cpp"        // CPU engine — always present (fallback + point-op home)
#if defined(MATRIX_USE_CUDA)
    #include "compute_cuda.cuh"    // GPU engine — present only in the CUDA build
#endif
#include "router.hpp"
#include <iostream>
#include <chrono>
#include <thread>
#include <memory>
#include <array>
#include <atomic>
#include <cassert>
#include <vector>
#include <algorithm>

#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    #include <pthread.h>
    #include <pthread/qos.h>        // ponytail: spec omitted these 3; needed to compile the
    #include <mach/mach.h>          // QoS + Mach affinity calls below on macOS
    #include <mach/thread_policy.h>
#elif defined(__linux__)
    #include <sched.h>
#endif

/**
 * @brief Platform-agnostic pipeline stalling barrier for low-overhead busy spins.
 */
inline void spin_stall() noexcept {
#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    // Instruction Synchronization Barrier (isb) forces instruction execution pipeline flush on ARM64
    asm volatile("isb" ::: "memory");
#elif defined(__x86_64__)
    // Standard pause execution hint
    asm volatile("pause" ::: "memory");
#else
    asm volatile("" ::: "memory");
#endif
}

/**
 * @brief Configures OS-specific thread-pinning and priority policies to ensure execution on performance cores.
 */
inline void pin_to_performance_core() {
#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    // Promotes the calling thread to the interactive QoS class to guarantee performance core placement on macOS
    pthread_set_qos_class_self_np(QOS_CLASS_USER_INTERACTIVE, 0);

    // Advisory affinity grouping configuration on Apple Silicon Mach kernels
    thread_port_t mach_thread = pthread_mach_thread_np(pthread_self());
    thread_affinity_policy_data_t policy = { 1 };
    thread_policy_set(mach_thread, THREAD_AFFINITY_POLICY, (thread_policy_t)&policy, THREAD_AFFINITY_POLICY_COUNT);
#elif defined(__linux__)
    cpu_set_t cpuset;
    CPU_ZERO(&cpuset);
    CPU_SET(0, &cpuset); // Pin strictly to logical Core 0
    pthread_setaffinity_np(pthread_self(), sizeof(cpu_set_t), &cpuset);
#endif
}

/**
 * @brief Raw SPSC handoff latency via ping-pong: the producer pushes exactly one
 * item, waits until the consumer has popped it, then pushes the next. This isolates
 * the pure enqueue->dequeue cost — the spec's "sub-microsecond" claim — with zero
 * queue backlog, unlike the pipeline's queue-residency number which is dominated by
 * burst backlog. Backend-independent: measures only the ring buffer.
 */
void measure_handoff_latency() {
    constexpr size_t WARMUP = 1000;   // discard cold-cache / thread-migration samples
    constexpr size_t SAMPLES = 100000;
    auto ring = std::make_unique<SPSCRingBuffer<DatabaseQuery, 4096>>();
    std::atomic<bool> running{true};
    std::vector<uint64_t> samples;
    samples.reserve(SAMPLES);

    // Consumer pops and timestamps; it is the sole writer of `samples` (joined before read).
    std::thread consumer([&]() {
        pin_to_performance_core();
        DatabaseQuery v;
        while (running.load(std::memory_order_relaxed)) {
            if (ring->try_pop(v)) {
                const uint64_t now_ns = static_cast<uint64_t>(
                    std::chrono::steady_clock::now().time_since_epoch().count());
                if (v.query_id >= WARMUP) {
                    samples.push_back(now_ns - v.timestamp_us);
                }
            }
        }
    });

    for (size_t i = 0; i < WARMUP + SAMPLES; ++i) {
        DatabaseQuery q{};
        q.query_id = i;
        q.timestamp_us = static_cast<uint64_t>(
            std::chrono::steady_clock::now().time_since_epoch().count());
        while (!ring->try_emplace(q)) spin_stall();
        while (!ring->empty()) spin_stall(); // wait for consumer to take it (ping-pong)
    }
    running.store(false);
    consumer.join();

    std::sort(samples.begin(), samples.end());
    const auto pct = [&](double p) { return samples[static_cast<size_t>(p * (samples.size() - 1))]; };
    std::cout << "Raw SPSC handoff (ns)    "
              << "p50=" << pct(0.50) << "  p99=" << pct(0.99)
              << "  p99.9=" << pct(0.999) << "  max=" << samples.back() << std::endl;
}

/**
 * @brief Throughput vs. batch size sweep — the key question for GPU viability.
 * Calls execute_batch() directly (no ring) so it measures pure engine cost per query
 * across batch sizes. Reveals fixed per-batch overhead and where (if ever) larger
 * batches amortize it. One run gives the whole curve — precious when GPU runs are remote.
 */
void sweep_batch_sizes(ComputeInterface& engine) {
    constexpr size_t QUERIES_PER_POINT = 400000; // enough work to dwarf timer noise
    const size_t sizes[] = {64, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536};

    std::cout << "--- Batch-size sweep (execute_batch only, " << QUERIES_PER_POINT
              << " queries/point) ---" << std::endl;

    std::vector<DatabaseQuery> batch(MATRIX_BATCH_MAX);
    for (size_t i = 0; i < MATRIX_BATCH_MAX; ++i) {
        batch[i] = DatabaseQuery{};
        batch[i].query_id = i;
        batch[i].opcode = (i % 2 == 0) ? OP_READ : OP_WRITE;
    }

    // Warm up the engine (first GPU call pays one-time CUDA init) so it doesn't
    // pollute the smallest batch point.
    engine.execute_batch(batch.data(), 256);

    for (size_t bs : sizes) {
        const size_t iters = QUERIES_PER_POINT / bs;
        const auto t0 = std::chrono::steady_clock::now();
        for (size_t it = 0; it < iters; ++it) {
            engine.execute_batch(batch.data(), bs);
        }
        const auto t1 = std::chrono::steady_clock::now();
        const double secs = std::chrono::duration<double>(t1 - t0).count();
        const double ops = static_cast<double>(iters * bs) / secs;
        const double us_per_batch = (secs / iters) * 1e6;
        std::cout << "  batch=" << bs
                  << "\t" << static_cast<uint64_t>(ops) << " ops/sec"
                  << "\t" << us_per_batch << " us/batch" << std::endl;
    }
}

/**
 * @brief Scan benchmark — the GPU's home turf. Filter-count over RESIDENT data of
 * growing size: small (fits CPU cache) to large (far exceeds it). The crossover where
 * GPU overtakes CPU is the whole point — exploiting bandwidth over big data, not
 * shipping tiny point-ops over PCIe. Verifies the count against a known oracle.
 */
void scan_benchmark(ComputeInterface& engine) {
    const size_t sizes[] = {1u<<16, 1u<<18, 1u<<20, 1u<<22, 1u<<24, 1u<<26}; // 64K .. 64M values
    std::cout << "--- Scan benchmark (filter-count over resident data) ---" << std::endl;
    for (size_t n : sizes) {
        const uint64_t threshold = n / 2;
        uint64_t c64 = 0, c32 = 0, c32x4 = 0, cipt = 0;
        const double s64 = engine.benchmark_scan(n, threshold, c64);
        const double s32 = engine.benchmark_scan_u32(n, static_cast<uint32_t>(threshold), c32);
        const double s32x4 = engine.benchmark_scan_u32x4(n, static_cast<uint32_t>(threshold), c32x4);
        const double sipt = engine.benchmark_scan_ipt(n, static_cast<uint32_t>(threshold), cipt);
        // value[i]=i, so count of i>threshold == n-1-threshold (oracle).
        assert(c64 == n - 1 - threshold && "u64 scan produced wrong count");
        assert(c32 == n - 1 - threshold && "u32 scan produced wrong count");
        assert(c32x4 == n - 1 - threshold && "u32x4 vectorized scan produced wrong count");
        assert(cipt == n - 1 - threshold && "items-per-thread scan produced wrong count");
        std::cout << "  n=" << n
                  << "\tu64 " << (n * sizeof(uint64_t)) / 1e9 / s64 << " GB/s"
                  << "\tu32 " << (n * sizeof(uint32_t)) / 1e9 / s32 << " GB/s"
                  << "\tu32x4 " << (n * sizeof(uint32_t)) / 1e9 / s32x4 << " GB/s"
                  << "\tipt8 " << (n * sizeof(uint32_t)) / 1e9 / sipt << " GB/s"
                  << " (" << static_cast<uint64_t>(n / sipt / 1e6) << "M vals/s)"
                  << std::endl;
    }
}

// Thesis demo: register two scan columns straddling the crossover plus point ops, run
// them through the Router, and report where each landed. Proves placement is by cost and
// results are correct regardless of home. (Latency comparison vs single-backend is a
// follow-up; this asserts correctness of routing first.)
void routing_demo(Router& router) {
    const char* sp[] = {"HOST", "DEVICE", "COLD", "UNIFIED"};
    const uint64_t small_id = router.place_scan_column(256u * 1024);       // 256 KB
    const uint64_t large_id = router.place_scan_column(64u * 1024 * 1024); // 64 MB
    std::cout << "Routing demo: 256KB column -> " << sp[(int)router.home_of(small_id)]
              << ", 64MB column -> " << sp[(int)router.home_of(large_id)] << std::endl;
}

// Demonstrates the full analytical stack live (not just in tests): load catalog columns into a
// RAM-constrained engine, run unified queries via execute_query, show auto-tiering hold a working
// set larger than RAM, and self-verify every result against an in-function brute-force oracle.
// Uses its own engine + catalog columns (id>0) — disjoint from the benchmark pipeline + scan oracle.
void analytical_query_demo() {
    const size_t N = 1u << 20;                 // 1M rows
    const uint32_t G = 4;                       // 4 groups
    const size_t S = N * sizeof(uint32_t);
    std::vector<uint32_t> keys(N), va(N), vb(N);
    for (size_t i = 0; i < N; ++i) {
        keys[i] = static_cast<uint32_t>(i % G);
        va[i]   = static_cast<uint32_t>(i % 1000);
        vb[i]   = static_cast<uint32_t>(i % 1000);
    }
    // Brute-force oracles (independent of the engine).
    std::vector<uint64_t> g_oracle(G, 0);
    uint64_t vb_sum = 0;
    for (size_t i = 0; i < N; ++i) { if (va[i] > 500) g_oracle[keys[i]] += va[i]; vb_sum += vb[i]; }

    CPUMockEngine demo(0, "", /*host_cap=*/2 * S);   // RAM holds 2 of the 3 columns
    demo.load_scan_column(1, keys.data(), N);        // key (group) column
    demo.load_scan_column(2, va.data(),  N);         // value column A
    demo.load_scan_column(3, vb.data(),  N);         // value column B

    std::cout << "\n=== Analytical query demo (tiered catalog + execute_query) ===" << std::endl;
    std::cout << "Loaded 3 columns (" << (3 * S) / (1024 * 1024) << " MB) into a "
              << (2 * S) / (1024 * 1024) << " MB RAM budget." << std::endl;

    // SELECT key, SUM(va) WHERE va > 500 GROUP BY key
    std::vector<uint64_t> grouped;
    demo.execute_query(MatrixQuery{.value_col = 2, .agg = AGG_SUM, .has_filter = true, .threshold = 500,
                                   .grouped = true, .key_col = 1, .num_groups = G}, grouped);
    assert(grouped == g_oracle && "grouped query result matches oracle");
    std::cout << "SELECT key, SUM(va) WHERE va>500 GROUP BY key:" << std::endl;
    for (uint32_t g = 0; g < G; ++g)
        std::cout << "   group " << g << " -> " << grouped[g] << std::endl;

    // Drive tiering: keep cols 1 & 2 hot (scalar scans), never touch col 3 -> col 3 (heat 0) is the
    // deterministic eviction victim once past MIN_RESIDENCY_TICKS.
    std::vector<uint64_t> scalar;
    for (int k = 0; k < 16; ++k) {
        demo.execute_query(MatrixQuery{.value_col = 1, .agg = AGG_COUNT}, scalar);
        demo.execute_query(MatrixQuery{.value_col = 2, .agg = AGG_COUNT}, scalar);
    }
#if defined(MATRIX_USE_CUDA)
    // GPU-3: with the DEVICE tier live, the cost model legitimately places columns differently than the
    // 2-tier (HOST/COLD) build — the hot columns promote to VRAM, which can relieve host pressure so the
    // idle column need not spill to SSD. Exact placement is the brain's call; the budget invariant still
    // holds: a 12 MB working set cannot all sit in an 8 MB host cap, so at least one column is off-HOST
    // (on DEVICE or COLD). Query correctness regardless of tier is checked below (vb_sum).
    assert((demo.column_tier(1) != MemorySpace::HOST || demo.column_tier(2) != MemorySpace::HOST
            || demo.column_tier(3) != MemorySpace::HOST) && "over-budget working set tiered off pure-HOST");
#else
    assert(demo.column_tier(3) == MemorySpace::COLD && "idle column auto-demoted to SSD");
    assert(demo.column_tier(1) == MemorySpace::HOST && demo.column_tier(2) == MemorySpace::HOST
           && "hot columns kept resident");
#endif

    const char* sp[] = {"HOST", "DEVICE", "COLD", "UNIFIED"};
    std::cout << "After a hot workload on cols 1 & 2, tier residency: col1="
              << sp[(int)demo.column_tier(1)] << " col2=" << sp[(int)demo.column_tier(2)]
              << " col3=" << sp[(int)demo.column_tier(3)]
              << "  (3-column working set in a 2-column RAM budget; the brain keeps the hot set in the fastest tier, spills the rest)" << std::endl;

    // Query the demoted column -> pulled back to RAM (or scanned in VRAM under CUDA), correct regardless of tier.
    demo.execute_query(MatrixQuery{.value_col = 3, .agg = AGG_SUM}, scalar);
    assert(scalar.size() == 1 && scalar[0] == vb_sum && "query over the tiered column is correct regardless of its tier");
    std::cout << "SELECT SUM(vb)  [col 3 tier=" << sp[(int)demo.column_tier(3)] << "] -> " << scalar[0]
              << "  (scanned in place / pulled back as needed, correct). Demo OK." << std::endl;
    const EngineStats st = demo.stats();
    std::cout << "engine stats: cold_borrows=" << st.cold_borrows
              << " rebalances=" << st.rebalances << " migrations=" << st.migrations
              << " catalog_cols=" << st.catalog_columns
              << " | resident HOST=" << st.host_resident_bytes / (1024 * 1024) << "MB"
              << " COLD=" << st.cold_resident_bytes / (1024 * 1024) << "MB" << std::endl;
    std::cout << "  query latency: count=" << st.query_count
              << " mean=" << (st.query_count ? st.total_query_ns / st.query_count / 1000 : 0) << "us"
              << " max=" << st.max_query_ns / 1000 << "us\n";
}

int main() {
    std::cout << "MatrixDB Bare-Metal Engine Booting..." << std::endl;

    // Microbenchmark first: pure ring handoff, before the pipeline saturates anything.
    measure_handoff_latency();

    constexpr size_t BATCH_SIZE_LIMIT = 512;
    constexpr uint64_t TEMPORAL_LIMIT_US = 50;
    constexpr size_t TOTAL_QUERIES = 10000;

    auto ring_buffer = std::make_unique<SPSCRingBuffer<DatabaseQuery, 4096>>();
    // Separate ring for scans (HTAP): a long scan in its own queue can't head-of-line-
    // block the point ops. Small — scans are infrequent and run one at a time.
    auto scan_ring = std::make_unique<SPSCRingBuffer<DatabaseQuery, 64>>();

    // Both engines live in one process; the Router places data and dispatches per query.
    auto cpu_engine = std::make_unique<CPUMockEngine>(4);
#if defined(MATRIX_USE_CUDA)
    auto gpu_engine = std::make_unique<CUDAGPUEngine>(4);
    ComputeInterface* gpu_ptr = gpu_engine.get();
    const bool gpu_available = true;
#else
    ComputeInterface* gpu_ptr = nullptr;
    const bool gpu_available = false;
#endif
    MemoryModel memmodel = MemoryModel::detect(gpu_available);
    Router router(cpu_engine.get(), gpu_ptr, CostModel(memmodel, gpu_available));

    // The scan column is a single dataset; the Router decides its home from its size.
    const uint64_t scan_col_id =
        router.place_scan_column(MATRIX_SCAN_COLUMN_SIZE * sizeof(uint32_t));

    // Counters for scans live on whichever engine actually ran them.
    ComputeInterface* scan_engine =
        (router.home_of(scan_col_id) == MemorySpace::DEVICE && gpu_ptr) ? gpu_ptr : cpu_engine.get();

    // Point-op counters/store always live on the CPU engine.
    ComputeInterface* point_op_engine = cpu_engine.get();

    routing_demo(router);

    // Sweep batch sizes before the pipeline run so one execution yields the whole
    // throughput-vs-batch curve (decisive for GPU viability; remote runs are costly).
    sweep_batch_sizes(*point_op_engine);
    scan_benchmark(*point_op_engine);

    std::atomic<bool> run_state{true};
    std::atomic<size_t> total_processed{0}; // ponytail: end-to-end check that every query flows through

    // Pre-reserved so the hot path never allocates. Filled by the single consumer only.
    std::vector<uint64_t> latencies_ns;
    latencies_ns.reserve(TOTAL_QUERIES);

    // Spin up the consumer thread to execute the ingestion busy-spin control loop
    std::thread consumer([&]() {
        pin_to_performance_core();

        std::array<DatabaseQuery, BATCH_SIZE_LIMIT> batch;
        size_t accumulated_queries = 0;
        auto batch_start_time = std::chrono::high_resolution_clock::now();

        while (run_state.load(std::memory_order_relaxed)) {
            DatabaseQuery incoming_query;
            const bool success = ring_buffer->try_pop(incoming_query);

            if (success) {
                // Queue residency = time from producer's enqueue stamp to this pop.
                const uint64_t now_ns = static_cast<uint64_t>(
                    std::chrono::steady_clock::now().time_since_epoch().count());
                latencies_ns.push_back(now_ns - incoming_query.timestamp_us);

                if (accumulated_queries == 0) {
                    batch_start_time = std::chrono::high_resolution_clock::now();
                }
                batch[accumulated_queries++] = incoming_query;
            }

            // Continuous evaluation of the Dual-Trigger Condition
            if (accumulated_queries > 0) {
                const auto current_time = std::chrono::high_resolution_clock::now();
                const auto duration_us = std::chrono::duration_cast<std::chrono::microseconds>(
                    current_time - batch_start_time
                ).count();

                // Trigger flush if batch is full OR time-based window has expired
                if (accumulated_queries >= BATCH_SIZE_LIMIT || duration_us >= static_cast<long long>(TEMPORAL_LIMIT_US)) {
                    router.route_batch(batch.data(), accumulated_queries);
                    total_processed.fetch_add(accumulated_queries, std::memory_order_relaxed);
                    accumulated_queries = 0;
                }
            }

            if (!success) {
                spin_stall();
            }
        }
    });

    // Scan consumer: dedicated thread + queue so a multi-ms scan runs concurrently with
    // point-op processing instead of stalling it. Scans run one at a time (no batching).
    std::atomic<size_t> scans_processed{0};
    std::thread scan_consumer([&]() {
        while (run_state.load(std::memory_order_relaxed)) {
            DatabaseQuery sq;
            if (scan_ring->try_pop(sq)) {
                router.route_scan(sq, scan_col_id);
                scans_processed.fetch_add(1, std::memory_order_relaxed);
            } else {
                spin_stall();
            }
        }
        // Drain any scans still queued at shutdown.
        DatabaseQuery sq;
        while (scan_ring->try_pop(sq)) {
            router.route_scan(sq, scan_col_id);
            scans_processed.fetch_add(1, std::memory_order_relaxed);
        }
    });

    // Snapshot engine counters after the sweep so the pipeline asserts below measure
    // only the pipeline's queries, not the sweep's traffic.
    const uint64_t reads_base = point_op_engine->reads();
    const uint64_t writes_base = point_op_engine->writes();
    const uint64_t applied_base = point_op_engine->commits();
    const uint64_t scans_base = scan_engine->scans();
    const uint64_t scan_sum_base = scan_engine->scan_result_sum();

    // Every Nth query is an analytical OP_SCAN with a fixed threshold; the rest are
    // point ops. Lets us prove scans flow through the same ring + batcher to the engine.
    constexpr size_t SCAN_EVERY = 1000;
    constexpr uint32_t SCAN_THRESHOLD = MATRIX_SCAN_COLUMN_SIZE / 2;
    size_t expected_scans = 0;
    for (size_t i = 0; i < TOTAL_QUERIES; ++i) if (i % SCAN_EVERY == 0) ++expected_scans;
    // Column is value[j]=j, so each scan counts (SIZE-1-threshold); sum = scans * that.
    const uint64_t expected_scan_sum =
        static_cast<uint64_t>(expected_scans) * (MATRIX_SCAN_COLUMN_SIZE - 1 - SCAN_THRESHOLD);

    // Simulate query production on a separate thread
    auto pipeline_start = std::chrono::steady_clock::now();
    std::thread producer([&]() {
        for (size_t i = 0; i < TOTAL_QUERIES; ++i) {
            // Stamp ingestion time (steady_clock ns) so the consumer can measure queue residency.
            const uint64_t stamp_ns = static_cast<uint64_t>(
                std::chrono::steady_clock::now().time_since_epoch().count());
            DatabaseQuery q{i, stamp_ns, 0, OP_READ, 8, {0}, {0}};
            if (i % SCAN_EVERY == 0) {
                matrix_set_scan_threshold(q, SCAN_THRESHOLD);       // analytical scan query
                while (!scan_ring->try_emplace(q)) spin_stall();    // -> dedicated scan queue
            } else {
                q.opcode = (i % 2 == 0) ? OP_READ : OP_WRITE;       // point ops
                while (!ring_buffer->try_emplace(q)) spin_stall();  // -> point-op queue
            }
        }
    });

    producer.join();

    // Wait until both queues have drained, then stamp the drain time. Point ops and
    // scans drain on independent threads; the slower of the two bounds the pipeline.
    const size_t expected_pointops = TOTAL_QUERIES - expected_scans;
    while (total_processed.load(std::memory_order_relaxed) < expected_pointops ||
           scans_processed.load(std::memory_order_relaxed) < expected_scans) {
        spin_stall();
    }
    auto pipeline_end = std::chrono::steady_clock::now();
    run_state.store(false);

    if (consumer.joinable()) consumer.join();
    if (scan_consumer.joinable()) scan_consumer.join();

    const double elapsed_s =
        std::chrono::duration<double>(pipeline_end - pipeline_start).count();

    const size_t processed = total_processed.load() + scans_processed.load();
    std::cout << "Processed " << processed << " / " << TOTAL_QUERIES << " queries." << std::endl;
    assert(processed == TOTAL_QUERIES && "pipeline dropped queries");

    // Verify the engine actually dispatched on opcode and committed every mutation.
    // Deltas since the pre-pipeline snapshot isolate the pipeline from the sweep.
    const uint64_t reads = point_op_engine->reads() - reads_base;
    const uint64_t writes = point_op_engine->writes() - writes_base;
    const uint64_t applied = point_op_engine->commits() - applied_base;
    const uint64_t scans = scan_engine->scans() - scans_base;
    const uint64_t scan_sum = scan_engine->scan_result_sum() - scan_sum_base;
    std::cout << "Engine: reads=" << reads << " writes=" << writes
              << " commits=" << applied << " scans=" << scans << std::endl;
    std::cout << "Store checksum: " << point_op_engine->store_checksum()
              << " (must match across CPU and GPU backends)" << std::endl;
    std::cout << "Scan result sum: " << scan_sum
              << " (oracle " << expected_scan_sum << ")" << std::endl;
    assert(reads + writes + scans == processed && "opcode dispatch missed queries");
    assert(applied == writes && "page-ownership commit dropped mutations");
    assert(scans == expected_scans && "scan queries dropped in pipeline");
    assert(scan_sum == expected_scan_sum && "GPU scan result mismatch vs oracle");

    // Report throughput split by workload. Point ops and scans now run on independent
    // threads/queues (HTAP separation), so point-op throughput is measured over the full
    // pipeline wall — it no longer pays scan time, which is the whole point of the split.
    const double scan_s = scan_engine->scan_time_s();
    const uint64_t point_ops = reads + writes;
    std::cout << "Throughput: point-ops " << static_cast<uint64_t>(point_ops / elapsed_s)
              << " ops/sec (" << point_ops << " in " << elapsed_s * 1e3 << " ms wall)";
    if (scans > 0) {
        std::cout << " | scans " << static_cast<uint64_t>(scans / scan_s)
                  << " scans/sec (" << scans << " in " << scan_s * 1e3 << " ms, "
                  << scan_s / scans * 1e3 << " ms/scan)";
    }
    std::cout << std::endl;

    // Attribute the scan cost: pure kernel time (cudaEvents) vs per-scan overhead
    // (memset + launch + result copy + sync + host jitter). The split tells us whether
    // closing the integrated-vs-standalone gap means batching scans or is already tight.
    if (scans > 0) {
        const double k_s = scan_engine->scan_kernel_time_s();
        const double overhead_s = scan_s - k_s;
        const double col_gb = (MATRIX_SCAN_COLUMN_SIZE * sizeof(uint32_t)) / 1e9;
        std::cout << "Scan breakdown: kernel " << k_s / scans * 1e3 << " ms/scan ("
                  << col_gb * scans / k_s << " GB/s) + overhead "
                  << overhead_s / scans * 1e3 << " ms/scan ("
                  << static_cast<uint64_t>(overhead_s / scan_s * 100) << "% of scan time)"
                  << std::endl;
    }

    // Queue residency under burst: time each query waits in the ring before it is
    // batched. Dominated by backlog (producer outruns the single consumer), so this
    // is NOT the raw handoff cost reported above — it is the saturated-pipeline view.
    if (!latencies_ns.empty()) {
        std::sort(latencies_ns.begin(), latencies_ns.end());
        const auto pct = [&](double p) {
            const size_t idx = static_cast<size_t>(p * (latencies_ns.size() - 1));
            return latencies_ns[idx];
        };
        std::cout << "Queue residency (ns)     "
                  << "p50=" << pct(0.50) << "  "
                  << "p99=" << pct(0.99) << "  "
                  << "p99.9=" << pct(0.999) << "  "
                  << "max=" << latencies_ns.back() << std::endl;
    }

    analytical_query_demo();
    std::cout << "Engine execution loop completed successfully." << std::endl;
    return 0;
}


In [ ]:
%%writefile test_scan_coverage.cpp
// Coverage test for the items-per-thread scan kernel's index arithmetic.
// The kernel's index math is pure integer logic — hardware-independent — so we can
// simulate it on the CPU and verify every index in [0,n) is visited exactly once.
// This is the test that WOULD have caught the GPU-only undercount bug.
//
// Build: clang++ -std=c++17 -O2 test_scan_coverage.cpp -o /tmp/tcov && /tmp/tcov
#include <cstdio>
#include <cstdint>
#include <cstdlib>
#include <vector>

// Simulate the kernel's visited-index set for a given base-init convention.
// blocked_init=true reproduces the BUG (base = blockIdx*blockDim*ITEMS + tid);
// blocked_init=false is the FIX (base = blockIdx*blockDim + tid).
static bool coverage_ok(size_t n, int GRID, int TPB, int ITEMS, bool blocked_init) {
    std::vector<int> visit(n, 0);
    const size_t base_stride = (size_t)GRID * TPB;
    const size_t chunk = base_stride * ITEMS;
    for (int b = 0; b < GRID; ++b) {
        for (int x = 0; x < TPB; ++x) {
            size_t base = blocked_init
                ? (size_t)b * TPB * ITEMS + x
                : (size_t)b * TPB + x;
            for (; base < n; base += chunk) {
                for (int k = 0; k < ITEMS; ++k) {
                    const size_t idx = base + (size_t)k * base_stride;
                    if (idx < n) visit[idx]++;
                }
            }
        }
    }
    size_t missed = 0, dup = 0;
    for (size_t i = 0; i < n; ++i) {
        if (visit[i] == 0) ++missed;
        else if (visit[i] > 1) ++dup;
    }
    printf("  n=%zu init=%s  missed=%zu dup=%zu  -> %s\n",
           n, blocked_init ? "blocked(BUG)" : "striped(FIX)", missed, dup,
           (missed == 0 && dup == 0) ? "OK" : "BROKEN");
    return missed == 0 && dup == 0;
}

int main() {
    const int GRID = 1024, TPB = 256, ITEMS = 8; // exactly what the engine launches
    const size_t sizes[] = {65536, 262144, 1048576, 4194304, 67108864};

    printf("Buggy (blocked init) — expect BROKEN:\n");
    bool any_bug_broken = false;
    for (size_t n : sizes) any_bug_broken |= !coverage_ok(n, GRID, TPB, ITEMS, true);

    printf("Fixed (striped init) — expect OK:\n");
    bool all_fix_ok = true;
    for (size_t n : sizes) all_fix_ok &= coverage_ok(n, GRID, TPB, ITEMS, false);

    if (!any_bug_broken) { printf("FAIL: bug did not reproduce\n"); return 1; }
    if (!all_fix_ok)     { printf("FAIL: fix does not give exact coverage\n"); return 1; }
    printf("PASS: bug reproduced, fix gives exact coverage\n");
    return 0;
}


In [ ]:
%%writefile test_cost_model.cpp
// CPU-only unit test for the cost model + placement. No GPU, no engines.
// Build: clang++ -std=c++20 -O2 test_cost_model.cpp -o /tmp/tcm && /tmp/tcm
#include "cost_model.hpp"
#include "memory_model.hpp"
#include <cstdio>
#include <cassert>
#include "router.hpp"
#include "compute.hpp"
#include <vector>
#include "tier_model.hpp"

int main() {
    const MemoryModel discrete = MemoryModel::detect(/*gpu_available=*/true);
    CostModel cm(discrete);

    // 1. Point ops NEVER go to DEVICE (measured: PCIe < CPU cache).
    assert(cm.place_point() == MemorySpace::HOST && "point op must place HOST");

    // 2. A tiny scan stays HOST (GPU launch floor dominates below the crossover).
    assert(cm.place_scan(1024) == MemorySpace::HOST && "1KB scan must place HOST");

    // 3. A large scan goes DEVICE (GPU bandwidth wins far above the crossover).
    assert(cm.place_scan(64u * 1024 * 1024) == MemorySpace::DEVICE && "64MB scan must place DEVICE");

    // 4. The crossover is monotonic: there is a single size boundary, HOST below it,
    //    DEVICE at/above it (no oscillation).
    bool seen_device = false;
    MemorySpace prev = MemorySpace::HOST;
    for (size_t b = 256; b <= (1u << 27); b *= 2) {
        MemorySpace s = cm.place_scan(b);
        if (s == MemorySpace::DEVICE) seen_device = true;
        // once DEVICE, never back to HOST as size grows
        assert(!(prev == MemorySpace::DEVICE && s == MemorySpace::HOST) && "crossover not monotonic");
        prev = s;
    }
    assert(seen_device && "some large size must reach DEVICE");

    // 5. No-GPU machine: everything is HOST.
    CostModel cm_nogpu(discrete, /*gpu_available=*/false);
    assert(cm_nogpu.place_scan(64u * 1024 * 1024) == MemorySpace::HOST && "no-GPU must place HOST");

    // --- Router routing test with a fake engine that records what it received ---
    struct FakeEngine : ComputeInterface {
        uint64_t batch_calls = 0, scan_calls = 0, last_batch_n = 0;
        void execute_batch(DatabaseQuery*, size_t n) override { ++batch_calls; last_batch_n = n; }
        void execute_scan(DatabaseQuery& q) override { ++scan_calls; q.transaction_id = 42; }
        uint64_t reads() const override { return 0; }
        uint64_t writes() const override { return 0; }
        uint64_t commits() const override { return 0; }
        uint64_t scans() const override { return scan_calls; }
        uint64_t scan_result_sum() const override { return 0; }
        double scan_time_s() const override { return 0; }
        double scan_kernel_time_s() const override { return 0; }
        uint64_t store_checksum() const override { return 0; }
        double benchmark_scan(size_t, uint64_t, uint64_t&) override { return 0; }
        double benchmark_scan_u32(size_t, uint32_t, uint64_t&) override { return 0; }
        double benchmark_scan_u32x4(size_t, uint32_t, uint64_t&) override { return 0; }
        double benchmark_scan_ipt(size_t, uint32_t, uint64_t&) override { return 0; }
    };

    FakeEngine cpu_eng, gpu_eng;
    Router router(&cpu_eng, &gpu_eng, CostModel(discrete));

    // Point-op batch always routes to CPU.
    std::vector<DatabaseQuery> pts(4);
    for (auto& q : pts) q.opcode = OP_READ;
    router.route_batch(pts.data(), pts.size());
    assert(cpu_eng.batch_calls == 1 && gpu_eng.batch_calls == 0 && "point batch must hit CPU");

    // A scan over a DEVICE-placed column routes to GPU.
    const uint64_t big_id = router.place_scan_column(64u * 1024 * 1024); // returns a dataset id
    DatabaseQuery sbig{}; matrix_set_scan_threshold(sbig, 1);
    router.route_scan(sbig, big_id);
    assert(gpu_eng.scan_calls == 1 && cpu_eng.scan_calls == 0 && "big scan must hit GPU");

    // A scan over a HOST-placed column routes to CPU.
    const uint64_t small_id = router.place_scan_column(1024);
    DatabaseQuery ssmall{}; matrix_set_scan_threshold(ssmall, 1);
    router.route_scan(ssmall, small_id);
    assert(cpu_eng.scan_calls == 1 && "small scan must hit CPU");

    // --- Tier-aware cost: scan time per tier, and migration cost ---
    {
        CostModel tcm(discrete);
        const size_t bytes = 64u * 1024 * 1024; // 64 MB

        // Per-tier scan time ranks by bandwidth: DEVICE (VRAM) fastest, COLD (SSD) slowest.
        const double dev = tcm.scan_us(MemorySpace::DEVICE, bytes);
        const double host = tcm.scan_us(MemorySpace::HOST, bytes);
        const double cold = tcm.scan_us(MemorySpace::COLD, bytes);
        assert(dev < host && host < cold && "scan time must rank DEVICE < HOST < COLD");

        // Migration cost is non-negative and zero when source==dest (no move).
        assert(tcm.migration_us(MemorySpace::HOST, MemorySpace::HOST, bytes) == 0.0
               && "no-op migration costs 0");
        assert(tcm.migration_us(MemorySpace::HOST, MemorySpace::DEVICE, bytes) > 0.0
               && "real migration costs > 0");
    }

    std::printf("PASS: cost model placement decisions correct\n");
    return 0;
}


In [ ]:
%%writefile test_kv_store.cpp
// CPU unit test for KVStore: the DM-1 fix. Proves distinct keys never overwrite each
// other (the silent-data-loss bug), get/put round-trips, and a full table is an explicit
// error, not corruption.
// Build: clang++ -std=c++20 -O2 test_kv_store.cpp -o /tmp/tkv && /tmp/tkv
#include "kv_store.hpp"
#include <cstdio>
#include <cassert>

int main() {
    KVStore kv(1024); // capacity 1024 slots

    // 1. put/get round-trip.
    kv.put(42, 100);
    uint64_t v = 0;
    assert(kv.get(42, v) && v == 100 && "get must return the put value");

    // 2. Missing key returns false.
    assert(!kv.get(999, v) && "absent key must report miss");

    // 3. Overwrite same key updates value (not a collision — same key).
    kv.put(42, 200);
    assert(kv.get(42, v) && v == 200 && "same-key put updates value");
    assert(kv.size() == 1 && "same-key put does not grow size");

    // 4. THE DM-1 BUG: two DISTINCT keys that collide under masking must BOTH survive.
    //    With a 1024 table, keys 7 and 7+1024 collide on the initial slot. Old code
    //    (key & MASK) silently overwrote; the hash table must probe and keep both.
    kv.put(7, 70);
    kv.put(7 + 1024, 71);
    uint64_t a = 0, b = 0;
    assert(kv.get(7, a) && a == 70 && "colliding key 7 must survive");
    assert(kv.get(7 + 1024, b) && b == 71 && "colliding key 7+1024 must survive");
    assert(kv.size() == 3 && "two distinct colliding keys are two entries");

    // 5. Full table is an explicit error (return false), never silent loss.
    KVStore small(2); // 2 slots
    assert(small.put(1, 1) && "first put fits");
    assert(small.put(2, 2) && "second put fits");
    assert(!small.put(3, 3) && "third put must FAIL (full), not overwrite");
    uint64_t x = 0;
    assert(small.get(1, x) && x == 1 && "existing entries intact after full");
    assert(small.get(2, x) && x == 2 && "existing entries intact after full");

    // 6. checksum is order-independent sum of stored values (used by the engine).
    KVStore c(16);
    c.put(1, 10); c.put(2, 20); c.put(3, 30);
    assert(c.checksum() == 60 && "checksum is sum of values");

    std::printf("PASS: KVStore correctness (no silent overwrite, full=error)\n");
    return 0;
}


In [ ]:
%%writefile test_tier_manager.cpp
// CPU unit test for TierManager (the auto-tiering brain). No GPU, no I/O.
// Build: clang++ -std=c++20 -O2 test_tier_manager.cpp -o /tmp/ttm && /tmp/ttm
#include "tier_manager.hpp"
#include "cost_model.hpp"
#include "memory_model.hpp"
#include <cstdio>
#include <cassert>
#include <vector>

static CostModel make_cm() { return CostModel(MemoryModel::detect(true), true); }

int main() {
    // --- Task 1: registration & placement ---
    {
        TierManager tm(make_cm(), /*device_cap*/ 1u<<30, /*host_cap*/ 1u<<30);
        tm.register_column(1, 1024, MemorySpace::COLD);
        tm.register_column(2, 2048, MemorySpace::HOST);
        assert(tm.tier_of(1) == MemorySpace::COLD && "registered tier is reported");
        assert(tm.tier_of(2) == MemorySpace::HOST && "registered tier is reported");
        assert(tm.resident_bytes(MemorySpace::COLD) == 1024 && "COLD resident bytes");
        assert(tm.resident_bytes(MemorySpace::HOST) == 2048 && "HOST resident bytes");
        assert(tm.heat_of(1) == 0.0 && "fresh column starts cold (heat 0)");
    }

    // --- Task 2: heat tracking ---
    {
        TierManager tm(make_cm(), 1u<<30, 1u<<30);
        tm.register_column(1, 1000, MemorySpace::HOST);

        // Access accumulates; heat only updates on rebalance (EWMA aging).
        tm.record_access(1, 1000);
        tm.record_access(1, 1000);
        assert(tm.heat_of(1) == 0.0 && "heat unchanged until rebalance");

        tm.rebalance(); // heat = 0.5*recent + 0.5*old = 0.5*2000 + 0 = 1000
        assert(tm.heat_of(1) == 1000.0 && "first rebalance sets heat to alpha*recent");

        // No access this round: heat decays toward 0.
        tm.rebalance(); // heat = 0.5*0 + 0.5*1000 = 500
        assert(tm.heat_of(1) == 500.0 && "idle column heat decays");
    }

    // --- Task 3: cost-benefit promotion ---
    {
        TierManager tm(make_cm(), 1u<<30, 1u<<30); // ample capacity: no eviction pressure
        // A large, hot column on COLD should be promoted toward HOST then DEVICE.
        tm.register_column(1, 64u*1024*1024, MemorySpace::COLD);
        for (int i = 0; i < 50; ++i) tm.record_access(1, 64u*1024*1024);

        auto d1 = tm.rebalance();
        assert(tm.tier_of(1) == MemorySpace::HOST && "hot COLD column promotes to HOST");
        bool found = false;
        for (auto& d : d1) if (d.column_id==1 && d.from==MemorySpace::COLD && d.to==MemorySpace::HOST) found = true;
        assert(found && "promotion emitted as a decision");

        // Keep it hot; next rebalance climbs HOST -> DEVICE.
        for (int i = 0; i < 50; ++i) tm.record_access(1, 64u*1024*1024);
        tm.rebalance();
        assert(tm.tier_of(1) == MemorySpace::DEVICE && "still-hot column climbs to DEVICE");

        // A cold (rarely accessed) column is NOT promoted.
        TierManager tm2(make_cm(), 1u<<30, 1u<<30);
        tm2.register_column(2, 64u*1024*1024, MemorySpace::COLD);
        tm2.record_access(2, 1); // negligible heat
        tm2.rebalance();
        assert(tm2.tier_of(2) == MemorySpace::COLD && "cold column stays put");
    }

    // --- Task 4: capacity eviction (cost-benefit, not pure LRU) + anti-thrash ---
    {
        const size_t COL = 64u*1024*1024;
        // DEVICE capacity holds ONE column, but BOTH start resident on DEVICE (e.g. loaded
        // hot). Col 1 is kept hot; col 2 goes cold. Eviction must demote the cold one so
        // DEVICE fits its capacity — and keep the hot one (cost-benefit, not pure LRU).
        TierManager tm(make_cm(), /*device_cap*/ COL, /*host_cap*/ 1u<<30);
        tm.register_column(1, COL, MemorySpace::DEVICE); // hot — must stay
        tm.register_column(2, COL, MemorySpace::DEVICE); // cold — must be evicted

        for (int r = 0; r < 3; ++r) {
            for (int i = 0; i < 50; ++i) tm.record_access(1, COL); // col 1 very hot
            // col 2 gets no accesses → cold → lowest keep_score → the eviction victim
            tm.rebalance();
        }
        assert(tm.tier_of(1) == MemorySpace::DEVICE && "hot column kept on scarce DEVICE");
        assert(tm.tier_of(2) != MemorySpace::DEVICE && "cold column evicted from full DEVICE");
        assert(tm.resident_bytes(MemorySpace::DEVICE) <= COL && "DEVICE never over capacity");
    }
    {
        // Anti-thrash: when a tier is OVER capacity but the only over-capacity resident is
        // within MIN_RESIDENCY_TICKS of arriving, eviction is SUPPRESSED this pass (it is
        // not thrashed straight back out). This test genuinely exercises the min-residency
        // guard: DEVICE holds one column's worth, both columns are freshly registered on
        // DEVICE, so the first rebalance (tick 1, arrived_tick 0 → age 1 < 2) must NOT evict
        // — DEVICE stays over capacity for one tick. (It would evict if MIN_RESIDENCY were 0.)
        const size_t COL = 8u*1024*1024;
        TierManager tm(make_cm(), /*device_cap*/ COL, /*host_cap*/ 1u<<30);
        tm.register_column(1, COL, MemorySpace::DEVICE);
        tm.register_column(2, COL, MemorySpace::DEVICE);
        auto d = tm.rebalance(); // tick 1: both age 1 < MIN_RESIDENCY_TICKS(2) → no eviction
        assert(d.empty() && "no eviction within min-residency, even when over capacity");
        assert(tm.resident_bytes(MemorySpace::DEVICE) == 2*COL && "still over cap this tick");
        tm.rebalance(); // tick 2: now age 2 >= 2 → eviction permitted, DEVICE fits
        assert(tm.resident_bytes(MemorySpace::DEVICE) <= COL && "evicted once past min-residency");
    }

    // --- Capacity-gated promotion: the plan must never over-subscribe a bounded tier ---
    {
        // DEVICE holds ONE column; THREE equally-hot columns all qualify to promote.
        // Promotion must be capacity-gated so the brain never emits an infeasible plan
        // (more bytes on DEVICE than it holds). Without gating, all three promote and
        // DEVICE sits multi-x over capacity in steady state.
        const size_t COL = 64u*1024*1024;
        TierManager tm(make_cm(), /*device_cap*/ COL, /*host_cap*/ 1u<<30);
        tm.register_column(1, COL, MemorySpace::COLD);
        tm.register_column(2, COL, MemorySpace::COLD);
        tm.register_column(3, COL, MemorySpace::COLD);
        for (int r = 0; r < 8; ++r) {
            for (int i = 0; i < 50; ++i) {
                tm.record_access(1, COL); tm.record_access(2, COL); tm.record_access(3, COL);
            }
            tm.rebalance();
            // The hard invariant the eviction/gating logic exists to guarantee:
            assert(tm.resident_bytes(MemorySpace::DEVICE) <= COL
                   && "DEVICE plan must never exceed capacity (promotion is capacity-gated)");
            assert(tm.resident_bytes(MemorySpace::HOST) <= (1u<<30)
                   && "HOST plan must never exceed capacity");
        }
        // Exactly one of the three should occupy DEVICE; the rest sit on HOST.
        int on_device = 0;
        for (uint64_t id : {1u, 2u, 3u}) if (tm.tier_of(id) == MemorySpace::DEVICE) ++on_device;
        assert(on_device == 1 && "exactly one column fits the 1-column DEVICE");
    }

    // --- Swap-on-promote: a re-hot lower-tier column displaces a COLDER resident of a full tier ---
    {
        const size_t COL = 64u*1024*1024;
        // HOST holds exactly 2 columns; DEVICE inert (cap 1 => nothing fits, like the CPU engine).
        TierManager tm(make_cm(), /*device_cap*/ 1, /*host_cap*/ 2*COL);
        tm.register_column(1, COL, MemorySpace::HOST); // kept hot -> stays
        tm.register_column(2, COL, MemorySpace::HOST); // goes idle -> the victim
        tm.register_column(3, COL, MemorySpace::COLD); // re-hot candidate -> must swap in
        for (int r = 0; r < 4; ++r) {
            for (int i = 0; i < 50; ++i) { tm.record_access(1, COL); tm.record_access(3, COL); }
            // col 2 idle: heat decays to ~0 -> lowest keep_score -> the displaced victim
            tm.rebalance();
        }
        assert(tm.tier_of(3) == MemorySpace::HOST && "hot COLD column swapped into the full HOST tier");
        assert(tm.tier_of(2) == MemorySpace::COLD && "idle HOST resident displaced to COLD");
        assert(tm.tier_of(1) == MemorySpace::HOST && "hot resident kept (cost-benefit, not LRU)");
        assert(tm.resident_bytes(MemorySpace::HOST) <= 2*COL && "HOST within budget after the swap");
    }
    // --- Swap margin (anti-thrash): an equally-hot candidate canNOT displace a resident ---
    {
        const size_t COL = 64u*1024*1024;
        TierManager tm(make_cm(), /*device_cap*/ 1, /*host_cap*/ 2*COL);
        tm.register_column(1, COL, MemorySpace::HOST);
        tm.register_column(2, COL, MemorySpace::HOST);
        tm.register_column(3, COL, MemorySpace::COLD);
        for (int r = 0; r < 4; ++r) {
            // all three equally hot -> value(col3) == value(col2) -> not > 1.5x -> no swap
            for (int i = 0; i < 50; ++i) { tm.record_access(1, COL); tm.record_access(2, COL); tm.record_access(3, COL); }
            tm.rebalance();
        }
        assert(tm.tier_of(3) == MemorySpace::COLD && "equally-hot candidate cannot clear the SWAP_MARGIN");
        assert(tm.resident_bytes(MemorySpace::HOST) <= 2*COL && "HOST within budget");
    }
    // --- Task 5: determinism — identical access sequence -> identical decisions ---
    {
        auto run = []() {
            TierManager tm(make_cm(), 64u*1024*1024, 1u<<30);
            tm.register_column(1, 32u*1024*1024, MemorySpace::COLD);
            tm.register_column(2, 32u*1024*1024, MemorySpace::COLD);
            std::vector<MigrationDecision> all;
            for (int r = 0; r < 4; ++r) {
                for (int i = 0; i < 20; ++i) { tm.record_access(1, 32u*1024*1024); tm.record_access(2, 16u*1024*1024); }
                auto d = tm.rebalance();
                for (auto& x : d) all.push_back(x);
            }
            return all;
        };
        auto a = run();
        auto b = run();
        assert(a.size() == b.size() && "deterministic decision count");
        for (size_t i = 0; i < a.size(); ++i) {
            assert(a[i].column_id == b[i].column_id && a[i].from == b[i].from && a[i].to == b[i].to
                   && "deterministic decision sequence");
        }
    }

    std::printf("PASS: tier manager decisions correct\n");
    return 0;
}


In [ ]:
%%writefile test_cold_store.cpp
// CPU unit test for ColdStore (the SSD WAL). Uses real temp files.
// Build: clang++ -std=c++20 -O2 test_cold_store.cpp -o /tmp/tcs && /tmp/tcs
#include "cold_store.hpp"
#include <cstdio>
#include <cassert>
#include <vector>
#include <string>
#include <unistd.h>    // ::truncate

static const char* PATH = "/tmp/matrixdb_wal_test.log";

int main() {
    // --- Task 1: append + replay round-trip, last-writer-wins ---
    {
        std::remove(PATH); // fresh
        {
            ColdStore cs(PATH, SyncPolicy::SYNC_EACH);
            cs.append_put(10, 100);
            cs.append_put(20, 200);
            cs.append_put(10, 101); // overwrite key 10 (later record wins on replay)
            assert(cs.records_written() == 3 && "three appends counted");
        }
        // Replay (a fresh ColdStore on the same path).
        ColdStore cs(PATH);
        std::vector<std::pair<uint64_t,uint64_t>> got;
        cs.replay([&](uint64_t k, uint64_t v){ got.push_back({k, v}); });
        assert(got.size() == 3 && "replay yields all three records in order");
        assert(got[0].first == 10 && got[0].second == 100);
        assert(got[1].first == 20 && got[1].second == 200);
        assert(got[2].first == 10 && got[2].second == 101 && "last write for key 10 replays last");
        std::remove(PATH);
    }

    // --- Task 2a: survives restart (write with one ColdStore, read with a fresh one) ---
    {
        std::remove(PATH);
        { ColdStore w(PATH); w.append_put(7, 70); w.append_put(8, 80); } // closed = "restart"
        ColdStore r(PATH);
        uint64_t sum = 0; int n = 0;
        r.replay([&](uint64_t, uint64_t v){ sum += v; ++n; });
        assert(n == 2 && sum == 150 && "data persists across ColdStore instances (restart)");
        std::remove(PATH);
    }

    // --- Task 2b: torn tail — truncating mid-last-record drops only that record ---
    {
        std::remove(PATH);
        { ColdStore w(PATH); w.append_put(1, 11); w.append_put(2, 22); w.append_put(3, 33); }
        // Each record on disk = 4 (len) + 4 (crc) + 20 (payload) = 28 bytes. Truncate to
        // 2 full records + a partial third (cut the third's payload).
        FILE* f = std::fopen(PATH, "rb");
        std::fseek(f, 0, SEEK_END); long size = std::ftell(f); std::fclose(f);
        assert(size == 3*28 && "three 28-byte records");
        if (::truncate(PATH, 2*28 + 10) != 0) std::perror("truncate"); // 2 whole records + 10 bytes of the third (torn)
        ColdStore r(PATH);
        int n = 0; uint64_t last = 0;
        r.replay([&](uint64_t k, uint64_t){ ++n; last = k; });
        assert(n == 2 && last == 2 && "torn third record dropped; first two recovered");
        std::remove(PATH);
    }

    // --- Task 2c: CRC corruption — a flipped payload byte stops replay at that record ---
    {
        std::remove(PATH);
        { ColdStore w(PATH); w.append_put(5, 55); w.append_put(6, 66); }
        // Flip a byte inside the FIRST record's payload (offset 8 = start of payload).
        FILE* f = std::fopen(PATH, "rb+");
        std::fseek(f, 8, SEEK_SET);            // first record's payload byte 0
        unsigned char b = 0; if (std::fread(&b, 1, 1, f) != 1) std::perror("fread");
        b ^= 0xFF;
        std::fseek(f, 8, SEEK_SET); std::fwrite(&b, 1, 1, f);
        std::fclose(f);
        ColdStore r(PATH);
        int n = 0;
        r.replay([&](uint64_t, uint64_t){ ++n; });
        assert(n == 0 && "corruption in the first record stops replay immediately");
        std::remove(PATH);
    }

    // --- Task 2d: empty / missing file replays nothing, no error ---
    {
        std::remove(PATH); // ensure missing
        ColdStore r(PATH); // "ab" creates an empty file
        int n = 0;
        r.replay([&](uint64_t, uint64_t){ ++n; });
        assert(n == 0 && "empty/missing log replays nothing");
        std::remove(PATH);
    }

    std::printf("PASS: cold store WAL correct\n");
    return 0;
}


In [ ]:
%%writefile test_engine_restart.cpp
// End-to-end durability: point-op writes through one CPUMockEngine survive into a fresh
// engine constructed on the same WAL path (simulates process restart).
// Build: clang++ -std=c++20 -O2 test_engine_restart.cpp -o /tmp/ter && /tmp/ter
#include "compute_mock.cpp"   // pulls in CPUMockEngine (used header-style, as in main.cpp)
#include <cstdio>
#include <cassert>
#include <vector>

static const char* WAL = "/tmp/matrixdb_engine_restart.log";

// Build a batch of OP_WRITE queries for keys [1..n].
static std::vector<DatabaseQuery> write_batch(uint64_t n) {
    std::vector<DatabaseQuery> b(n);
    for (uint64_t i = 0; i < n; ++i) {
        b[i] = DatabaseQuery{};
        b[i].query_id = i + 1;       // keys 1..n; value == key (mock projection)
        b[i].opcode = OP_WRITE;
    }
    return b;
}

int main() {
    std::remove(WAL);

    uint64_t checksum_before = 0;
    {
        CPUMockEngine eng(4, WAL);          // durability ON
        auto batch = write_batch(100);
        eng.execute_batch(batch.data(), batch.size());
        checksum_before = eng.store_checksum();
        assert(checksum_before == 5050 && "sum of values 1..100 (value==key)");
    } // engine destroyed = "process exit"

    {
        CPUMockEngine eng2(4, WAL);         // fresh engine, same WAL → recovery on construct
        const uint64_t checksum_after = eng2.store_checksum();
        assert(checksum_after == checksum_before
               && "writes survived restart: recovered store matches pre-restart");
    }

    std::remove(WAL);
    std::printf("PASS: engine survives restart (WAL recovery)\n");
    return 0;
}


In [ ]:
%%writefile test_migration.cpp
// CPU unit test for cross-tier migration (HOST<->COLD; DEVICE is Colab-verified).
// Build: clang++ -std=c++20 -O2 test_migration.cpp -o /tmp/tmig && /tmp/tmig
#include "tiered_column.hpp"
#include "migration_executor.hpp"
#include "tier_manager.hpp"
#include "cost_model.hpp"
#include <unordered_map>
#include <cstdio>
#include <cassert>
#include <vector>
#include <numeric>

int main() {
    // --- Task 1: HOST<->COLD round-trip + integrity ---
    {
        std::vector<unsigned char> data(4096);
        for (size_t i = 0; i < data.size(); ++i) data[i] = static_cast<unsigned char>(i * 7 + 1);
        TieredColumn col(1, data.data(), data.size());
        const uint64_t want = col.checksum();
        assert(col.tier() == MemorySpace::HOST && "born in HOST");
        assert(col.size_bytes() == 4096);

        col.migrate_to(MemorySpace::COLD);
        assert(col.tier() == MemorySpace::COLD && "moved to COLD");
        assert(col.checksum() == want && "checksum invariant HOST->COLD");

        col.migrate_to(MemorySpace::HOST);
        assert(col.tier() == MemorySpace::HOST && "moved back to HOST");
        assert(col.checksum() == want && "checksum invariant COLD->HOST");

        // Integrity across a chain.
        col.migrate_to(MemorySpace::COLD);
        col.migrate_to(MemorySpace::HOST);
        col.migrate_to(MemorySpace::COLD);
        assert(col.checksum() == want && "checksum invariant across a HOST/COLD chain");
        col.migrate_to(MemorySpace::HOST); // leave on HOST so dtor frees the vector (no temp file left)
    }

    // --- Task 2: TierManager decisions actually move columns (the auto-tiering loop) ---
    {
        // Hand-built plan (decision-driven loop; TierManager produces exactly this shape).
        const size_t N = 4096;
        std::vector<unsigned char> bytes(N, 0xAB);
        TieredColumn hot(1, bytes.data(), N);   // stays
        TieredColumn cold(2, bytes.data(), N);  // will be demoted to COLD
        const uint64_t hot_sum = hot.checksum(), cold_sum = cold.checksum();

        std::unordered_map<uint64_t, TieredColumn*> columns{{1, &hot}, {2, &cold}};

        std::vector<MigrationDecision> plan{
            MigrationDecision{2, MemorySpace::HOST, MemorySpace::COLD},
            MigrationDecision{99, MemorySpace::HOST, MemorySpace::COLD}, // absent id -> skipped
        };

        MigrationExecutor exec;
        const size_t applied = exec.apply(plan, columns);
        assert(applied == 1 && "one valid decision applied, absent id skipped");
        assert(cold.tier() == MemorySpace::COLD && "cold column physically demoted");
        assert(hot.tier() == MemorySpace::HOST && "untouched column stays");
        assert(cold.checksum() == cold_sum && "demoted column bytes intact");
        assert(hot.checksum() == hot_sum && "untouched column bytes intact");
        cold.migrate_to(MemorySpace::HOST); // cleanup: leave on HOST so no temp file remains
    }

    // --- Task 2b: a real TierManager rebalance feeds the executor ---
    {
        const size_t N = 4096;
        std::vector<unsigned char> bytes(N, 0x5C);
        TieredColumn a(10, bytes.data(), N);
        TieredColumn b(11, bytes.data(), N);
        std::unordered_map<uint64_t, TieredColumn*> columns{{10, &a}, {11, &b}};

        // Brain: both registered on HOST; HOST cap holds ONE column, so the colder is evicted.
        TierManager tm(CostModel(MemoryModel::detect(true), true),
                       /*device_cap*/ 1u<<30, /*host_cap*/ N);
        tm.register_column(10, N, MemorySpace::HOST);
        tm.register_column(11, N, MemorySpace::HOST);
        for (int r = 0; r < 3; ++r) {
            for (int i = 0; i < 50; ++i) tm.record_access(10, N); // col 10 hot
            auto plan = tm.rebalance();
            MigrationExecutor exec;
            exec.apply(plan, columns);
        }
        assert(a.tier() == tm.tier_of(10) && "executor synced column 10 to brain's tier");
        assert(b.tier() == tm.tier_of(11) && "executor synced column 11 to brain's tier");
        assert(a.checksum() == b.checksum() && "both columns' bytes intact (same fill)");
        a.migrate_to(MemorySpace::HOST); b.migrate_to(MemorySpace::HOST); // cleanup temp files
    }

    std::printf("PASS: migration correct\n");
    return 0;
}


In [ ]:
%%writefile test_live_tiering.cpp
// CPU test for the live tiering integration. Grows across tasks; one main() runs all.
#include "compute_mock.cpp"   // CPUMockEngine + compute.hpp (codec) + tiering headers
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>
#include "tiered_column.hpp"

static void test_codec() {
    DatabaseQuery q{};
    matrix_set_scan_target(q, 42u, 7ull);
    assert(q.opcode == OP_SCAN);
    assert(matrix_get_scan_threshold(q) == 42u);
    assert(matrix_get_scan_column_id(q) == 7ull);

    DatabaseQuery q2{};
    matrix_set_scan_threshold(q2, 99u);              // legacy delegates to target(...,0)
    assert(matrix_get_scan_threshold(q2) == 99u);
    assert(matrix_get_scan_column_id(q2) == 0ull);   // legacy id is explicitly 0
    std::cout << "[codec] ok\n";
}

static void test_host_ptr() {
    std::vector<uint32_t> data(4);
    for (uint32_t i = 0; i < 4; ++i) data[i] = i * 10;          // 0,10,20,30
    TieredColumn col(101, reinterpret_cast<const unsigned char*>(data.data()),
                     data.size() * sizeof(uint32_t));
    assert(col.tier() == MemorySpace::HOST);
    const uint32_t* v = reinterpret_cast<const uint32_t*>(col.host_ptr());
    assert(v != nullptr && v[0] == 0 && v[1] == 10 && v[2] == 20 && v[3] == 30);

    const uint64_t cks = col.checksum();
    col.migrate_to(MemorySpace::COLD);
    assert(col.host_ptr() == nullptr);                          // bytes on SSD now
    col.migrate_to(MemorySpace::HOST);
    const uint32_t* v2 = reinterpret_cast<const uint32_t*>(col.host_ptr());
    assert(v2 != nullptr && v2[0] == 0 && v2[3] == 30);
    assert(col.checksum() == cks);                              // round-trip integrity
    std::cout << "[host_ptr] ok\n";
}

static void test_cold_path_uniqueness() {
    // Two columns with the SAME id must NOT share a COLD file (else one's demote silently
    // clobbers the other's bytes). Distinct values; round-trip both through COLD independently.
    std::vector<uint32_t> a(4), b(4);
    for (uint32_t i = 0; i < 4; ++i) { a[i] = i; b[i] = 100 + i; }
    TieredColumn ca(7, reinterpret_cast<const unsigned char*>(a.data()), a.size() * sizeof(uint32_t));
    TieredColumn cb(7, reinterpret_cast<const unsigned char*>(b.data()), b.size() * sizeof(uint32_t)); // same id 7
    const uint64_t cka = ca.checksum(), ckb = cb.checksum();
    assert(cka != ckb);
    ca.migrate_to(MemorySpace::COLD);
    cb.migrate_to(MemorySpace::COLD);             // shared path would overwrite ca's file here
    ca.migrate_to(MemorySpace::HOST);
    cb.migrate_to(MemorySpace::HOST);
    assert(ca.checksum() == cka && "column A intact — not clobbered by same-id column B");
    assert(cb.checksum() == ckb && "column B intact");
    std::cout << "[cold-path uniqueness] ok\n";
}

static void test_tiered_single_column() {
    const size_t N = 1000;
    std::vector<uint32_t> col(N);
    for (size_t i = 0; i < N; ++i) col[i] = static_cast<uint32_t>(i); // value[i]=i
    CPUMockEngine eng(0, "", /*host_cap=*/SIZE_MAX);                   // generous: no eviction
    eng.load_scan_column(1, col.data(), N);

    DatabaseQuery q{};
    const uint32_t T = 500;
    matrix_set_scan_target(q, T, 1);
    eng.execute_scan(q);
    assert(q.transaction_id == N - 1 - T);                            // count of i>T == 499
    assert(eng.column_tier(1) == MemorySpace::HOST);                  // fits budget, stays resident

    // legacy id==0 path still scans the fixed column correctly
    DatabaseQuery ql{};
    const uint32_t TL = 1000;
    matrix_set_scan_threshold(ql, TL);
    eng.execute_scan(ql);
    assert(ql.transaction_id == MATRIX_SCAN_COLUMN_SIZE - 1 - TL);
    std::cout << "[tiered single-column + legacy] ok\n";
}

static void test_eviction_holds_more_than_ram() {
    const size_t N = 1000;
    const size_t S = N * sizeof(uint32_t);            // 4000 bytes per column
    std::vector<uint32_t> col(N);
    for (size_t i = 0; i < N; ++i) col[i] = static_cast<uint32_t>(i);

    CPUMockEngine eng(0, "", /*host_cap=*/2 * S);      // room for 2 of 3 columns
    eng.load_scan_column(1, col.data(), N);
    eng.load_scan_column(2, col.data(), N);
    eng.load_scan_column(3, col.data(), N);            // 3*S > 2*S: more than fits in RAM

    // Hot/cold: scan cols 1 & 2 each round, NEVER col 3 -> col 3 heat stays 0 (the victim).
    const uint32_t T = 250;
    for (int round = 0; round < 8; ++round) {
        for (uint64_t id : {1ull, 2ull}) {
            DatabaseQuery q{};
            matrix_set_scan_target(q, T, id);
            eng.execute_scan(q);
            assert(q.transaction_id == N - 1 - T);     // hot scans stay correct (749)
        }
    }
    // 16 tiered scans -> 4 rebalances (REBALANCE_EVERY=4). MIN_RESIDENCY_TICKS=2 means the
    // 1st rebalance evicts nothing; col 3 demotes on the 2nd. Final state is deterministic:
    assert(eng.manager_tier(3) == MemorySpace::COLD);  // brain demoted the cold column
    assert(eng.column_tier(3)  == MemorySpace::COLD);  // and the bytes are actually on SSD
    assert(eng.manager_tier(1) == MemorySpace::HOST);  // hot columns retained
    assert(eng.manager_tier(2) == MemorySpace::HOST);
    assert(eng.host_resident_bytes() <= 2 * S);        // budget respected
    static_assert(3 * (1000 * sizeof(uint32_t)) > 2 * (1000 * sizeof(uint32_t)),
                  "catalog holds more bytes than the RAM budget");

    // Scan the COLD column: borrowed to HOST, scanned, returned to COLD; result still correct.
    const uint64_t cks_before = eng.column_checksum(3);
    DatabaseQuery q3{};
    matrix_set_scan_target(q3, T, 3);
    eng.execute_scan(q3);
    assert(q3.transaction_id == N - 1 - T);            // pull-back-correct (749)
    assert(eng.column_tier(3) == MemorySpace::COLD);   // borrow returned: rest tier == brain
    assert(eng.column_checksum(3) == cks_before);      // demote+borrow preserved the bytes
    std::cout << "[eviction holds-more-than-RAM + borrow] ok\n";
}

static void test_repromotion_under_pressure() {
    const size_t N = 1000;
    const size_t S = N * sizeof(uint32_t);
    std::vector<uint32_t> col(N);
    for (size_t i = 0; i < N; ++i) col[i] = static_cast<uint32_t>(i);

    CPUMockEngine eng(0, "", /*host_cap=*/2 * S);   // room for 2 of 3 columns
    eng.load_scan_column(1, col.data(), N);
    eng.load_scan_column(2, col.data(), N);
    eng.load_scan_column(3, col.data(), N);

    const uint32_t T = 250;
    auto scan = [&](uint64_t id) {
        DatabaseQuery q{};
        matrix_set_scan_target(q, T, id);
        eng.execute_scan(q);
        assert(q.transaction_id == N - 1 - T);       // 749, regardless of tier
    };

    // Phase 1: cols 1 & 2 hot, col 3 never -> col 3 demoted to COLD (the INT-1 baseline).
    for (int r = 0; r < 8; ++r) { scan(1); scan(2); }
    assert(eng.column_tier(3) == MemorySpace::COLD && "phase 1: col 3 demoted to SSD");

    // Phase 2: FLIP the heat — col 3 scanned 3x/round (decisively hot, clears the COLD->HOST
    // promotion gate), col 1 once, col 2 NEVER. Col 2 cools to keep_score 0; col 3 re-heats and
    // is RE-PROMOTED to resident HOST (swap-on-promote displaces the now-cold col 2).
    for (int r = 0; r < 16; ++r) { scan(3); scan(1); scan(3); scan(3); }
    assert(eng.column_tier(3)  == MemorySpace::HOST && "col 3 re-promoted to RESIDENT RAM (not just borrowed)");
    assert(eng.manager_tier(3) == MemorySpace::HOST && "brain agrees col 3 is resident");
    assert(eng.column_tier(2)  == MemorySpace::COLD && "col 2 displaced to SSD");
    assert(eng.host_resident_bytes() <= 2 * S       && "HOST within budget");
    std::cout << "[re-promotion under pressure] ok\n";
}

int main() {
    test_codec();
    test_host_ptr();
    test_cold_path_uniqueness();
    test_tiered_single_column();
    test_eviction_holds_more_than_ram();
    test_repromotion_under_pressure();
    std::cout << "ALL LIVE-TIERING TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_aggregations.cpp
// CPU test for analytical aggregations. Grows across tasks; one main() runs all.
#include "compute_mock.cpp"   // CPUMockEngine + compute.hpp (codec + reducer)
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

static void test_reduce_closed_form() {
    const size_t N = 1000;
    std::vector<uint32_t> v(N);
    for (size_t i = 0; i < N; ++i) v[i] = static_cast<uint32_t>(i); // value[i]=i
    const uint32_t T = 600;
    const uint64_t count = N - 1 - T;            // # of i>T in [0,N)
    const uint64_t mn = T + 1;
    const uint64_t mx = N - 1;
    uint64_t sum = 0; for (uint64_t i = T + 1; i <= N - 1; ++i) sum += i;
    assert(matrix_cpu_reduce(v.data(), N, T, AGG_COUNT) == count);
    assert(matrix_cpu_reduce(v.data(), N, T, AGG_SUM)   == sum);
    assert(matrix_cpu_reduce(v.data(), N, T, AGG_MIN)   == mn);
    assert(matrix_cpu_reduce(v.data(), N, T, AGG_MAX)   == mx);
    assert(sum != count); // non-vacuity: a stub returning count for every op would fail here
    std::cout << "[reduce closed-form] ok\n";
}

static void test_reduce_empty() {
    const size_t N = 1000;
    std::vector<uint32_t> v(N);
    for (size_t i = 0; i < N; ++i) v[i] = static_cast<uint32_t>(i);
    const uint32_t T = N - 1; // nothing is > N-1
    assert(matrix_cpu_reduce(v.data(), N, T, AGG_COUNT) == 0);
    assert(matrix_cpu_reduce(v.data(), N, T, AGG_SUM)   == 0);
    assert(matrix_cpu_reduce(v.data(), N, T, AGG_MIN)   == UINT64_MAX);
    assert(matrix_cpu_reduce(v.data(), N, T, AGG_MAX)   == 0);
    std::cout << "[reduce empty-set] ok\n";
}

static void test_agg_codec() {
    DatabaseQuery q{};
    matrix_set_scan_target(q, 50u, 9ull);
    assert(matrix_get_scan_agg_op(q) == AGG_COUNT);  // default 0 when not set
    matrix_set_scan_agg_op(q, AGG_SUM);
    assert(matrix_get_scan_agg_op(q) == AGG_SUM);
    assert(matrix_get_scan_threshold(q) == 50u);     // not disturbed
    assert(matrix_get_scan_column_id(q) == 9ull);    // not disturbed
    std::cout << "[agg codec] ok\n";
}

static void test_engine_agg_legacy() {
    CPUMockEngine eng(0);                          // legacy fixed column (id 0)
    const uint64_t SIZE = MATRIX_SCAN_COLUMN_SIZE;
    const uint32_t T = 8000000;                    // < SIZE
    auto run = [&](MatrixAggOp op) {
        DatabaseQuery q{}; matrix_set_scan_target(q, T, 0); matrix_set_scan_agg_op(q, op);
        eng.execute_scan(q); return q.transaction_id;
    };
    const uint64_t cnt = SIZE - 1 - T;
    assert(run(AGG_COUNT) == cnt);
    assert(run(AGG_MIN)   == static_cast<uint64_t>(T) + 1);
    assert(run(AGG_MAX)   == SIZE - 1);
    const uint64_t sum = (static_cast<uint64_t>(SIZE - 1) + (static_cast<uint64_t>(T) + 1)) * cnt / 2;
    assert(run(AGG_SUM)   == sum);
    std::cout << "[engine agg legacy] ok\n";
}

static void test_engine_agg_tiered() {
    const size_t N = 1000;
    std::vector<uint32_t> col(N);
    for (size_t i = 0; i < N; ++i) col[i] = static_cast<uint32_t>(i);
    CPUMockEngine eng(0, "", /*host_cap=*/SIZE_MAX);
    eng.load_scan_column(1, col.data(), N);
    const uint32_t T = 600;
    auto run = [&](MatrixAggOp op) {
        DatabaseQuery q{}; matrix_set_scan_target(q, T, 1); matrix_set_scan_agg_op(q, op);
        eng.execute_scan(q); return q.transaction_id;
    };
    const uint64_t cnt = N - 1 - T;
    assert(run(AGG_COUNT) == cnt);
    assert(run(AGG_MIN)   == static_cast<uint64_t>(T) + 1);
    assert(run(AGG_MAX)   == N - 1);
    uint64_t sum = 0; for (uint64_t i = T + 1; i <= N - 1; ++i) sum += i;
    assert(run(AGG_SUM)   == sum);
    std::cout << "[engine agg tiered] ok\n";
}

static void test_engine_agg_cold_borrow() {
    // Aggregate a NON-COUNT op over a column that has actually been demoted to COLD — exercises
    // the reducer through the borrow path (SSD->RAM->reduce->SSD), the headline use case, not just
    // a HOST-resident column.
    const size_t N = 1000;
    const size_t S = N * sizeof(uint32_t);
    std::vector<uint32_t> col(N);
    for (size_t i = 0; i < N; ++i) col[i] = static_cast<uint32_t>(i);
    CPUMockEngine eng(0, "", /*host_cap=*/S);          // RAM holds ONE column
    eng.load_scan_column(1, col.data(), N);
    eng.load_scan_column(2, col.data(), N);            // 2*S > budget: one must be demoted
    // Hammer col 2, never col 1 -> col 1 (heat 0) is the eviction victim -> demoted to COLD.
    for (int r = 0; r < 12; ++r) {
        DatabaseQuery q{}; matrix_set_scan_target(q, 0u, 2); eng.execute_scan(q);
    }
    assert(eng.column_tier(1) == MemorySpace::COLD && "col 1 demoted to SSD");
    // SUM over the COLD column: borrowed to HOST, reduced, returned to COLD.
    const uint32_t T = 600;
    DatabaseQuery qs{}; matrix_set_scan_target(qs, T, 1); matrix_set_scan_agg_op(qs, AGG_SUM);
    eng.execute_scan(qs);
    uint64_t sum = 0; for (uint64_t i = T + 1; i <= N - 1; ++i) sum += i;
    assert(qs.transaction_id == sum && "SUM correct over a COLD-borrowed column");
    assert(eng.column_tier(1) == MemorySpace::COLD && "col 1 returned to SSD after the borrow");
    std::cout << "[engine agg cold-borrow] ok\n";
}

int main() {
    test_reduce_closed_form();
    test_reduce_empty();
    test_agg_codec();
    test_engine_agg_legacy();
    test_engine_agg_tiered();
    test_engine_agg_cold_borrow();
    std::cout << "ALL AGGREGATION TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_group_by.cpp
// CPU test for grouped aggregation (GROUP BY). Grows across tasks; one main() runs all.
#include "compute_mock.cpp"   // CPUMockEngine + compute.hpp (matrix_cpu_group_reduce)
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

// Hand-worked: keys/vals below, G=3 groups.
//   g0 rows = {5,9,15} (indices 0,2,5);  g1 = {7,13} (1,4);  g2 = {11} (3)
static void test_group_reduce_handworked() {
    const std::vector<uint32_t> keys = {0, 1, 0, 2, 1, 0};
    const std::vector<uint32_t> vals = {5, 7, 9, 11, 13, 15};
    const uint32_t G = 3;
    std::vector<uint64_t> out(G);
    matrix_cpu_group_reduce(keys.data(), vals.data(), keys.size(), G, AGG_COUNT, out.data());
    assert((out == std::vector<uint64_t>{3, 2, 1}));
    matrix_cpu_group_reduce(keys.data(), vals.data(), keys.size(), G, AGG_SUM, out.data());
    assert((out == std::vector<uint64_t>{29, 20, 11}));   // 5+9+15, 7+13, 11
    matrix_cpu_group_reduce(keys.data(), vals.data(), keys.size(), G, AGG_MIN, out.data());
    assert((out == std::vector<uint64_t>{5, 7, 11}));
    matrix_cpu_group_reduce(keys.data(), vals.data(), keys.size(), G, AGG_MAX, out.data());
    assert((out == std::vector<uint64_t>{15, 13, 11}));
    std::cout << "[group reduce hand-worked] ok\n";
}

static void test_group_reduce_edge() {
    // G=4 -> group 3 is empty; plus an out-of-range key (10) that must be ignored.
    const std::vector<uint32_t> keys = {0, 1, 0, 2, 1, 0, 10};
    const std::vector<uint32_t> vals = {5, 7, 9, 11, 13, 15, 999};
    const uint32_t G = 4;
    std::vector<uint64_t> out(G);
    matrix_cpu_group_reduce(keys.data(), vals.data(), keys.size(), G, AGG_COUNT, out.data());
    assert((out == std::vector<uint64_t>{3, 2, 1, 0}));        // group 3 empty -> 0; key 10 ignored
    matrix_cpu_group_reduce(keys.data(), vals.data(), keys.size(), G, AGG_SUM, out.data());
    assert((out == std::vector<uint64_t>{29, 20, 11, 0}));     // 999 (key 10) NOT summed anywhere
    matrix_cpu_group_reduce(keys.data(), vals.data(), keys.size(), G, AGG_MIN, out.data());
    assert((out == std::vector<uint64_t>{5, 7, 11, UINT64_MAX})); // empty-group MIN sentinel
    matrix_cpu_group_reduce(keys.data(), vals.data(), keys.size(), G, AGG_MAX, out.data());
    assert((out == std::vector<uint64_t>{15, 13, 11, 0}));     // empty-group MAX sentinel
    std::cout << "[group reduce edge] ok\n";
}

// Independent reference for the scale/COLD test (brute-force grouping).
static std::vector<uint64_t> brute_group(const std::vector<uint32_t>& keys,
                                         const std::vector<uint32_t>& vals,
                                         uint32_t G, MatrixAggOp op) {
    std::vector<uint64_t> out(G, op == AGG_MIN ? UINT64_MAX : 0);
    for (size_t i = 0; i < keys.size(); ++i) {
        const uint32_t k = keys[i]; if (k >= G) continue;
        if (op == AGG_SUM) out[k] += vals[i];
        else if (op == AGG_MIN) { if (vals[i] < out[k]) out[k] = vals[i]; }
        else if (op == AGG_MAX) { if (vals[i] > out[k]) out[k] = vals[i]; }
        else out[k] += 1;
    }
    return out;
}

static void test_engine_group_by_host() {
    const std::vector<uint32_t> keys = {0, 1, 0, 2, 1, 0};
    const std::vector<uint32_t> vals = {5, 7, 9, 11, 13, 15};
    CPUMockEngine eng(0, "", /*host_cap=*/SIZE_MAX);
    eng.load_scan_column(1, keys.data(), keys.size());     // key column
    eng.load_scan_column(2, vals.data(), vals.size());     // value column
    std::vector<uint64_t> out;
    eng.grouped_aggregate(1, 2, /*num_groups=*/3, AGG_SUM, out);
    assert((out == std::vector<uint64_t>{29, 20, 11}));
    eng.grouped_aggregate(1, 2, 3, AGG_MAX, out);
    assert((out == std::vector<uint64_t>{15, 13, 11}));
    eng.grouped_aggregate(1, 2, 3, AGG_MIN, out);            // MIN through the engine (incl. its sentinel path)
    assert((out == std::vector<uint64_t>{5, 7, 11}));
    std::cout << "[engine group-by HOST] ok\n";
}

static void test_engine_group_by_cold() {
    // Both key and value demoted to COLD, then grouped — exercises the DOUBLE borrow-and-return.
    const size_t N = 1000;
    const uint32_t G = 8;
    std::vector<uint32_t> keys(N), vals(N), dummy(N, 0);
    for (size_t i = 0; i < N; ++i) { keys[i] = static_cast<uint32_t>(i % G); vals[i] = static_cast<uint32_t>(i); }
    CPUMockEngine eng(0, "", /*host_cap=*/N * sizeof(uint32_t));  // RAM holds ONE column
    eng.load_scan_column(1, keys.data(), N);
    eng.load_scan_column(2, vals.data(), N);
    eng.load_scan_column(3, dummy.data(), N);
    // Hammer only the dummy column -> keys(1) and vals(2) go idle -> both demoted to COLD.
    for (int r = 0; r < 12; ++r) { DatabaseQuery q{}; matrix_set_scan_target(q, 0u, 3); eng.execute_scan(q); }
    assert(eng.column_tier(1) == MemorySpace::COLD && eng.column_tier(2) == MemorySpace::COLD);
    std::vector<uint64_t> out;
    eng.grouped_aggregate(1, 2, G, AGG_SUM, out);              // double borrow COLD->HOST->reduce->COLD
    assert(out == brute_group(keys, vals, G, AGG_SUM));
    eng.grouped_aggregate(1, 2, G, AGG_COUNT, out);
    assert(out == brute_group(keys, vals, G, AGG_COUNT));
    assert(eng.column_tier(1) == MemorySpace::COLD && eng.column_tier(2) == MemorySpace::COLD); // borrows returned
    std::cout << "[engine group-by COLD double-borrow] ok\n";
}

static void test_group_reduce_where() {
    const std::vector<uint32_t> keys = {0, 1, 0, 2, 1, 0};
    const std::vector<uint32_t> vals = {5, 7, 9, 11, 13, 15};
    const uint32_t G = 3;
    std::vector<uint64_t> out(G);
    // WHERE value > 8 -> kept: g0{9,15}, g1{13}, g2{11}
    matrix_cpu_group_reduce_where(keys.data(), vals.data(), keys.size(), G, AGG_COUNT, 8, out.data());
    assert((out == std::vector<uint64_t>{2, 1, 1}));
    matrix_cpu_group_reduce_where(keys.data(), vals.data(), keys.size(), G, AGG_SUM, 8, out.data());
    assert((out == std::vector<uint64_t>{24, 13, 11}));   // 9+15, 13, 11
    matrix_cpu_group_reduce_where(keys.data(), vals.data(), keys.size(), G, AGG_MIN, 8, out.data());
    assert((out == std::vector<uint64_t>{9, 13, 11}));
    matrix_cpu_group_reduce_where(keys.data(), vals.data(), keys.size(), G, AGG_MAX, 8, out.data());
    assert((out == std::vector<uint64_t>{15, 13, 11}));
    // non-vacuity + regression: unfiltered wrapper still gives GBY-1's results
    matrix_cpu_group_reduce(keys.data(), vals.data(), keys.size(), G, AGG_SUM, out.data());
    assert((out == std::vector<uint64_t>{29, 20, 11}));
    // WHERE value > 12 -> g2 emptied BY the filter
    matrix_cpu_group_reduce_where(keys.data(), vals.data(), keys.size(), G, AGG_COUNT, 12, out.data());
    assert((out == std::vector<uint64_t>{1, 1, 0}));
    matrix_cpu_group_reduce_where(keys.data(), vals.data(), keys.size(), G, AGG_MIN, 12, out.data());
    assert((out == std::vector<uint64_t>{15, 13, UINT64_MAX}));
    std::cout << "[group reduce WHERE] ok\n";
}

static void test_engine_group_by_where() {
    const std::vector<uint32_t> keys = {0, 1, 0, 2, 1, 0};
    const std::vector<uint32_t> vals = {5, 7, 9, 11, 13, 15};
    CPUMockEngine eng(0, "", /*host_cap=*/SIZE_MAX);
    eng.load_scan_column(1, keys.data(), keys.size());
    eng.load_scan_column(2, vals.data(), vals.size());
    std::vector<uint64_t> out;
    eng.grouped_aggregate_where(1, 2, /*num_groups=*/3, AGG_SUM, /*threshold=*/8, out);
    assert((out == std::vector<uint64_t>{24, 13, 11}));
    eng.grouped_aggregate_where(1, 2, 3, AGG_COUNT, 8, out);
    assert((out == std::vector<uint64_t>{2, 1, 1}));
    std::cout << "[engine group-by WHERE] ok\n";
}

int main() {
    test_group_reduce_handworked();
    test_group_reduce_edge();
    test_engine_group_by_host();
    test_engine_group_by_cold();
    test_group_reduce_where();
    test_engine_group_by_where();
    std::cout << "ALL GROUP-BY TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_query.cpp
// CPU test for the unified query API (MatrixQuery / execute_query). Grows; one main().
#include "compute_mock.cpp"   // CPUMockEngine + compute.hpp (MatrixQuery, matrix_cpu_reduce_all)
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

static std::vector<uint64_t> brute_group(const std::vector<uint32_t>& keys,
                                         const std::vector<uint32_t>& vals,
                                         uint32_t G, MatrixAggOp op, bool filt, uint32_t T) {
    std::vector<uint64_t> out(G, op == AGG_MIN ? UINT64_MAX : 0);
    for (size_t i = 0; i < keys.size(); ++i) {
        const uint32_t k = keys[i]; if (k >= G) continue;
        if (filt && vals[i] <= T) continue;
        if (op == AGG_SUM) out[k] += vals[i];
        else if (op == AGG_MIN) { if (vals[i] < out[k]) out[k] = vals[i]; }
        else if (op == AGG_MAX) { if (vals[i] > out[k]) out[k] = vals[i]; }
        else out[k] += 1;
    }
    return out;
}

static void test_reduce_all() {
    std::vector<uint32_t> v(1000);
    for (size_t i = 0; i < v.size(); ++i) v[i] = static_cast<uint32_t>(i);
    assert(matrix_cpu_reduce_all(v.data(), v.size(), AGG_COUNT) == 1000);
    assert(matrix_cpu_reduce_all(v.data(), v.size(), AGG_SUM)   == 499500ull); // 999*1000/2
    assert(matrix_cpu_reduce_all(v.data(), v.size(), AGG_MIN)   == 0);
    assert(matrix_cpu_reduce_all(v.data(), v.size(), AGG_MAX)   == 999);
    std::cout << "[reduce_all] ok\n";
}

static void test_query_routes() {
    const size_t N = 1000; const uint32_t G = 8;
    std::vector<uint32_t> vals(N), keys(N);
    for (size_t i = 0; i < N; ++i) { vals[i] = static_cast<uint32_t>(i); keys[i] = static_cast<uint32_t>(i % G); }
    CPUMockEngine eng(0, "", /*host_cap=*/SIZE_MAX);
    eng.load_scan_column(1, keys.data(), N);   // key column
    eng.load_scan_column(2, vals.data(), N);   // value column
    std::vector<uint64_t> out;

    eng.execute_query(MatrixQuery{.value_col = 2, .agg = AGG_SUM}, out);
    assert(out.size() == 1 && out[0] == 499500ull);
    eng.execute_query(MatrixQuery{.value_col = 2, .agg = AGG_SUM, .has_filter = true, .threshold = 500}, out);
    assert(out.size() == 1 && out[0] == 374250ull);   // 501..999; differs from unfiltered (non-vacuous)
    eng.execute_query(MatrixQuery{.value_col = 2, .agg = AGG_SUM, .grouped = true, .key_col = 1, .num_groups = G}, out);
    assert(out == brute_group(keys, vals, G, AGG_SUM, false, 0));
    eng.execute_query(MatrixQuery{.value_col = 2, .agg = AGG_SUM, .has_filter = true, .threshold = 500,
                                  .grouped = true, .key_col = 1, .num_groups = G}, out);
    assert(out == brute_group(keys, vals, G, AGG_SUM, true, 500));
    std::cout << "[query routes] ok\n";
}

static void test_query_cold() {
    const size_t N = 1000;
    std::vector<uint32_t> vals(N), dummy(N, 0);
    for (size_t i = 0; i < N; ++i) vals[i] = static_cast<uint32_t>(i);
    CPUMockEngine eng(0, "", /*host_cap=*/N * sizeof(uint32_t));  // RAM holds ONE column
    eng.load_scan_column(2, vals.data(), N);
    eng.load_scan_column(3, dummy.data(), N);
    for (int r = 0; r < 12; ++r) { DatabaseQuery q{}; matrix_set_scan_target(q, 0u, 3); eng.execute_scan(q); }
    assert(eng.column_tier(2) == MemorySpace::COLD && "value column demoted to SSD");
    std::vector<uint64_t> out;
    eng.execute_query(MatrixQuery{.value_col = 2, .agg = AGG_SUM}, out);   // unfiltered, COLD borrow
    assert(out.size() == 1 && out[0] == 499500ull);
    assert(eng.column_tier(2) == MemorySpace::COLD && "value column returned to SSD after the borrow");
    std::cout << "[query cold-borrow] ok\n";
}

int main() {
    test_reduce_all();
    test_query_routes();
    test_query_cold();
    std::cout << "ALL QUERY TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_observability.cpp
// CPU test for engine observability (EngineStats / stats()).
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

static void test_stats() {
    const size_t N = 1000;
    const size_t S = N * sizeof(uint32_t);
    std::vector<uint32_t> col(N);
    for (size_t i = 0; i < N; ++i) col[i] = static_cast<uint32_t>(i);
    CPUMockEngine eng(0, "", /*host_cap=*/2 * S);   // 2-column budget
    eng.load_scan_column(1, col.data(), N);
    eng.load_scan_column(2, col.data(), N);
    eng.load_scan_column(3, col.data(), N);
    {
        EngineStats s = eng.stats();
        assert(s.catalog_columns == 3);
        assert(s.cold_resident_bytes == 0);
        assert(s.rebalances == 0 && s.migrations == 0 && s.cold_borrows == 0);
    }
    const uint32_t T = 250;
    for (int r = 0; r < 8; ++r)
        for (uint64_t id : {1ull, 2ull}) {
            DatabaseQuery q{}; matrix_set_scan_target(q, T, id); eng.execute_scan(q);
        }
    {
        EngineStats s = eng.stats();
        assert(s.rebalances == 16 / 4);                 // 16 tiered scans / REBALANCE_EVERY(4)
        assert(s.migrations >= 1);                       // col 3 demoted (non-vacuous)
        assert(s.cold_resident_bytes == S);              // one column on SSD
        assert(s.host_resident_bytes == 2 * S);          // two columns in RAM
        assert(s.cold_borrows == 0);                     // cols 1,2 stayed HOST -> no borrow
    }
    { DatabaseQuery q{}; matrix_set_scan_target(q, T, 3); eng.execute_scan(q); }
    assert(eng.stats().cold_borrows == 1);
    std::cout << "[engine stats] ok\n";
}

int main() {
    test_stats();
    std::cout << "ALL OBSERVABILITY TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_column_io.cpp
// CPU test for binary column persistence (column_io.hpp + engine save/load).
#include "compute_mock.cpp"
#include "column_io.hpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <cstdio>
#include <iostream>

static void test_direct_roundtrip() {
    std::vector<uint32_t> in(500);
    for (size_t i = 0; i < in.size(); ++i) in[i] = static_cast<uint32_t>(i * 3 + 1);
    const std::string path = "/tmp/matrixdb_coltest_500.bin";
    matrix_write_column(path, in.data(), in.size());
    std::vector<uint32_t> out;
    matrix_read_column(path, out);
    assert(out == in && "round-trip preserves the column");
    FILE* f = std::fopen(path.c_str(), "rb"); assert(f);
    uint32_t magic = 0; assert(std::fread(&magic, sizeof magic, 1, f) == 1); std::fclose(f);
    assert(magic == MATRIX_COLUMN_MAGIC && "file carries the column magic");
    std::remove(path.c_str());
    std::vector<uint32_t> empty, eout;
    matrix_write_column("/tmp/matrixdb_coltest_0.bin", empty.data(), 0);
    matrix_read_column("/tmp/matrixdb_coltest_0.bin", eout);
    assert(eout.empty());
    std::remove("/tmp/matrixdb_coltest_0.bin");
    std::cout << "[column_io direct round-trip] ok\n";
}

static uint64_t sum_to(uint64_t hi) { uint64_t s = 0; for (uint64_t i = 0; i <= hi; ++i) s += i; return s; }

static void test_engine_save_load_query() {
    const size_t N = 1000;
    std::vector<uint32_t> col(N);
    for (size_t i = 0; i < N; ++i) col[i] = static_cast<uint32_t>(i);
    const std::string path = "/tmp/matrixdb_coltest_eng.bin";
    CPUMockEngine a(0, "", SIZE_MAX);
    a.load_scan_column(7, col.data(), N);
    a.save_column(7, path);
    CPUMockEngine b(0, "", SIZE_MAX);
    b.load_column_from_file(7, path);
    std::vector<uint64_t> out;
    b.execute_query(MatrixQuery{.value_col = 7, .agg = AGG_SUM}, out);
    assert(out.size() == 1 && out[0] == sum_to(N - 1) && "persisted column reloads + queries identically");
    std::remove(path.c_str());
    std::cout << "[engine save->load->query] ok\n";
}

static void test_engine_save_cold_column() {
    const size_t N = 1000;
    const size_t S = N * sizeof(uint32_t);
    std::vector<uint32_t> col(N), dummy(N, 0);
    for (size_t i = 0; i < N; ++i) col[i] = static_cast<uint32_t>(i);
    const std::string path = "/tmp/matrixdb_coltest_cold.bin";
    CPUMockEngine a(0, "", /*host_cap=*/S);   // one-column budget
    a.load_scan_column(7, col.data(), N);
    a.load_scan_column(8, dummy.data(), N);
    for (int r = 0; r < 12; ++r) { DatabaseQuery q{}; matrix_set_scan_target(q, 0u, 8); a.execute_scan(q); }
    assert(a.column_tier(7) == MemorySpace::COLD && "col 7 demoted to SSD");
    a.save_column(7, path);
    assert(a.column_tier(7) == MemorySpace::COLD && "save returned the borrow");
    CPUMockEngine b(0, "", SIZE_MAX);
    b.load_column_from_file(7, path);
    std::vector<uint64_t> out;
    b.execute_query(MatrixQuery{.value_col = 7, .agg = AGG_SUM}, out);
    assert(out.size() == 1 && out[0] == sum_to(N - 1) && "COLD column persisted + reloaded correctly");
    std::remove(path.c_str());
    std::cout << "[engine save COLD column] ok\n";
}

int main() {
    test_direct_roundtrip();
    test_engine_save_load_query();
    test_engine_save_cold_column();
    std::cout << "ALL COLUMN-IO TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_catalog_snapshot.cpp
// CPU test for catalog snapshot durability (save_catalog / load_catalog).
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <cstdio>
#include <iostream>

static uint64_t sum_of(const std::vector<uint32_t>& v) { uint64_t s = 0; for (uint32_t x : v) s += x; return s; }
static uint64_t engine_sum(CPUMockEngine& e, uint64_t id) {
    std::vector<uint64_t> out; e.execute_query(MatrixQuery{.value_col = id, .agg = AGG_SUM}, out); return out[0];
}

static void test_multicolumn_roundtrip() {
    const size_t N = 1000;
    std::vector<uint32_t> a(N), b(N), c(N);
    for (size_t i = 0; i < N; ++i) { a[i] = i; b[i] = i + 1000; c[i] = static_cast<uint32_t>(i * 2); }
    const std::string path = "/tmp/matrixdb_catsnap.bin";
    CPUMockEngine src(0, "", SIZE_MAX);
    src.load_scan_column(1, a.data(), N);
    src.load_scan_column(2, b.data(), N);
    src.load_scan_column(3, c.data(), N);
    src.save_catalog(path);

    CPUMockEngine dst(0, "", SIZE_MAX);
    dst.load_catalog(path);
    assert(dst.stats().catalog_columns == 3);
    assert(engine_sum(dst, 1) == sum_of(a));
    assert(engine_sum(dst, 2) == sum_of(b));
    assert(engine_sum(dst, 3) == sum_of(c));
    assert(sum_of(a) != sum_of(b) && sum_of(b) != sum_of(c));
    std::remove(path.c_str());
    std::cout << "[catalog multi-column round-trip] ok\n";
}

static void test_snapshot_with_cold_column() {
    const size_t N = 1000;
    const size_t S = N * sizeof(uint32_t);
    std::vector<uint32_t> a(N), dummy(N, 0);
    for (size_t i = 0; i < N; ++i) a[i] = i;
    const std::string path = "/tmp/matrixdb_catsnap_cold.bin";
    CPUMockEngine src(0, "", /*host_cap=*/S);
    src.load_scan_column(1, a.data(), N);
    src.load_scan_column(9, dummy.data(), N);
    for (int r = 0; r < 12; ++r) { DatabaseQuery q{}; matrix_set_scan_target(q, 0u, 9); src.execute_scan(q); }
    assert(src.column_tier(1) == MemorySpace::COLD && "col 1 demoted before snapshot");
    src.save_catalog(path);
    assert(src.column_tier(1) == MemorySpace::COLD && "save returned the borrow");

    CPUMockEngine dst(0, "", SIZE_MAX);
    dst.load_catalog(path);
    assert(engine_sum(dst, 1) == sum_of(a) && "COLD column restored correctly");
    assert(engine_sum(dst, 9) == 0 && "dummy column restored");
    std::remove(path.c_str());
    std::cout << "[catalog snapshot with COLD column] ok\n";
}

static void test_empty_catalog() {
    const std::string path = "/tmp/matrixdb_catsnap_empty.bin";
    CPUMockEngine src(0, "", SIZE_MAX);
    src.save_catalog(path);
    CPUMockEngine dst(0, "", SIZE_MAX);
    dst.load_catalog(path);
    assert(dst.stats().catalog_columns == 0);
    std::remove(path.c_str());
    std::cout << "[empty catalog snapshot] ok\n";
}

int main() {
    test_multicolumn_roundtrip();
    test_snapshot_with_cold_column();
    test_empty_catalog();
    std::cout << "ALL CATALOG-SNAPSHOT TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_query_validation.cpp
// CPU test for query input validation (execute_query returns MatrixQueryStatus, never crashes).
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

static void test_validation() {
    const size_t N = 1000;
    std::vector<uint32_t> a(N), b(500), keys(500);
    for (size_t i = 0; i < N; ++i) a[i] = static_cast<uint32_t>(i);
    for (size_t i = 0; i < 500; ++i) { b[i] = static_cast<uint32_t>(i); keys[i] = static_cast<uint32_t>(i % 4); }
    CPUMockEngine eng(0, "", SIZE_MAX);
    eng.load_scan_column(5, a.data(), N);        // len 1000
    eng.load_scan_column(6, b.data(), 500);      // len 500
    eng.load_scan_column(7, keys.data(), 500);   // len 500, keys i%4
    std::vector<uint64_t> out;
    using S = MatrixQueryStatus;

    assert(eng.execute_query(MatrixQuery{.value_col = 5, .agg = AGG_SUM}, out) == S::OK && out.size() == 1);
    assert(eng.execute_query(MatrixQuery{.value_col = 0, .agg = AGG_SUM}, out) == S::ERR_UNKNOWN_COLUMN && out.empty());
    assert(eng.execute_query(MatrixQuery{.value_col = 999, .agg = AGG_SUM}, out) == S::ERR_UNKNOWN_COLUMN && out.empty());
    assert(eng.execute_query(MatrixQuery{.value_col = 5, .agg = AGG_SUM, .grouped = true, .key_col = 5, .num_groups = 4}, out) == S::ERR_INVALID_GROUP);
    assert(eng.execute_query(MatrixQuery{.value_col = 5, .agg = AGG_SUM, .grouped = true, .key_col = 999, .num_groups = 4}, out) == S::ERR_INVALID_GROUP);
    assert(eng.execute_query(MatrixQuery{.value_col = 6, .agg = AGG_SUM, .grouped = true, .key_col = 7, .num_groups = 0}, out) == S::ERR_INVALID_GROUP);
    assert(eng.execute_query(MatrixQuery{.value_col = 5, .agg = AGG_SUM, .grouped = true, .key_col = 7, .num_groups = 4}, out) == S::ERR_INVALID_GROUP); // len 1000 vs 500
    assert(eng.execute_query(MatrixQuery{.value_col = 6, .agg = AGG_SUM, .grouped = true, .key_col = 7, .num_groups = (1u << 28) + 1}, out) == S::ERR_TOO_MANY_GROUPS);
    assert(eng.execute_query(MatrixQuery{.value_col = 6, .agg = AGG_SUM, .grouped = true, .key_col = 7, .num_groups = 4}, out) == S::OK && out.size() == 4);
    std::cout << "[query validation] ok\n";
}

int main() {
    test_validation();
    std::cout << "ALL QUERY-VALIDATION TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_transactions.cpp
// CPU test for atomic transactions (WAL group commit). Grows across tasks; one main().
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <cstdio>
#include <iostream>

static std::vector<std::pair<uint64_t,uint64_t>> replay_all(const std::string& path) {
    std::vector<std::pair<uint64_t,uint64_t>> applied;
    ColdStore cs(path, SyncPolicy::SYNC_OFF);
    cs.replay([&](uint64_t k, uint64_t v){ applied.emplace_back(k, v); });
    return applied;
}

static void test_wal_commit_atomicity() {
    const std::string path = "/tmp/matrixdb_txn_wal.bin";
    std::remove(path.c_str());
    {
        ColdStore cs(path, SyncPolicy::SYNC_OFF);
        cs.append_txn_put(10, 100);
        cs.append_txn_put(11, 110);
        cs.append_commit();              // committed group {10,11}
        cs.append_txn_put(12, 120);      // a "crash": txn-put with no following commit marker
    }
    auto applied = replay_all(path);
    assert(applied.size() == 2 && "only the committed group replays");
    assert(applied[0].first == 10 && applied[1].first == 11);
    bool saw12 = false; for (auto& kv : applied) if (kv.first == 12) saw12 = true;
    assert(!saw12 && "uncommitted txn-put is discarded (crash-atomic)");
    std::remove(path.c_str());
    std::cout << "[wal commit atomicity] ok\n";
}

static void test_wal_autocommit_unchanged() {
    const std::string path = "/tmp/matrixdb_txn_auto.bin";
    std::remove(path.c_str());
    {
        ColdStore cs(path, SyncPolicy::SYNC_OFF);
        cs.append_put(1, 11);
        cs.append_put(2, 22);
    }
    auto applied = replay_all(path);
    assert(applied.size() == 2 && applied[0].first == 1 && applied[1].first == 2
           && "auto-commit puts still apply immediately (backward compat)");
    std::remove(path.c_str());
    std::cout << "[wal auto-commit unchanged] ok\n";
}

static void test_engine_commit_durable() {
    const std::string wal = "/tmp/matrixdb_txn_eng.bin";
    std::remove(wal.c_str());
    {
        CPUMockEngine eng(0, wal);            // durability ON
        eng.begin();
        eng.txn_put(100, 1000);
        eng.txn_put(101, 1010);
        eng.txn_put(102, 1020);
        eng.commit();
        uint64_t v = 0;
        assert(eng.kv_get(100, v) && v == 1000);
        assert(eng.kv_get(102, v) && v == 1020);
    }
    {
        CPUMockEngine eng2(0, wal);           // replay the committed txn on restart
        uint64_t v = 0;
        assert(eng2.kv_get(100, v) && v == 1000 && "committed txn survives restart");
        assert(eng2.kv_get(101, v) && v == 1010);
        assert(eng2.kv_get(102, v) && v == 1020);
    }
    std::remove(wal.c_str());
    std::cout << "[engine commit durable] ok\n";
}

static void test_engine_rollback() {
    const std::string wal = "/tmp/matrixdb_txn_rb.bin";
    std::remove(wal.c_str());
    {
        CPUMockEngine eng(0, wal);
        eng.begin();
        eng.txn_put(200, 2000);
        eng.rollback();
        uint64_t v = 0;
        assert(!eng.kv_get(200, v) && "rolled-back write is not visible");
    }
    {
        CPUMockEngine eng2(0, wal);
        uint64_t v = 0;
        assert(!eng2.kv_get(200, v) && "rolled-back write was never persisted");
    }
    std::remove(wal.c_str());
    std::cout << "[engine rollback] ok\n";
}

int main() {
    test_wal_commit_atomicity();
    test_wal_autocommit_unchanged();
    test_engine_commit_durable();
    test_engine_rollback();
    std::cout << "ALL TRANSACTION TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_server.cpp
// CPU test for the server core (request/response protocol + matrix_serve dispatch).
#include "server.hpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <cstdio>
#include <iostream>

static void test_request_roundtrip() {
    MatrixRequest r; r.kind = ReqKind::QUERY; r.key = 7; r.value = 70;
    r.query = MatrixQuery{.value_col=2, .agg=AGG_SUM, .has_filter=true, .threshold=5,
                          .grouped=true, .key_col=1, .num_groups=4};
    auto b = matrix_serialize_request(r);
    MatrixRequest r2;
    assert(matrix_deserialize_request(b, r2));
    assert(r2.kind == ReqKind::QUERY && r2.key == 7 && r2.value == 70);
    assert(r2.query.value_col == 2 && r2.query.agg == AGG_SUM && r2.query.has_filter
           && r2.query.threshold == 5 && r2.query.grouped && r2.query.key_col == 1 && r2.query.num_groups == 4);
    b.pop_back(); assert(!matrix_deserialize_request(b, r2) && "truncated request rejected");
    std::cout << "[request round-trip] ok\n";
}

static void test_response_roundtrip() {
    MatrixResponse r; r.status = 3; r.results = {10, 20, 30};
    auto b = matrix_serialize_response(r);
    MatrixResponse r2;
    assert(matrix_deserialize_response(b, r2) && r2.status == 3 && (r2.results == std::vector<uint64_t>{10,20,30}));
    MatrixResponse e; auto be = matrix_serialize_response(e);
    MatrixResponse e2; assert(matrix_deserialize_response(be, e2) && e2.status == 0 && e2.results.empty());
    b.pop_back(); assert(!matrix_deserialize_response(b, r2) && "truncated response rejected");
    std::cout << "[response round-trip] ok\n";
}

static void test_serve_get_put() {
    const std::string wal = "/tmp/matrixdb_srv.bin"; std::remove(wal.c_str());
    {
        CPUMockEngine eng(0, wal);
        MatrixRequest put; put.kind = ReqKind::PUT; put.key = 5; put.value = 500;
        MatrixResponse pr; assert(matrix_deserialize_response(matrix_serve(eng, matrix_serialize_request(put)), pr) && pr.status == 0);
        MatrixRequest get; get.kind = ReqKind::GET; get.key = 5;
        MatrixResponse gr; assert(matrix_deserialize_response(matrix_serve(eng, matrix_serialize_request(get)), gr));
        assert(gr.results.size() == 1 && gr.results[0] == 500);
        MatrixRequest miss; miss.kind = ReqKind::GET; miss.key = 999;
        MatrixResponse mr; assert(matrix_deserialize_response(matrix_serve(eng, matrix_serialize_request(miss)), mr) && mr.results.empty());
    }
    {
        CPUMockEngine eng2(0, wal);
        MatrixRequest get; get.kind = ReqKind::GET; get.key = 5;
        MatrixResponse gr; assert(matrix_deserialize_response(matrix_serve(eng2, matrix_serialize_request(get)), gr));
        assert(gr.results.size() == 1 && gr.results[0] == 500 && "PUT durable through the wire");
    }
    std::remove(wal.c_str());
    std::cout << "[serve get/put + durable] ok\n";
}

static void test_serve_query() {
    const size_t N = 1000; std::vector<uint32_t> col(N);
    for (size_t i = 0; i < N; ++i) col[i] = static_cast<uint32_t>(i);
    CPUMockEngine eng(0, "", SIZE_MAX); eng.load_scan_column(2, col.data(), N);
    MatrixRequest q; q.kind = ReqKind::QUERY; q.query = MatrixQuery{.value_col = 2, .agg = AGG_SUM};
    MatrixResponse r; assert(matrix_deserialize_response(matrix_serve(eng, matrix_serialize_request(q)), r));
    assert(r.status == 0 && r.results.size() == 1);
    std::vector<uint64_t> direct; eng.execute_query(MatrixQuery{.value_col = 2, .agg = AGG_SUM}, direct);
    assert(r.results == direct && "serve QUERY == direct execute_query");
    MatrixRequest bad; bad.kind = ReqKind::QUERY; bad.query = MatrixQuery{.value_col = 999, .agg = AGG_SUM};
    MatrixResponse br; assert(matrix_deserialize_response(matrix_serve(eng, matrix_serialize_request(bad)), br));
    assert(br.status == static_cast<uint32_t>(MatrixQueryStatus::ERR_UNKNOWN_COLUMN) && br.results.empty());
    std::cout << "[serve query] ok\n";
}

static void test_bad_request() {
    CPUMockEngine eng(0, "", SIZE_MAX);
    std::vector<uint8_t> garbage = {1, 2, 3};
    MatrixResponse r; assert(matrix_deserialize_response(matrix_serve(eng, garbage), r));
    assert(r.status == static_cast<uint32_t>(ServerStatus::ERR_BADREQUEST) && "bad request -> ERR_BADREQUEST, no crash");
    std::cout << "[bad request] ok\n";
}

static void test_serve_health() {
    const size_t N = 50; std::vector<uint32_t> col(N, 1);
    CPUMockEngine eng(0, "", SIZE_MAX); eng.load_scan_column(2, col.data(), N);
    MatrixRequest hq; hq.kind = ReqKind::HEALTH;
    MatrixResponse hr; assert(matrix_deserialize_response(matrix_serve(eng, matrix_serialize_request(hq)), hr));
    // results layout: [ready, durable, catalog_columns, host_resident_bytes, wal_pending, dropped]
    assert(hr.status == 0 && hr.results.size() == 6 && "HEALTH -> 6-field snapshot");
    HealthStatus h = eng.health();
    assert(hr.results[0] == (h.ready ? 1u : 0u) && hr.results[1] == (h.durable ? 1u : 0u));
    assert(hr.results[2] == 1 && "one column in the catalog");
    assert(hr.results[3] == N * sizeof(uint32_t) && "resident bytes over the wire");
    assert(hr.results[0] == 1 && hr.results[5] == 0 && "ready, no dropped writes");
    // HEALTH is probeable even by a principal with NO grants (operational status, not data)
    AccessPolicy locked;                                       // grants nothing
    MatrixResponse lr; assert(matrix_deserialize_response(matrix_serve(eng, locked, /*principal=*/42,
                                                          matrix_serialize_request(hq)), lr));
    assert(lr.status == 0 && lr.results.size() == 6 && "HEALTH allowed without grants");
    // a GET by the same locked principal IS denied (proves the policy is otherwise restrictive)
    MatrixRequest g; g.kind = ReqKind::GET; g.key = 1;
    MatrixResponse gr; assert(matrix_deserialize_response(matrix_serve(eng, locked, 42, matrix_serialize_request(g)), gr));
    assert(gr.status == static_cast<uint32_t>(ServerStatus::ERR_FORBIDDEN) && "GET still forbidden for the locked principal");
    std::cout << "[serve health] ok\n";
}

static void test_serve_stats() {
    const size_t N = 100; std::vector<uint32_t> col(N);
    for (size_t i = 0; i < N; ++i) col[i] = static_cast<uint32_t>(i);
    CPUMockEngine eng(0, "", SIZE_MAX); eng.load_scan_column(2, col.data(), N);
    // run a few queries so the metrics are non-zero
    for (int i = 0; i < 3; ++i) { std::vector<uint64_t> o; eng.execute_query(MatrixQuery{.value_col = 2, .agg = AGG_SUM}, o); }
    MatrixRequest sq; sq.kind = ReqKind::STATS;
    MatrixResponse sr; assert(matrix_deserialize_response(matrix_serve(eng, matrix_serialize_request(sq)), sr));
    assert(sr.status == 0 && sr.results.size() == 12 && "STATS -> 12-field metrics snapshot");
    EngineStats s = eng.stats();
    assert(sr.results[3] == 1 && "catalog_columns over the wire");
    assert(sr.results[6] == s.query_count && sr.results[6] >= 3 && "query_count over the wire (>=3 ran)");
    assert(sr.results[10] >= sr.results[9] && "p99 >= p50 (monotone percentiles)");
    assert(sr.results[11] == eng.version_u64() && sr.results[11] != 0 && "server version over the wire");
    // STATS, like HEALTH, is readable without grants (operational metrics, no row data)
    AccessPolicy locked;
    MatrixResponse lr; assert(matrix_deserialize_response(matrix_serve(eng, locked, 42, matrix_serialize_request(sq)), lr));
    assert(lr.status == 0 && lr.results.size() == 12 && "STATS allowed without grants");
    std::cout << "[serve stats] ok\n";
}

int main() {
    test_request_roundtrip();
    test_response_roundtrip();
    test_serve_get_put();
    test_serve_query();
    test_serve_health();
    test_serve_stats();
    test_bad_request();
    std::cout << "ALL SERVER TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_security.cpp
// CPU test for authorization / access control (AccessPolicy + authorizing matrix_serve).
#include "server.hpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <string>
#include <cstdio>
#include <iostream>

static MatrixResponse serve_req(CPUMockEngine& e, const AccessPolicy& p, uint64_t principal, const MatrixRequest& r) {
    MatrixResponse resp;
    matrix_deserialize_response(matrix_serve(e, p, principal, matrix_serialize_request(r)), resp);
    return resp;
}
static MatrixRequest mk(ReqKind kind) { MatrixRequest r; r.kind = kind; return r; }

static void test_access_control() {
    const size_t N = 1000; std::vector<uint32_t> v(N), k(N);
    for (size_t i = 0; i < N; ++i) { v[i] = static_cast<uint32_t>(i); k[i] = static_cast<uint32_t>(i % 4); }
    const std::string wal = "/tmp/matrixdb_sec.bin"; std::remove(wal.c_str());
    CPUMockEngine eng(0, wal);
    eng.load_scan_column(1, k.data(), N);
    eng.load_scan_column(2, v.data(), N);
    AccessPolicy p; p.allow_column(1, 2); p.allow_read(1); p.allow_write(1);
    using S = ServerStatus;

    { MatrixRequest g = mk(ReqKind::GET); g.key = 5;
      assert(serve_req(eng, p, 1, g).status == static_cast<uint32_t>(S::OK));
      assert(serve_req(eng, p, 2, g).status == static_cast<uint32_t>(S::ERR_FORBIDDEN)); }

    { MatrixRequest put = mk(ReqKind::PUT); put.key = 5; put.value = 500;
      assert(serve_req(eng, p, 2, put).status == static_cast<uint32_t>(S::ERR_FORBIDDEN));
      MatrixRequest g = mk(ReqKind::GET); g.key = 5;
      assert(serve_req(eng, p, 1, g).results.empty() && "denied PUT wrote nothing");
      assert(serve_req(eng, p, 1, put).status == static_cast<uint32_t>(S::OK));
      auto after = serve_req(eng, p, 1, g);
      assert(after.results.size() == 1 && after.results[0] == 500); }

    { MatrixRequest q = mk(ReqKind::QUERY); q.query = MatrixQuery{.value_col = 2, .agg = AGG_SUM};
      assert(serve_req(eng, p, 1, q).status == static_cast<uint32_t>(MatrixQueryStatus::OK));
      assert(serve_req(eng, p, 2, q).status == static_cast<uint32_t>(S::ERR_FORBIDDEN));
      MatrixRequest q3 = mk(ReqKind::QUERY); q3.query = MatrixQuery{.value_col = 3, .agg = AGG_SUM};
      assert(serve_req(eng, p, 1, q3).status == static_cast<uint32_t>(S::ERR_FORBIDDEN) && "col 3 not granted (authz before existence)"); }

    { MatrixRequest qg = mk(ReqKind::QUERY);
      qg.query = MatrixQuery{.value_col = 2, .agg = AGG_SUM, .grouped = true, .key_col = 1, .num_groups = 4};
      assert(serve_req(eng, p, 1, qg).status == static_cast<uint32_t>(S::ERR_FORBIDDEN) && "key col 1 not granted yet");
      p.allow_column(1, 1);
      assert(serve_req(eng, p, 1, qg).status == static_cast<uint32_t>(MatrixQueryStatus::OK) && "both cols granted -> OK"); }

    { AccessPolicy perm = AccessPolicy::permissive(); MatrixRequest g = mk(ReqKind::GET); g.key = 5;
      assert(serve_req(eng, perm, 2, g).status == static_cast<uint32_t>(S::OK) && "permissive allows anon"); }

    std::remove(wal.c_str());
    std::cout << "[access control] ok\n";
}

// SE-1: the authenticating matrix_serve validates a bearer token -> principal BEFORE authz; a bad/empty
// token is rejected with ERR_UNAUTHENTICATED and no engine work, a valid token serves as its principal.
static void test_authentication() {
    const size_t N = 100; std::vector<uint32_t> v(N);
    for (size_t i = 0; i < N; ++i) v[i] = static_cast<uint32_t>(i);
    CPUMockEngine eng; eng.load_scan_column(2, v.data(), N);
    using S = ServerStatus;

    Authenticator auth;
    auth.add_credential("s3cr3t-alice", 1);                    // token -> principal 1
    AccessPolicy p; p.allow_column(1, 2);                      // principal 1 may query col 2

    MatrixRequest q = mk(ReqKind::QUERY); q.query = MatrixQuery{.value_col = 2, .agg = AGG_SUM};
    const std::vector<uint8_t> qb = matrix_serialize_request(q);
    auto serve = [&](const std::string& tok) {
        MatrixResponse r; matrix_deserialize_response(matrix_serve(eng, p, auth, tok, qb), r); return r;
    };
    // valid token -> authenticated as principal 1 -> authorized -> OK
    assert(serve("s3cr3t-alice").status == static_cast<uint32_t>(MatrixQueryStatus::OK) && "valid token authenticates");
    // unknown / empty token -> ERR_UNAUTHENTICATED, no results
    assert(serve("wrong-token").status == static_cast<uint32_t>(S::ERR_UNAUTHENTICATED) && "bad token rejected");
    assert(serve("").status == static_cast<uint32_t>(S::ERR_UNAUTHENTICATED) && "empty token rejected");
    assert(serve("wrong-token").results.empty() && "rejected request returns no data");

    // authn precedes authz: a valid token for a principal WITHOUT the grant -> ERR_FORBIDDEN (not UNAUTH)
    auth.add_credential("s3cr3t-bob", 2);                      // principal 2 has no column grant
    assert(serve("s3cr3t-bob").status == static_cast<uint32_t>(S::ERR_FORBIDDEN) && "authenticated but unauthorized");
    std::cout << "[authentication] ok\n";
}

int main() {
    test_access_control();
    test_authentication();
    std::cout << "ALL SECURITY TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_audit.cpp
// CPU test for audit logging (AuditLog + matrix_serve audit overload).
#include "server.hpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <string>
#include <cstdio>
#include <iostream>

static MatrixRequest mk(ReqKind kind) { MatrixRequest r; r.kind = kind; return r; }

static void test_audit() {
    const size_t N = 1000; std::vector<uint32_t> v(N);
    for (size_t i = 0; i < N; ++i) v[i] = static_cast<uint32_t>(i);
    const std::string wal = "/tmp/matrixdb_audit.bin"; std::remove(wal.c_str());
    CPUMockEngine eng(0, wal);
    eng.load_scan_column(2, v.data(), N);
    AccessPolicy p; p.allow_column(1, 2); p.allow_read(1); p.allow_write(1);
    AuditLog audit;

    MatrixRequest get = mk(ReqKind::GET); get.key = 5;
    matrix_serve(eng, p, 1, matrix_serialize_request(get), audit);
    MatrixRequest put = mk(ReqKind::PUT); put.key = 5; put.value = 500;
    matrix_serve(eng, p, 2, matrix_serialize_request(put), audit);
    MatrixRequest q = mk(ReqKind::QUERY); q.query = MatrixQuery{.value_col = 2, .agg = AGG_SUM};
    matrix_serve(eng, p, 1, matrix_serialize_request(q), audit);
    std::vector<uint8_t> garbage = {1, 2, 3};
    matrix_serve(eng, p, 7, garbage, audit);

    const auto& e = audit.entries();
    assert(audit.size() == 4 && "every served request is audited");
    assert(e[0].principal == 1 && e[0].kind == static_cast<uint32_t>(ReqKind::GET)
           && e[0].status == static_cast<uint32_t>(ServerStatus::OK) && e[0].key == 5);
    assert(e[1].principal == 2 && e[1].kind == static_cast<uint32_t>(ReqKind::PUT)
           && e[1].status == static_cast<uint32_t>(ServerStatus::ERR_FORBIDDEN) && e[1].key == 5);
    assert(e[2].principal == 1 && e[2].kind == static_cast<uint32_t>(ReqKind::QUERY)
           && e[2].status == static_cast<uint32_t>(MatrixQueryStatus::OK) && e[2].column == 2);
    assert(e[3].principal == 7 && e[3].kind == 0
           && e[3].status == static_cast<uint32_t>(ServerStatus::ERR_BADREQUEST));
    std::remove(wal.c_str());
    std::cout << "[audit log] ok\n";
}

int main() {
    test_audit();
    std::cout << "ALL AUDIT TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_csv_ingest.cpp
// CPU test for CSV ingest (matrix_read_csv_column + CPUMockEngine::load_column_from_csv).
#include "server.hpp"        // pulls in compute_mock.cpp (engine) — execute_query, MatrixQueryStatus
#include "csv_ingest.hpp"
#include <cassert>
#include <cstdint>
#include <cstdio>
#include <fstream>
#include <string>
#include <vector>
#include <iostream>

static std::string write_tmp(const std::string& name, const std::string& body) {
    const std::string path = "/tmp/" + name;
    std::ofstream(path) << body;
    return path;
}

static void test_parser() {
    std::vector<uint32_t> out;

    // Basic: two columns, pick each.
    std::string p = write_tmp("mdb_csv_basic.csv", "1,10\n2,20\n3,30\n");
    assert(matrix_read_csv_column(p, 1, false, ',', out) && out == (std::vector<uint32_t>{10, 20, 30}));
    assert(matrix_read_csv_column(p, 0, false, ',', out) && out == (std::vector<uint32_t>{1, 2, 3}));

    // Header skip.
    p = write_tmp("mdb_csv_hdr.csv", "key,val\n5,50\n6,60\n");
    assert(matrix_read_csv_column(p, 1, true, ',', out) && out == (std::vector<uint32_t>{50, 60}));

    // Custom delimiter.
    p = write_tmp("mdb_csv_semi.csv", "7;70\n8;80\n");
    assert(matrix_read_csv_column(p, 1, false, ';', out) && out == (std::vector<uint32_t>{70, 80}));

    // Empty file -> true, empty. Header-only -> true, empty.
    p = write_tmp("mdb_csv_empty.csv", "");
    assert(matrix_read_csv_column(p, 0, false, ',', out) && out.empty());
    p = write_tmp("mdb_csv_hdronly.csv", "k,v\n");
    assert(matrix_read_csv_column(p, 0, true, ',', out) && out.empty());

    // CRLF tolerance.
    p = write_tmp("mdb_csv_crlf.csv", "1,2\r\n3,4\r\n");
    assert(matrix_read_csv_column(p, 0, false, ',', out) && out == (std::vector<uint32_t>{1, 3}));

    // Graceful failures: each returns false and clears out.
    assert(!matrix_read_csv_column("/tmp/mdb_csv_does_not_exist.csv", 0, false, ',', out));
    p = write_tmp("mdb_csv_short.csv", "1,2\n3\n");          // row 2 has no field index 1
    assert(!matrix_read_csv_column(p, 1, false, ',', out) && out.empty());
    p = write_tmp("mdb_csv_nonint.csv", "1,x\n");
    assert(!matrix_read_csv_column(p, 1, false, ',', out));
    p = write_tmp("mdb_csv_junk.csv", "12x\n");              // trailing junk -> not a full integer
    assert(!matrix_read_csv_column(p, 0, false, ',', out));
    p = write_tmp("mdb_csv_over.csv", "5000000000\n");       // > UINT32_MAX
    assert(!matrix_read_csv_column(p, 0, false, ',', out));
    std::cout << "[csv parser] ok\n";
}

static void test_engine_ingest() {
    const std::string wal = "/tmp/mdb_csv_eng.bin"; std::remove(wal.c_str());
    CPUMockEngine eng(0, wal);
    const std::string p = write_tmp("mdb_csv_eng.csv", "k,v\n10,100\n20,200\n30,300\n");

    assert(eng.load_column_from_csv(7, p, 1, /*has_header=*/true));      // values {100,200,300}
    MatrixQuery q{}; q.value_col = 7; q.agg = AGG_SUM;
    std::vector<uint64_t> r;
    assert(eng.execute_query(q, r) == MatrixQueryStatus::OK);
    assert(r.size() == 1 && r[0] == 600 && "SUM of ingested column");   // 100+200+300

    // Malformed CSV -> false, and NO catalog entry created (query on id 8 is unknown).
    const std::string bad = write_tmp("mdb_csv_bad.csv", "1,x\n");
    assert(!eng.load_column_from_csv(8, bad, 1));
    MatrixQuery q8{}; q8.value_col = 8; q8.agg = AGG_COUNT;
    std::vector<uint64_t> r8;
    assert(eng.execute_query(q8, r8) == MatrixQueryStatus::ERR_UNKNOWN_COLUMN && "no entry from bad CSV");
    std::remove(wal.c_str());
    std::cout << "[csv engine ingest] ok\n";
}

int main() {
    test_parser();
    test_engine_ingest();
    std::cout << "ALL CSV INGEST TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_checkpoint.cpp
// CPU test for WAL checkpoint / compaction (DU-4): snapshot kv_ + truncate WAL; recover from both.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <cstdio>
#include <string>
#include <vector>
#include <iostream>

// Remove a WAL path and its derived checkpoint/temp files so each sub-test starts clean.
static void clean(const std::string& wal) {
    std::remove(wal.c_str());
    std::remove((wal + ".ckpt").c_str());
    std::remove((wal + ".ckpt.tmp").c_str());
}

// Write keys [lo, hi) as value=key*10+1, each in its own committed transaction.
static void write_range(CPUMockEngine& e, uint64_t lo, uint64_t hi) {
    for (uint64_t k = lo; k < hi; ++k) { e.begin(); e.txn_put(k, k * 10 + 1); e.commit(); }
}

static void test_wal_shrinks_and_recovers() {
    const std::string wal = "/tmp/mdb_ckpt_a.wal"; clean(wal);
    {
        CPUMockEngine e(0, wal);
        e.begin(); for (uint64_t k = 0; k < 100; ++k) e.txn_put(k, k * 10 + 1); e.commit();
        assert(e.wal_records() == 101 && "100 txn-puts + 1 commit marker");
        e.checkpoint();
        assert(e.wal_records() == 0 && e.checkpoints() == 1 && "WAL truncated after checkpoint");
        write_range(e, 100, 110);                       // 10 more txns post-checkpoint
        assert(e.wal_records() == 20 && "10 txn-puts + 10 commit markers since checkpoint");
    }
    {   // restart on the same WAL: checkpoint (0..99) + replayed WAL (100..109)
        CPUMockEngine e(0, wal);
        for (uint64_t k = 0; k < 110; ++k) {
            uint64_t v = 0; assert(e.kv_get(k, v) && v == k * 10 + 1 && "all 110 keys survive restart");
        }
    }
    clean(wal);
    std::cout << "[checkpoint shrink+recover] ok\n";
}

static void test_checkpoint_alone_carries_data() {   // NON-VACUITY: data must come from the checkpoint file
    const std::string wal = "/tmp/mdb_ckpt_b.wal"; clean(wal);
    {
        CPUMockEngine e(0, wal);
        e.begin(); for (uint64_t k = 0; k < 100; ++k) e.txn_put(k, k * 10 + 1); e.commit();
        e.checkpoint();                                 // WAL now empty; data only in the .ckpt file
        assert(e.wal_records() == 0);
    }
    {
        CPUMockEngine e(0, wal);                        // replay sees an EMPTY WAL — only the checkpoint can restore
        size_t found = 0;
        for (uint64_t k = 0; k < 100; ++k) { uint64_t v = 0; if (e.kv_get(k, v) && v == k * 10 + 1) ++found; }
        assert(found == 100 && "checkpoint file alone restored every key");
    }
    clean(wal);
    std::cout << "[checkpoint alone] ok\n";
}

static void test_idempotent_overlap() {                 // a key written before AND after checkpoint
    const std::string wal = "/tmp/mdb_ckpt_c.wal"; clean(wal);
    {
        CPUMockEngine e(0, wal);
        e.begin(); e.txn_put(7, 700); e.commit();
        e.checkpoint();
        e.begin(); e.txn_put(7, 777); e.commit();       // post-checkpoint overwrite
    }
    {
        CPUMockEngine e(0, wal);
        uint64_t v = 0; assert(e.kv_get(7, v) && v == 777 && "last write wins; checkpoint value not stale/doubled");
    }
    clean(wal);
    std::cout << "[idempotent overlap] ok\n";
}

static void test_no_wal_noop() {
    CPUMockEngine e(0, "");                              // durability off
    e.checkpoint();                                     // must not crash
    assert(e.checkpoints() == 0 && e.wal_records() == 0 && "checkpoint is a no-op without a WAL");
    std::cout << "[no-wal no-op] ok\n";
}

int main() {
    test_wal_shrinks_and_recovers();
    test_checkpoint_alone_carries_data();
    test_idempotent_overlap();
    test_no_wal_noop();
    std::cout << "ALL CHECKPOINT TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_query_predicates.cpp
// CPU test for richer scan predicates (QRY-3): MatrixCmp / MatrixPredicate, matrix_cpu_reduce_pred,
// and execute_query with GE/LT/LE/EQ/NE/BETWEEN. Oracles compute the comparison EXPLICITLY (not via
// matrix_pred_match) so a bug in the predicate dispatch is caught independently.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

// Independent predicate evaluation (explicit ops; does NOT call matrix_pred_match).
static bool ref_match(uint32_t v, MatrixCmp c, uint32_t a, uint32_t b) {
    switch (c) {
        case MatrixCmp::GT: return v > a;   case MatrixCmp::GE: return v >= a;
        case MatrixCmp::LT: return v < a;   case MatrixCmp::LE: return v <= a;
        case MatrixCmp::EQ: return v == a;  case MatrixCmp::NE: return v != a;
        case MatrixCmp::BETWEEN: return v >= a && v <= b;
    }
    return false;
}
static uint64_t ref_reduce(const std::vector<uint32_t>& v, MatrixCmp c, uint32_t a, uint32_t b, MatrixAggOp op) {
    uint64_t cnt = 0, sum = 0, mn = UINT64_MAX, mx = 0;
    for (uint32_t x : v) if (ref_match(x, c, a, b)) { ++cnt; sum += x; if (x < mn) mn = x; if (x > mx) mx = x; }
    switch (op) { case AGG_SUM: return sum; case AGG_MIN: return mn; case AGG_MAX: return mx;
                  case AGG_COUNT: default: return cnt; }
}

static void test_pred_match() {
    assert(matrix_pred_match(6, {MatrixCmp::GT, 5, 0}) && !matrix_pred_match(5, {MatrixCmp::GT, 5, 0}));
    assert(matrix_pred_match(5, {MatrixCmp::GE, 5, 0}) && !matrix_pred_match(4, {MatrixCmp::GE, 5, 0}));
    assert(matrix_pred_match(4, {MatrixCmp::LT, 5, 0}) && !matrix_pred_match(5, {MatrixCmp::LT, 5, 0}));
    assert(matrix_pred_match(5, {MatrixCmp::LE, 5, 0}) && !matrix_pred_match(6, {MatrixCmp::LE, 5, 0}));
    assert(matrix_pred_match(5, {MatrixCmp::EQ, 5, 0}) && !matrix_pred_match(6, {MatrixCmp::EQ, 5, 0}));
    assert(matrix_pred_match(6, {MatrixCmp::NE, 5, 0}) && !matrix_pred_match(5, {MatrixCmp::NE, 5, 0}));
    assert(matrix_pred_match(3, {MatrixCmp::BETWEEN, 3, 7}) && matrix_pred_match(7, {MatrixCmp::BETWEEN, 3, 7}));
    assert(!matrix_pred_match(2, {MatrixCmp::BETWEEN, 3, 7}) && !matrix_pred_match(8, {MatrixCmp::BETWEEN, 3, 7}));
    std::cout << "[pred match] ok\n";
}

static void test_reduce_pred() {
    const std::vector<uint32_t> v = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 5, 5};
    const std::pair<MatrixCmp, std::pair<uint32_t,uint32_t>> cases[] = {
        {MatrixCmp::GT,{5,0}}, {MatrixCmp::GE,{5,0}}, {MatrixCmp::LT,{5,0}}, {MatrixCmp::LE,{5,0}},
        {MatrixCmp::EQ,{5,0}}, {MatrixCmp::NE,{5,0}}, {MatrixCmp::BETWEEN,{3,7}}, {MatrixCmp::LT,{1,0}} };
    for (auto& cs : cases)
        for (MatrixAggOp op : {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX}) {
            MatrixPredicate p{cs.first, cs.second.first, cs.second.second};
            assert(matrix_cpu_reduce_pred(v.data(), v.size(), p, op)
                   == ref_reduce(v, cs.first, cs.second.first, cs.second.second, op));
        }
    // MAX-0 caveat: LT 1 matches only the value 0 -> MAX returns 0; COUNT distinguishes (==1, not empty).
    MatrixPredicate lt1{MatrixCmp::LT, 1, 0};
    assert(matrix_cpu_reduce_pred(v.data(), v.size(), lt1, AGG_MAX) == 0);
    assert(matrix_cpu_reduce_pred(v.data(), v.size(), lt1, AGG_COUNT) == 1);
    std::cout << "[reduce pred] ok\n";
}

static void test_execute_query_scalar() {
    std::vector<uint32_t> v(200);
    for (size_t i = 0; i < v.size(); ++i) v[i] = static_cast<uint32_t>(i % 50);   // 0..49 repeating
    CPUMockEngine eng;
    eng.load_scan_column(2, v.data(), v.size());
    struct Case { MatrixCmp c; uint32_t a, b; MatrixAggOp op; };
    const Case cases[] = {
        {MatrixCmp::LT, 10, 0, AGG_COUNT}, {MatrixCmp::EQ, 7, 0, AGG_COUNT}, {MatrixCmp::NE, 7, 0, AGG_SUM},
        {MatrixCmp::BETWEEN, 20, 30, AGG_SUM}, {MatrixCmp::GE, 45, 0, AGG_MIN}, {MatrixCmp::LE, 5, 0, AGG_MAX} };
    for (auto& cs : cases) {
        MatrixQuery q{}; q.value_col = 2; q.agg = cs.op; q.has_filter = true;
        q.cmp = cs.c; q.threshold = cs.a; q.upper = cs.b;
        std::vector<uint64_t> out;
        assert(eng.execute_query(q, out) == MatrixQueryStatus::OK && out.size() == 1);
        assert(out[0] == ref_reduce(v, cs.c, cs.a, cs.b, cs.op) && "scalar predicate query matches oracle");
    }
    // Backward-compat: default cmp (GT) == old value>threshold.
    MatrixQuery old_style{}; old_style.value_col = 2; old_style.agg = AGG_COUNT; old_style.has_filter = true; old_style.threshold = 25;
    std::vector<uint64_t> o1; eng.execute_query(old_style, o1);
    assert(o1[0] == ref_reduce(v, MatrixCmp::GT, 25, 0, AGG_COUNT) && "default cmp is GT");
    // Non-vacuity: EQ 25 differs from GT 25.
    MatrixQuery eq{}; eq.value_col = 2; eq.agg = AGG_COUNT; eq.has_filter = true; eq.cmp = MatrixCmp::EQ; eq.threshold = 25;
    std::vector<uint64_t> o2; eng.execute_query(eq, o2);
    assert(o2[0] != o1[0] && "EQ is actually applied, not treated as GT");
    std::cout << "[execute_query scalar] ok\n";
}

static void test_execute_query_grouped() {
    std::vector<uint32_t> keys(300), vals(300);
    for (size_t i = 0; i < 300; ++i) { keys[i] = static_cast<uint32_t>(i % 4); vals[i] = static_cast<uint32_t>(i % 60); }
    CPUMockEngine eng;
    eng.load_scan_column(1, keys.data(), keys.size());
    eng.load_scan_column(2, vals.data(), vals.size());
    MatrixQuery q{}; q.value_col = 2; q.key_col = 1; q.num_groups = 4; q.agg = AGG_SUM;
    q.grouped = true; q.has_filter = true; q.cmp = MatrixCmp::BETWEEN; q.threshold = 20; q.upper = 40;
    std::vector<uint64_t> out;
    assert(eng.execute_query(q, out) == MatrixQueryStatus::OK && out.size() == 4);
    uint64_t ref[4] = {0, 0, 0, 0};
    for (size_t i = 0; i < 300; ++i) if (vals[i] >= 20 && vals[i] <= 40) ref[keys[i]] += vals[i];
    for (int g = 0; g < 4; ++g) assert(out[g] == ref[g] && "grouped BETWEEN matches oracle");
    std::cout << "[execute_query grouped] ok\n";
}

int main() {
    test_pred_match();
    test_reduce_pred();
    test_execute_query_scalar();
    test_execute_query_grouped();
    std::cout << "ALL PREDICATE TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_typed_columns.cpp
// CPU test for int64 typed columns (DM-3a): MatrixType, matrix_cpu_reduce_all_i64,
// load_scan_column_i64, and execute_query int64 scalar dispatch (incl. graceful ERR_UNSUPPORTED_TYPE).
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

// Hand oracle over the int64 data (independent of the SUT reducer). [[maybe_unused]]: unreferenced under -DNDEBUG.
[[maybe_unused]] static int64_t oracle(const std::vector<int64_t>& v, MatrixAggOp op) {
    int64_t cnt = static_cast<int64_t>(v.size()), sum = 0, mn = INT64_MAX, mx = INT64_MIN;
    for (int64_t x : v) { sum += x; if (x < mn) mn = x; if (x > mx) mx = x; }
    switch (op) { case AGG_SUM: return sum; case AGG_MIN: return mn; case AGG_MAX: return mx;
                  case AGG_COUNT: default: return cnt; }
}

static void test_reduce_i64() {
    const std::vector<int64_t> v = {-7, 0, 5, 5000000000LL, -3, 2147483648LL};   // negatives + > UINT32_MAX
    for ([[maybe_unused]] MatrixAggOp op : {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX})
        assert(matrix_cpu_reduce_all_i64(v.data(), v.size(), op) == oracle(v, op));
    // Empty-set sentinels.
    assert(matrix_cpu_reduce_all_i64(nullptr, 0, AGG_COUNT) == 0);
    assert(matrix_cpu_reduce_all_i64(nullptr, 0, AGG_SUM) == 0);
    assert(matrix_cpu_reduce_all_i64(nullptr, 0, AGG_MIN) == INT64_MAX);
    assert(matrix_cpu_reduce_all_i64(nullptr, 0, AGG_MAX) == INT64_MIN);
    std::cout << "[reduce i64] ok\n";
}

static void test_engine_i64() {
    const std::vector<int64_t> v = {-7, 0, 5, 5000000000LL, -3, 2147483648LL, 100, -100};
    CPUMockEngine eng;
    eng.load_scan_column_i64(7, v.data(), v.size());
    assert(eng.column_type(7) == MatrixType::I64);
    for (MatrixAggOp op : {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX}) {
        MatrixQuery q{}; q.value_col = 7; q.agg = op;
        std::vector<uint64_t> out;
        assert(eng.execute_query(q, out) == MatrixQueryStatus::OK && out.size() == 1);
        assert(static_cast<int64_t>(out[0]) == oracle(v, op) && "int64 scalar aggregate matches oracle");
    }
    // The > UINT32_MAX value proves genuine 64-bit: a uint32 read of 5000000000 would be 705032704.
    { MatrixQuery q{}; q.value_col = 7; q.agg = AGG_MAX; std::vector<uint64_t> o; eng.execute_query(q, o);
      assert(static_cast<int64_t>(o[0]) == 5000000000LL && static_cast<int64_t>(o[0]) != 705032704); }
    // DM-3b: filtered int64 now works (default cmp GT, lo_i64 0 -> value > 0); grouped int64 stays unsupported (DM-3c).
    { MatrixQuery q{}; q.value_col = 7; q.agg = AGG_SUM; q.has_filter = true; q.lo_i64 = 0;   // WHERE value > 0
      std::vector<uint64_t> o; assert(eng.execute_query(q, o) == MatrixQueryStatus::OK
        && static_cast<int64_t>(o[0]) == 5000000000LL + 2147483648LL + 5 + 100); }   // 5,5e9,2147483648,100
    { MatrixQuery q{}; q.value_col = 7; q.agg = AGG_COUNT; q.grouped = true; q.key_col = 7; q.num_groups = 2;
      std::vector<uint64_t> o; assert(eng.execute_query(q, o) == MatrixQueryStatus::ERR_UNSUPPORTED_TYPE); }
    std::cout << "[engine i64] ok\n";
}

// Regression for the typed-GROUP-BY-key corruption path: an int64 key (N rows = 8N bytes) has the SAME
// byte length as a uint32 value of 2N rows, so it passes execute_query's length guard — but reading the
// int64 key bytes as uint32 would invent phantom groups. Must be rejected, not silently mis-grouped.
static void test_typed_key_rejected() {
    std::vector<int64_t> keys = {0, 1, 2, 3};                     // 4 rows  -> 32 bytes
    std::vector<uint32_t> vals = {5, 6, 7, 8, 9, 10, 11, 12};     // 8 rows  -> 32 bytes (equal length)
    CPUMockEngine eng;
    eng.load_scan_column_i64(8, keys.data(), keys.size());
    eng.load_scan_column(9, vals.data(), vals.size());
    MatrixQuery q{}; q.value_col = 9; q.key_col = 8; q.num_groups = 4; q.agg = AGG_COUNT; q.grouped = true;
    std::vector<uint64_t> out;
    assert(eng.execute_query(q, out) == MatrixQueryStatus::ERR_UNSUPPORTED_TYPE
           && out.empty() && "int64 GROUP BY key rejected, not reinterpreted as uint32");
    std::cout << "[typed key rejected] ok\n";
}

static void test_u32_untouched() {
    std::vector<uint32_t> v(100);
    for (size_t i = 0; i < v.size(); ++i) v[i] = static_cast<uint32_t>(i);
    CPUMockEngine eng;
    eng.load_scan_column(3, v.data(), v.size());
    assert(eng.column_type(3) == MatrixType::U32 && "untagged column defaults to U32");
    MatrixQuery q{}; q.value_col = 3; q.agg = AGG_SUM; std::vector<uint64_t> out;
    assert(eng.execute_query(q, out) == MatrixQueryStatus::OK && out[0] == 4950);   // 0+...+99
    MatrixQuery qf{}; qf.value_col = 3; qf.agg = AGG_COUNT; qf.has_filter = true; qf.threshold = 50;
    std::vector<uint64_t> of; assert(eng.execute_query(qf, of) == MatrixQueryStatus::OK && of[0] == 49); // 51..99
    std::cout << "[u32 untouched] ok\n";
}

int main() {
    test_reduce_i64();
    test_engine_i64();
    test_typed_key_rejected();
    test_u32_untouched();
    std::cout << "ALL TYPED-COLUMN TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_typed_predicates.cpp
// CPU test for int64 filtered aggregation (DM-3b): MatrixPredicateI64, matrix_pred_match_i64,
// matrix_cpu_reduce_pred_i64, execute_query int64 filtered dispatch, and int64 scans driving rebalance.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

// Independent signed predicate (explicit ops; does NOT call matrix_pred_match_i64).
static bool ref_match(int64_t v, MatrixCmp c, int64_t a, int64_t b) {
    switch (c) {
        case MatrixCmp::GT: return v > a;   case MatrixCmp::GE: return v >= a;
        case MatrixCmp::LT: return v < a;   case MatrixCmp::LE: return v <= a;
        case MatrixCmp::EQ: return v == a;  case MatrixCmp::NE: return v != a;
        case MatrixCmp::BETWEEN: return v >= a && v <= b;
    }
    return false;
}
static int64_t ref_reduce(const std::vector<int64_t>& v, MatrixCmp c, int64_t a, int64_t b, MatrixAggOp op) {
    int64_t cnt = 0, sum = 0, mn = INT64_MAX, mx = INT64_MIN;
    for (int64_t x : v) if (ref_match(x, c, a, b)) { ++cnt; sum += x; if (x < mn) mn = x; if (x > mx) mx = x; }
    switch (op) { case AGG_SUM: return sum; case AGG_MIN: return mn; case AGG_MAX: return mx;
                  case AGG_COUNT: default: return cnt; }
}

static void test_pred_match_i64() {
    assert(matrix_pred_match_i64(-4, {MatrixCmp::GT, -5, 0}) && !matrix_pred_match_i64(-5, {MatrixCmp::GT, -5, 0}));
    assert(matrix_pred_match_i64(-1, {MatrixCmp::LT, 0, 0}) && !matrix_pred_match_i64(0, {MatrixCmp::LT, 0, 0}));
    assert(matrix_pred_match_i64(-10, {MatrixCmp::BETWEEN, -10, 10}) && matrix_pred_match_i64(10, {MatrixCmp::BETWEEN, -10, 10}));
    assert(!matrix_pred_match_i64(-11, {MatrixCmp::BETWEEN, -10, 10}) && !matrix_pred_match_i64(11, {MatrixCmp::BETWEEN, -10, 10}));
    assert(matrix_pred_match_i64(5000000000LL, {MatrixCmp::GT, 4000000000LL, 0})
           && !matrix_pred_match_i64(3000000000LL, {MatrixCmp::GT, 4000000000LL, 0}));   // > UINT32_MAX bound
    assert(matrix_pred_match_i64(7, {MatrixCmp::GE, 7, 0}) && matrix_pred_match_i64(7, {MatrixCmp::LE, 7, 0})
           && matrix_pred_match_i64(7, {MatrixCmp::EQ, 7, 0}) && !matrix_pred_match_i64(7, {MatrixCmp::NE, 7, 0}));
    std::cout << "[pred match i64] ok\n";
}

static void test_reduce_pred_i64() {
    const std::vector<int64_t> v = {-7, 0, 5, 5000000000LL, -3, 2147483648LL, 100, -100, 5};
    const std::pair<MatrixCmp, std::pair<int64_t,int64_t>> cases[] = {
        {MatrixCmp::GT,{0,0}}, {MatrixCmp::LT,{0,0}}, {MatrixCmp::GE,{5,0}}, {MatrixCmp::LE,{-3,0}},
        {MatrixCmp::EQ,{5,0}}, {MatrixCmp::NE,{5,0}}, {MatrixCmp::BETWEEN,{-7,100}}, {MatrixCmp::GT,{4000000000LL,0}} };
    for (auto& cs : cases)
        for (MatrixAggOp op : {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX}) {
            MatrixPredicateI64 p{cs.first, cs.second.first, cs.second.second};
            assert(matrix_cpu_reduce_pred_i64(v.data(), v.size(), p, op)
                   == ref_reduce(v, cs.first, cs.second.first, cs.second.second, op));
        }
    std::cout << "[reduce pred i64] ok\n";
}

static void test_engine_i64_filtered() {
    const std::vector<int64_t> v = {-7, 0, 5, 5000000000LL, -3, 2147483648LL, 100, -100, 5};
    CPUMockEngine eng;
    eng.load_scan_column_i64(7, v.data(), v.size());
    struct Case { MatrixCmp c; int64_t a, b; MatrixAggOp op; };
    const Case cases[] = {
        {MatrixCmp::LT, 0, 0, AGG_COUNT}, {MatrixCmp::EQ, 5, 0, AGG_COUNT}, {MatrixCmp::GE, 100, 0, AGG_SUM},
        {MatrixCmp::BETWEEN, -7, 100, AGG_SUM}, {MatrixCmp::GT, 4000000000LL, 0, AGG_MAX} };
    for (auto& cs : cases) {
        MatrixQuery q{}; q.value_col = 7; q.agg = cs.op; q.has_filter = true;
        q.cmp = cs.c; q.lo_i64 = cs.a; q.hi_i64 = cs.b;
        std::vector<uint64_t> out;
        assert(eng.execute_query(q, out) == MatrixQueryStatus::OK && out.size() == 1);
        assert(static_cast<int64_t>(out[0]) == ref_reduce(v, cs.c, cs.a, cs.b, cs.op) && "int64 filtered matches oracle");
    }
    // Non-vacuity: a > UINT32_MAX bound is honored as int64 (only 5000000000 exceeds it -> MAX is it).
    { MatrixQuery q{}; q.value_col = 7; q.agg = AGG_MAX; q.has_filter = true; q.cmp = MatrixCmp::GT; q.lo_i64 = 4000000000LL;
      std::vector<uint64_t> o; eng.execute_query(q, o); assert(static_cast<int64_t>(o[0]) == 5000000000LL); }
    // EQ differs from GT (predicate actually applied).
    { MatrixQuery eq{}; eq.value_col = 7; eq.agg = AGG_COUNT; eq.has_filter = true; eq.cmp = MatrixCmp::EQ; eq.lo_i64 = 5;
      MatrixQuery gt{}; gt.value_col = 7; gt.agg = AGG_COUNT; gt.has_filter = true; gt.cmp = MatrixCmp::GT; gt.lo_i64 = 5;
      std::vector<uint64_t> a, b; eng.execute_query(eq, a); eng.execute_query(gt, b); assert(a[0] != b[0]); }
    // Grouped int64 still rejected (DM-3c).
    { MatrixQuery q{}; q.value_col = 7; q.agg = AGG_COUNT; q.grouped = true; q.key_col = 7; q.num_groups = 2;
      std::vector<uint64_t> o; assert(eng.execute_query(q, o) == MatrixQueryStatus::ERR_UNSUPPORTED_TYPE); }
    std::cout << "[engine i64 filtered] ok\n";
}

static void test_i64_drives_rebalance() {
    std::vector<int64_t> v(1000, 1);
    CPUMockEngine eng;
    eng.load_scan_column_i64(7, v.data(), v.size());
    for (int i = 0; i < 5; ++i) {   // > REBALANCE_EVERY (4) int64 scalar queries
        MatrixQuery q{}; q.value_col = 7; q.agg = AGG_SUM; std::vector<uint64_t> o; eng.execute_query(q, o);
    }
    assert(eng.stats().rebalances >= 1 && "int64 scans drive the rebalance cadence (DM-3a follow-up)");
    std::cout << "[i64 drives rebalance] ok\n";
}

int main() {
    test_pred_match_i64();
    test_reduce_pred_i64();
    test_engine_i64_filtered();
    test_i64_drives_rebalance();
    std::cout << "ALL TYPED-PREDICATE TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_typed_grouped.cpp
// CPU test for grouped int64 aggregation (DM-3c): matrix_group_reduce_i64[_pred], grouped_aggregate_i64,
// execute_query int64 GROUP BY (u32 key, int64 value), incl. mixed-width + int64-key-rejection guards.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

static bool refm(int64_t v, MatrixCmp c, int64_t a, int64_t b) {
    switch (c) { case MatrixCmp::GT: return v>a; case MatrixCmp::GE: return v>=a; case MatrixCmp::LT: return v<a;
                 case MatrixCmp::LE: return v<=a; case MatrixCmp::EQ: return v==a; case MatrixCmp::NE: return v!=a;
                 case MatrixCmp::BETWEEN: return v>=a && v<=b; } return false; }
// Brute grouped oracle: out[g] for the given op over rows matching (filtered ? pred : all).
static void ref_group(const std::vector<uint32_t>& keys, const std::vector<int64_t>& vals, uint32_t ng,
                      MatrixAggOp op, bool filt, MatrixCmp c, int64_t a, int64_t b, std::vector<int64_t>& out) {
    out.assign(ng, op == AGG_MIN ? INT64_MAX : (op == AGG_MAX ? INT64_MIN : 0));
    for (size_t i = 0; i < keys.size(); ++i) {
        if (keys[i] >= ng) continue;
        if (filt && !refm(vals[i], c, a, b)) continue;
        int64_t& o = out[keys[i]];
        switch (op) { case AGG_SUM: o += vals[i]; break; case AGG_MIN: if (vals[i]<o) o=vals[i]; break;
                      case AGG_MAX: if (vals[i]>o) o=vals[i]; break; case AGG_COUNT: default: o += 1; break; } }
}

static void test_group_reduce_i64() {
    // keys 0..2; group 1 is ALL NEGATIVE (the MAX-init guard); group 2 has a > UINT32_MAX value.
    std::vector<uint32_t> keys = {0, 1, 1, 2, 0, 1, 2};
    std::vector<int64_t>  vals = {5, -3, -100, 5000000000LL, 7, -1, -2};
    for (bool filt : {false, true})
        for (MatrixAggOp op : {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX}) {
            std::vector<int64_t> got(3), exp;
            MatrixPredicateI64 p{MatrixCmp::GE, -3, 0};   // filt: value >= -3
            if (filt) matrix_cpu_group_reduce_i64_pred(keys.data(), vals.data(), keys.size(), 3, op, p, got.data());
            else      matrix_cpu_group_reduce_i64(keys.data(), vals.data(), keys.size(), 3, op, got.data());
            ref_group(keys, vals, 3, op, filt, MatrixCmp::GE, -3, 0, exp);
            assert(got == exp && "grouped int64 reduce matches oracle");
        }
    // Explicit MAX-init guard: group 1 (all negative) MAX must be -3, NOT 0.
    std::vector<int64_t> mx(3);
    matrix_cpu_group_reduce_i64(keys.data(), vals.data(), keys.size(), 3, AGG_MAX, mx.data());
    assert(mx[1] == -1 && "MAX of a negative group is the max negative (-1), not 0");   // group1 vals {-3,-100,-1}
    std::cout << "[group reduce i64] ok\n";
}

static void test_engine_grouped_i64() {
    std::vector<uint32_t> keys = {0, 1, 1, 2, 0, 1, 2};
    std::vector<int64_t>  vals = {5, -3, -100, 5000000000LL, 7, -1, -2};
    CPUMockEngine eng;
    eng.load_scan_column(1, keys.data(), keys.size());        // u32 key
    eng.load_scan_column_i64(7, vals.data(), vals.size());    // int64 value (equal ROW count, 4N vs 8N bytes)
    for (bool filt : {false, true})
        for (MatrixAggOp op : {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX}) {
            MatrixQuery q{}; q.value_col = 7; q.key_col = 1; q.num_groups = 3; q.agg = op; q.grouped = true;
            q.has_filter = filt; q.cmp = MatrixCmp::GE; q.lo_i64 = -3;
            std::vector<uint64_t> out;
            assert(eng.execute_query(q, out) == MatrixQueryStatus::OK && out.size() == 3);
            std::vector<int64_t> exp; ref_group(keys, vals, 3, op, filt, MatrixCmp::GE, -3, 0, exp);
            for (uint32_t g = 0; g < 3; ++g) assert(static_cast<int64_t>(out[g]) == exp[g] && "engine grouped int64 == oracle");
        }
    // Non-vacuity: group 2's SUM includes the > UINT32_MAX value (genuine int64).
    { MatrixQuery q{}; q.value_col = 7; q.key_col = 1; q.num_groups = 3; q.agg = AGG_SUM; q.grouped = true;
      std::vector<uint64_t> o; eng.execute_query(q, o); assert(static_cast<int64_t>(o[2]) == 5000000000LL - 2); }
    std::cout << "[engine grouped i64] ok\n";
}

static void test_grouped_i64_guards() {
    std::vector<uint32_t> keys = {0, 1, 0, 1};
    std::vector<int64_t>  vals = {1, 2, 3, 4};
    CPUMockEngine eng;
    eng.load_scan_column(1, keys.data(), keys.size());        // u32 key, 4 rows
    eng.load_scan_column_i64(7, vals.data(), vals.size());    // i64 value, 4 rows (equal ROW count)
    // int64 KEY rejected: i64 value + i64 key.
    std::vector<int64_t> keys64 = {0, 1, 0, 1};
    eng.load_scan_column_i64(8, keys64.data(), keys64.size());
    { MatrixQuery q{}; q.value_col = 7; q.key_col = 8; q.num_groups = 2; q.agg = AGG_COUNT; q.grouped = true;
      std::vector<uint64_t> o; assert(eng.execute_query(q, o) == MatrixQueryStatus::ERR_UNSUPPORTED_TYPE && "int64 key rejected"); }
    // Row-count mismatch -> ERR_INVALID_GROUP (i64 value 4 rows, u32 key 5 rows).
    std::vector<uint32_t> keys5 = {0, 1, 0, 1, 0};
    eng.load_scan_column(2, keys5.data(), keys5.size());
    { MatrixQuery q{}; q.value_col = 7; q.key_col = 2; q.num_groups = 2; q.agg = AGG_COUNT; q.grouped = true;
      std::vector<uint64_t> o; assert(eng.execute_query(q, o) == MatrixQueryStatus::ERR_INVALID_GROUP && "row-count mismatch rejected"); }
    std::cout << "[grouped i64 guards] ok\n";
}

int main() {
    test_group_reduce_i64();
    test_engine_grouped_i64();
    test_grouped_i64_guards();
    std::cout << "ALL TYPED-GROUPED TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_typed_snapshot.cpp
// CPU test for typed catalog snapshot (DM-3d): save_catalog/load_catalog round-trip a mixed
// u32 + int64 catalog (incl. negatives and > UINT32_MAX), with types and values preserved.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <cstdio>
#include <vector>
#include <iostream>

static int64_t i64_sum(const std::vector<int64_t>& v) { int64_t s = 0; for (int64_t x : v) s += x; return s; }
static uint64_t u32_sum(const std::vector<uint32_t>& v) { uint64_t s = 0; for (uint32_t x : v) s += x; return s; }

static void test_mixed_roundtrip() {
    const std::string path = "/tmp/mdb_typed_catalog.bin"; std::remove(path.c_str());
    std::vector<uint32_t> u(100); for (size_t i = 0; i < u.size(); ++i) u[i] = static_cast<uint32_t>(i);
    std::vector<int64_t>  s = {-7, 0, 5, 5000000000LL, -3, 2147483648LL, 100, -100};
    {
        CPUMockEngine eng;
        eng.load_scan_column(3, u.data(), u.size());          // u32
        eng.load_scan_column_i64(7, s.data(), s.size());      // int64 (negatives + > UINT32_MAX)
        eng.save_catalog(path);
    }
    {
        CPUMockEngine eng;                                    // fresh engine
        eng.load_catalog(path);
        assert(eng.column_type(3) == MatrixType::U32 && eng.column_type(7) == MatrixType::I64 && "types restored");
        // u32 column value check
        MatrixQuery qu{}; qu.value_col = 3; qu.agg = AGG_SUM; std::vector<uint64_t> ou;
        assert(eng.execute_query(qu, ou) == MatrixQueryStatus::OK && ou[0] == u32_sum(u) && "u32 column restored");
        // int64 column value checks (SUM/MIN/MAX) — the negatives + large value must survive
        MatrixQuery qs{}; qs.value_col = 7; qs.agg = AGG_SUM; std::vector<uint64_t> os;
        assert(eng.execute_query(qs, os) == MatrixQueryStatus::OK && static_cast<int64_t>(os[0]) == i64_sum(s) && "int64 SUM restored");
        MatrixQuery qmax{}; qmax.value_col = 7; qmax.agg = AGG_MAX; std::vector<uint64_t> omax;
        eng.execute_query(qmax, omax); assert(static_cast<int64_t>(omax[0]) == 5000000000LL && "int64 MAX (>UINT32_MAX) restored");
        MatrixQuery qmin{}; qmin.value_col = 7; qmin.agg = AGG_MIN; std::vector<uint64_t> omin;
        eng.execute_query(qmin, omin); assert(static_cast<int64_t>(omin[0]) == -100 && "int64 MIN (negative) restored");
    }
    std::remove(path.c_str());
    std::cout << "[mixed catalog roundtrip] ok\n";
}

static void test_empty_catalog() {
    const std::string path = "/tmp/mdb_empty_catalog.bin"; std::remove(path.c_str());
    { CPUMockEngine eng; eng.save_catalog(path); }
    { CPUMockEngine eng; eng.load_catalog(path); }   // must not crash
    std::remove(path.c_str());
    std::cout << "[empty catalog] ok\n";
}

int main() {
    test_mixed_roundtrip();
    test_empty_catalog();
    std::cout << "ALL TYPED-SNAPSHOT TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_typed_double.cpp
// CPU test for double (float64) columns (DM-3e): MatrixType::F64, matrix_cpu_reduce_all_f64 / _pred,
// matrix_pred_match_f64 (incl. NaN), load_scan_column_f64, execute_query scalar dispatch, durability.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <cstdio>
#include <bit>
#include <cmath>
#include <limits>
#include <vector>
#include <iostream>

static bool refm(double v, MatrixCmp c, double a, double b) {
    switch (c) { case MatrixCmp::GT: return v>a; case MatrixCmp::GE: return v>=a; case MatrixCmp::LT: return v<a;
                 case MatrixCmp::LE: return v<=a; case MatrixCmp::EQ: return v==a; case MatrixCmp::NE: return v!=a;
                 case MatrixCmp::BETWEEN: return v>=a && v<=b; } return false; }
static double ref_reduce(const std::vector<double>& v, bool filt, MatrixCmp c, double a, double b, MatrixAggOp op) {
    double cnt=0, sum=0, mn=std::numeric_limits<double>::infinity(), mx=-std::numeric_limits<double>::infinity();
    for (double x : v) if (!filt || refm(x,c,a,b)) { cnt+=1; sum+=x; if (x<mn) mn=x; if (x>mx) mx=x; }
    switch (op) { case AGG_SUM: return sum; case AGG_MIN: return mn; case AGG_MAX: return mx; case AGG_COUNT: default: return cnt; } }

// Exactly-representable doubles + matching order -> == is exact.
static const std::vector<double> V = {1.5, -3.0, 0.5, 2.25, 5000000000.0, -0.25, 100.0, -100.0};

static void test_reduce_f64() {
    for (MatrixAggOp op : {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX})
        assert(matrix_cpu_reduce_all_f64(V.data(), V.size(), op) == ref_reduce(V, false, MatrixCmp::GT, 0, 0, op));
    const std::pair<MatrixCmp, std::pair<double,double>> cs[] = {
        {MatrixCmp::LT,{0.0,0}}, {MatrixCmp::GE,{0.5,0}}, {MatrixCmp::EQ,{1.5,0}}, {MatrixCmp::BETWEEN,{-3.0,2.25}} };
    for (auto& c : cs) for (MatrixAggOp op : {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX}) {
        MatrixPredicateF64 p{c.first, c.second.first, c.second.second};
        assert(matrix_cpu_reduce_pred_f64(V.data(), V.size(), p, op) == ref_reduce(V, true, c.first, c.second.first, c.second.second, op)); }
    // empty sentinels
    assert(matrix_cpu_reduce_all_f64(nullptr, 0, AGG_MIN) == std::numeric_limits<double>::infinity());
    assert(matrix_cpu_reduce_all_f64(nullptr, 0, AGG_MAX) == -std::numeric_limits<double>::infinity());
    std::cout << "[reduce f64] ok\n";
}

static void test_pred_match_f64() {
    assert(matrix_pred_match_f64(1.6, {MatrixCmp::GT, 1.5, 0}) && !matrix_pred_match_f64(1.5, {MatrixCmp::GT, 1.5, 0}));
    assert(matrix_pred_match_f64(-0.25, {MatrixCmp::BETWEEN, -3.0, 0.0}) && !matrix_pred_match_f64(0.5, {MatrixCmp::BETWEEN, -3.0, 0.0}));
    const double nan = std::numeric_limits<double>::quiet_NaN();
    assert(!matrix_pred_match_f64(nan, {MatrixCmp::GT, 0, 0}) && !matrix_pred_match_f64(nan, {MatrixCmp::LE, 0, 0})
           && !matrix_pred_match_f64(nan, {MatrixCmp::EQ, nan, 0}) && matrix_pred_match_f64(nan, {MatrixCmp::NE, 0, 0}));
    std::cout << "[pred match f64] ok\n";
}

static void test_engine_f64() {
    CPUMockEngine eng;
    eng.load_scan_column_f64(7, V.data(), V.size());
    assert(eng.column_type(7) == MatrixType::F64);
    for (MatrixAggOp op : {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX}) {
        MatrixQuery q{}; q.value_col = 7; q.agg = op; std::vector<uint64_t> o;
        assert(eng.execute_query(q, o) == MatrixQueryStatus::OK && o.size() == 1);
        assert(std::bit_cast<double>(o[0]) == ref_reduce(V, false, MatrixCmp::GT, 0, 0, op) && "f64 scalar == oracle"); }
    // filtered (value > 0)
    { MatrixQuery q{}; q.value_col = 7; q.agg = AGG_SUM; q.has_filter = true; q.cmp = MatrixCmp::GT; q.lo_f64 = 0.0;
      std::vector<uint64_t> o; eng.execute_query(q, o);
      assert(std::bit_cast<double>(o[0]) == ref_reduce(V, true, MatrixCmp::GT, 0, 0, AGG_SUM)); }
    // fractional value survives (non-vacuity: a uint32/int64 path would truncate 1.5)
    { MatrixQuery q{}; q.value_col = 7; q.agg = AGG_MIN; q.has_filter = true; q.cmp = MatrixCmp::GT; q.lo_f64 = 0.0;
      std::vector<uint64_t> o; eng.execute_query(q, o); assert(std::bit_cast<double>(o[0]) == 0.5); }
    // grouped double still rejected (DM-3f)
    { MatrixQuery q{}; q.value_col = 7; q.agg = AGG_COUNT; q.grouped = true; q.key_col = 7; q.num_groups = 2;
      std::vector<uint64_t> o; assert(eng.execute_query(q, o) == MatrixQueryStatus::ERR_UNSUPPORTED_TYPE); }
    std::cout << "[engine f64] ok\n";
}

static void test_f64_durability() {
    const std::string path = "/tmp/mdb_f64_catalog.bin"; std::remove(path.c_str());
    std::vector<uint32_t> u = {1, 2, 3};
    { CPUMockEngine eng; eng.load_scan_column(3, u.data(), u.size()); eng.load_scan_column_f64(7, V.data(), V.size()); eng.save_catalog(path); }
    { CPUMockEngine eng; eng.load_catalog(path);
      assert(eng.column_type(7) == MatrixType::F64 && eng.column_type(3) == MatrixType::U32 && "types restored");
      MatrixQuery q{}; q.value_col = 7; q.agg = AGG_SUM; std::vector<uint64_t> o; eng.execute_query(q, o);
      assert(std::bit_cast<double>(o[0]) == ref_reduce(V, false, MatrixCmp::GT, 0, 0, AGG_SUM) && "f64 column survived restart"); }
    std::remove(path.c_str());
    std::cout << "[f64 durability] ok\n";
}

int main() { test_reduce_f64(); test_pred_match_f64(); test_engine_f64(); test_f64_durability();
    std::cout << "ALL TYPED-DOUBLE TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_typed_double_grouped.cpp
// CPU test for grouped double aggregation (DM-3f): matrix_group_reduce_f64[_pred], grouped_aggregate_f64,
// execute_query double GROUP BY (u32 key, double value), incl. negative-group MAX + guards. Mirrors DM-3c.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <bit>
#include <limits>
#include <vector>
#include <iostream>

static bool refm(double v, MatrixCmp c, double a, double b) {
    switch (c) { case MatrixCmp::GT: return v>a; case MatrixCmp::GE: return v>=a; case MatrixCmp::LT: return v<a;
                 case MatrixCmp::LE: return v<=a; case MatrixCmp::EQ: return v==a; case MatrixCmp::NE: return v!=a;
                 case MatrixCmp::BETWEEN: return v>=a && v<=b; } return false; }
static void ref_group(const std::vector<uint32_t>& keys, const std::vector<double>& vals, uint32_t ng,
                      MatrixAggOp op, bool filt, MatrixCmp c, double a, double b, std::vector<double>& out) {
    const double inf = std::numeric_limits<double>::infinity();
    out.assign(ng, op == AGG_MIN ? inf : (op == AGG_MAX ? -inf : 0.0));
    for (size_t i = 0; i < keys.size(); ++i) {
        if (keys[i] >= ng) continue;
        if (filt && !refm(vals[i], c, a, b)) continue;
        double& o = out[keys[i]];
        switch (op) { case AGG_SUM: o += vals[i]; break; case AGG_MIN: if (vals[i]<o) o=vals[i]; break;
                      case AGG_MAX: if (vals[i]>o) o=vals[i]; break; case AGG_COUNT: default: o += 1.0; break; } } }

// keys 0..2; group 1 all-negative (MAX-init guard); group 2 has a large + fractional value.
static const std::vector<uint32_t> K = {0, 1, 1, 2, 0, 1, 2};
static const std::vector<double>   V = {1.5, -3.0, -100.0, 5000000000.0, 0.5, -0.25, 2.25};

static void test_group_reduce_f64() {
    for (bool filt : {false, true})
        for (MatrixAggOp op : {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX}) {
            std::vector<double> got(3), exp;
            MatrixPredicateF64 p{MatrixCmp::GE, -3.0, 0};   // value >= -3.0
            if (filt) matrix_cpu_group_reduce_f64_pred(K.data(), V.data(), K.size(), 3, op, p, got.data());
            else      matrix_cpu_group_reduce_f64(K.data(), V.data(), K.size(), 3, op, got.data());
            ref_group(K, V, 3, op, filt, MatrixCmp::GE, -3.0, 0, exp);
            assert(got == exp && "grouped double reduce matches oracle"); }
    // MAX-init guard: group 1 (all negative {-3,-100,-0.25}) MAX must be -0.25, NOT 0.
    std::vector<double> mx(3); matrix_cpu_group_reduce_f64(K.data(), V.data(), K.size(), 3, AGG_MAX, mx.data());
    assert(mx[1] == -0.25 && "MAX of a negative double group is the max negative, not 0");
    std::cout << "[group reduce f64] ok\n";
}

static void test_engine_grouped_f64() {
    CPUMockEngine eng;
    eng.load_scan_column(1, K.data(), K.size());          // u32 key
    eng.load_scan_column_f64(7, V.data(), V.size());      // double value (same ROW count, 4N vs 8N bytes)
    for (bool filt : {false, true})
        for (MatrixAggOp op : {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX}) {
            MatrixQuery q{}; q.value_col = 7; q.key_col = 1; q.num_groups = 3; q.agg = op; q.grouped = true;
            q.has_filter = filt; q.cmp = MatrixCmp::GE; q.lo_f64 = -3.0;
            std::vector<uint64_t> out;
            assert(eng.execute_query(q, out) == MatrixQueryStatus::OK && out.size() == 3);
            std::vector<double> exp; ref_group(K, V, 3, op, filt, MatrixCmp::GE, -3.0, 0, exp);
            for (uint32_t g = 0; g < 3; ++g) assert(std::bit_cast<double>(out[g]) == exp[g] && "engine grouped double == oracle"); }
    // Non-vacuity: group 2 SUM includes the large + fractional values (5e9 + 2.25).
    { MatrixQuery q{}; q.value_col = 7; q.key_col = 1; q.num_groups = 3; q.agg = AGG_SUM; q.grouped = true;
      std::vector<uint64_t> o; eng.execute_query(q, o); assert(std::bit_cast<double>(o[2]) == 5000000000.0 + 2.25); }
    std::cout << "[engine grouped f64] ok\n";
}

static void test_grouped_f64_guards() {
    CPUMockEngine eng;
    eng.load_scan_column(1, K.data(), K.size());
    eng.load_scan_column_f64(7, V.data(), V.size());
    std::vector<double> dkeys = {0, 1, 0, 1, 0, 1, 0};
    eng.load_scan_column_f64(8, dkeys.data(), dkeys.size());     // double key
    { MatrixQuery q{}; q.value_col = 7; q.key_col = 8; q.num_groups = 2; q.agg = AGG_COUNT; q.grouped = true;
      std::vector<uint64_t> o; assert(eng.execute_query(q, o) == MatrixQueryStatus::ERR_UNSUPPORTED_TYPE && "double key rejected"); }
    std::vector<uint32_t> k8 = {0, 1, 0, 1, 0, 1, 0, 1};         // 8 rows vs V's 7 -> mismatch
    eng.load_scan_column(2, k8.data(), k8.size());
    { MatrixQuery q{}; q.value_col = 7; q.key_col = 2; q.num_groups = 2; q.agg = AGG_COUNT; q.grouped = true;
      std::vector<uint64_t> o; assert(eng.execute_query(q, o) == MatrixQueryStatus::ERR_INVALID_GROUP && "row-count mismatch rejected"); }
    std::cout << "[grouped f64 guards] ok\n";
}

int main() { test_group_reduce_f64(); test_engine_grouped_f64(); test_grouped_f64_guards();
    std::cout << "ALL TYPED-DOUBLE-GROUPED TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_typed_csv.cpp
// CPU test for typed CSV ingest (DM-3g): matrix_read_csv_column_i64/_f64 + load_column_from_csv_i64/_f64.
#include "compute_mock.cpp"
#include "csv_ingest.hpp"
#include <cassert>
#include <cstdint>
#include <cstdio>
#include <bit>
#include <fstream>
#include <string>
#include <vector>
#include <iostream>

static std::string wr(const std::string& name, const std::string& body) {
    const std::string p = "/tmp/" + name; std::ofstream(p) << body; return p; }

static void test_csv_i64() {
    std::vector<int64_t> out;
    std::string p = wr("mdb_csv_i64.csv", "k,v\n-7,-7\n10,5000000000\n3,3\n");   // negatives + > UINT32_MAX
    assert(matrix_read_csv_column_i64(p, 1, true, ',', out) && out == (std::vector<int64_t>{-7, 5000000000LL, 3}));
    // graceful failures
    assert(!matrix_read_csv_column_i64("/tmp/mdb_csv_nope.csv", 0, false, ',', out));
    p = wr("mdb_csv_i64_bad.csv", "1,x\n");      assert(!matrix_read_csv_column_i64(p, 1, false, ',', out));
    p = wr("mdb_csv_i64_over.csv", "99999999999999999999\n"); assert(!matrix_read_csv_column_i64(p, 0, false, ',', out));
    p = wr("mdb_csv_i64_junk.csv", "12x\n");     assert(!matrix_read_csv_column_i64(p, 0, false, ',', out));
    p = wr("mdb_csv_i64_short.csv", "1,2\n3\n"); assert(!matrix_read_csv_column_i64(p, 1, false, ',', out) && out.empty());
    std::cout << "[csv i64] ok\n";
}

static void test_csv_f64() {
    std::vector<double> out;
    std::string p = wr("mdb_csv_f64.csv", "1.5\n-3.25\n5e9\n0.5\n");   // fractional + negative + exponent
    assert(matrix_read_csv_column_f64(p, 0, false, ',', out) && out == (std::vector<double>{1.5, -3.25, 5e9, 0.5}));
    assert(!matrix_read_csv_column_f64("/tmp/mdb_csv_nope2.csv", 0, false, ',', out));
    p = wr("mdb_csv_f64_bad.csv", "x\n");     assert(!matrix_read_csv_column_f64(p, 0, false, ',', out));
    p = wr("mdb_csv_f64_junk.csv", "1.5x\n"); assert(!matrix_read_csv_column_f64(p, 0, false, ',', out));
    p = wr("mdb_csv_f64_empty.csv", "1,\n");  assert(!matrix_read_csv_column_f64(p, 1, false, ',', out));  // empty field
    std::cout << "[csv f64] ok\n";
}

static void test_engine_typed_csv() {
    CPUMockEngine eng;
    std::string pi = wr("mdb_eng_i64.csv", "k,v\n-7,-7\n10,5000000000\n3,3\n");
    assert(eng.load_column_from_csv_i64(7, pi, 1, true));
    assert(eng.column_type(7) == MatrixType::I64);
    { MatrixQuery q{}; q.value_col = 7; q.agg = AGG_SUM; std::vector<uint64_t> o;
      assert(eng.execute_query(q, o) == MatrixQueryStatus::OK && static_cast<int64_t>(o[0]) == -7 + 5000000000LL + 3); }
    std::string pf = wr("mdb_eng_f64.csv", "1.5\n-3.25\n0.25\n");
    assert(eng.load_column_from_csv_f64(8, pf, 0));
    assert(eng.column_type(8) == MatrixType::F64);
    { MatrixQuery q{}; q.value_col = 8; q.agg = AGG_SUM; std::vector<uint64_t> o;
      eng.execute_query(q, o); assert(std::bit_cast<double>(o[0]) == 1.5 - 3.25 + 0.25); }
    // malformed -> false, no catalog entry
    std::string pb = wr("mdb_eng_bad.csv", "1,x\n");
    assert(!eng.load_column_from_csv_i64(9, pb, 1));
    { MatrixQuery q{}; q.value_col = 9; q.agg = AGG_COUNT; std::vector<uint64_t> o;
      assert(eng.execute_query(q, o) == MatrixQueryStatus::ERR_UNKNOWN_COLUMN && "no entry from bad CSV"); }
    std::cout << "[engine typed csv] ok\n";
}

int main() { test_csv_i64(); test_csv_f64(); test_engine_typed_csv();
    std::cout << "ALL TYPED-CSV TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_query_latency.cpp
// CPU test for per-query latency metrics (OB-2): execute_query records count/total_ns/max_ns in EngineStats.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

static void test_query_latency() {
    std::vector<uint32_t> v(200000);
    for (size_t i = 0; i < v.size(); ++i) v[i] = static_cast<uint32_t>(i);
    CPUMockEngine eng;
    eng.load_scan_column(2, v.data(), v.size());

    // 5 OK queries (real work on 200k rows) + 2 that return ERR — all must be counted/timed.
    for (int i = 0; i < 5; ++i) {
        MatrixQuery q{}; q.value_col = 2; q.agg = AGG_SUM; q.has_filter = true; q.threshold = 100;
        std::vector<uint64_t> o; assert(eng.execute_query(q, o) == MatrixQueryStatus::OK);
    }
    { MatrixQuery q{}; q.value_col = 999; q.agg = AGG_COUNT; std::vector<uint64_t> o;   // unknown column
      assert(eng.execute_query(q, o) == MatrixQueryStatus::ERR_UNKNOWN_COLUMN); }
    { MatrixQuery q{}; q.value_col = 2; q.key_col = 2; q.num_groups = 4; q.grouped = true; q.agg = AGG_COUNT;
      std::vector<uint64_t> o; assert(eng.execute_query(q, o) == MatrixQueryStatus::ERR_INVALID_GROUP); } // key==value

    const EngineStats s = eng.stats();
    assert(s.query_count == 7 && "every execute_query call counted (OK and ERR)");
    assert(s.total_query_ns > 0 && "real queries took measurable time");
    assert(s.max_query_ns > 0 && s.max_query_ns <= s.total_query_ns && "max is one sample of the total");
    std::cout << "[query latency] ok (count=" << s.query_count
              << " mean_ns=" << (s.total_query_ns / s.query_count) << " max_ns=" << s.max_query_ns << ")\n";
    // OB-2b: latency histogram + percentiles
    auto hist = eng.query_latency_histogram();
    uint64_t hsum = 0; for (uint64_t b : hist) hsum += b;
    assert(hsum == s.query_count && "histogram sums to query_count");
    const uint64_t p50 = eng.query_latency_percentile_ns(0.50), p99 = eng.query_latency_percentile_ns(0.99);
    assert(p99 >= p50 && "p99 >= p50 (cumulative histogram is monotone)");
    assert(p99 > 0 && "real queries -> nonzero p99 latency");
    std::cout << "[latency histogram] ok (p50~" << p50 << "ns p99~" << p99 << "ns)\n";
}

int main() { test_query_latency(); std::cout << "ALL QUERY-LATENCY TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_backup.cpp
// CPU test for backup/restore (DU-6): backup() writes <prefix>.catalog + <prefix>.kv; restore() loads
// both into a fresh engine — analytical columns (u32 + int64) and point-op writes all survive.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <cstdio>
#include <vector>
#include <iostream>

static void test_backup_roundtrip() {
    const std::string pfx = "/tmp/mdb_backup_rt";
    std::remove((pfx + ".catalog").c_str()); std::remove((pfx + ".kv").c_str());
    std::vector<uint32_t> u(50); for (size_t i = 0; i < u.size(); ++i) u[i] = static_cast<uint32_t>(i);
    std::vector<int64_t>  s = {-7, 5000000000LL, 3, -100};
    {
        CPUMockEngine eng;                                   // no WAL — backup must still capture kv_
        eng.load_scan_column(3, u.data(), u.size());
        eng.load_scan_column_i64(7, s.data(), s.size());
        eng.begin(); eng.txn_put(11, 111); eng.txn_put(22, 222); eng.commit();
        eng.begin(); eng.txn_put(33, 333); eng.commit();
        eng.backup(pfx);
    }
    {
        CPUMockEngine eng;                                   // fresh engine
        eng.restore(pfx);
        // analytical columns restored (type + values)
        assert(eng.column_type(3) == MatrixType::U32 && eng.column_type(7) == MatrixType::I64);
        MatrixQuery qu{}; qu.value_col = 3; qu.agg = AGG_SUM; std::vector<uint64_t> ou;
        assert(eng.execute_query(qu, ou) == MatrixQueryStatus::OK && ou[0] == (49u * 50u / 2u)); // 0..49 = 1225
        MatrixQuery qs{}; qs.value_col = 7; qs.agg = AGG_SUM; std::vector<uint64_t> os;
        eng.execute_query(qs, os); assert(static_cast<int64_t>(os[0]) == -7 + 5000000000LL + 3 - 100);
        // point-op writes restored
        uint64_t v = 0;
        assert(eng.kv_get(11, v) && v == 111);
        assert(eng.kv_get(22, v) && v == 222);
        assert(eng.kv_get(33, v) && v == 333);
        assert(!eng.kv_get(99, v) && "absent key still absent");
    }
    std::remove((pfx + ".catalog").c_str()); std::remove((pfx + ".kv").c_str());
    std::cout << "[backup roundtrip] ok\n";
}

static void test_backup_empty() {
    const std::string pfx = "/tmp/mdb_backup_empty";
    std::remove((pfx + ".catalog").c_str()); std::remove((pfx + ".kv").c_str());
    { CPUMockEngine eng; eng.backup(pfx); }
    { CPUMockEngine eng; eng.restore(pfx); }   // empty catalog + empty kv -> no crash
    std::remove((pfx + ".catalog").c_str()); std::remove((pfx + ".kv").c_str());
    std::cout << "[backup empty] ok\n";
}

int main() { test_backup_roundtrip(); test_backup_empty();
    std::cout << "ALL BACKUP TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_schema.cpp
// CPU test for named columns + catalog introspection (DM-2): name_column / column_id / column_name /
// catalog_columns over u32 + int64 + double columns.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <algorithm>
#include <string>
#include <vector>
#include <iostream>

static void test_naming() {
    std::vector<uint32_t> u = {1, 2, 3};
    std::vector<int64_t>  s = {-7, 5000000000LL, 3, -100};
    std::vector<double>   d = {1.5, -3.25};
    CPUMockEngine eng;
    eng.load_scan_column(3, u.data(), u.size());
    eng.load_scan_column_i64(7, s.data(), s.size());
    eng.load_scan_column_f64(9, d.data(), d.size());
    eng.name_column(3, "qty"); eng.name_column(7, "revenue"); eng.name_column(9, "rate");
    assert(eng.column_id("qty") == 3 && eng.column_id("revenue") == 7 && eng.column_id("rate") == 9);
    assert(eng.column_id("nonexistent") == 0 && "unknown name -> 0");
    assert(eng.column_name(7) == "revenue");
    // load a 4th unnamed column; its name is ""
    std::vector<uint32_t> u2 = {9, 9};
    eng.load_scan_column(4, u2.data(), u2.size());
    assert(eng.column_name(4).empty() && "unnamed column -> empty name");
    std::cout << "[naming] ok\n";
}

static void test_catalog_columns() {
    std::vector<uint32_t> u = {1, 2, 3, 4, 5};       // 5 rows
    std::vector<int64_t>  s = {-7, 5000000000LL, 3}; // 3 rows
    std::vector<double>   d = {1.5, -3.25};          // 2 rows
    CPUMockEngine eng;
    eng.load_scan_column(3, u.data(), u.size());     eng.name_column(3, "qty");
    eng.load_scan_column_i64(7, s.data(), s.size()); eng.name_column(7, "revenue");
    eng.load_scan_column_f64(9, d.data(), d.size()); // unnamed
    std::vector<ColumnInfo> info = eng.catalog_columns();
    assert(info.size() == 3);
    std::sort(info.begin(), info.end(), [](const ColumnInfo& a, const ColumnInfo& b){ return a.id < b.id; });
    assert(info[0].id == 3 && info[0].name == "qty"     && info[0].type == MatrixType::U32 && info[0].rows == 5);
    assert(info[1].id == 7 && info[1].name == "revenue" && info[1].type == MatrixType::I64 && info[1].rows == 3);
    assert(info[2].id == 9 && info[2].name == ""        && info[2].type == MatrixType::F64 && info[2].rows == 2);
    for (const auto& ci : info) assert(ci.tier == MemorySpace::HOST && "freshly loaded -> HOST");
    std::cout << "[catalog columns] ok\n";
}

static void test_resolve_then_query() {
    std::vector<int64_t> s = {-7, 5000000000LL, 3, -100};
    CPUMockEngine eng;
    eng.load_scan_column_i64(7, s.data(), s.size());
    eng.name_column(7, "revenue");
    MatrixQuery q{}; q.value_col = eng.column_id("revenue"); q.agg = AGG_SUM;   // resolve name -> id
    std::vector<uint64_t> o;
    assert(eng.execute_query(q, o) == MatrixQueryStatus::OK);
    assert(static_cast<int64_t>(o[0]) == -7 + 5000000000LL + 3 - 100 && "query by resolved name");
    std::cout << "[resolve then query] ok\n";
}

int main() { test_naming(); test_catalog_columns(); test_resolve_then_query();
    std::cout << "ALL SCHEMA TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_append.cpp
// CPU test for append / dynamic column growth (DM-9): append_to_column[_i64/_f64] grow a column;
// rows become queryable; works across the COLD tier (borrow-append-return).
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <bit>
#include <vector>
#include <iostream>

static void test_append_u32() {
    std::vector<uint32_t> v(10); for (uint32_t i = 0; i < 10; ++i) v[i] = i;   // 0..9, sum 45
    CPUMockEngine eng;
    eng.load_scan_column(3, v.data(), v.size());
    MatrixQuery q{}; q.value_col = 3; q.agg = AGG_SUM; std::vector<uint64_t> o;
    eng.execute_query(q, o); assert(o[0] == 45);
    uint32_t more[] = {10, 11, 12};
    eng.append_to_column(3, more, 3);                       // 0..12, sum 78, 13 rows
    assert(eng.column_rows(3) == 13);
    eng.execute_query(q, o); assert(o[0] == 78 && "appended rows included in SUM");
    std::cout << "[append u32] ok\n";
}

static void test_append_typed() {
    CPUMockEngine eng;
    std::vector<int64_t> s = {-7, 5};               eng.load_scan_column_i64(7, s.data(), s.size());
    std::vector<double>  d = {1.5, -0.5};           eng.load_scan_column_f64(9, d.data(), d.size());
    int64_t mi[] = {5000000000LL, -100};            eng.append_to_column_i64(7, mi, 2);   // sum -7+5+5e9-100
    double  md[] = {2.25, -3.0};                    eng.append_to_column_f64(9, md, 2);   // sum 1.5-0.5+2.25-3.0
    assert(eng.column_rows(7) == 4 && eng.column_rows(9) == 4);
    { MatrixQuery q{}; q.value_col = 7; q.agg = AGG_SUM; std::vector<uint64_t> o; eng.execute_query(q, o);
      assert(static_cast<int64_t>(o[0]) == -7 + 5 + 5000000000LL - 100); }
    { MatrixQuery q{}; q.value_col = 9; q.agg = AGG_SUM; std::vector<uint64_t> o; eng.execute_query(q, o);
      assert(std::bit_cast<double>(o[0]) == 1.5 - 0.5 + 2.25 - 3.0); }
    std::cout << "[append typed] ok\n";
}

static void test_append_cold() {
    // Small host budget: holds ~one 400KB column, not two -> the idle one demotes to COLD.
    CPUMockEngine eng(0, "", 600 * 1024);
    std::vector<uint32_t> a(100000, 1), b(100000, 1);       // 400KB each
    eng.load_scan_column(1, a.data(), a.size());
    eng.load_scan_column(2, b.data(), b.size());
    for (int i = 0; i < 8; ++i) {                           // scan col1 hard -> rebalance demotes idle col2
        MatrixQuery q{}; q.value_col = 1; q.agg = AGG_COUNT; std::vector<uint64_t> o; eng.execute_query(q, o);
    }
    assert(eng.column_tier(2) == MemorySpace::COLD && "idle column demoted to SSD");
    uint32_t more[] = {7, 7, 7};
    eng.append_to_column(2, more, 3);                       // append to the COLD column
    MatrixQuery q{}; q.value_col = 2; q.agg = AGG_COUNT; std::vector<uint64_t> o;
    assert(eng.execute_query(q, o) == MatrixQueryStatus::OK && o[0] == 100003 && "COLD column grew + queryable");
    std::cout << "[append cold] ok\n";
}

int main() { test_append_u32(); test_append_typed(); test_append_cold();
    std::cout << "ALL APPEND TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_kv_range.cpp
// CPU test for key range scan over the point-op store (DM-7): kv_range(lo, hi) returns every
// (key, value) with lo <= key <= hi (inclusive), reusing KVStore::for_each.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <algorithm>
#include <utility>
#include <vector>
#include <iostream>

static std::vector<std::pair<uint64_t, uint64_t>> sorted(std::vector<std::pair<uint64_t, uint64_t>> v) {
    std::sort(v.begin(), v.end());
    return v;
}

static void test_kv_range() {
    CPUMockEngine eng;                                  // no WAL needed
    // value = key*10 so we can check values too.
    eng.begin();
    for (uint64_t k : {5ull, 10ull, 15ull, 20ull, 100ull}) eng.txn_put(k, k * 10);
    eng.commit();

    // inclusive [10, 20] -> 10,15,20
    auto r = sorted(eng.kv_range(10, 20));
    assert((r == std::vector<std::pair<uint64_t,uint64_t>>{{10,100},{15,150},{20,200}}) && "inclusive range");
    // full range -> all 5
    assert(eng.kv_range(0, UINT64_MAX).size() == 5 && "full range = all keys");
    // empty interior range
    assert(eng.kv_range(11, 14).empty() && "no keys in (10,15)");
    // single-key boundary
    auto one = eng.kv_range(5, 5);
    const std::pair<uint64_t, uint64_t> expect{5, 50};   // (in a var: a comma inside assert(...) args breaks the macro)
    assert(one.size() == 1 && one[0] == expect && "inclusive single-key");
    // non-vacuity: 100 is OUTSIDE [10,20] (excluded), and each boundary IS included
    for (auto& kv : r) assert(kv.first != 100 && "out-of-range key excluded");
    assert(sorted(eng.kv_range(10, 20)).front().first == 10 && sorted(eng.kv_range(10, 20)).back().first == 20
           && "both boundaries inclusive");
    std::cout << "[kv range] ok\n";
}

int main() { test_kv_range(); std::cout << "ALL KV-RANGE TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_query_parser.cpp
// CPU test for the text query parser (DM-4): parse_query parses scalar SELECT/WHERE into a MatrixQuery
// (name resolution + type-aware bound placement); graceful ERR_PARSE / ERR_UNKNOWN_COLUMN on bad input.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <bit>
#include <string>
#include <vector>
#include <iostream>

static void test_happy() {
    std::vector<uint32_t> q = {1, 2, 3, 4, 5};                 // qty, sum 15
    std::vector<int64_t>  r = {-100, 5000000000LL, 50, 2000000000LL};  // revenue
    std::vector<double>   t = {1.5, 3.5, 4.0, 2.0};            // rate
    CPUMockEngine eng;
    eng.load_scan_column(3, q.data(), q.size());     eng.name_column(3, "qty");
    eng.load_scan_column_i64(7, r.data(), r.size()); eng.name_column(7, "revenue");
    eng.load_scan_column_f64(9, t.data(), t.size()); eng.name_column(9, "rate");

    MatrixQuery m; std::vector<uint64_t> o;
    // unfiltered scalar
    assert(eng.parse_query("SELECT SUM(qty)", m) == MatrixQueryStatus::OK);
    assert(m.value_col == 3 && m.agg == AGG_SUM && !m.has_filter);
    eng.execute_query(m, o); assert(o[0] == 15);
    // int64 GT bound -> lo_i64 (NOT threshold)
    assert(eng.parse_query("SELECT COUNT(revenue) WHERE revenue > 1000000000", m) == MatrixQueryStatus::OK);
    assert(m.value_col == 7 && m.agg == AGG_COUNT && m.has_filter && m.cmp == MatrixCmp::GT
           && m.lo_i64 == 1000000000LL && m.threshold == 0 && "int64 bound in lo_i64");
    eng.execute_query(m, o); assert(o[0] == 2 && "revenue>1e9 -> {5e9, 2e9}");
    // double LE bound -> lo_f64
    assert(eng.parse_query("SELECT MAX(rate) WHERE rate <= 3.5", m) == MatrixQueryStatus::OK);
    assert(m.cmp == MatrixCmp::LE && m.lo_f64 == 3.5 && "double bound in lo_f64");
    eng.execute_query(m, o); assert(std::bit_cast<double>(o[0]) == 3.5);
    // BETWEEN (int64)
    assert(eng.parse_query("SELECT SUM(revenue) WHERE revenue BETWEEN -100 AND 5000000000", m) == MatrixQueryStatus::OK);
    assert(m.cmp == MatrixCmp::BETWEEN && m.lo_i64 == -100 && m.hi_i64 == 5000000000LL);
    eng.execute_query(m, o); assert(static_cast<int64_t>(o[0]) == -100 + 5000000000LL + 50 + 2000000000LL);
    // case-insensitive + spacing
    assert(eng.parse_query("select sum ( qty )", m) == MatrixQueryStatus::OK && m.value_col == 3);
    std::cout << "[parser happy] ok\n";
}

static void test_errors() {
    std::vector<uint32_t> q = {1, 2, 3};
    CPUMockEngine eng; eng.load_scan_column(3, q.data(), q.size()); eng.name_column(3, "qty");
    MatrixQuery m;
    assert(eng.parse_query("", m) == MatrixQueryStatus::ERR_PARSE);
    assert(eng.parse_query("SELECT FOO(qty)", m) == MatrixQueryStatus::ERR_PARSE);
    assert(eng.parse_query("SELECT SUM(nosuchcol)", m) == MatrixQueryStatus::ERR_UNKNOWN_COLUMN);
    assert(eng.parse_query("SELECT SUM(qty", m) == MatrixQueryStatus::ERR_PARSE);          // missing )
    assert(eng.parse_query("SELECT SUM(qty) WHERE qty >", m) == MatrixQueryStatus::ERR_PARSE);   // missing value
    assert(eng.parse_query("SELECT SUM(qty) WHERE qty > x", m) == MatrixQueryStatus::ERR_PARSE); // non-numeric
    assert(eng.parse_query("SELECT SUM(qty) WHERE qty BETWEEN 1 5", m) == MatrixQueryStatus::ERR_PARSE); // missing AND
    assert(eng.parse_query("SELECT SUM(qty) GROUP qty", m) == MatrixQueryStatus::ERR_PARSE);       // GROUP without BY
    assert(eng.parse_query("SELECT SUM(qty) GROUP BY nosuchkey", m) == MatrixQueryStatus::ERR_UNKNOWN_COLUMN);
    assert(eng.parse_query("SELECT SUM(qty) extra", m) == MatrixQueryStatus::ERR_PARSE);    // trailing junk
    // out is reset on a MID-parse failure (no partial state leaks to a caller that ignores the status).
    MatrixQuery dirty;
    assert(eng.parse_query("SELECT SUM(qty) WHERE qty BETWEEN 1 AND x", dirty) == MatrixQueryStatus::ERR_PARSE);
    assert(dirty.value_col == 0 && !dirty.has_filter && dirty.cmp == MatrixCmp::GT && "out reset on mid-parse error");
    std::cout << "[parser errors] ok\n";
}

static void test_groupby() {
    std::vector<uint32_t> region = {0, 1, 0, 2}, amount = {10, 20, 30, 40};
    CPUMockEngine eng;
    eng.load_scan_column(2, region.data(), region.size()); eng.name_column(2, "region");
    eng.load_scan_column(3, amount.data(), amount.size()); eng.name_column(3, "amount");
    MatrixQuery m; std::vector<uint64_t> o;
    // SUM(amount) GROUP BY region — num_groups derived as max(region)+1 = 3
    assert(eng.parse_query("SELECT SUM(amount) GROUP BY region", m) == MatrixQueryStatus::OK);
    assert(m.grouped && m.value_col == 3 && m.key_col == 2 && m.num_groups == 3 && "GROUP BY parsed; num_groups = max key + 1");
    assert(eng.execute_query(m, o) == MatrixQueryStatus::OK && o.size() == 3);
    assert(o[0] == 40 && o[1] == 20 && o[2] == 40 && "grouped SUM: r0=10+30, r1=20, r2=40");
    // filtered grouped: COUNT(amount) WHERE amount > 15 GROUP BY region
    assert(eng.parse_query("SELECT COUNT(amount) WHERE amount > 15 GROUP BY region", m) == MatrixQueryStatus::OK);
    assert(m.grouped && m.has_filter && m.cmp == MatrixCmp::GT && m.threshold == 15 && m.key_col == 2);
    assert(eng.execute_query(m, o) == MatrixQueryStatus::OK);
    assert(o[0] == 1 && o[1] == 1 && o[2] == 1 && "amount>15 per region: r0{30}, r1{20}, r2{40}");
    std::cout << "[parser group by] ok\n";
}

int main() { test_happy(); test_errors(); test_groupby();
    std::cout << "ALL QUERY-PARSER TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_join.cpp
// CPU test for the equi-join primitive (DM-8): hash_join(left,right) returns matching (left_row,
// right_row) pairs; verified against a brute O(n*m) oracle, incl. duplicates + a COLD column.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <algorithm>
#include <utility>
#include <vector>
#include <iostream>

using Pairs = std::vector<std::pair<uint64_t, uint64_t>>;
static Pairs sorted(Pairs v) { std::sort(v.begin(), v.end()); return v; }
// Brute reference: every (i,j) with l[i]==r[j].
static Pairs brute(const std::vector<uint32_t>& l, const std::vector<uint32_t>& r) {
    Pairs out;
    for (uint64_t i = 0; i < l.size(); ++i) for (uint64_t j = 0; j < r.size(); ++j)
        if (l[i] == r[j]) out.emplace_back(i, j);
    return sorted(out);
}

static void test_join_basic() {
    std::vector<uint32_t> L = {10, 20, 30, 20}, R = {20, 40, 10, 20};
    CPUMockEngine eng;
    eng.load_scan_column(1, L.data(), L.size());
    eng.load_scan_column(2, R.data(), R.size());
    Pairs got = sorted(eng.hash_join(1, 2));
    Pairs exp = brute(L, R);
    assert(got == exp && "hash join == brute oracle");
    // explicit expected set: (0,2),(1,0),(1,3),(3,0),(3,3)
    assert((got == Pairs{{0,2},{1,0},{1,3},{3,0},{3,3}}) && "exact pairs incl. duplicate-key Cartesian");
    // count-only join (resource-safe): equals the materialized cardinality, no pairs stored.
    assert(eng.hash_join_count(1, 2) == got.size() && eng.hash_join_count(1, 2) == 5 && "hash_join_count == cardinality");
    std::cout << "[join basic] ok\n";
}

static void test_join_edges() {
    CPUMockEngine eng;
    std::vector<uint32_t> a = {1, 2, 3}, b = {4, 5, 6};         // disjoint
    eng.load_scan_column(1, a.data(), a.size());
    eng.load_scan_column(2, b.data(), b.size());
    assert(eng.hash_join(1, 2).empty() && "no matches -> empty");
    std::vector<uint32_t> c = {7, 7}, d = {7};                 // dup left, single right
    eng.load_scan_column(3, c.data(), c.size());
    eng.load_scan_column(4, d.data(), d.size());
    assert((sorted(eng.hash_join(3, 4)) == Pairs{{0,0},{1,0}}) && "Cartesian of dup match");
    std::cout << "[join edges] ok\n";
}

static void test_join_cold() {
    // Tiny host budget (bytes): L(16B)+R(8B)=24 > 16, so the idle column demotes to COLD; the join
    // borrows it back. Scanning L keeps it hot so R is the eviction victim.
    CPUMockEngine eng(0, "", 16);
    std::vector<uint32_t> L = {5, 9, 5, 1}, R = {9, 5};        // 16B, 8B
    eng.load_scan_column(1, L.data(), L.size());
    eng.load_scan_column(2, R.data(), R.size());
    for (int i = 0; i < 8; ++i) {                              // heat col1 -> rebalance demotes idle col2
        MatrixQuery q{}; q.value_col = 1; q.agg = AGG_COUNT; std::vector<uint64_t> o; eng.execute_query(q, o);
    }
    assert(eng.column_tier(2) == MemorySpace::COLD && "idle join column demoted -> exercises the borrow path");
    Pairs got = sorted(eng.hash_join(1, 2));
    assert(got == brute(L, R) && "join correct across the COLD tier (borrow)");
    assert((got == Pairs{{0,1},{1,0},{2,1}}) && "L{5,9,5,1} x R{9,5}: 5@0=5@1, 9@1=9@0, 5@2=5@1");
    std::cout << "[join cold] ok\n";
}

static Pairs brute_i64(const std::vector<int64_t>& l, const std::vector<int64_t>& r) {
    Pairs out;
    for (uint64_t i = 0; i < l.size(); ++i) for (uint64_t j = 0; j < r.size(); ++j)
        if (l[i] == r[j]) out.emplace_back(i, j);
    return sorted(out);
}
static void test_join_i64() {
    std::vector<int64_t> L = {-7, 5000000000LL, 30, 5000000000LL}, R = {5000000000LL, 40, -7};
    CPUMockEngine eng;
    eng.load_scan_column_i64(1, L.data(), L.size());
    eng.load_scan_column_i64(2, R.data(), R.size());
    Pairs got = sorted(eng.hash_join_i64(1, 2));
    assert(got == brute_i64(L, R) && "int64 hash join == brute oracle");
    // -7@0=-7@2, 5e9@1=5e9@0, 5e9@3=5e9@0 — >UINT32_MAX + negative keys (a u32 join would mismatch these)
    assert((got == Pairs{{0,2},{1,0},{3,0}}) && "int64-key join incl. > UINT32_MAX + negative keys");
    std::cout << "[join i64] ok\n";
}

int main() { test_join_basic(); test_join_edges(); test_join_cold(); test_join_i64();
    std::cout << "ALL JOIN TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_integration.cpp
// End-to-end INTEGRATION test (QA-2): exercises the major features COMPOSING in one realistic flow —
// typed CSV ingest -> naming -> catalog introspection -> parse_query -> execute -> append -> hash_join
// -> backup/restore. Asserts at each stage. (Unit tests cover each feature in isolation; this proves
// they work together.) Also documents one finding: column names are RAM-only (not in backup/restore).
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <algorithm>
#include <fstream>
#include <string>
#include <utility>
#include <vector>
#include <iostream>

int main() {
    const std::string csv = "/tmp/mdb_integ.csv";
    std::ofstream(csv) << "region,revenue\n0,500000000\n1,2000000000\n0,1500000000\n2,3000000000\n";
    const std::string bk = "/tmp/mdb_integ_backup";
    std::remove((bk + ".catalog").c_str()); std::remove((bk + ".kv").c_str());

    // --- Stage 1: typed CSV ingest (u32 region col 0, int64 revenue col 1) + naming ---
    CPUMockEngine eng;
    assert(eng.load_column_from_csv(2, csv, 0, /*has_header=*/true) && "region (u32) from CSV");
    assert(eng.load_column_from_csv_i64(7, csv, 1, /*has_header=*/true) && "revenue (int64) from CSV");
    eng.name_column(2, "region"); eng.name_column(7, "revenue");
    assert(eng.column_type(2) == MatrixType::U32 && eng.column_type(7) == MatrixType::I64);

    // --- Stage 2: catalog introspection ---
    {
        auto info = eng.catalog_columns();
        std::sort(info.begin(), info.end(), [](const ColumnInfo& a, const ColumnInfo& b){ return a.id < b.id; });
        assert(info.size() == 2);
        assert(info[0].id == 2 && info[0].name == "region"  && info[0].type == MatrixType::U32 && info[0].rows == 4);
        assert(info[1].id == 7 && info[1].name == "revenue" && info[1].type == MatrixType::I64 && info[1].rows == 4);
    }

    // --- Stage 3: parse + execute (SUM revenue WHERE revenue > 1e9) ---
    MatrixQuery q;
    assert(eng.parse_query("SELECT SUM(revenue) WHERE revenue > 1000000000", q) == MatrixQueryStatus::OK);
    assert(q.value_col == 7 && q.cmp == MatrixCmp::GT && q.lo_i64 == 1000000000LL);
    std::vector<uint64_t> o;
    assert(eng.execute_query(q, o) == MatrixQueryStatus::OK);
    assert(static_cast<int64_t>(o[0]) == 2000000000LL + 1500000000LL + 3000000000LL && "revenue>1e9 sum");

    // --- Stage 4: append a row, re-run the same parsed query (sum grows by the appended 4e9) ---
    int64_t more[] = {4000000000LL};
    eng.append_to_column_i64(7, more, 1);
    assert(eng.column_rows(7) == 5);
    assert(eng.execute_query(q, o) == MatrixQueryStatus::OK);
    assert(static_cast<int64_t>(o[0]) == 2000000000LL + 1500000000LL + 3000000000LL + 4000000000LL && "appended row joins the filter");

    // --- Stage 5: equi-join region against a lookup of valid regions ---
    std::vector<uint32_t> valid = {0, 2};
    eng.load_scan_column(3, valid.data(), valid.size()); eng.name_column(3, "valid_region");
    // region (rows now... region wasn't appended, still {0,1,0,2}); join region(2) x valid(3)
    auto pairs = eng.hash_join(2, 3);
    std::sort(pairs.begin(), pairs.end());
    // region {0,1,0,2} x valid {0,2}: 0@0=0@0, 0@2=0@0, 2@3=2@1
    assert((pairs == std::vector<std::pair<uint64_t,uint64_t>>{{0,0},{2,0},{3,1}}) && "join region x valid");

    // --- Stage 6: backup -> restore into a fresh engine -> revenue survived (with the appended row) ---
    eng.backup(bk);
    {
        CPUMockEngine fresh;
        fresh.restore(bk);
        assert(fresh.column_type(7) == MatrixType::I64 && fresh.column_rows(7) == 5 && "revenue restored incl. appended row");
        // Names now survive backup/restore (DM-2b: the catalog snapshot carries them) — query BY NAME.
        assert(fresh.column_id("revenue") == 7 && fresh.column_id("region") == 2 && "names restored");
        MatrixQuery qa{}; qa.value_col = fresh.column_id("revenue"); qa.agg = AGG_SUM;   // resolve name in the RESTORED engine
        std::vector<uint64_t> oa;
        assert(fresh.execute_query(qa, oa) == MatrixQueryStatus::OK);
        assert(static_cast<int64_t>(oa[0]) == 500000000LL + 2000000000LL + 1500000000LL + 3000000000LL + 4000000000LL
               && "full revenue (incl. appended) survived backup/restore, queried by restored name");
    }

    std::remove(csv.c_str()); std::remove((bk + ".catalog").c_str()); std::remove((bk + ".kv").c_str());
    std::cout << "ALL INTEGRATION TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_typed_column_io.cpp
// CPU test for typed single-column binary I/O (DM-3h): save_column / load_column_from_file round-trip
// int64 and double columns (not just uint32) via the typed file format; matrix_write/read_column_typed.
#include "compute_mock.cpp"
#include "column_io.hpp"
#include <cassert>
#include <cstdint>
#include <cstdio>
#include <bit>
#include <vector>
#include <string>
#include <iostream>

static void test_direct_typed_io() {
    const std::string p = "/tmp/mdb_typed_col.bin"; std::remove(p.c_str());
    std::vector<int64_t> v = {-7, 5000000000LL, 3};
    matrix_write_column_typed(p, v.data(), v.size() * sizeof(int64_t), static_cast<uint32_t>(MatrixType::I64));
    std::vector<unsigned char> bytes; uint32_t type = 99;
    matrix_read_column_typed(p, bytes, type);
    assert(type == static_cast<uint32_t>(MatrixType::I64) && bytes.size() == v.size() * sizeof(int64_t));
    assert(std::memcmp(bytes.data(), v.data(), bytes.size()) == 0 && "typed bytes round-trip");
    std::remove(p.c_str());
    std::cout << "[direct typed io] ok\n";
}

static void test_engine_save_load_typed() {
    const std::string pi = "/tmp/mdb_sc_i64.bin", pf = "/tmp/mdb_sc_f64.bin", pu = "/tmp/mdb_sc_u32.bin";
    for (auto* p : {&pi, &pf, &pu}) std::remove(p->c_str());
    std::vector<int64_t>  s = {-100, 5000000000LL, 7, -3};
    std::vector<double>   d = {1.5, -3.25, 2.0};
    std::vector<uint32_t> u = {10, 20, 30, 40, 50};
    {
        CPUMockEngine eng;
        eng.load_scan_column_i64(7, s.data(), s.size());
        eng.load_scan_column_f64(8, d.data(), d.size());
        eng.load_scan_column(9, u.data(), u.size());
        eng.save_column(7, pi); eng.save_column(8, pf); eng.save_column(9, pu);   // int64 no longer aborts
    }
    {
        CPUMockEngine eng;
        eng.load_column_from_file(7, pi); eng.load_column_from_file(8, pf); eng.load_column_from_file(9, pu);
        assert(eng.column_type(7) == MatrixType::I64 && eng.column_type(8) == MatrixType::F64 && eng.column_type(9) == MatrixType::U32);
        // values via query
        MatrixQuery qi{}; qi.value_col = 7; qi.agg = AGG_SUM; std::vector<uint64_t> oi; eng.execute_query(qi, oi);
        assert(static_cast<int64_t>(oi[0]) == -100 + 5000000000LL + 7 - 3 && "int64 column survived single-file round-trip");
        MatrixQuery qf{}; qf.value_col = 8; qf.agg = AGG_SUM; std::vector<uint64_t> of; eng.execute_query(qf, of);
        assert(std::bit_cast<double>(of[0]) == 1.5 - 3.25 + 2.0 && "double column survived");
        MatrixQuery qu{}; qu.value_col = 9; qu.agg = AGG_SUM; std::vector<uint64_t> ou; eng.execute_query(qu, ou);
        assert(ou[0] == 150 && "u32 column survived (typed format, type 0)");
    }
    for (auto* p : {&pi, &pf, &pu}) std::remove(p->c_str());
    std::cout << "[engine save/load typed] ok\n";
}

int main() { test_direct_typed_io(); test_engine_save_load_typed();
    std::cout << "ALL TYPED-COLUMN-IO TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_kv_index.cpp
// CPU test for the sorted secondary index (DM-7): kv_range_sorted returns range keys in ASCENDING order
// via the ordered index (vs kv_range's unordered O(n) scan); index is maintained on commit + rebuilt on
// recovery (survives restart).
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <algorithm>
#include <cstdio>
#include <string>
#include <utility>
#include <vector>
#include <iostream>

using Pairs = std::vector<std::pair<uint64_t, uint64_t>>;
static Pairs sorted(Pairs v) { std::sort(v.begin(), v.end()); return v; }

static void test_sorted_range() {
    CPUMockEngine eng;
    eng.begin();
    for (uint64_t k : {50ull, 10ull, 30ull, 20ull, 40ull, 100ull}) eng.txn_put(k, k * 10);   // inserted out of order
    eng.commit();
    // [20,40] -> 20,30,40 in ASCENDING order
    Pairs r = eng.kv_range_sorted(20, 40);
    assert((r == Pairs{{20,200},{30,300},{40,400}}) && "kv_range_sorted is ascending + inclusive");
    assert(std::is_sorted(r.begin(), r.end()) && "ascending order");
    // cross-check vs the unordered kv_range (same set, sorted)
    assert(r == sorted(eng.kv_range(20, 40)) && "kv_range_sorted == sorted(kv_range)");
    assert(eng.kv_range_sorted(0, UINT64_MAX).size() == 6 && "full range = all keys");
    assert(eng.kv_range_sorted(21, 29).empty() && "empty interior range");
    assert((eng.kv_range_sorted(50, 50) == Pairs{{50,500}}) && "single-key inclusive");
    std::cout << "[sorted range] ok\n";
}

static void test_index_survives_restart() {
    const std::string wal = "/tmp/mdb_kvindex.wal";
    std::remove(wal.c_str()); std::remove((wal + ".ckpt").c_str());
    {
        CPUMockEngine eng(0, wal);
        eng.begin(); for (uint64_t k : {7ull, 3ull, 9ull, 1ull}) eng.txn_put(k, k + 1000); eng.commit();
    }
    {
        CPUMockEngine eng(0, wal);                 // fresh engine: index rebuilt from recovered kv_
        Pairs r = eng.kv_range_sorted(2, 8);       // 3,7
        assert((r == Pairs{{3,1003},{7,1007}}) && "index rebuilt on recovery -> sorted range works after restart");
    }
    std::remove(wal.c_str()); std::remove((wal + ".ckpt").c_str());
    std::cout << "[index survives restart] ok\n";
}

int main() { test_sorted_range(); test_index_survives_restart();
    std::cout << "ALL KV-INDEX TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_gather.cpp
// CPU test for gather (DM-8d, the "project" step): gather(col, rows) returns column values at the given
// row indices (typed: U32 value / I64+F64 bit-pattern), and composes with hash_join for join-then-project.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <algorithm>
#include <bit>
#include <vector>
#include <iostream>

static void test_gather_typed() {
    CPUMockEngine eng;
    std::vector<uint32_t> u = {10, 20, 30, 40};
    std::vector<int64_t>  s = {-7, 5000000000LL};
    std::vector<double>   d = {1.5, -2.5};
    eng.load_scan_column(1, u.data(), u.size());
    eng.load_scan_column_i64(2, s.data(), s.size());
    eng.load_scan_column_f64(3, d.data(), d.size());
    // u32: index order preserved, reorder + dup
    assert((eng.gather(1, {2, 0, 3, 0}) == std::vector<uint64_t>{30, 10, 40, 10}) && "u32 gather in index order");
    // i64: bit-pattern -> static_cast<int64_t>
    { auto g = eng.gather(2, {1, 0}); assert(static_cast<int64_t>(g[0]) == 5000000000LL && static_cast<int64_t>(g[1]) == -7); }
    // f64: bit-pattern -> bit_cast<double>
    { auto g = eng.gather(3, {0, 1}); assert(std::bit_cast<double>(g[0]) == 1.5 && std::bit_cast<double>(g[1]) == -2.5); }
    std::cout << "[gather typed] ok\n";
}

static void test_join_then_project() {
    // region[row] aligned with amount[row]; join region against a lookup of valid regions, project amount.
    std::vector<uint32_t> region = {0, 1, 0, 2}, lookup = {0, 2}, amount = {100, 200, 300, 400};
    CPUMockEngine eng;
    eng.load_scan_column(1, region.data(), region.size());
    eng.load_scan_column(2, lookup.data(), lookup.size());
    eng.load_scan_column(3, amount.data(), amount.size());
    auto pairs = eng.hash_join(1, 2);                         // (region_row, lookup_row) where region in {0,2}
    std::vector<uint64_t> left_rows;
    for (auto& p : pairs) left_rows.push_back(p.first);       // region rows 0,2 (=0) and 3 (=2)
    auto amounts = eng.gather(3, left_rows);                  // project amount at the matched region rows
    std::sort(amounts.begin(), amounts.end());
    assert((amounts == std::vector<uint64_t>{100, 300, 400}) && "join region x valid, project amount: rows 0,2,3");
    std::cout << "[join then project] ok\n";
}

int main() { test_gather_typed(); test_join_then_project();
    std::cout << "ALL GATHER TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_tables.cpp
// CPU test for named tables (DM-2c): create_table groups equal-length columns into a named, validated,
// introspectable unit; table_columns / tables list them; row-mismatch + unknown-column are rejected.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <algorithm>
#include <string>
#include <vector>
#include <iostream>

static void test_tables() {
    std::vector<uint32_t> region = {0, 1, 0, 2}, amount = {10, 20, 30, 40};   // 4 rows
    std::vector<int64_t>  note   = {-1, -2, -3, -4};                          // 4 rows (int64)
    std::vector<uint32_t> small  = {1, 2};                                    // 2 rows (mismatch)
    CPUMockEngine eng;
    eng.load_scan_column(2, region.data(), region.size()); eng.name_column(2, "region");
    eng.load_scan_column(3, amount.data(), amount.size()); eng.name_column(3, "amount");
    eng.load_scan_column_i64(7, note.data(), note.size());  eng.name_column(7, "note");
    eng.load_scan_column(5, small.data(), small.size());

    // valid table: 3 equal-length columns
    assert(eng.create_table("sales", {2, 3, 7}) && "equal-length columns -> table created");
    auto cols = eng.table_columns("sales");
    assert(cols.size() == 3);
    assert(cols[0].name == "region" && cols[0].type == MatrixType::U32 && cols[0].rows == 4);
    assert(cols[1].name == "amount" && cols[1].type == MatrixType::U32);
    assert(cols[2].name == "note"   && cols[2].type == MatrixType::I64 && "table preserves declared order + mixed types");
    assert(eng.tables() == std::vector<std::string>{"sales"});

    // rejections (no table created, graceful false)
    assert(!eng.create_table("bad_unknown", {2, 99}) && "unknown column id rejected");
    assert(!eng.create_table("bad_rows", {2, 5}) && "row-count mismatch rejected (4 vs 2)");
    assert(!eng.create_table("bad_empty", {}) && "empty column list rejected");
    assert(eng.table_columns("nosuchtable").empty() && "unknown table -> empty");
    assert(eng.tables().size() == 1 && "rejected tables were not registered");
    std::cout << "[named tables] ok\n";
}

int main() { test_tables(); std::cout << "ALL TABLE TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_string_columns.cpp
// CPU test for minimal string columns (DM-3i): load_string_column + row count + equality-filter count
// + element access. A self-contained string store (separate from the fixed-width byte catalog).
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <string>
#include <vector>
#include <iostream>

static void test_string_columns() {
    CPUMockEngine eng;
    std::vector<std::string> category = {"books", "toys", "books", "food", "books"};
    eng.load_string_column(5, category);
    assert(eng.string_column_rows(5) == 5);
    // equality-filter COUNT (the string WHERE col = 'literal' count)
    assert(eng.string_count_where_eq(5, "books") == 3 && "3 rows == 'books'");
    assert(eng.string_count_where_eq(5, "toys")  == 1);
    assert(eng.string_count_where_eq(5, "none")  == 0 && "no match -> 0");
    // element access
    assert(eng.string_column_at(5, 0) == "books" && eng.string_column_at(5, 1) == "toys");
    // unknown id -> empty / 0 (graceful)
    assert(eng.string_column_rows(99) == 0 && eng.string_count_where_eq(99, "x") == 0 && "unknown string column -> 0");
    // non-vacuity: the equality count actually discriminates by value
    assert(eng.string_count_where_eq(5, "books") != eng.string_count_where_eq(5, "toys"));
    std::cout << "[string columns] ok\n";
}

int main() { test_string_columns(); std::cout << "ALL STRING-COLUMN TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_nullable.cpp
// CPU test for nullable columns (DM-3j): set_column_nulls marks NULL rows; unfiltered scalar aggregates
// skip them (SQL NULL semantics). A maskless column is byte-identical to before (oracle-safe).
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <bit>
#include <vector>
#include <iostream>

static uint64_t agg_query(CPUMockEngine& eng, uint64_t id, MatrixAggOp op) {
    MatrixQuery q{}; q.value_col = id; q.agg = op; std::vector<uint64_t> o;
    assert(eng.execute_query(q, o) == MatrixQueryStatus::OK && o.size() == 1);
    return o[0];
}

static void test_nullable() {
    std::vector<uint32_t> v = {10, 20, 30, 40, 50};
    CPUMockEngine eng;
    eng.load_scan_column(2, v.data(), v.size());
    // before any mask: full aggregates (existing path, unchanged)
    assert(agg_query(eng, 2, AGG_SUM) == 150 && agg_query(eng, 2, AGG_COUNT) == 5 && "maskless == full");
    // mark rows 1 and 3 NULL (values 20, 40) -> non-null = {10, 30, 50}
    eng.set_column_nulls(2, {0, 1, 0, 1, 0});
    assert(agg_query(eng, 2, AGG_COUNT) == 3 && "COUNT counts non-null only");
    assert(agg_query(eng, 2, AGG_SUM)   == 10 + 30 + 50 && "SUM skips nulls");
    assert(agg_query(eng, 2, AGG_MIN)   == 10 && agg_query(eng, 2, AGG_MAX) == 50 && "MIN/MAX over non-null");
    // non-vacuity: with nulls the SUM differs from the full SUM
    assert(agg_query(eng, 2, AGG_SUM) != 150 && "nulls actually excluded");

    // all-null column -> empty-set sentinels
    std::vector<uint32_t> w = {7, 8, 9};
    eng.load_scan_column(3, w.data(), w.size());
    eng.set_column_nulls(3, {1, 1, 1});
    assert(agg_query(eng, 3, AGG_COUNT) == 0 && agg_query(eng, 3, AGG_SUM) == 0);
    assert(agg_query(eng, 3, AGG_MIN) == UINT64_MAX && agg_query(eng, 3, AGG_MAX) == 0 && "all-null sentinels");
    std::cout << "[nullable] ok\n";
}

static void test_nullable_typed() {
    CPUMockEngine eng;
    std::vector<int64_t> s = {-7, 100, 5000000000LL, -3};
    eng.load_scan_column_i64(7, s.data(), s.size());
    eng.set_column_nulls(7, {0, 1, 0, 1});                 // null rows 1,3 -> non-null {-7, 5e9}
    { MatrixQuery q{}; q.value_col = 7; q.agg = AGG_SUM; std::vector<uint64_t> o; eng.execute_query(q, o);
      assert(static_cast<int64_t>(o[0]) == -7 + 5000000000LL && "int64 nullable SUM skips nulls"); }
    { MatrixQuery q{}; q.value_col = 7; q.agg = AGG_COUNT; std::vector<uint64_t> o; eng.execute_query(q, o);
      assert(o[0] == 2 && "int64 nullable COUNT = non-null"); }

    std::vector<double> d = {1.5, -2.5, 3.0};
    eng.load_scan_column_f64(8, d.data(), d.size());
    eng.set_column_nulls(8, {0, 0, 1});                    // null row 2 -> non-null {1.5, -2.5}
    { MatrixQuery q{}; q.value_col = 8; q.agg = AGG_SUM; std::vector<uint64_t> o; eng.execute_query(q, o);
      assert(std::bit_cast<double>(o[0]) == 1.5 - 2.5 && "double nullable SUM skips nulls"); }
    std::cout << "[nullable typed] ok\n";
}

// Filtered scalar aggregates now also skip NULLs: WHERE + a null mask excludes a row whether or not it
// matches the predicate. Across U32/I64/F64.
static void test_filtered_nullable() {
    CPUMockEngine eng;
    std::vector<uint32_t> v = {10, 20, 30, 40, 50};
    eng.load_scan_column(2, v.data(), v.size());
    // WHERE value >= 30 with NO mask -> {30,40,50}: SUM 120 (baseline for non-vacuity)
    { MatrixQuery q{}; q.value_col = 2; q.agg = AGG_SUM; q.has_filter = true; q.cmp = MatrixCmp::GE; q.threshold = 30;
      std::vector<uint64_t> o; eng.execute_query(q, o); assert(o[0] == 120 && "maskless filtered baseline"); }
    // mask rows 1,4 NULL (values 20,50). WHERE value >= 30 -> non-null AND >=30 = {30,40}: SUM 70, COUNT 2
    eng.set_column_nulls(2, {0, 1, 0, 0, 1});
    { MatrixQuery q{}; q.value_col = 2; q.agg = AGG_SUM; q.has_filter = true; q.cmp = MatrixCmp::GE; q.threshold = 30;
      std::vector<uint64_t> o; eng.execute_query(q, o);
      assert(o[0] == 70 && "filtered SUM skips null row 4 (50 excluded: 120 -> 70)"); }
    { MatrixQuery q{}; q.value_col = 2; q.agg = AGG_COUNT; q.has_filter = true; q.cmp = MatrixCmp::GE; q.threshold = 30;
      std::vector<uint64_t> o; eng.execute_query(q, o); assert(o[0] == 2 && "filtered COUNT of non-null matches"); }

    // int64 filtered + nullable: {-5,100,7,200}, null row1 (100), WHERE value > 0 -> {7,200}: SUM 207
    std::vector<int64_t> s = {-5, 100, 7, 200};
    eng.load_scan_column_i64(3, s.data(), s.size());
    eng.set_column_nulls(3, {0, 1, 0, 0});
    { MatrixQuery q{}; q.value_col = 3; q.agg = AGG_SUM; q.has_filter = true; q.cmp = MatrixCmp::GT; q.lo_i64 = 0;
      std::vector<uint64_t> o; eng.execute_query(q, o);
      assert(static_cast<int64_t>(o[0]) == 207 && "int64 filtered+nullable SUM"); }

    // double filtered + nullable: {1.5,2.5,3.5}, null row2 (3.5), WHERE value < 3.0 -> {1.5,2.5}: SUM 4.0
    std::vector<double> d = {1.5, 2.5, 3.5};
    eng.load_scan_column_f64(4, d.data(), d.size());
    eng.set_column_nulls(4, {0, 0, 1});
    { MatrixQuery q{}; q.value_col = 4; q.agg = AGG_SUM; q.has_filter = true; q.cmp = MatrixCmp::LT; q.lo_f64 = 3.0;
      std::vector<uint64_t> o; eng.execute_query(q, o);
      assert(std::bit_cast<double>(o[0]) == 4.0 && "double filtered+nullable SUM"); }
    std::cout << "[filtered nullable] ok\n";
}

static void grp(CPUMockEngine& eng, uint64_t key, uint64_t val, uint32_t ng, MatrixAggOp op, std::vector<uint64_t>& o) {
    MatrixQuery q{}; q.value_col = val; q.key_col = key; q.num_groups = ng; q.grouped = true; q.agg = op;
    assert(eng.execute_query(q, o) == MatrixQueryStatus::OK);
}

// Grouped aggregates now skip NULL rows too (the documented follow-up to scalar null-awareness): a NULL
// row contributes to no group. Closes the scalar-vs-grouped asymmetry across U32/I64/F64.
static void test_grouped_nullable() {
    CPUMockEngine eng;
    std::vector<uint32_t> region = {0, 1, 0, 2, 1}, amount = {10, 20, 30, 40, 50};
    eng.load_scan_column(1, region.data(), region.size());
    eng.load_scan_column(2, amount.data(), amount.size());
    std::vector<uint64_t> o;
    // baseline (maskless): g0 = 10+30 = 40, g1 = 20+50 = 70, g2 = 40  (oracle-safe: unchanged path)
    grp(eng, 1, 2, 3, AGG_SUM, o);
    assert(o[0] == 40 && o[1] == 70 && o[2] == 40 && "maskless grouped SUM unchanged");
    // mark row 2 (region 0, amount 30) NULL -> g0 loses it
    eng.set_column_nulls(2, {0, 0, 1, 0, 0});
    grp(eng, 1, 2, 3, AGG_SUM, o);
    assert(o[0] == 10 && o[1] == 70 && o[2] == 40 && "grouped SUM skips the null row (g0: 40 -> 10)");
    grp(eng, 1, 2, 3, AGG_COUNT, o);
    assert(o[0] == 1 && o[1] == 2 && o[2] == 1 && "grouped COUNT counts non-null per group");
    grp(eng, 1, 2, 3, AGG_MAX, o);
    assert(o[0] == 10 && "grouped MAX over g0 non-null is 10 (30 excluded)");

    // int64 grouped nullable
    std::vector<int64_t> iv = {-5, 100, 7, 5000000000LL};            // regions {0,0,1,1}
    std::vector<uint32_t> ik = {0, 0, 1, 1};
    eng.load_scan_column(3, ik.data(), ik.size());
    eng.load_scan_column_i64(4, iv.data(), iv.size());
    eng.set_column_nulls(4, {0, 1, 0, 0});                           // null row1 (region 0, val 100)
    grp(eng, 3, 4, 2, AGG_SUM, o);
    assert(static_cast<int64_t>(o[0]) == -5 && static_cast<int64_t>(o[1]) == 7 + 5000000000LL
           && "int64 grouped SUM skips null (g0: -5 only)");

    // double grouped nullable
    std::vector<double> dv = {1.5, 2.5, 4.0, 8.0};                   // regions {0,0,1,1}
    eng.load_scan_column(5, ik.data(), ik.size());
    eng.load_scan_column_f64(6, dv.data(), dv.size());
    eng.set_column_nulls(6, {1, 0, 0, 0});                           // null row0 (region 0, val 1.5)
    grp(eng, 5, 6, 2, AGG_SUM, o);
    assert(std::bit_cast<double>(o[0]) == 2.5 && std::bit_cast<double>(o[1]) == 12.0
           && "double grouped SUM skips null (g0: 2.5 only)");
    std::cout << "[grouped nullable] ok\n";
}

int main() { test_nullable(); test_nullable_typed(); test_filtered_nullable(); test_grouped_nullable(); std::cout << "ALL NULLABLE TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_top_groups.cpp
// CPU test for top-N groups (DM-4 ORDER BY ... DESC LIMIT k): top_groups runs a grouped query and
// returns the k (group, value) pairs with the largest aggregate, descending.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <utility>
#include <vector>
#include <iostream>

using Pairs = std::vector<std::pair<uint64_t, uint64_t>>;

static void test_top_groups() {
    std::vector<uint32_t> region = {0, 1, 0, 2, 1, 0}, amount = {10, 20, 30, 40, 50, 60};
    CPUMockEngine eng;
    eng.load_scan_column(1, region.data(), region.size());
    eng.load_scan_column(2, amount.data(), amount.size());
    MatrixQuery q{}; q.value_col = 2; q.key_col = 1; q.num_groups = 3; q.agg = AGG_SUM; q.grouped = true;
    // group sums: r0 = 10+30+60 = 100, r1 = 20+50 = 70, r2 = 40
    assert((eng.top_groups(q, 2) == Pairs{{0, 100}, {1, 70}}) && "top-2 groups by SUM, descending");
    assert((eng.top_groups(q, 10) == Pairs{{0, 100}, {1, 70}, {2, 40}}) && "k > num_groups -> all groups, sorted");
    const std::pair<uint64_t, uint64_t> top1{0, 100};
    assert(eng.top_groups(q, 1).front() == top1 && "top-1 is the max group");   // comma in <> breaks assert()
    assert(eng.top_groups(q, 0).empty() && "k=0 -> empty");
    // a non-grouped query -> empty (top_groups is for grouped queries)
    MatrixQuery scalar{}; scalar.value_col = 2; scalar.agg = AGG_SUM;
    assert(eng.top_groups(scalar, 3).empty() && "non-grouped query -> empty");
    std::cout << "[top groups] ok\n";
}

// top_query: the string entry point — "... GROUP BY k ORDER BY <agg> DESC LIMIT n" -> parse + top_groups.
static void test_top_query() {
    std::vector<uint32_t> region = {0, 1, 0, 2, 1, 0}, amount = {10, 20, 30, 40, 50, 60};
    CPUMockEngine eng;
    eng.load_scan_column(1, region.data(), region.size());
    eng.load_scan_column(2, amount.data(), amount.size());
    eng.name_column(1, "region"); eng.name_column(2, "amount");   // parser resolves by name
    // r0=100, r1=70, r2=40 -> top-2 desc = [(0,100),(1,70)]
    assert((eng.top_query("SELECT SUM(amount) GROUP BY region ORDER BY SUM DESC LIMIT 2") == Pairs{{0, 100}, {1, 70}})
           && "ORDER BY <agg> DESC LIMIT parses + ranks");
    assert((eng.top_query("SELECT SUM(amount) GROUP BY region ORDER BY amount DESC LIMIT 2") == Pairs{{0, 100}, {1, 70}})
           && "sort key may name the value column too");
    // no LIMIT / not grouped / ASC / bad n -> empty (top_query is top-N only)
    assert(eng.top_query("SELECT SUM(amount) GROUP BY region").empty() && "no LIMIT -> not a top-N query");
    assert(eng.top_query("SELECT SUM(amount)").empty() && "non-grouped -> empty");
    assert(eng.top_query("SELECT SUM(amount) GROUP BY region ORDER BY SUM ASC LIMIT 2").empty() && "ASC rejected");
    assert(eng.top_query("SELECT SUM(amount) GROUP BY region ORDER BY SUM DESC LIMIT 0").empty() && "LIMIT 0 rejected");
    std::cout << "[top query] ok\n";
}

int main() { test_top_groups(); test_top_query(); std::cout << "ALL TOP-GROUPS TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_having.cpp
// CPU test for HAVING (DM-4): having() runs a grouped query and returns the (group, value) pairs whose
// aggregate satisfies a comparison — the SQL HAVING clause. Group-id order preserved.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <utility>
#include <vector>
#include <iostream>

using Pairs = std::vector<std::pair<uint64_t, uint64_t>>;

static void test_having() {
    std::vector<uint32_t> region = {0, 1, 0, 2, 1, 0}, amount = {10, 20, 30, 40, 50, 60};
    CPUMockEngine eng;
    eng.load_scan_column(1, region.data(), region.size());
    eng.load_scan_column(2, amount.data(), amount.size());
    MatrixQuery q{}; q.value_col = 2; q.key_col = 1; q.num_groups = 3; q.agg = AGG_SUM; q.grouped = true;
    // group sums: g0=100, g1=70, g2=40
    assert((eng.having(q, MatrixCmp::GT, 50) == Pairs{{0, 100}, {1, 70}}) && "HAVING SUM > 50");
    assert((eng.having(q, MatrixCmp::LT, 50) == Pairs{{2, 40}}) && "HAVING SUM < 50");
    assert((eng.having(q, MatrixCmp::EQ, 40) == Pairs{{2, 40}}) && "HAVING SUM = 40");
    assert((eng.having(q, MatrixCmp::BETWEEN, 40, 70) == Pairs{{1, 70}, {2, 40}}) && "HAVING SUM BETWEEN 40 AND 70");
    assert(eng.having(q, MatrixCmp::GT, 1000).empty() && "HAVING with no matches -> empty");

    // COUNT per group: g0=3, g1=2, g2=1 -> HAVING COUNT >= 2 keeps g0, g1
    MatrixQuery c = q; c.agg = AGG_COUNT;
    assert((eng.having(c, MatrixCmp::GE, 2) == Pairs{{0, 3}, {1, 2}}) && "HAVING COUNT >= 2");

    // non-grouped query -> empty (HAVING is for grouped queries)
    MatrixQuery scalar{}; scalar.value_col = 2; scalar.agg = AGG_SUM;
    assert(eng.having(scalar, MatrixCmp::GT, 0).empty() && "non-grouped -> empty");
    std::cout << "[having] ok\n";
}

// having_query: the string entry point — "SELECT SUM(x) GROUP BY k HAVING SUM <cmp> v" -> parse + having().
static void test_having_query() {
    std::vector<uint32_t> region = {0, 1, 0, 2, 1, 0}, amount = {10, 20, 30, 40, 50, 60};
    CPUMockEngine eng;
    eng.load_scan_column(1, region.data(), region.size());
    eng.load_scan_column(2, amount.data(), amount.size());
    eng.name_column(1, "region"); eng.name_column(2, "amount");   // g0=100, g1=70, g2=40
    assert((eng.having_query("SELECT SUM(amount) GROUP BY region HAVING SUM > 50") == Pairs{{0, 100}, {1, 70}})
           && "HAVING SUM > 50 (agg-name key)");
    assert((eng.having_query("SELECT SUM(amount) GROUP BY region HAVING amount < 50") == Pairs{{2, 40}})
           && "sort key may name the value column too");
    assert((eng.having_query("SELECT COUNT(amount) GROUP BY region HAVING COUNT >= 2") == Pairs{{0, 3}, {1, 2}})
           && "HAVING COUNT >= 2");
    // not a having query / malformed -> empty
    assert(eng.having_query("SELECT SUM(amount) GROUP BY region").empty() && "no HAVING -> empty");
    assert(eng.having_query("SELECT SUM(amount)").empty() && "non-grouped -> empty");
    assert(eng.having_query("SELECT SUM(amount) GROUP BY region HAVING SUM > ten").empty() && "bad value -> empty");
    assert(eng.having_query("SELECT SUM(amount) GROUP BY region HAVING bogus > 5").empty() && "bad sort key -> empty");
    std::cout << "[having query] ok\n";
}

int main() { test_having(); test_having_query(); std::cout << "ALL HAVING TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_average.cpp
// CPU test for AVG (DM-4): average() derives SUM/COUNT as double(s). Scalar -> one element (NULL-aware,
// since scalar SUM/COUNT skip nulls); grouped -> one per group; zero-count -> NaN. Typed across U32/I64/F64.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <cmath>
#include <vector>
#include <iostream>

static double scalar_avg(CPUMockEngine& eng, uint64_t id) {
    MatrixQuery q{}; q.value_col = id;
    auto a = eng.average(q);
    assert(a.size() == 1 && "scalar AVG -> one element");
    return a[0];
}

static void test_avg_scalar() {
    CPUMockEngine eng;
    std::vector<uint32_t> u = {10, 20, 30, 40, 50};                 // sum 150, count 5 -> 30
    eng.load_scan_column(2, u.data(), u.size());
    assert(scalar_avg(eng, 2) == 30.0 && "u32 scalar AVG = sum/count");

    std::vector<int64_t> s = {-10, 0, 10, 5000000000LL};            // sum 5000000000, count 4 -> 1.25e9
    eng.load_scan_column_i64(3, s.data(), s.size());
    assert(scalar_avg(eng, 3) == 1250000000.0 && "i64 scalar AVG (negatives, big)");

    std::vector<double> d = {1.0, 2.0, 6.0};                        // sum 9, count 3 -> 3.0
    eng.load_scan_column_f64(4, d.data(), d.size());
    assert(scalar_avg(eng, 4) == 3.0 && "f64 scalar AVG");

    // NULL-aware (scalar): mark rows 0,4 null on the u32 col -> {20,30,40} -> 30
    eng.set_column_nulls(2, {1, 0, 0, 0, 1});
    assert(scalar_avg(eng, 2) == 30.0 && "scalar AVG skips nulls (sum 90 / count 3)");
    // non-vacuity: a different mask gives a different average
    eng.set_column_nulls(2, {0, 0, 0, 0, 1});                       // {10,20,30,40} -> 25
    assert(scalar_avg(eng, 2) == 25.0 && "mask actually changes the average");

    // all-null -> count 0 -> NaN
    std::vector<uint32_t> w = {7, 8, 9};
    eng.load_scan_column(5, w.data(), w.size());
    eng.set_column_nulls(5, {1, 1, 1});
    assert(std::isnan(scalar_avg(eng, 5)) && "all-null -> NaN");
    std::cout << "[avg scalar] ok\n";
}

static void test_avg_grouped() {
    CPUMockEngine eng;
    std::vector<uint32_t> region = {0, 1, 0, 2}, amount = {10, 20, 30, 40};
    eng.load_scan_column(1, region.data(), region.size());
    eng.load_scan_column(2, amount.data(), amount.size());
    MatrixQuery q{}; q.value_col = 2; q.key_col = 1; q.num_groups = 3; q.grouped = true;
    auto a = eng.average(q);                                        // g0=(10+30)/2=20, g1=20, g2=40
    assert(a.size() == 3 && a[0] == 20.0 && a[1] == 20.0 && a[2] == 40.0 && "grouped AVG per group");
    // grouped AVG is NULL-aware too (grouped SUM+COUNT both skip nulls): null row 2 (region 0, amount 30)
    // -> g0 = 10/1 = 10 (not 40/2 = 20)
    eng.set_column_nulls(2, {0, 0, 1, 0});
    auto an = eng.average(q);
    assert(an.size() == 3 && an[0] == 10.0 && an[1] == 20.0 && an[2] == 40.0 && "grouped AVG skips null row");
    std::cout << "[avg grouped] ok\n";
}

// avg_query: the string entry point — "SELECT AVG(col) [WHERE ...] [GROUP BY k]" -> rewrite AVG->SUM,
// parse, derive. Round-trips through the full parser (WHERE + GROUP BY supported).
static void test_avg_query() {
    CPUMockEngine eng;
    std::vector<uint32_t> region = {0, 1, 0, 2}, amount = {10, 20, 30, 40};
    eng.load_scan_column(1, region.data(), region.size());
    eng.load_scan_column(2, amount.data(), amount.size());
    eng.name_column(1, "region"); eng.name_column(2, "amount");
    auto s = eng.avg_query("SELECT AVG(amount)");                  // (10+20+30+40)/4 = 25
    assert(s.size() == 1 && s[0] == 25.0 && "scalar AVG from string");
    auto w = eng.avg_query("SELECT AVG(amount) WHERE amount >= 30");// (30+40)/2 = 35
    assert(w.size() == 1 && w[0] == 35.0 && "filtered AVG from string");
    auto g = eng.avg_query("SELECT AVG(amount) GROUP BY region");  // g0=(10+30)/2=20, g1=20, g2=40
    assert(g.size() == 3 && g[0] == 20.0 && g[1] == 20.0 && g[2] == 40.0 && "grouped AVG from string");
    assert(eng.avg_query("SELECT SUM(amount)").empty() && "non-AVG query -> empty");
    assert(eng.avg_query("garbage").empty() && "junk -> empty");
    std::cout << "[avg query] ok\n";
}

int main() { test_avg_scalar(); test_avg_grouped(); test_avg_query(); std::cout << "ALL AVG TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_count_distinct.cpp
// CPU test for COUNT(DISTINCT col): count_distinct returns the number of distinct non-NULL values,
// typed (U32/I64/F64) and null-aware (masked rows excluded).
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

static void test_count_distinct() {
    CPUMockEngine eng;
    std::vector<uint32_t> u = {1, 2, 2, 3, 3, 3};                   // distinct {1,2,3} = 3
    eng.load_scan_column(2, u.data(), u.size());
    assert(eng.count_distinct(2) == 3 && "u32 distinct count");
    // null row 0 (value 1) -> distinct {2,3} = 2
    eng.set_column_nulls(2, {1, 0, 0, 0, 0, 0});
    assert(eng.count_distinct(2) == 2 && "distinct skips the null-only value");

    std::vector<int64_t> s = {-5, -5, 100, 100, 7};                // distinct {-5,100,7} = 3
    eng.load_scan_column_i64(3, s.data(), s.size());
    assert(eng.count_distinct(3) == 3 && "int64 distinct count");

    std::vector<double> d = {1.5, 1.5, 2.5};                       // distinct {1.5,2.5} = 2
    eng.load_scan_column_f64(4, d.data(), d.size());
    assert(eng.count_distinct(4) == 2 && "double distinct count");

    std::vector<uint32_t> all = {10, 20, 30};                      // all unique
    eng.load_scan_column(5, all.data(), all.size());
    assert(eng.count_distinct(5) == 3 && "all-distinct");

    // single repeated value -> 1
    std::vector<uint32_t> one = {7, 7, 7, 7};
    eng.load_scan_column(6, one.data(), one.size());
    assert(eng.count_distinct(6) == 1 && "single distinct value");
    std::cout << "[count distinct] ok\n";
}

// distinct_query: the string entry point — "SELECT COUNT(DISTINCT col)" -> count_distinct.
static void test_distinct_query() {
    std::vector<uint32_t> u = {1, 2, 2, 3, 3, 3};                   // distinct = 3
    CPUMockEngine eng;
    eng.load_scan_column(2, u.data(), u.size());
    eng.name_column(2, "amount");
    uint64_t n = 0;
    assert(eng.distinct_query("SELECT COUNT(DISTINCT amount)", n) && n == 3 && "COUNT(DISTINCT) from string");
    // malformed / non-distinct / unknown -> false, n untouched
    n = 999;
    assert(!eng.distinct_query("SELECT COUNT(amount)", n) && n == 999 && "plain COUNT is not a distinct query");
    assert(!eng.distinct_query("SELECT COUNT(DISTINCT nope)", n) && "unknown column -> false");
    assert(!eng.distinct_query("garbage", n) && "junk -> false");
    std::cout << "[distinct query] ok\n";
}

int main() { test_count_distinct(); test_distinct_query(); std::cout << "ALL COUNT-DISTINCT TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_admission_control.cpp
// CPU test for RM-2 admission control: set_max_query_groups caps a single grouped query's group count
// (result-memory guard). A query above the cap returns ERR_TOO_MANY_GROUPS with no allocation; the
// default (2^28) leaves existing queries unaffected.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

static void test_admission_control() {
    std::vector<uint32_t> region = {0, 1, 2, 1, 0}, amount = {10, 20, 30, 40, 50};
    CPUMockEngine eng;
    eng.load_scan_column(1, region.data(), region.size());
    eng.load_scan_column(2, amount.data(), amount.size());
    MatrixQuery q{}; q.value_col = 2; q.key_col = 1; q.num_groups = 3; q.agg = AGG_SUM; q.grouped = true;

    // default cap (2^28): the 3-group query runs fine
    assert(eng.max_query_groups() == (1u << 28) && "default cap is 2^28");
    std::vector<uint64_t> o;
    assert(eng.execute_query(q, o) == MatrixQueryStatus::OK && o.size() == 3 && "under default cap -> OK");

    // tighten the cap below the query's group count -> rejected, no allocation
    eng.set_max_query_groups(2);
    std::vector<uint64_t> o2;
    assert(eng.execute_query(q, o2) == MatrixQueryStatus::ERR_TOO_MANY_GROUPS && "3 groups > cap 2 -> rejected");
    assert(o2.empty() && "rejected query allocates nothing");

    // cap exactly at the group count -> allowed (boundary: > cap, not >=)
    eng.set_max_query_groups(3);
    assert(eng.execute_query(q, o) == MatrixQueryStatus::OK && "num_groups == cap -> OK (strict >)");

    // raise it back -> OK again (knob is live)
    eng.set_max_query_groups(1u << 28);
    assert(eng.execute_query(q, o) == MatrixQueryStatus::OK && "cap raised -> OK again");
    std::cout << "[admission control] ok\n";
}

int main() { test_admission_control(); std::cout << "ALL ADMISSION-CONTROL TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_rebalance_config.cpp
// CPU test for OB-4 runtime config: set_rebalance_interval tunes the heat-rebalance cadence at runtime.
// A scalar scan triggers maybe_rebalance; stats().rebalances counts the passes that fired.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

static void scan(CPUMockEngine& e) {
    MatrixQuery q{}; q.value_col = 2; q.agg = AGG_COUNT; std::vector<uint64_t> o;
    e.execute_query(q, o);
}

static void test_rebalance_config() {
    // default cadence is 4: rebalance fires only on the 4th scan
    {
        CPUMockEngine eng; std::vector<uint32_t> v(10, 1); eng.load_scan_column(2, v.data(), v.size());
        assert(eng.rebalance_interval() == 4 && "default cadence == REBALANCE_EVERY");
        scan(eng); scan(eng); scan(eng);
        assert(eng.stats().rebalances == 0 && "no rebalance before the 4th scan");
        scan(eng);
        assert(eng.stats().rebalances == 1 && "rebalance fires on the 4th scan");
    }
    // aggressive: interval 1 -> a rebalance pass every scan
    {
        CPUMockEngine eng; std::vector<uint32_t> v(10, 1); eng.load_scan_column(2, v.data(), v.size());
        eng.set_rebalance_interval(1);
        scan(eng); scan(eng); scan(eng);
        assert(eng.stats().rebalances == 3 && "interval 1 -> rebalance every scan");
    }
    // relaxed: a large interval suppresses rebalancing over many scans
    {
        CPUMockEngine eng; std::vector<uint32_t> v(10, 1); eng.load_scan_column(2, v.data(), v.size());
        eng.set_rebalance_interval(1000);
        for (int i = 0; i < 5; ++i) scan(eng);
        assert(eng.stats().rebalances == 0 && "large interval -> no rebalance over 5 scans");
        eng.set_rebalance_interval(0);
        assert(eng.rebalance_interval() == 1 && "0 clamps to 1 (rebalance every scan)");
    }
    std::cout << "[rebalance config] ok\n";
}

int main() { test_rebalance_config(); std::cout << "ALL REBALANCE-CONFIG TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_shutdown.cpp
// CPU test for RM-4 graceful shutdown: shutdown() rolls back any open txn, then checkpoints the WAL so a
// restart replays an ~empty log (bounded recovery). Committed writes survive; uncommitted are discarded.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <cstdio>
#include <string>
#include <iostream>

static void clean(const std::string& wal) {
    std::remove(wal.c_str());
    std::remove((wal + ".ckpt").c_str());
    std::remove((wal + ".ckpt.tmp").c_str());
}

static uint64_t get(CPUMockEngine& e, uint64_t k) { uint64_t v = 0; e.kv_get(k, v); return v; }

static void test_shutdown_checkpoints_and_recovers() {
    const std::string wal = "/tmp/mdb_rm4.wal"; clean(wal);
    {
        CPUMockEngine e(0, wal);
        e.begin(); e.txn_put(1, 111); e.commit();
        e.begin(); e.txn_put(2, 222); e.commit();
        assert(e.wal_records() > 0 && "writes appended to the WAL");
        e.shutdown();                                            // checkpoint: snapshot + truncate
        assert(e.wal_records() == 0 && e.checkpoints() == 1 && "shutdown truncated the WAL via a checkpoint");
    }
    // restart on the same path: recovers from the checkpoint with an empty WAL replay
    {
        CPUMockEngine e(0, wal);
        assert(get(e, 1) == 111 && get(e, 2) == 222 && "committed writes survive shutdown+restart");
        assert(e.wal_records() == 0 && "restart replayed an ~empty WAL (fast recovery)");
    }
    clean(wal);
    std::cout << "[shutdown checkpoints+recovers] ok\n";
}

static void test_shutdown_rolls_back_open_txn() {
    const std::string wal = "/tmp/mdb_rm4b.wal"; clean(wal);
    {
        CPUMockEngine e(0, wal);
        e.begin(); e.txn_put(7, 777); e.commit();
        e.begin(); e.txn_put(8, 888);                            // NOT committed
        e.shutdown();                                            // rolls back the open txn, then checkpoints
    }
    {
        CPUMockEngine e(0, wal);
        assert(get(e, 7) == 777 && "committed write survives");
        assert(get(e, 8) == 0 && "uncommitted write discarded on graceful shutdown");
    }
    clean(wal);
    std::cout << "[shutdown rolls back open txn] ok\n";
}

static void test_shutdown_idempotent_and_no_wal() {
    // idempotent: a second shutdown is safe (and a third)
    const std::string wal = "/tmp/mdb_rm4c.wal"; clean(wal);
    { CPUMockEngine e(0, wal); e.begin(); e.txn_put(1, 1); e.commit();
      e.shutdown(); e.shutdown(); assert(e.wal_records() == 0 && "repeated shutdown stays clean"); }
    clean(wal);
    // no WAL attached -> shutdown is a no-op, never crashes
    { CPUMockEngine e; e.shutdown(); assert(e.checkpoints() == 0 && "shutdown without a WAL is a no-op"); }
    std::cout << "[shutdown idempotent + no-WAL] ok\n";
}

int main() {
    test_shutdown_checkpoints_and_recovers();
    test_shutdown_rolls_back_open_txn();
    test_shutdown_idempotent_and_no_wal();
    std::cout << "ALL SHUTDOWN TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_durability_levels.cpp
// CPU test for DU-5 durability levels: the engine constructor selects the WAL fsync policy. SYNC_EACH
// (default) fsyncs each commit (survives power loss); SYNC_OFF buffers for throughput. Both recover
// committed writes on a clean restart — the power-loss difference is inherent to fsync, not unit-testable
// (a clean process exit flushes either way), so this pins plumbing + recovery, and documents the rest.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <cstdio>
#include <string>
#include <iostream>

static void clean(const std::string& wal) {
    std::remove(wal.c_str()); std::remove((wal + ".ckpt").c_str()); std::remove((wal + ".ckpt.tmp").c_str());
}
static uint64_t get(CPUMockEngine& e, uint64_t k) { uint64_t v = 0; e.kv_get(k, v); return v; }

static void test_default_is_sync_each() {
    const std::string wal = "/tmp/mdb_du5_a.wal"; clean(wal);
    CPUMockEngine e(0, wal);                                    // default sync policy
    assert(e.durability_level() == SyncPolicy::SYNC_EACH && "default WAL durability is SYNC_EACH (fsync per commit)");
    CPUMockEngine mem;                                         // no WAL
    assert(mem.durability_level() == SyncPolicy::SYNC_OFF && "no WAL -> nothing to sync");
    clean(wal);
    std::cout << "[du5 default sync_each] ok\n";
}

static void test_policy_selectable_and_recovers() {
    const std::string wal = "/tmp/mdb_du5_b.wal"; clean(wal);
    {
        CPUMockEngine e(0, wal, SIZE_MAX, SyncPolicy::SYNC_OFF);   // operator opts into throughput mode
        assert(e.durability_level() == SyncPolicy::SYNC_OFF && "SYNC_OFF selected via constructor");
        e.begin(); e.txn_put(1, 111); e.commit();
        e.begin(); e.txn_put(2, 222); e.commit();
    }
    // clean restart recovers committed writes regardless of policy (OS flushed buffers on close)
    {
        CPUMockEngine e(0, wal, SIZE_MAX, SyncPolicy::SYNC_OFF);
        assert(get(e, 1) == 111 && get(e, 2) == 222 && "committed writes recover under SYNC_OFF on a clean restart");
    }
    clean(wal);
    std::cout << "[du5 selectable + recovers] ok\n";
}

int main() { test_default_is_sync_each(); test_policy_selectable_and_recovers();
             std::cout << "ALL DURABILITY-LEVEL TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_fault_injection.cpp
// CPU test for QA-5 fault injection (engine-level WAL corruption recovery): a fresh CPUMockEngine built
// on a CORRUPT WAL must recover the intact prefix, discard the corrupt tail, and stay usable — never
// crash, never apply garbage. (test_cold_store covers the ColdStore unit; this is the engine integration.)
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <cstdio>
#include <string>
#include <iostream>

static void clean(const std::string& wal) {
    std::remove(wal.c_str()); std::remove((wal + ".ckpt").c_str()); std::remove((wal + ".ckpt.tmp").c_str());
}
static uint64_t get(CPUMockEngine& e, uint64_t k) { uint64_t v = 0; e.kv_get(k, v); return v; }

// Write k=k*10 as single-key committed transactions for k in [1, n].
static void write_n(const std::string& wal, uint64_t n) {
    CPUMockEngine e(0, wal);
    for (uint64_t k = 1; k <= n; ++k) { e.begin(); e.txn_put(k, k * 10); e.commit(); }
}

static void test_torn_tail_recovers_prefix() {
    const std::string wal = "/tmp/mdb_fault_a.wal"; clean(wal);
    write_n(wal, 5);
    // simulate a crash mid-write: append a torn record (a few bytes — not a whole [len][crc][payload])
    { std::FILE* f = std::fopen(wal.c_str(), "ab"); assert(f);
      const unsigned char garbage[5] = {0xDE, 0xAD, 0xBE, 0xEF, 0x01};
      std::fwrite(garbage, 1, sizeof garbage, f); std::fclose(f); }
    // fresh engine: recovers all 5 committed writes, ignores the torn tail, no crash
    CPUMockEngine e(0, wal);
    for (uint64_t k = 1; k <= 5; ++k) assert(get(e, k) == k * 10 && "committed write survives a torn tail");
    // and the engine is still usable after recovery
    e.begin(); e.txn_put(99, 990); e.commit();
    assert(get(e, 99) == 990 && "engine usable after recovering past a torn tail");
    clean(wal);
    std::cout << "[torn tail recovers prefix] ok\n";
}

static void test_early_corruption_stops_clean() {
    const std::string wal = "/tmp/mdb_fault_b.wal"; clean(wal);
    write_n(wal, 4);
    // flip a byte inside the FIRST record's payload (offset 8 = past the [u32 len][u32 crc] header) ->
    // CRC mismatch on record 1 -> replay stops immediately, recovering nothing.
    { std::FILE* f = std::fopen(wal.c_str(), "r+b"); assert(f);
      std::fseek(f, 8, SEEK_SET);
      unsigned char b = 0; size_t r = std::fread(&b, 1, 1, f); assert(r == 1);
      b ^= 0xFF; std::fseek(f, 8, SEEK_SET); std::fwrite(&b, 1, 1, f); std::fclose(f); }
    // fresh engine: corruption at record 1 -> nothing recovered, but NO crash and the engine is usable
    CPUMockEngine e(0, wal);
    assert(get(e, 1) == 0 && "corrupt first record -> not recovered (replay stopped at it)");
    e.begin(); e.txn_put(7, 77); e.commit();
    assert(get(e, 7) == 77 && "engine fully usable after a corrupt-WAL recovery");
    clean(wal);
    std::cout << "[early corruption stops clean] ok\n";
}

int main() {
    test_torn_tail_recovers_prefix();
    test_early_corruption_stops_clean();
    std::cout << "ALL FAULT-INJECTION TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_health.cpp
// CPU test for OB-3 health/readiness probe: health() reports a ready verdict + the gauges behind it.
// Exercises the gauges (catalog size, durable flag, pending-WAL, resident bytes) and pins the verdict
// invariant ready == (dropped_writes == 0). The degraded path (ready=false) needs a release build + a
// full KVStore (the overflow asserts in debug), so it's covered by the invariant, not a forced overflow.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <cstdio>
#include <string>
#include <iostream>

static void clean(const std::string& wal) {
    std::remove(wal.c_str()); std::remove((wal + ".ckpt").c_str()); std::remove((wal + ".ckpt.tmp").c_str());
}

static void test_health_fresh() {
    CPUMockEngine eng;                                          // no WAL
    HealthStatus h = eng.health();
    assert(h.ready && "fresh engine is ready");
    assert(!h.durable && "no WAL -> not durable");
    assert(h.catalog_columns == 0 && h.host_resident_bytes == 0 && "empty catalog");
    assert(h.wal_records_pending == 0 && h.dropped_writes == 0 && "nothing pending, nothing dropped");
    assert(h.ready == (h.dropped_writes == 0) && "ready verdict == no-dropped-writes invariant");

    std::vector<uint32_t> a = {1, 2, 3, 4}, b = {5, 6, 7, 8};
    eng.load_scan_column(1, a.data(), a.size());
    eng.load_scan_column(2, b.data(), b.size());
    h = eng.health();
    assert(h.catalog_columns == 2 && "two columns registered");
    assert(h.host_resident_bytes == (a.size() + b.size()) * sizeof(uint32_t) && "resident bytes track the catalog");
    std::cout << "[health fresh] ok\n";
}

static void test_health_durable() {
    const std::string wal = "/tmp/mdb_health.wal"; clean(wal);
    {
        CPUMockEngine eng(0, wal);
        assert(eng.health().durable && "WAL attached -> durable");
        eng.begin(); eng.txn_put(1, 10); eng.commit();
        eng.begin(); eng.txn_put(2, 20); eng.commit();
        HealthStatus h = eng.health();
        assert(h.wal_records_pending > 0 && "un-checkpointed commits are pending");
        assert(h.ready && "still ready");
        eng.shutdown();                                         // checkpoint compacts the WAL
        assert(eng.health().wal_records_pending == 0 && "after shutdown the WAL is compacted");
    }
    clean(wal);
    std::cout << "[health durable] ok\n";
}

int main() { test_health_fresh(); test_health_durable(); std::cout << "ALL HEALTH TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_fuzz.cpp
// CPU fuzz harness (QA-4): hammer the UNTRUSTED-INPUT paths with seeded pseudo-random + mutated inputs
// and assert they never crash (graceful status/bool, no UB/OOB). Run it under ASan/UBSan (SAN=1
// ./run_tests.sh) to also catch memory bugs the fixed-input unit tests miss. Deterministic (seeded LCG,
// no wall-clock/RNG) so failures reproduce.
#include "server.hpp"        // CPUMockEngine + matrix_deserialize_request
#include "csv_ingest.hpp"
#include <cassert>
#include <cstdint>
#include <cstdio>
#include <fstream>
#include <string>
#include <vector>
#include <iostream>

int main() {
    uint64_t st = 0x123456789abcdef0ull;
    auto rnd = [&] { st = st * 6364136223846793005ull + 1442695040888963407ull; return st >> 33; };

    CPUMockEngine eng;
    std::vector<uint32_t> v = {1, 2, 3};
    eng.load_scan_column(2, v.data(), v.size()); eng.name_column(2, "qty");

    const std::string valid = "SELECT SUM(qty) WHERE qty > 5";
    uint64_t parsed_ok = 0, deser_ok = 0;

    // 1) random strings -> parse_query  (must return a status, never crash)
    // 2) random bytes  -> matrix_deserialize_request  (must return bool, never crash)
    for (int i = 0; i < 40000; ++i) {
        std::string s;
        const size_t len = rnd() % 48;
        for (size_t j = 0; j < len; ++j) s.push_back(static_cast<char>(rnd() % 128));   // any byte incl. control/NUL
        MatrixQuery q;
        if (eng.parse_query(s, q) == MatrixQueryStatus::OK) ++parsed_ok;

        std::vector<uint8_t> b;
        const size_t blen = rnd() % 80;
        for (size_t j = 0; j < blen; ++j) b.push_back(static_cast<uint8_t>(rnd() & 0xff));
        MatrixRequest req;
        if (matrix_deserialize_request(b, req)) ++deser_ok;
    }

    // 3) mutate a VALID query (flip/insert/delete a byte) -> reaches deeper into the parser
    for (int i = 0; i < 20000; ++i) {
        std::string s = valid;
        const int muts = 1 + static_cast<int>(rnd() % 4);
        for (int m = 0; m < muts && !s.empty(); ++m) {
            const size_t pos = rnd() % s.size();
            switch (rnd() % 3) {
                case 0: s[pos] = static_cast<char>(rnd() % 128); break;          // flip
                case 1: s.insert(pos, 1, static_cast<char>(rnd() % 128)); break; // insert
                default: s.erase(pos, 1); break;                                 // delete
            }
        }
        MatrixQuery q;
        if (eng.parse_query(s, q) == MatrixQueryStatus::OK) ++parsed_ok;
    }

    // 4) random CSV files -> matrix_read_csv_column (must return bool, never crash)
    const std::string path = "/tmp/mdb_fuzz.csv";
    for (int i = 0; i < 4000; ++i) {
        std::string body;
        const size_t len = rnd() % 64;
        for (size_t j = 0; j < len; ++j) body.push_back(static_cast<char>(rnd() % 128));
        std::ofstream(path) << body;
        std::vector<uint32_t> out;
        matrix_read_csv_column(path, rnd() % 3, false, ',', out);
    }
    std::remove(path.c_str());

    // The whole point is reaching this line — no input crashed. The counts confirm the fuzzer exercised
    // the success paths too (not just instant rejections), so the deep code was actually hit.
    std::cout << "fuzz: " << parsed_ok << " queries parsed OK, " << deser_ok
              << " requests deserialized (over 64k inputs, no crash)\n";
    assert(parsed_ok > 0 && "mutation fuzzer hit some valid parses (exercised deep parser paths)");
    std::cout << "ALL FUZZ TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_stress.cpp
// CPU stress / load test (QA-5): sustained query load + heavy tiering churn + append + join at scale,
// every result checked against a closed-form oracle (value[i] = i). Catches scale bugs (overflow,
// residency accounting, churn) the small fixed-input unit tests miss. Sized to run in ~1-2s.
#include "compute_mock.cpp"
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

int main() {
    constexpr uint32_t N = 50000;                 // rows per column (value[i] = i)
    std::vector<uint32_t> col(N);
    for (uint32_t i = 0; i < N; ++i) col[i] = i;

    // Small host budget vs 4x 200KB columns -> the engine must churn (borrow/demote/promote) under load.
    CPUMockEngine eng(0, "", 300 * 1024);
    for (uint64_t id = 1; id <= 4; ++id) eng.load_scan_column(id, col.data(), col.size());

    // Closed-form: for value[i]=i over [0,N), SUM(value > t) = (N-1)N/2 - t(t+1)/2 ; COUNT(value > t) = N-1-t.
    auto sum_gt = [&](uint64_t t) { return (uint64_t)(N - 1) * N / 2 - t * (t + 1) / 2; };

    // 300 filtered queries cycling the 4 columns — sustained load + rebalance/borrow churn.
    uint64_t seed = 1;
    for (int q = 0; q < 300; ++q) {
        const uint64_t id = 1 + (q % 4);
        seed = seed * 6364136223846793005ull + 1; const uint32_t t = (seed >> 40) % N;   // varied threshold
        MatrixQuery mq{}; mq.value_col = id; mq.agg = AGG_SUM; mq.has_filter = true; mq.cmp = MatrixCmp::GT; mq.threshold = t;
        std::vector<uint64_t> o;
        assert(eng.execute_query(mq, o) == MatrixQueryStatus::OK && o[0] == sum_gt(t) && "filtered SUM correct under load");
        MatrixQuery cq{}; cq.value_col = id; cq.agg = AGG_COUNT; cq.has_filter = true; cq.cmp = MatrixCmp::GT; cq.threshold = t;
        std::vector<uint64_t> oc; eng.execute_query(cq, oc);
        assert(oc[0] == (uint64_t)(N - 1 - t) && "filtered COUNT correct under load");
    }

    // Append stress: grow column 1 by 500 small batches; final SUM includes the appended rows.
    uint64_t appended_sum = 0; uint32_t extra[8];
    for (int b = 0; b < 500; ++b) {
        for (int j = 0; j < 8; ++j) { extra[j] = 1000000u + b * 8 + j; appended_sum += extra[j]; }
        eng.append_to_column(1, extra, 8);
    }
    assert(eng.column_rows(1) == N + 500 * 8 && "appended rows present");
    { MatrixQuery mq{}; mq.value_col = 1; mq.agg = AGG_SUM; std::vector<uint64_t> o; eng.execute_query(mq, o);
      assert(o[0] == (uint64_t)(N - 1) * N / 2 + appended_sum && "full SUM incl. appended rows correct"); }

    // Join at scale: col2 (=col, value[i]=i) joined with itself-shaped col3 -> N matching pairs (each i once).
    assert(eng.hash_join_count(2, 3) == N && "self-aligned join cardinality == N");

    // The tiering must have actually churned under this load (not a no-op).
    const EngineStats s = eng.stats();
    assert(s.rebalances > 0 && s.cold_borrows > 0 && "tiering churned under sustained load");
    assert(s.query_count >= 300 && "all queries counted");
    std::cout << "stress: 300 queries + 500 appends + " << N << "-row join, churn rebalances=" << s.rebalances
              << " cold_borrows=" << s.cold_borrows << " (all results oracle-correct)\n";
    std::cout << "ALL STRESS TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_server_tcp.cpp
// CPU test for the TCP transport adapter (NW transport): the length-prefixed wire protocol +
// matrix_serve_conn are runtime-verified over a socketpair (no bind needed) — a framed request served
// over a real socket yields exactly the same response bytes as a direct matrix_serve. (The bind/accept
// loop, matrix_serve_tcp, is HOST-ONLY and only compile-verified here — see server_tcp.hpp.)
#include "server_tcp.hpp"
#include <sys/socket.h>
#include <unistd.h>
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

static MatrixRequest mk(ReqKind k) { MatrixRequest r; r.kind = k; return r; }

static void framed_write(int fd, const std::vector<uint8_t>& b) {
    uint32_t len = static_cast<uint32_t>(b.size());
    assert(matrixsrv_detail::send_all(fd, &len, sizeof len));
    if (len) assert(matrixsrv_detail::send_all(fd, b.data(), len));
}
static std::vector<uint8_t> framed_read(int fd) {
    uint32_t len = 0; assert(matrixsrv_detail::recv_all(fd, &len, sizeof len));
    std::vector<uint8_t> b(len); if (len) assert(matrixsrv_detail::recv_all(fd, b.data(), len));
    return b;
}

// Serve `req` over a socketpair and assert the framed response equals a direct matrix_serve of the same bytes.
static void roundtrip(CPUMockEngine& eng, const AccessPolicy& pol, uint64_t principal, const MatrixRequest& req) {
    int fd[2]; assert(::socketpair(AF_UNIX, SOCK_STREAM, 0, fd) == 0);
    const std::vector<uint8_t> reqb = matrix_serialize_request(req);
    framed_write(fd[0], reqb);                                  // client -> framed request
    assert(matrix_serve_conn(eng, pol, principal, fd[1]) && "server served one framed request");
    const std::vector<uint8_t> got = framed_read(fd[0]);        // client <- framed response
    const std::vector<uint8_t> expect = matrix_serve(eng, pol, principal, reqb);   // direct serve, same bytes
    assert(got == expect && "TCP-framed serve-over-socket == direct matrix_serve");
    ::close(fd[0]); ::close(fd[1]);
}

int main() {
    std::vector<uint32_t> v(1000);
    for (size_t i = 0; i < v.size(); ++i) v[i] = static_cast<uint32_t>(i);
    CPUMockEngine eng;                                  // durability off (PUT applies to kv_ in memory)
    eng.load_scan_column(2, v.data(), v.size());
    AccessPolicy pol; pol.allow_column(1, 2); pol.allow_read(1); pol.allow_write(1);   // principal 1 only

    MatrixRequest g = mk(ReqKind::GET);   g.key = 5;
    MatrixRequest p = mk(ReqKind::PUT);   p.key = 5; p.value = 500;
    MatrixRequest q = mk(ReqKind::QUERY); q.query = MatrixQuery{}; q.query.value_col = 2; q.query.agg = AGG_SUM;
    roundtrip(eng, pol, 1, g);    // allowed GET
    roundtrip(eng, pol, 1, p);    // allowed PUT
    roundtrip(eng, pol, 1, q);    // allowed QUERY (sum over 1000 values)
    roundtrip(eng, pol, 2, p);    // DENIED (principal 2) -> ERR_FORBIDDEN, faithfully framed over the socket
    // a malformed/empty framed request still serves a (bad-request) response without hanging/crashing
    { int fd[2]; assert(::socketpair(AF_UNIX, SOCK_STREAM, 0, fd) == 0);
      framed_write(fd[0], {1, 2, 3});                   // garbage payload
      assert(matrix_serve_conn(eng, pol, 7, fd[1]));
      assert(!framed_read(fd[0]).empty() && "malformed request -> a (bad-request) response, no hang");
      ::close(fd[0]); ::close(fd[1]); }

    std::cout << "ALL TCP-TRANSPORT TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_recv_timeout.cpp
// CPU test for NW-5 connection read timeout: a client that connects but never sends must NOT hang the
// single-owner serve loop — SO_RCVTIMEO makes recv fail, so matrix_serve_conn returns false and the loop
// drops the stuck connection. A normal request still serves fine with a timeout set.
#include "server_tcp.hpp"
#include <sys/socket.h>
#include <unistd.h>
#include <cassert>
#include <cstdint>
#include <vector>
#include <iostream>

static void test_recv_timeout() {
    std::vector<uint32_t> v(10, 1);
    CPUMockEngine eng; eng.load_scan_column(2, v.data(), v.size());
    const AccessPolicy pol = AccessPolicy::permissive();

    // (1) slowloris: client connects, sends NOTHING. With a short recv timeout, serve_conn must return
    // false (not block forever). The 200ms timeout bounds this — if it hung, the test would never finish.
    {
        int fd[2]; assert(::socketpair(AF_UNIX, SOCK_STREAM, 0, fd) == 0);
        assert(matrix_set_recv_timeout(fd[1], 200) && "set recv timeout");
        assert(!matrix_serve_conn(eng, pol, 0, fd[1]) && "stuck client (no data) -> serve_conn returns false, no hang");
        ::close(fd[0]); ::close(fd[1]);
    }
    // (2) a normal framed request still serves OK with a timeout set (data arrives before the timeout)
    {
        int fd[2]; assert(::socketpair(AF_UNIX, SOCK_STREAM, 0, fd) == 0);
        assert(matrix_set_recv_timeout(fd[1], 2000));
        MatrixRequest q; q.kind = ReqKind::QUERY; q.query = MatrixQuery{}; q.query.value_col = 2; q.query.agg = AGG_SUM;
        const std::vector<uint8_t> rb = matrix_serialize_request(q);
        const uint32_t len = static_cast<uint32_t>(rb.size());
        assert(matrixsrv_detail::send_all(fd[0], &len, sizeof len) && matrixsrv_detail::send_all(fd[0], rb.data(), len));
        assert(matrix_serve_conn(eng, pol, 0, fd[1]) && "a request that arrives in time still serves");
        uint32_t rlen = 0; assert(matrixsrv_detail::recv_all(fd[0], &rlen, sizeof rlen) && rlen > 0 && "got a response");
        ::close(fd[0]); ::close(fd[1]);
    }
    // (3) send timeout: the symmetric setter works, and a normal request/response still completes with
    // both timeouts set (the send buffers aren't full, so send returns immediately). The block-past-
    // deadline path is the same SO_*TIMEO kernel mechanism proven for recv in (1).
    {
        int fd[2]; assert(::socketpair(AF_UNIX, SOCK_STREAM, 0, fd) == 0);
        assert(matrix_set_recv_timeout(fd[1], 2000) && matrix_set_send_timeout(fd[1], 2000) && "both timeouts set");
        MatrixRequest q; q.kind = ReqKind::QUERY; q.query = MatrixQuery{}; q.query.value_col = 2; q.query.agg = AGG_COUNT;
        const std::vector<uint8_t> rb = matrix_serialize_request(q);
        const uint32_t len = static_cast<uint32_t>(rb.size());
        assert(matrixsrv_detail::send_all(fd[0], &len, sizeof len) && matrixsrv_detail::send_all(fd[0], rb.data(), len));
        assert(matrix_serve_conn(eng, pol, 0, fd[1]) && "serve completes with both timeouts set");
        uint32_t rlen = 0; assert(matrixsrv_detail::recv_all(fd[0], &rlen, sizeof rlen) && rlen > 0);
        ::close(fd[0]); ::close(fd[1]);
    }
    std::cout << "[recv timeout] ok\n";
}

int main() { test_recv_timeout(); std::cout << "ALL RECV-TIMEOUT TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_client.cpp
// CPU test for the NW-4 client driver: MatrixClient drives the wire protocol end-to-end over a socketpair.
// The server serves the connection in a thread; the client thread only touches the socket (no engine race).
// Exercises every op (PUT/GET/QUERY/HEALTH/STATS) and confirms client results == direct engine calls.
#include "client.hpp"
#include <sys/socket.h>
#include <unistd.h>
#include <csignal>
#include <cassert>
#include <cstdint>
#include <thread>
#include <vector>
#include <iostream>

static void test_client_roundtrip() {
    int fd[2]; assert(::socketpair(AF_UNIX, SOCK_STREAM, 0, fd) == 0);
    std::vector<uint32_t> v(100);
    for (size_t i = 0; i < v.size(); ++i) v[i] = static_cast<uint32_t>(i);
    CPUMockEngine eng; eng.load_scan_column(2, v.data(), v.size());
    const AccessPolicy pol = AccessPolicy::permissive();

    // server: serve this connection's requests until the client closes its end
    std::thread srv([&] { while (matrix_serve_conn(eng, pol, /*principal=*/1, fd[1])) {} });

    MatrixClient cli(fd[0]);
    // PUT then GET it back over the wire
    assert(cli.put(5, 500) && "PUT over the wire");
    uint64_t got = 0;
    assert(cli.get(5, got) && got == 500 && "GET returns the PUT value");
    uint64_t miss = 123;
    assert(!cli.get(99999, miss) && miss == 123 && "GET miss -> false, out untouched");

    // QUERY: SUM over the column == direct engine call
    MatrixQuery q{}; q.value_col = 2; q.agg = AGG_SUM;
    MatrixQueryStatus st; std::vector<uint64_t> out;
    assert(cli.query(q, st, out) && st == MatrixQueryStatus::OK && out.size() == 1);
    std::vector<uint64_t> direct; eng.execute_query(q, direct);
    assert(out == direct && "client QUERY == direct execute_query");
    // a query-level error rides through as a status (not a transport failure)
    MatrixQuery bad{}; bad.value_col = 999; bad.agg = AGG_SUM;
    assert(cli.query(bad, st, out) && st == MatrixQueryStatus::ERR_UNKNOWN_COLUMN && "query error -> status, transport ok");

    // HEALTH over the wire == the engine's own snapshot
    HealthStatus h;
    assert(cli.health(h) && h.ready && h.catalog_columns == 1 && "HEALTH over the wire");
    assert(h.host_resident_bytes == eng.health().host_resident_bytes && "health gauges match the engine");

    // STATS over the wire: 12 fields, query_count reflects the queries served
    std::vector<uint64_t> s;
    assert(cli.stats(s) && s.size() == 12 && "STATS over the wire");
    assert(s[6] == eng.stats().query_count && "client-observed query_count == engine's");
    assert(cli.server_version() == eng.version_u64() && cli.server_version() != 0 && "server version over the wire");

    ::close(fd[0]);                                            // signals the server loop to end
    srv.join();
    ::close(fd[1]);
    std::cout << "[client round-trip] ok\n";
}

// SE-1 over the transport: the connection authenticates once (token frame), then serves as that principal.
static void test_client_authenticated() {
    std::vector<uint32_t> v(20, 1);
    CPUMockEngine eng; eng.load_scan_column(2, v.data(), v.size());
    Authenticator auth; auth.add_credential("alice-token", 1);
    AccessPolicy pol; pol.allow_column(1, 2);                  // principal 1 may query col 2

    // valid token: handshake succeeds, then the authenticated principal can query
    {
        int fd[2]; assert(::socketpair(AF_UNIX, SOCK_STREAM, 0, fd) == 0);
        // server owns fd[1] and closes it when done (as matrix_serve_tcp's accept loop closes each conn)
        std::thread srv([&] { matrix_serve_conn_auth(eng, pol, auth, fd[1]); ::close(fd[1]); });
        MatrixClient cli(fd[0]);
        assert(cli.authenticate("alice-token") && "send the token frame");
        MatrixQuery q{}; q.value_col = 2; q.agg = AGG_COUNT;
        MatrixQueryStatus st; std::vector<uint64_t> out;
        assert(cli.query(q, st, out) && st == MatrixQueryStatus::OK && out[0] == 20 && "authenticated query OK");
        ::close(fd[0]); srv.join();
    }
    // bad token: server rejects the handshake and closes -> the client's query gets EOF (no response)
    {
        int fd[2]; assert(::socketpair(AF_UNIX, SOCK_STREAM, 0, fd) == 0);
        std::thread srv([&] { matrix_serve_conn_auth(eng, pol, auth, fd[1]); ::close(fd[1]); });
        MatrixClient cli(fd[0]);
        assert(cli.authenticate("bad-token") && "token frame sent (server will reject it)");
        MatrixQuery q{}; q.value_col = 2; q.agg = AGG_COUNT;
        MatrixQueryStatus st; std::vector<uint64_t> out;
        assert(!cli.query(q, st, out) && "unauthenticated connection -> no response (server closed)");
        ::close(fd[0]); srv.join();
    }
    std::cout << "[client authenticated] ok\n";
}

int main() {
    std::signal(SIGPIPE, SIG_IGN);   // a write to a closed peer must yield EPIPE (false), not kill the process
    test_client_roundtrip();
    test_client_authenticated();
    std::cout << "ALL CLIENT TESTS PASSED\n";
    return 0;
}


In [ ]:
%%writefile test_version.cpp
// CPU test for BP-3 versioning: the version constant, packed form, and the engine getter agree.
#include "compute_mock.cpp"   // pulls in version.hpp + the engine version() getter
#include <cassert>
#include <cstdint>
#include <string>
#include <iostream>

static void test_version() {
    // string + numeric constants are consistent
    assert(std::string(matrixdb_version()) == "0.1.0" && "version string");
    assert(MATRIXDB_VERSION_MAJOR == 0 && MATRIXDB_VERSION_MINOR == 1 && MATRIXDB_VERSION_PATCH == 0);
    // packed form: major<<32 | minor<<16 | patch
    const uint64_t expect = (uint64_t(0) << 32) | (uint64_t(1) << 16) | uint64_t(0);
    assert(matrixdb_version_u64() == expect && matrixdb_version_u64() != 0 && "packed version");
    // a higher version packs strictly greater (ordering holds — what a client comparison relies on)
    const uint64_t v0_1_0 = (uint64_t(0) << 32) | (uint64_t(1) << 16) | 0;
    const uint64_t v0_2_0 = (uint64_t(0) << 32) | (uint64_t(2) << 16) | 0;
    const uint64_t v1_0_0 = (uint64_t(1) << 32);
    assert(v0_1_0 < v0_2_0 && v0_2_0 < v1_0_0 && "packed versions order like semver");
    // the engine reports the same build version
    CPUMockEngine eng;
    assert(std::string(eng.version()) == matrixdb_version() && eng.version_u64() == matrixdb_version_u64()
           && "engine reports the build version");
    std::cout << "[version] ok (" << eng.version() << ")\n";
}

int main() { test_version(); std::cout << "ALL VERSION TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_logging.cpp
// CPU test for OB-1 structured logging: leveled logger with a runtime threshold. Tests the filter
// (enabled()) and the actual stderr emission (by redirecting cerr's rdbuf).
#include "logging.hpp"
#include <cassert>
#include <sstream>
#include <string>
#include <iostream>

static void test_level_filter() {
    Log::set_level(LogLevel::WARN);                            // default
    assert(!Log::enabled(LogLevel::DEBUG) && !Log::enabled(LogLevel::INFO) && "below threshold -> filtered");
    assert(Log::enabled(LogLevel::WARN) && Log::enabled(LogLevel::ERROR) && "at/above threshold -> enabled");
    Log::set_level(LogLevel::DEBUG);                           // verbose
    assert(Log::enabled(LogLevel::DEBUG) && Log::get_level() == LogLevel::DEBUG && "DEBUG threshold -> all enabled");
    Log::set_level(LogLevel::ERROR);                           // quiet
    assert(!Log::enabled(LogLevel::WARN) && Log::enabled(LogLevel::ERROR) && "ERROR threshold -> only ERROR");
    std::cout << "[level filter] ok\n";
}

static void test_emission() {
    // capture cerr
    std::ostringstream cap; std::streambuf* old = std::cerr.rdbuf(cap.rdbuf());
    Log::set_level(LogLevel::WARN);
    Log::emit(LogLevel::DEBUG, "debug-suppressed");            // below threshold -> nothing
    Log::emit(LogLevel::INFO,  "info-suppressed");             // below threshold -> nothing
    Log::emit(LogLevel::WARN,  "a warning");                   // emitted
    Log::emit(LogLevel::ERROR, "an error");                    // emitted
    std::cerr.rdbuf(old);                                      // restore
    const std::string out = cap.str();
    assert(out.find("debug-suppressed") == std::string::npos && "below-threshold suppressed");
    assert(out.find("info-suppressed") == std::string::npos && "below-threshold suppressed");
    assert(out.find("[WARN] a warning") != std::string::npos && "WARN emitted with prefix");
    assert(out.find("[ERROR] an error") != std::string::npos && "ERROR emitted with prefix");
    Log::set_level(LogLevel::WARN);                            // restore default for other tests/links
    std::cout << "[emission] ok\n";
}

int main() { test_level_filter(); test_emission(); std::cout << "ALL LOGGING TESTS PASSED\n"; return 0; }


In [ ]:
%%writefile test_migration_gpu.cpp
// GPU proof (Colab): a column migrated HOST->DEVICE is byte-intact and GPU-scannable in
// place. Build: nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread \
//   -DMATRIX_USE_CUDA test_migration_gpu.cpp -o test_migration_gpu && ./test_migration_gpu
#include "tiered_column.hpp"
#include "compute_cuda.cuh"   // matrix_scan_kernel_u32x4 + CUDA_CHECK
#include <cstdio>
#include <cassert>
#include <vector>

int main() {
    // Build a uint32 column value[i]=i (n divisible by 4 for uint4 scan).
    const size_t N = 1u << 20; // 1,048,576 values
    std::vector<uint32_t> vals(N);
    for (size_t i = 0; i < N; ++i) vals[i] = static_cast<uint32_t>(i);
    const uint32_t threshold = N / 2;

    // CPU reference count of value > threshold.
    uint64_t cpu_count = 0;
    for (size_t i = 0; i < N; ++i) cpu_count += (vals[i] > threshold);

    // Make a column from those bytes; checksum before.
    TieredColumn col(1, reinterpret_cast<const unsigned char*>(vals.data()), N * sizeof(uint32_t));
    const uint64_t want = col.checksum();

    // Migrate HOST -> DEVICE and scan IN PLACE on the device pointer.
    col.migrate_to(MemorySpace::DEVICE);
    assert(col.tier() == MemorySpace::DEVICE && "promoted to DEVICE");

    unsigned long long* d_count = nullptr;
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(unsigned long long)));
    CUDA_CHECK(cudaMemset(d_count, 0, sizeof(unsigned long long)));
    const uint4* col4 = reinterpret_cast<const uint4*>(col.device_ptr());
    const size_t n4 = (N * sizeof(uint32_t)) / sizeof(uint4); // bytes/16
    matrix_scan_kernel_u32x4<<<1024, 256>>>(col4, n4, threshold, d_count);
    CUDA_CHECK(cudaGetLastError());
    unsigned long long gpu_count = 0;
    CUDA_CHECK(cudaMemcpy(&gpu_count, d_count, sizeof(gpu_count), cudaMemcpyDeviceToHost));
    cudaFree(d_count);

    assert(gpu_count == cpu_count && "GPU scan of the promoted column matches the CPU count");

    // Migrate back; bytes must be byte-identical (integrity across the round trip).
    col.migrate_to(MemorySpace::HOST);
    assert(col.tier() == MemorySpace::HOST && "demoted back to HOST");
    assert(col.checksum() == want && "checksum invariant HOST->DEVICE->HOST");

    std::printf("PASS: VRAM-promoted column is GPU-scannable and byte-intact "
                "(gpu_count=%llu cpu_count=%llu)\n",
                static_cast<unsigned long long>(gpu_count), static_cast<unsigned long long>(cpu_count));
    return 0;
}


In [ ]:
%%writefile test_gpu_agg.cu
// GPU AGG-2 cross-backend merge gate (Colab/nvcc only): the GPU SUM/MIN/MAX/COUNT reduction kernels must
// equal matrix_cpu_reduce over the SAME bytes — the correctness anchor for the whole GPU phase. Tests a
// matching predicate and an empty-match predicate (the sentinels). Merge GPU-1 to main only when green.
// Build (Colab T4):
//   nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_gpu_agg.cu -o test_gpu_agg && ./test_gpu_agg
#include "compute_cuda.cuh"   // matrix_{sum,min,max}_kernel_u32 + matrix_scan_kernel_u32x4 + matrix_cpu_reduce + CUDA_CHECK
#include <cstdio>
#include <cassert>
#include <cstdint>
#include <vector>

// Run one aggregate on the GPU over the device column `d` (n values), return the uint64 result. Inits the
// device accumulator to the same empty-sentinel the CPU reducer uses (SUM/COUNT/MAX -> 0; MIN -> all-ones).
static unsigned long long gpu_agg(const uint32_t* d, size_t n, uint32_t threshold, MatrixAggOp op) {
    unsigned long long* d_out = nullptr;
    CUDA_CHECK(cudaMalloc(&d_out, sizeof(unsigned long long)));
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM) {
        CUDA_CHECK(cudaMemset(d_out, 0x00, sizeof(unsigned long long)));
        matrix_sum_kernel_u32<<<BLOCKS, TPB>>>(d, n, threshold, d_out);
    } else if (op == AGG_MIN) {
        CUDA_CHECK(cudaMemset(d_out, 0xFF, sizeof(unsigned long long)));   // UINT64_MAX
        matrix_min_kernel_u32<<<BLOCKS, TPB>>>(d, n, threshold, d_out);
    } else if (op == AGG_MAX) {
        CUDA_CHECK(cudaMemset(d_out, 0x00, sizeof(unsigned long long)));
        matrix_max_kernel_u32<<<BLOCKS, TPB>>>(d, n, threshold, d_out);
    } else {                                                              // AGG_COUNT (u32x4, the oracle path)
        CUDA_CHECK(cudaMemset(d_out, 0x00, sizeof(unsigned long long)));
        const uint4* d4 = reinterpret_cast<const uint4*>(d);
        matrix_scan_kernel_u32x4<<<BLOCKS, TPB>>>(d4, n / 4, threshold, d_out);
    }
    CUDA_CHECK(cudaGetLastError());
    unsigned long long h = 0;
    CUDA_CHECK(cudaMemcpy(&h, d_out, sizeof(h), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
    return h;
}

static void check_all_ops(const uint32_t* host, const uint32_t* d, size_t n, uint32_t threshold) {
    const MatrixAggOp ops[] = {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX};
    const char* names[]     = {"COUNT", "SUM", "MIN", "MAX"};
    for (int i = 0; i < 4; ++i) {
        const unsigned long long gpu = gpu_agg(d, n, threshold, ops[i]);
        const unsigned long long cpu = static_cast<unsigned long long>(matrix_cpu_reduce(host, n, threshold, ops[i]));
        std::printf("  thr=%-10u %-5s GPU=%llu CPU=%llu  %s\n", threshold, names[i], gpu, cpu,
                    gpu == cpu ? "OK" : "*** MISMATCH ***");
        assert(gpu == cpu && "GPU aggregate must equal matrix_cpu_reduce over the same bytes");
    }
}

int main() {
    const size_t N = 1u << 20;                       // 1,048,576 values (divisible by 4 for the u32x4 COUNT)
    std::vector<uint32_t> vals(N);
    for (size_t i = 0; i < N; ++i) vals[i] = static_cast<uint32_t>(i);
    uint32_t* d = nullptr;
    CUDA_CHECK(cudaMalloc(&d, N * sizeof(uint32_t)));
    CUDA_CHECK(cudaMemcpy(d, vals.data(), N * sizeof(uint32_t), cudaMemcpyHostToDevice));

    std::printf("GPU AGG-2 cross-backend check (N=%zu) — GPU kernels vs matrix_cpu_reduce:\n", N);
    check_all_ops(vals.data(), d, N, N / 2);          // ~half the rows match value > threshold
    check_all_ops(vals.data(), d, N, 0u);             // all but value 0 match
    check_all_ops(vals.data(), d, N, 0xFFFFFFFFu);    // empty match -> exercises the empty sentinels

    CUDA_CHECK(cudaFree(d));
    std::printf("ALL GPU-AGG TESTS PASSED\n");
    return 0;
}


In [ ]:
%%writefile test_gpu_pred.cu
// GPU-4 predicate cross-check (Colab/nvcc only): the GPU predicate-filtered reductions must equal
// matrix_cpu_reduce_pred over the SAME bytes, for every MatrixCmp op x {COUNT,SUM,MIN,MAX} + an empty match.
// Merge gate for GPU-4. Build (Colab T4):
//   nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_gpu_pred.cu -o test_gpu_pred && ./test_gpu_pred
#include "compute_cuda.cuh"   // matrix_{count,sum,min,max}_kernel_pred_u32 + matrix_cpu_reduce_pred + CUDA_CHECK
#include <cstdio>
#include <cassert>
#include <cstdint>
#include <vector>

static unsigned long long gpu_pred(const uint32_t* d, size_t n, MatrixCmp c, uint32_t a, uint32_t b, MatrixAggOp op) {
    unsigned long long* d_out = nullptr;
    CUDA_CHECK(cudaMalloc(&d_out, sizeof(unsigned long long)));
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM) {
        CUDA_CHECK(cudaMemset(d_out, 0x00, sizeof(unsigned long long)));
        matrix_sum_kernel_pred_u32<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    } else if (op == AGG_MIN) {
        CUDA_CHECK(cudaMemset(d_out, 0xFF, sizeof(unsigned long long)));   // UINT64_MAX
        matrix_min_kernel_pred_u32<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    } else if (op == AGG_MAX) {
        CUDA_CHECK(cudaMemset(d_out, 0x00, sizeof(unsigned long long)));
        matrix_max_kernel_pred_u32<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    } else {
        CUDA_CHECK(cudaMemset(d_out, 0x00, sizeof(unsigned long long)));
        matrix_count_kernel_pred_u32<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    }
    CUDA_CHECK(cudaGetLastError());
    unsigned long long h = 0;
    CUDA_CHECK(cudaMemcpy(&h, d_out, sizeof(h), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
    return h;
}

int main() {
    const size_t N = 1u << 20;
    std::vector<uint32_t> v(N);
    for (size_t i = 0; i < N; ++i) v[i] = static_cast<uint32_t>(i % 1000);   // values 0..999, predicates select known subsets
    uint32_t* d = nullptr;
    CUDA_CHECK(cudaMalloc(&d, N * sizeof(uint32_t)));
    CUDA_CHECK(cudaMemcpy(d, v.data(), N * sizeof(uint32_t), cudaMemcpyHostToDevice));

    struct Case { MatrixCmp c; uint32_t a, b; const char* nm; };
    const Case cases[] = {
        {MatrixCmp::GT, 500, 0, "GT500"}, {MatrixCmp::GE, 500, 0, "GE500"},
        {MatrixCmp::LT, 500, 0, "LT500"}, {MatrixCmp::LE, 500, 0, "LE500"},
        {MatrixCmp::EQ, 42, 0, "EQ42"},   {MatrixCmp::NE, 42, 0, "NE42"},
        {MatrixCmp::BETWEEN, 200, 800, "BETW200_800"}, {MatrixCmp::GT, 99999, 0, "empty"} };
    const MatrixAggOp ops[] = {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX};
    const char* on[] = {"COUNT", "SUM", "MIN", "MAX"};

    std::printf("GPU-4 predicate cross-check (N=%zu) — GPU kernels vs matrix_cpu_reduce_pred:\n", N);
    for (const auto& cs : cases)
        for (int i = 0; i < 4; ++i) {
            const unsigned long long g = gpu_pred(d, N, cs.c, cs.a, cs.b, ops[i]);
            const unsigned long long cpu = static_cast<unsigned long long>(
                matrix_cpu_reduce_pred(v.data(), N, MatrixPredicate{cs.c, cs.a, cs.b}, ops[i]));
            std::printf("  %-11s %-5s GPU=%llu CPU=%llu  %s\n", cs.nm, on[i], g, cpu, g == cpu ? "OK" : "*** MISMATCH ***");
            assert(g == cpu && "GPU predicate aggregate must equal matrix_cpu_reduce_pred");
        }
    CUDA_CHECK(cudaFree(d));
    std::printf("ALL GPU-PRED TESTS PASSED\n");
    return 0;
}


In [ ]:
%%writefile test_gpu_grouped.cu
// GPU-2 grouped cross-check (Colab/nvcc only): the GPU per-group reduction must equal
// matrix_cpu_group_reduce over the SAME keys+values, for {COUNT,SUM,MIN,MAX}. The dataset includes
// out-of-range keys (>= num_groups) to verify both backends ignore them. Merge gate for GPU-2. Build:
//   nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_gpu_grouped.cu -o test_gpu_grouped && ./test_gpu_grouped
#include "compute_cuda.cuh"   // matrix_group_{count,sum,min,max}_kernel + matrix_cpu_group_reduce + CUDA_CHECK
#include <cstdio>
#include <cassert>
#include <cstdint>
#include <vector>

static std::vector<unsigned long long> gpu_group(const uint32_t* dk, const uint32_t* dv, size_t n,
                                                 uint32_t G, MatrixAggOp op) {
    unsigned long long* d_out = nullptr;
    CUDA_CHECK(cudaMalloc(&d_out, G * sizeof(unsigned long long)));
    // Per-op sentinel init (matches matrix_cpu_group_reduce): MIN -> all-ones (UINT64_MAX), else 0.
    CUDA_CHECK(cudaMemset(d_out, (op == AGG_MIN) ? 0xFF : 0x00, G * sizeof(unsigned long long)));
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      matrix_group_sum_kernel<<<BLOCKS, TPB>>>(dk, dv, n, G, d_out);
    else if (op == AGG_MIN) matrix_group_min_kernel<<<BLOCKS, TPB>>>(dk, dv, n, G, d_out);
    else if (op == AGG_MAX) matrix_group_max_kernel<<<BLOCKS, TPB>>>(dk, dv, n, G, d_out);
    else                    matrix_group_count_kernel<<<BLOCKS, TPB>>>(dk, n, G, d_out);
    CUDA_CHECK(cudaGetLastError());
    std::vector<unsigned long long> h(G);
    CUDA_CHECK(cudaMemcpy(h.data(), d_out, G * sizeof(unsigned long long), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
    return h;
}

int main() {
    const size_t N = 1u << 20;
    const uint32_t G = 8;
    std::vector<uint32_t> keys(N), vals(N);
    for (size_t i = 0; i < N; ++i) {
        keys[i] = static_cast<uint32_t>(i % (G + 2));   // 0..G+1 -> keys G and G+1 are out-of-range (must be ignored)
        vals[i] = static_cast<uint32_t>(i % 1000);
    }
    uint32_t *dk = nullptr, *dv = nullptr;
    CUDA_CHECK(cudaMalloc(&dk, N * sizeof(uint32_t)));
    CUDA_CHECK(cudaMalloc(&dv, N * sizeof(uint32_t)));
    CUDA_CHECK(cudaMemcpy(dk, keys.data(), N * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dv, vals.data(), N * sizeof(uint32_t), cudaMemcpyHostToDevice));

    const MatrixAggOp ops[] = {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX};
    const char* on[] = {"COUNT", "SUM", "MIN", "MAX"};
    std::printf("GPU-2 grouped cross-check (N=%zu, G=%u, keys incl. out-of-range) — GPU vs matrix_cpu_group_reduce:\n", N, G);
    for (int i = 0; i < 4; ++i) {
        const std::vector<unsigned long long> g = gpu_group(dk, dv, N, G, ops[i]);
        std::vector<uint64_t> cpu(G);
        matrix_cpu_group_reduce(keys.data(), vals.data(), N, G, ops[i], cpu.data());
        bool ok = true;
        for (uint32_t k = 0; k < G; ++k) if (g[k] != static_cast<unsigned long long>(cpu[k])) ok = false;
        std::printf("  %-5s match=%s  (g0 GPU=%llu CPU=%llu)\n", on[i], ok ? "OK" : "*** MISMATCH ***",
                    g[0], static_cast<unsigned long long>(cpu[0]));
        for (uint32_t k = 0; k < G; ++k)
            assert(g[k] == static_cast<unsigned long long>(cpu[k]) && "GPU grouped == matrix_cpu_group_reduce");
    }
    CUDA_CHECK(cudaFree(dk)); CUDA_CHECK(cudaFree(dv));
    std::printf("ALL GPU-GROUPED TESTS PASSED\n");
    return 0;
}


In [ ]:
%%writefile test_gpu_typed.cu
// GPU-5 typed cross-check (Colab/nvcc only): the GPU int64 + double predicate reductions must equal
// matrix_cpu_reduce_pred_i64 / _f64 over the SAME bytes, for several predicates x {COUNT,SUM,MIN,MAX}.
// Merge gate for GPU-5. Build (Colab T4):
//   nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_gpu_typed.cu -o test_gpu_typed && ./test_gpu_typed
//
// NOTE on double SUM: floating-point addition is order-dependent and GPU atomics accumulate in
// nondeterministic order, so GPU==CPU is only bit-exact when every partial sum is exact. The dataset uses
// HALF-INTEGER values (k*0.5) with a bounded integer running sum (< 2^53), so all partial sums are exact
// and the cross-check holds regardless of accumulation order. int64 is exact integer arithmetic (no issue).
#include "compute_cuda.cuh"   // matrix_{count,sum,min,max}_kernel_pred_{i64,f64} + matrix_cpu_reduce_pred_{i64,f64} + CUDA_CHECK
#include <cstdio>
#include <cassert>
#include <cstdint>
#include <limits>
#include <vector>

static long long gpu_pred_i64(const long long* d, size_t n, MatrixCmp c, long long a, long long b, MatrixAggOp op) {
    long long* d_out = nullptr; CUDA_CHECK(cudaMalloc(&d_out, sizeof(long long)));
    long long init = (op == AGG_MIN) ? INT64_MAX : (op == AGG_MAX ? INT64_MIN : 0);
    CUDA_CHECK(cudaMemcpy(d_out, &init, sizeof(long long), cudaMemcpyHostToDevice));   // sentinels aren't memset-able
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      matrix_sum_kernel_pred_i64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else if (op == AGG_MIN) matrix_min_kernel_pred_i64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else if (op == AGG_MAX) matrix_max_kernel_pred_i64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else                    matrix_count_kernel_pred_i64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    CUDA_CHECK(cudaGetLastError());
    long long h = 0; CUDA_CHECK(cudaMemcpy(&h, d_out, sizeof(long long), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
    return h;
}
static double gpu_pred_f64(const double* d, size_t n, MatrixCmp c, double a, double b, MatrixAggOp op) {
    double* d_out = nullptr; CUDA_CHECK(cudaMalloc(&d_out, sizeof(double)));
    const double inf = std::numeric_limits<double>::infinity();
    double init = (op == AGG_MIN) ? inf : (op == AGG_MAX ? -inf : 0.0);
    CUDA_CHECK(cudaMemcpy(d_out, &init, sizeof(double), cudaMemcpyHostToDevice));
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      matrix_sum_kernel_pred_f64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else if (op == AGG_MIN) matrix_min_kernel_pred_f64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else if (op == AGG_MAX) matrix_max_kernel_pred_f64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else                    matrix_count_kernel_pred_f64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    CUDA_CHECK(cudaGetLastError());
    double h = 0; CUDA_CHECK(cudaMemcpy(&h, d_out, sizeof(double), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
    return h;
}

int main() {
    const size_t N = 1u << 20;
    const MatrixAggOp ops[] = {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX};
    const char* on[] = {"COUNT", "SUM", "MIN", "MAX"};

    // --- int64: negatives + values beyond UINT32_MAX (exact integer arithmetic) ---
    std::vector<int64_t> vi(N);
    for (size_t i = 0; i < N; ++i) vi[i] = ((int64_t)(i % 4000) - 2000) * 3000000LL;   // range ~[-6e9, 6e9], incl. >2^32
    int64_t* di = nullptr; CUDA_CHECK(cudaMalloc(&di, N * sizeof(int64_t)));
    CUDA_CHECK(cudaMemcpy(di, vi.data(), N * sizeof(int64_t), cudaMemcpyHostToDevice));
    struct Ci { MatrixCmp c; long long a, b; const char* nm; };
    const Ci ci[] = { {MatrixCmp::GT, 0, 0, "GT0"}, {MatrixCmp::LT, 0, 0, "LT0"},
                      {MatrixCmp::GE, 3000000000LL, 0, "GE3e9"}, {MatrixCmp::EQ, 0, 0, "EQ0"},
                      {MatrixCmp::BETWEEN, -1000000000LL, 1000000000LL, "BETW"}, {MatrixCmp::GT, 999999999999LL, 0, "empty"} };
    std::printf("GPU-5 int64 cross-check (N=%zu) vs matrix_cpu_reduce_pred_i64:\n", N);
    for (const auto& cs : ci) for (int k = 0; k < 4; ++k) {
        const long long g = gpu_pred_i64((const long long*)di, N, cs.c, cs.a, cs.b, ops[k]);
        const long long cpu = matrix_cpu_reduce_pred_i64(vi.data(), N, MatrixPredicateI64{cs.c, cs.a, cs.b}, ops[k]);
        std::printf("  %-6s %-5s GPU=%lld CPU=%lld  %s\n", cs.nm, on[k], g, cpu, g == cpu ? "OK" : "*** MISMATCH ***");
        assert(g == cpu && "GPU int64 predicate aggregate == matrix_cpu_reduce_pred_i64");
    }
    CUDA_CHECK(cudaFree(di));

    // --- double: half-integers (exact partial sums -> order-independent SUM) incl. negatives + fractions ---
    std::vector<double> vf(N);
    for (size_t i = 0; i < N; ++i) vf[i] = ((double)((int64_t)(i % 2000) - 1000)) * 0.5;   // [-500, 499.5] in 0.5 steps
    double* df = nullptr; CUDA_CHECK(cudaMalloc(&df, N * sizeof(double)));
    CUDA_CHECK(cudaMemcpy(df, vf.data(), N * sizeof(double), cudaMemcpyHostToDevice));
    struct Cf { MatrixCmp c; double a, b; const char* nm; };
    const Cf cf[] = { {MatrixCmp::GT, 0.0, 0, "GT0"}, {MatrixCmp::LT, 0.0, 0, "LT0"},
                      {MatrixCmp::LE, -100.0, 0, "LE-100"}, {MatrixCmp::EQ, 0.0, 0, "EQ0"},
                      {MatrixCmp::BETWEEN, -50.0, 50.0, "BETW"}, {MatrixCmp::GT, 1e9, 0, "empty"} };
    std::printf("GPU-5 double cross-check (N=%zu, half-integers) vs matrix_cpu_reduce_pred_f64:\n", N);
    for (const auto& cs : cf) for (int k = 0; k < 4; ++k) {
        const double g = gpu_pred_f64(df, N, cs.c, cs.a, cs.b, ops[k]);
        const double cpu = matrix_cpu_reduce_pred_f64(vf.data(), N, MatrixPredicateF64{cs.c, cs.a, cs.b}, ops[k]);
        std::printf("  %-6s %-5s GPU=%g CPU=%g  %s\n", cs.nm, on[k], g, cpu, g == cpu ? "OK" : "*** MISMATCH ***");
        assert(g == cpu && "GPU double predicate aggregate == matrix_cpu_reduce_pred_f64");
    }
    CUDA_CHECK(cudaFree(df));

    std::printf("ALL GPU-TYPED TESTS PASSED\n");
    return 0;
}


In [ ]:
%%writefile test_gpu_catalog.cu
// GPU-3 cross-check (Colab/nvcc only): execute_query over a DEVICE/VRAM-resident catalog column must
// equal the CPU reducer over the same bytes — for u32/i64/f64 x {COUNT,SUM,MIN,MAX} x {filtered,unfiltered}.
// This proves the *analytical* path (not just the legacy scan column) runs on the GPU and matches the
// oracle: pin a column to VRAM (CPUMockEngine::pin_device), run execute_query (which now dispatches the
// verified matrix_gpu_reduce_dev_* kernels when tier()==DEVICE), and compare to matrix_cpu_reduce_*.
// Merge gate for GPU-3. Build (Colab T4):
//   nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_gpu_catalog.cu -o test_gpu_catalog && ./test_gpu_catalog
#include "compute_mock.cpp"   // CPUMockEngine (execute_query, pin_device) + matrix_cpu_reduce_* oracle
#include "compute_cuda.cuh"   // defines matrix_gpu_reduce_dev_* (forward-declared in compute_mock.cpp)
#include <cstdio>
#include <cassert>
#include <cstdint>
#include <vector>

static const MatrixAggOp OPS[] = {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX};
static const char* ON[] = {"COUNT", "SUM", "MIN", "MAX"};

int main() {
    const size_t N = 1u << 20;

    // ---- u32: values 0..999, filter GT 500 (a known subset) + unfiltered ----
    {
        std::vector<uint32_t> v(N);
        for (size_t i = 0; i < N; ++i) v[i] = static_cast<uint32_t>(i % 1000);
        CPUMockEngine eng;
        eng.load_scan_column(1, v.data(), N);
        assert(eng.pin_device(1) && "u32 column must pin to VRAM");   // now DEVICE-resident
        std::printf("GPU-3 u32 execute_query (DEVICE-resident) vs matrix_cpu_reduce*:\n");
        for (int filt = 0; filt < 2; ++filt) for (int k = 0; k < 4; ++k) {
            MatrixQuery q; q.value_col = 1; q.agg = OPS[k];
            q.has_filter = (filt == 1); q.cmp = MatrixCmp::GT; q.threshold = 500;
            std::vector<uint64_t> out;
            const MatrixQueryStatus st = eng.execute_query(q, out);
            assert(st == MatrixQueryStatus::OK && out.size() == 1);
            const uint64_t gpu = out[0];
            const uint64_t cpu = q.has_filter
                ? matrix_cpu_reduce_pred(v.data(), N, MatrixPredicate{MatrixCmp::GT, 500, 0}, OPS[k])
                : matrix_cpu_reduce_all(v.data(), N, OPS[k]);
            std::printf("  %-9s %-5s GPU=%llu CPU=%llu  %s\n", filt ? "GT500" : "all", ON[k],
                        (unsigned long long)gpu, (unsigned long long)cpu, gpu == cpu ? "OK" : "*** MISMATCH ***");
            assert(gpu == cpu && "u32 execute_query DEVICE path == matrix_cpu_reduce*");
        }
    }

    // ---- i64: negatives + values > 2^32, filter GT 0 + unfiltered ----
    {
        std::vector<int64_t> v(N);
        for (size_t i = 0; i < N; ++i) v[i] = ((int64_t)(i % 4000) - 2000) * 3000000LL;
        CPUMockEngine eng;
        eng.load_scan_column_i64(1, v.data(), N);
        assert(eng.pin_device(1) && "i64 column must pin to VRAM");
        std::printf("GPU-3 i64 execute_query (DEVICE-resident) vs matrix_cpu_reduce*_i64:\n");
        for (int filt = 0; filt < 2; ++filt) for (int k = 0; k < 4; ++k) {
            MatrixQuery q; q.value_col = 1; q.agg = OPS[k];
            q.has_filter = (filt == 1); q.cmp = MatrixCmp::GT; q.lo_i64 = 0; q.hi_i64 = 0;
            std::vector<uint64_t> out;
            const MatrixQueryStatus st = eng.execute_query(q, out);
            assert(st == MatrixQueryStatus::OK && out.size() == 1);
            const int64_t gpu = static_cast<int64_t>(out[0]);
            const int64_t cpu = q.has_filter
                ? matrix_cpu_reduce_pred_i64(v.data(), N, MatrixPredicateI64{MatrixCmp::GT, 0, 0}, OPS[k])
                : matrix_cpu_reduce_all_i64(v.data(), N, OPS[k]);
            std::printf("  %-9s %-5s GPU=%lld CPU=%lld  %s\n", filt ? "GT0" : "all", ON[k],
                        (long long)gpu, (long long)cpu, gpu == cpu ? "OK" : "*** MISMATCH ***");
            assert(gpu == cpu && "i64 execute_query DEVICE path == matrix_cpu_reduce*_i64");
        }
    }

    // ---- f64: half-integers (exact order-independent SUM), filter GT 0 + unfiltered ----
    {
        std::vector<double> v(N);
        for (size_t i = 0; i < N; ++i) v[i] = ((double)((int64_t)(i % 2000) - 1000)) * 0.5;
        CPUMockEngine eng;
        eng.load_scan_column_f64(1, v.data(), N);
        assert(eng.pin_device(1) && "f64 column must pin to VRAM");
        std::printf("GPU-3 f64 execute_query (DEVICE-resident, half-integers) vs matrix_cpu_reduce*_f64:\n");
        for (int filt = 0; filt < 2; ++filt) for (int k = 0; k < 4; ++k) {
            MatrixQuery q; q.value_col = 1; q.agg = OPS[k];
            q.has_filter = (filt == 1); q.cmp = MatrixCmp::GT; q.lo_f64 = 0.0; q.hi_f64 = 0.0;
            std::vector<uint64_t> out;
            const MatrixQueryStatus st = eng.execute_query(q, out);
            assert(st == MatrixQueryStatus::OK && out.size() == 1);
            const double gpu = matrix_bit_cast<double>(out[0]);
            const double cpu = q.has_filter
                ? matrix_cpu_reduce_pred_f64(v.data(), N, MatrixPredicateF64{MatrixCmp::GT, 0.0, 0.0}, OPS[k])
                : matrix_cpu_reduce_all_f64(v.data(), N, OPS[k]);
            std::printf("  %-9s %-5s GPU=%g CPU=%g  %s\n", filt ? "GT0" : "all", ON[k],
                        gpu, cpu, gpu == cpu ? "OK" : "*** MISMATCH ***");
            assert(gpu == cpu && "f64 execute_query DEVICE path == matrix_cpu_reduce*_f64");
        }
    }

    std::printf("ALL GPU-CATALOG TESTS PASSED\n");
    return 0;
}


In [ ]:
%%writefile test_gpu_catalog_grouped.cu
// GPU-3g cross-check (Colab/nvcc only): a GROUP BY over DEVICE/VRAM-resident key+value columns must equal
// matrix_cpu_group_reduce[_i64/_f64](_pred) over the same bytes — for u32/i64/f64 values x {COUNT,SUM,MIN,MAX}
// x {filtered,unfiltered}. Proves the grouped analytical path (all value types) runs on the GPU: pin
// key+value to VRAM (pin_device), run execute_query GROUP BY (which dispatches matrix_gpu_group_reduce_dev*
// when both columns are tier()==DEVICE) and compare per-group to the CPU oracle. Merge gate. Build:
//   nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_gpu_catalog_grouped.cu -o test_gpu_catalog_grouped && ./test_gpu_catalog_grouped
#include "compute_mock.cpp"   // CPUMockEngine (execute_query, pin_device) + matrix_cpu_group_reduce* oracle
#include "compute_cuda.cuh"   // matrix_gpu_group_reduce_dev{,_i64,_f64} + the grouped kernels
#include <cstdio>
#include <cassert>
#include <cstdint>
#include <vector>

static const MatrixAggOp OPS[] = {AGG_COUNT, AGG_SUM, AGG_MIN, AGG_MAX};
static const char* ON[] = {"COUNT", "SUM", "MIN", "MAX"};

int main() {
    const size_t N = 1u << 20;
    const uint32_t G = 8;
    std::vector<uint32_t> keys(N);
    for (size_t i = 0; i < N; ++i) keys[i] = static_cast<uint32_t>(i % G);

    // ---- u32 value, filter GT 500 + unfiltered ----
    {
        std::vector<uint32_t> vals(N);
        for (size_t i = 0; i < N; ++i) vals[i] = static_cast<uint32_t>(i % 1000);
        CPUMockEngine eng;
        eng.load_scan_column(1, keys.data(), N);
        eng.load_scan_column(2, vals.data(), N);
        assert(eng.pin_device(1) && eng.pin_device(2));
        std::printf("GPU-3g u32 GROUP BY (DEVICE-resident key+value, G=%u) vs matrix_cpu_group_reduce:\n", G);
        for (int filt = 0; filt < 2; ++filt) for (int k = 0; k < 4; ++k) {
            MatrixQuery q; q.value_col = 2; q.agg = OPS[k]; q.grouped = true; q.key_col = 1; q.num_groups = G;
            q.has_filter = (filt == 1); q.cmp = MatrixCmp::GT; q.threshold = 500;
            std::vector<uint64_t> gpu; assert(eng.execute_query(q, gpu) == MatrixQueryStatus::OK && gpu.size() == G);
            std::vector<uint64_t> cpu(G);
            if (q.has_filter) matrix_cpu_group_reduce_pred(keys.data(), vals.data(), N, G, OPS[k], MatrixPredicate{MatrixCmp::GT, 500, 0}, cpu.data());
            else              matrix_cpu_group_reduce(keys.data(), vals.data(), N, G, OPS[k], cpu.data());
            std::printf("  %-6s %-5s match=%s (g0 GPU=%llu CPU=%llu)\n", filt ? "GT500" : "all", ON[k],
                        gpu == cpu ? "OK" : "*** MISMATCH ***", (unsigned long long)gpu[0], (unsigned long long)cpu[0]);
            assert(gpu == cpu && "u32 GROUP BY DEVICE path == matrix_cpu_group_reduce");
        }
    }

    // ---- i64 value (negatives + > 2^32), filter GT 0 + unfiltered ----
    {
        std::vector<int64_t> vals(N);
        for (size_t i = 0; i < N; ++i) vals[i] = ((int64_t)(i % 4000) - 2000) * 3000000LL;
        CPUMockEngine eng;
        eng.load_scan_column(1, keys.data(), N);
        eng.load_scan_column_i64(2, vals.data(), N);
        assert(eng.pin_device(1) && eng.pin_device(2));
        std::printf("GPU-3g i64 GROUP BY (DEVICE-resident) vs matrix_cpu_group_reduce_i64:\n");
        for (int filt = 0; filt < 2; ++filt) for (int k = 0; k < 4; ++k) {
            MatrixQuery q; q.value_col = 2; q.agg = OPS[k]; q.grouped = true; q.key_col = 1; q.num_groups = G;
            q.has_filter = (filt == 1); q.cmp = MatrixCmp::GT; q.lo_i64 = 0; q.hi_i64 = 0;
            std::vector<uint64_t> gpu; assert(eng.execute_query(q, gpu) == MatrixQueryStatus::OK && gpu.size() == G);
            std::vector<int64_t> tmp(G);
            if (q.has_filter) matrix_cpu_group_reduce_i64_pred(keys.data(), vals.data(), N, G, OPS[k], MatrixPredicateI64{MatrixCmp::GT, 0, 0}, tmp.data());
            else              matrix_cpu_group_reduce_i64(keys.data(), vals.data(), N, G, OPS[k], tmp.data());
            std::vector<uint64_t> cpu(G); for (uint32_t g = 0; g < G; ++g) cpu[g] = static_cast<uint64_t>(tmp[g]);
            std::printf("  %-6s %-5s match=%s (g0 GPU=%lld CPU=%lld)\n", filt ? "GT0" : "all", ON[k],
                        gpu == cpu ? "OK" : "*** MISMATCH ***", (long long)gpu[0], (long long)tmp[0]);
            assert(gpu == cpu && "i64 GROUP BY DEVICE path == matrix_cpu_group_reduce_i64");
        }
    }

    // ---- f64 value (half-integers -> exact order-independent SUM), filter GT 0 + unfiltered ----
    {
        std::vector<double> vals(N);
        for (size_t i = 0; i < N; ++i) vals[i] = ((double)((int64_t)(i % 2000) - 1000)) * 0.5;
        CPUMockEngine eng;
        eng.load_scan_column(1, keys.data(), N);
        eng.load_scan_column_f64(2, vals.data(), N);
        assert(eng.pin_device(1) && eng.pin_device(2));
        std::printf("GPU-3g f64 GROUP BY (DEVICE-resident, half-integers) vs matrix_cpu_group_reduce_f64:\n");
        for (int filt = 0; filt < 2; ++filt) for (int k = 0; k < 4; ++k) {
            MatrixQuery q; q.value_col = 2; q.agg = OPS[k]; q.grouped = true; q.key_col = 1; q.num_groups = G;
            q.has_filter = (filt == 1); q.cmp = MatrixCmp::GT; q.lo_f64 = 0.0; q.hi_f64 = 0.0;
            std::vector<uint64_t> gpu; assert(eng.execute_query(q, gpu) == MatrixQueryStatus::OK && gpu.size() == G);
            std::vector<double> tmp(G);
            if (q.has_filter) matrix_cpu_group_reduce_f64_pred(keys.data(), vals.data(), N, G, OPS[k], MatrixPredicateF64{MatrixCmp::GT, 0.0, 0.0}, tmp.data());
            else              matrix_cpu_group_reduce_f64(keys.data(), vals.data(), N, G, OPS[k], tmp.data());
            std::vector<uint64_t> cpu(G); for (uint32_t g = 0; g < G; ++g) cpu[g] = matrix_bit_cast<uint64_t>(tmp[g]);
            std::printf("  %-6s %-5s match=%s (g0 GPU=%g CPU=%g)\n", filt ? "GT0" : "all", ON[k],
                        gpu == cpu ? "OK" : "*** MISMATCH ***", matrix_bit_cast<double>(gpu[0]), tmp[0]);
            assert(gpu == cpu && "f64 GROUP BY DEVICE path == matrix_cpu_group_reduce_f64");
        }
    }

    std::printf("ALL GPU-CATALOG-GROUPED TESTS PASSED\n");
    return 0;
}


## 3. Kernel index-coverage test (CPU, no GPU needed)

Simulates the items-per-thread scan kernel's index arithmetic and asserts every index is visited exactly once. This is pure integer logic, so it runs on the CPU and catches GPU-only coverage bugs before spending a GPU build.

In [ ]:
!g++ -std=c++17 -O2 test_scan_coverage.cpp -o /tmp/tcov && /tmp/tcov

## 3a. KVStore unit test (CPU, no GPU)

Proves the DM-1 fix: distinct colliding keys never overwrite each other, and a full table is an explicit error, not silent data loss.

In [ ]:
!clang++ -std=c++20 -O2 test_kv_store.cpp -o /tmp/tkv 2>/dev/null || g++ -std=c++20 -O2 test_kv_store.cpp -o /tmp/tkv; /tmp/tkv

## 3c. TierManager unit test (CPU, no GPU)

Proves the auto-tiering brain: hot columns promote toward VRAM, cold stay put, scarce tiers evict by cost-benefit (not pure LRU), anti-thrash holds, decisions are deterministic.

In [ ]:
!clang++ -std=c++20 -O2 test_tier_manager.cpp -o /tmp/ttm 2>/dev/null || g++ -std=c++20 -O2 test_tier_manager.cpp -o /tmp/ttm; /tmp/ttm

## 3d. ColdStore WAL test (CPU, no GPU)

Proves durability: append+replay round-trip, data survives a fresh ColdStore instance (restart), torn tail dropped, CRC corruption stops replay.

In [ ]:
!clang++ -std=c++20 -O2 test_cold_store.cpp -o /tmp/tcs 2>/dev/null || g++ -std=c++20 -O2 test_cold_store.cpp -o /tmp/tcs; /tmp/tcs

## 3e. Engine restart test (CPU, no GPU)

End-to-end: point-op writes through one engine survive into a fresh engine on the same WAL path — MatrixDB no longer loses data on restart.

In [ ]:
!clang++ -std=c++20 -O2 test_engine_restart.cpp -o /tmp/ter 2>/dev/null || g++ -std=c++20 -O2 test_engine_restart.cpp -o /tmp/ter; /tmp/ter

## 3f. Migration test (CPU, no GPU)

Cross-tier byte movement: HOST<->COLD round-trip is checksum-invariant, and TierManager decisions drive the executor to physically move columns.

In [ ]:
!clang++ -std=c++20 -O2 test_migration.cpp -o /tmp/tmig 2>/dev/null || g++ -std=c++20 -O2 test_migration.cpp -o /tmp/tmig; /tmp/tmig

### Live tiering integration
The engine holds a catalog of analytical columns larger than its RAM budget; the TierManager demotes the coldest to SSD and a scan pulls a cold column back, results intact.

In [ ]:
!clang++ -std=c++20 -O2 test_live_tiering.cpp -o /tmp/tlt 2>/dev/null || g++ -std=c++20 -O2 test_live_tiering.cpp -o /tmp/tlt; /tmp/tlt

### Analytical aggregations
OP_SCAN computes COUNT / SUM / MIN / MAX over the values matching the predicate, on both the legacy column and a tiered catalog column — verified against closed-form oracles.

In [ ]:
!clang++ -std=c++20 -O2 test_aggregations.cpp -o /tmp/tagg 2>/dev/null || g++ -std=c++20 -O2 test_aggregations.cpp -o /tmp/tagg; /tmp/tagg

### Grouped aggregation (GROUP BY)
Per-group COUNT/SUM/MIN/MAX over two aligned tiered columns (a key and a value), verified against hand-worked and brute-force oracles — including over COLD-demoted columns.

In [ ]:
!clang++ -std=c++20 -O2 test_group_by.cpp -o /tmp/tgby 2>/dev/null || g++ -std=c++20 -O2 test_group_by.cpp -o /tmp/tgby; /tmp/tgby

### Unified query API
One `execute_query(MatrixQuery)` composes scalar/grouped × filtered/unfiltered aggregation over tiered catalog columns — verified against closed-form and brute-force oracles.

In [ ]:
!clang++ -std=c++20 -O2 test_query.cpp -o /tmp/tq 2>/dev/null || g++ -std=c++20 -O2 test_query.cpp -o /tmp/tq; /tmp/tq

### Engine observability
`stats()` exposes tiering activity (cold borrows, rebalances, migrations) + resident-bytes gauges — verified against a known eviction workload.

In [ ]:
!clang++ -std=c++20 -O2 test_observability.cpp -o /tmp/tobs 2>/dev/null || g++ -std=c++20 -O2 test_observability.cpp -o /tmp/tobs; /tmp/tobs

### Binary column persistence
save_column / load_column_from_file round-trip a column through a binary file (incl. a COLD-tier column); a reloaded column queries identically.

In [ ]:
!clang++ -std=c++20 -O2 test_column_io.cpp -o /tmp/tcio 2>/dev/null || g++ -std=c++20 -O2 test_column_io.cpp -o /tmp/tcio; /tmp/tcio

### Catalog snapshot durability
save_catalog / load_catalog snapshot the whole tiered catalog to one file and restore it into a fresh engine (incl. COLD columns) — the analytical store survives a restart.

In [ ]:
!clang++ -std=c++20 -O2 test_catalog_snapshot.cpp -o /tmp/tcs2 2>/dev/null || g++ -std=c++20 -O2 test_catalog_snapshot.cpp -o /tmp/tcs2; /tmp/tcs2

### Query input validation
execute_query rejects malformed queries (unknown/zero column, self-group, length mismatch, absurd group count) with a status code — gracefully, never crashing (verified under -DNDEBUG too).

In [ ]:
!clang++ -std=c++20 -O2 test_query_validation.cpp -o /tmp/tqv 2>/dev/null || g++ -std=c++20 -O2 test_query_validation.cpp -o /tmp/tqv; /tmp/tqv

### Atomic transactions (WAL group commit)
begin/txn_put/commit/rollback — a committed transaction is all-or-nothing across a crash (uncommitted txn-puts are discarded on replay); the auto-commit path is unchanged.

In [ ]:
!clang++ -std=c++20 -O2 test_transactions.cpp -o /tmp/ttx 2>/dev/null || g++ -std=c++20 -O2 test_transactions.cpp -o /tmp/ttx; /tmp/ttx

### Server core (request/response protocol)
A serializable GET/PUT/QUERY request + matrix_serve dispatcher make the engine request-serveable; serialize->serve->deserialize round-trips equal direct engine calls. (The TCP accept-loop adapter runs on a non-sandboxed machine.)

In [ ]:
!clang++ -std=c++20 -O2 test_server.cpp -o /tmp/tsv 2>/dev/null || g++ -std=c++20 -O2 test_server.cpp -o /tmp/tsv; /tmp/tsv

### Authorization / access control
AccessPolicy enforces per-principal column-level query access + read/write grants at the matrix_serve boundary; unpermitted requests are ERR_FORBIDDEN with no engine side-effects.

In [ ]:
!clang++ -std=c++20 -O2 test_security.cpp -o /tmp/tsec 2>/dev/null || g++ -std=c++20 -O2 test_security.cpp -o /tmp/tsec; /tmp/tsec

### Audit logging
AuditLog records every served request — allowed, denied (with the principal), and malformed — at the matrix_serve boundary; the forensic trail of who did what.

In [ ]:
!clang++ -std=c++20 -O2 test_audit.cpp -o /tmp/taud 2>/dev/null || g++ -std=c++20 -O2 test_audit.cpp -o /tmp/taud; /tmp/taud

### CSV ingest
load_column_from_csv reads a uint32 column straight out of a CSV file into the tiered catalog — and reports malformed input gracefully (false, no crash) rather than aborting.

In [ ]:
!clang++ -std=c++20 -O2 test_csv_ingest.cpp -o /tmp/tcsv 2>/dev/null || g++ -std=c++20 -O2 test_csv_ingest.cpp -o /tmp/tcsv; /tmp/tcsv

### WAL checkpoint / compaction
checkpoint() snapshots the point-op store and truncates the write-ahead log, so recovery loads the snapshot then replays only newer records — WAL size and restart time stay bounded.

In [ ]:
!clang++ -std=c++20 -O2 test_checkpoint.cpp -o /tmp/tckpt 2>/dev/null || g++ -std=c++20 -O2 test_checkpoint.cpp -o /tmp/tckpt; /tmp/tckpt

### Richer scan predicates
execute_query's WHERE clause supports GT / GE / LT / LE / EQ / NE / BETWEEN over catalog columns (not just value>threshold) — verified per-operator against brute-force oracles.

In [ ]:
!clang++ -std=c++20 -O2 test_query_predicates.cpp -o /tmp/tqp 2>/dev/null || g++ -std=c++20 -O2 test_query_predicates.cpp -o /tmp/tqp; /tmp/tqp

### Typed columns — int64
load_scan_column_i64 registers a signed 64-bit column (values beyond uint32 range, and negatives); execute_query aggregates it (COUNT/SUM/MIN/MAX) — the first slice of real types.

In [ ]:
!clang++ -std=c++20 -O2 test_typed_columns.cpp -o /tmp/ttyp 2>/dev/null || g++ -std=c++20 -O2 test_typed_columns.cpp -o /tmp/ttyp; /tmp/ttyp

### int64 filtered aggregation
int64 columns now support WHERE GT/GE/LT/LE/EQ/NE/BETWEEN with signed 64-bit bounds (negatives and values beyond uint32) — verified per-operator against brute-force oracles.

In [ ]:
!clang++ -std=c++20 -O2 test_typed_predicates.cpp -o /tmp/ttp 2>/dev/null || g++ -std=c++20 -O2 test_typed_predicates.cpp -o /tmp/ttp; /tmp/ttp

### Grouped int64 aggregation
GROUP BY a uint32 key over an int64 value column (filtered + unfiltered) — completing int64 query parity; verified incl. negative-group MAX and mixed-width row-count guards.

In [ ]:
!clang++ -std=c++20 -O2 test_typed_grouped.cpp -o /tmp/ttg 2>/dev/null || g++ -std=c++20 -O2 test_typed_grouped.cpp -o /tmp/ttg; /tmp/ttg

### Typed catalog snapshot (int64 durability)
save_catalog / load_catalog round-trip a mixed uint32 + int64 catalog (types + values), so an int64 analytical store survives a restart — not just RAM-resident.

In [ ]:
!clang++ -std=c++20 -O2 test_typed_snapshot.cpp -o /tmp/tts 2>/dev/null || g++ -std=c++20 -O2 test_typed_snapshot.cpp -o /tmp/tts; /tmp/tts

### double (float64) columns
load_scan_column_f64 registers a 64-bit float column; execute_query aggregates it (COUNT/SUM/MIN/MAX, filtered + unfiltered) and it survives a restart — fractional real data.

In [ ]:
!clang++ -std=c++20 -O2 test_typed_double.cpp -o /tmp/ttd 2>/dev/null || g++ -std=c++20 -O2 test_typed_double.cpp -o /tmp/ttd; /tmp/ttd

### Grouped double aggregation
GROUP BY a uint32 key over a double value column (filtered + unfiltered) — completing double query parity; verified incl. negative-group MAX and mixed-width row-count guards.

In [ ]:
!clang++ -std=c++20 -O2 test_typed_double_grouped.cpp -o /tmp/tdg 2>/dev/null || g++ -std=c++20 -O2 test_typed_double_grouped.cpp -o /tmp/tdg; /tmp/tdg

### Typed CSV ingest (int64 + double)
load_column_from_csv_i64 / _f64 ingest signed-64-bit and floating-point columns straight from CSV (negatives, values beyond uint32, fractions) — gracefully rejecting malformed input.

In [ ]:
!clang++ -std=c++20 -O2 test_typed_csv.cpp -o /tmp/ttc 2>/dev/null || g++ -std=c++20 -O2 test_typed_csv.cpp -o /tmp/ttc; /tmp/ttc

### Per-query latency metrics
EngineStats now reports query_count / total_query_ns / max_query_ns — execute_query times every call (OK and error) so query latency (count, mean, max) is observable, the #1 DB ops metric.

In [ ]:
!clang++ -std=c++20 -O2 test_query_latency.cpp -o /tmp/tql 2>/dev/null || g++ -std=c++20 -O2 test_query_latency.cpp -o /tmp/tql; /tmp/tql

### Backup / restore
backup(prefix) snapshots the whole durable state (analytical catalog + point-op store) under one path prefix; restore(prefix) brings it all back into a fresh engine — a basic ops capability.

In [ ]:
!clang++ -std=c++20 -O2 test_backup.cpp -o /tmp/tbk 2>/dev/null || g++ -std=c++20 -O2 test_backup.cpp -o /tmp/tbk; /tmp/tbk

### Named columns + catalog introspection
name_column / column_id / column_name attach names to columns, and catalog_columns() lists every column with its id, name, type, row count, and tier — a discoverable schema, not just numeric ids.

In [ ]:
!clang++ -std=c++20 -O2 test_schema.cpp -o /tmp/tsch 2>/dev/null || g++ -std=c++20 -O2 test_schema.cpp -o /tmp/tsch; /tmp/tsch

### Append / dynamic column growth
append_to_column[_i64/_f64] add rows to an existing column (growing it, even across the COLD tier) — the store is no longer load-once; appended rows are immediately queryable.

In [ ]:
!clang++ -std=c++20 -O2 test_append.cpp -o /tmp/tap 2>/dev/null || g++ -std=c++20 -O2 test_append.cpp -o /tmp/tap; /tmp/tap

### Key range scan (point-op store)
kv_range(lo, hi) returns every (key, value) with lo <= key <= hi — a range access path over the point-op store, beyond exact-key get. (O(n) scan; a sorted index is the deferred upgrade.)

In [ ]:
!clang++ -std=c++20 -O2 test_kv_range.cpp -o /tmp/tkr 2>/dev/null || g++ -std=c++20 -O2 test_kv_range.cpp -o /tmp/tkr; /tmp/tkr

### Text query parser
parse_query turns a SQL-ish string (SELECT AGG(col) [WHERE col <op> val [AND val]]) into a MatrixQuery — resolving column names and placing the bound by column type; malformed input is a graceful ERR_PARSE, never a crash.

In [ ]:
!clang++ -std=c++20 -O2 test_query_parser.cpp -o /tmp/tqparse 2>/dev/null || g++ -std=c++20 -O2 test_query_parser.cpp -o /tmp/tqparse; /tmp/tqparse

### Equi-join primitive
hash_join(left, right) inner-joins two uint32 key columns into matching (left_row, right_row) pairs (build-hash + probe) — multi-table correlation; verified against a brute-force oracle.

In [ ]:
!clang++ -std=c++20 -O2 test_join.cpp -o /tmp/tj 2>/dev/null || g++ -std=c++20 -O2 test_join.cpp -o /tmp/tj; /tmp/tj

### End-to-end integration
One realistic flow exercising the features composing: typed CSV ingest -> name -> catalog introspection -> parse_query -> execute -> append -> equi-join -> backup/restore -> re-query.

In [ ]:
!clang++ -std=c++20 -O2 test_integration.cpp -o /tmp/tint 2>/dev/null || g++ -std=c++20 -O2 test_integration.cpp -o /tmp/tint; /tmp/tint

### Typed single-column files
save_column / load_column_from_file round-trip int64 and double columns (not just uint32) via a typed single-file format — the int64-abort guard is gone; the element type is carried in the file.

In [ ]:
!clang++ -std=c++20 -O2 test_typed_column_io.cpp -o /tmp/ttci 2>/dev/null || g++ -std=c++20 -O2 test_typed_column_io.cpp -o /tmp/ttci; /tmp/ttci

### Sorted secondary index
kv_range_sorted returns a key range in ascending order via an ordered index (O(log n) locate + O(result) walk, vs kv_range's O(n) scan); maintained on commit, rebuilt on recovery.

In [ ]:
!clang++ -std=c++20 -O2 test_kv_index.cpp -o /tmp/tki 2>/dev/null || g++ -std=c++20 -O2 test_kv_index.cpp -o /tmp/tki; /tmp/tki

### Gather / join-then-project
gather(col, rows) projects a column's values at given row indices (typed) — composes with hash_join so a join yields joined data, not just index pairs.

In [ ]:
!clang++ -std=c++20 -O2 test_gather.cpp -o /tmp/tg 2>/dev/null || g++ -std=c++20 -O2 test_gather.cpp -o /tmp/tg; /tmp/tg

### Named tables
create_table groups equal-length columns into a named, validated, introspectable unit (table_columns / tables) — organizational schema over the named columns.

In [ ]:
!clang++ -std=c++20 -O2 test_tables.cpp -o /tmp/tt 2>/dev/null || g++ -std=c++20 -O2 test_tables.cpp -o /tmp/tt; /tmp/tt

### String columns
load_string_column + string_count_where_eq give the engine variable-length string data it had no support for — a minimal self-contained store (load, row count, equality-filter count).

In [ ]:
!clang++ -std=c++20 -O2 test_string_columns.cpp -o /tmp/tsc 2>/dev/null || g++ -std=c++20 -O2 test_string_columns.cpp -o /tmp/tsc; /tmp/tsc

### Nullable columns
set_column_nulls marks NULL rows; unfiltered scalar aggregates skip them (SQL NULL semantics) — COUNT counts non-null, SUM/MIN/MAX ignore nulls. A maskless column is unchanged.

In [ ]:
!clang++ -std=c++20 -O2 test_nullable.cpp -o /tmp/tn 2>/dev/null || g++ -std=c++20 -O2 test_nullable.cpp -o /tmp/tn; /tmp/tn

### Top-N groups (ORDER BY agg DESC LIMIT k)
top_groups runs a grouped query, then returns the k (group, value) pairs with the largest aggregate, descending — the "top 10 regions by revenue" staple. U32/COUNT grouping.

In [ ]:
!clang++ -std=c++20 -O2 test_top_groups.cpp -o /tmp/ttg 2>/dev/null || g++ -std=c++20 -O2 test_top_groups.cpp -o /tmp/ttg; /tmp/ttg

### HAVING (filter groups by aggregate)
having() runs a grouped query and returns the (group, value) pairs whose aggregate satisfies a comparison (GT/GE/LT/LE/EQ/NE/BETWEEN) — the SQL HAVING clause. Group-id order preserved. U32/COUNT.

In [ ]:
!clang++ -std=c++20 -O2 test_having.cpp -o /tmp/th 2>/dev/null || g++ -std=c++20 -O2 test_having.cpp -o /tmp/th; /tmp/th

### AVG aggregate
average() derives AVG = SUM/COUNT as double(s) from the two existing aggregates, so it inherits per-type handling (U32/I64/F64) and scalar NULL-skipping for free. Scalar -> one value; grouped -> one per group; zero-count -> NaN.

In [ ]:
!clang++ -std=c++20 -O2 test_average.cpp -o /tmp/tavg 2>/dev/null || g++ -std=c++20 -O2 test_average.cpp -o /tmp/tavg; /tmp/tavg

### COUNT(DISTINCT)
count_distinct returns the number of distinct non-NULL values in a column — exact (hash set), typed (U32/I64/F64), null-aware. A HyperLogLog sketch is the upgrade path for huge columns.

In [ ]:
!clang++ -std=c++20 -O2 test_count_distinct.cpp -o /tmp/tcd 2>/dev/null || g++ -std=c++20 -O2 test_count_distinct.cpp -o /tmp/tcd; /tmp/tcd

### Admission control (RM-2)
set_max_query_groups caps a single grouped query's group count — bounding its result memory (num_groups x 8 bytes) so one runaway GROUP BY can't OOM the box. Over the cap -> ERR_TOO_MANY_GROUPS, no allocation. Runtime-settable (toward OB-4).

In [ ]:
!clang++ -std=c++20 -O2 test_admission_control.cpp -o /tmp/tac 2>/dev/null || g++ -std=c++20 -O2 test_admission_control.cpp -o /tmp/tac; /tmp/tac

### Runtime config (OB-4)
set_rebalance_interval tunes the heat-rebalance cadence (run the brain every N tiered scans) at runtime — no recompile. Smaller N re-tiers more aggressively; larger N relaxes it. With the query group cap, the compile-time tiering knobs are becoming operator-tunable.

In [ ]:
!clang++ -std=c++20 -O2 test_rebalance_config.cpp -o /tmp/trc 2>/dev/null || g++ -std=c++20 -O2 test_rebalance_config.cpp -o /tmp/trc; /tmp/trc

### Graceful shutdown (RM-4)
shutdown() rolls back any open transaction, then checkpoints the WAL (snapshot + truncate) so a restart replays an ~empty log — bounded recovery. Committed writes survive; uncommitted are discarded. Idempotent; no-op without a WAL.

In [ ]:
!clang++ -std=c++20 -O2 test_shutdown.cpp -o /tmp/tsd 2>/dev/null || g++ -std=c++20 -O2 test_shutdown.cpp -o /tmp/tsd; /tmp/tsd

### Durability levels (DU-5)
The engine constructor selects the WAL fsync policy: SYNC_EACH (default) fsyncs each commit so a committed write survives power loss; SYNC_OFF buffers for throughput (a crash may lose the tail). durability_level() reports it. Both recover committed writes on a clean restart.

In [ ]:
!clang++ -std=c++20 -O2 test_durability_levels.cpp -o /tmp/tdl 2>/dev/null || g++ -std=c++20 -O2 test_durability_levels.cpp -o /tmp/tdl; /tmp/tdl

### Fault injection — corrupt-WAL recovery (QA-5)
A fresh engine built on a CORRUPT WAL must recover the intact prefix, discard the corrupt tail, and stay usable — never crash, never apply garbage. A torn tail -> all committed writes recover; an early flipped byte -> replay stops there (CRC), recovering nothing, engine still usable.

In [ ]:
!clang++ -std=c++20 -O2 test_fault_injection.cpp -o /tmp/tfi 2>/dev/null || g++ -std=c++20 -O2 test_fault_injection.cpp -o /tmp/tfi; /tmp/tfi

### Health / readiness probe (OB-3)
health() returns a ready verdict + the gauges behind it (catalog size, durable flag, pending-WAL records, resident bytes, dropped writes). ready is false when point-op writes have been dropped (store full) — a real degradation signal an orchestrator can act on.

In [ ]:
!clang++ -std=c++20 -O2 test_health.cpp -o /tmp/thl 2>/dev/null || g++ -std=c++20 -O2 test_health.cpp -o /tmp/thl; /tmp/thl

### Fuzz harness (untrusted-input crash-safety)
Seeded pseudo-random + mutated inputs hammer the untrusted paths (parse_query, deserialize_request, CSV) — they must never crash. Run under ASan/UBSan for memory safety.

In [ ]:
!clang++ -std=c++20 -O2 test_fuzz.cpp -o /tmp/tf 2>/dev/null || g++ -std=c++20 -O2 test_fuzz.cpp -o /tmp/tf; /tmp/tf

### Stress / load test
Sustained query load + heavy tiering churn + append + join at scale (50k-row columns in a tight RAM budget), every result checked against a closed-form oracle — behavior under load.

In [ ]:
!clang++ -std=c++20 -O2 test_stress.cpp -o /tmp/tstr 2>/dev/null || g++ -std=c++20 -O2 test_stress.cpp -o /tmp/tstr; /tmp/tstr

### TCP transport adapter
The length-prefixed wire protocol + matrix_serve_conn are verified over a socketpair (a framed request served over a real socket == a direct matrix_serve); matrix_serve_tcp is the host-runnable accept loop (bind is sandbox-blocked, so its loop is compile-verified here).

In [ ]:
!clang++ -std=c++20 -O2 test_server_tcp.cpp -o /tmp/ttcp 2>/dev/null || g++ -std=c++20 -O2 test_server_tcp.cpp -o /tmp/ttcp; /tmp/ttcp

### Connection read timeout (NW-5)
matrix_set_recv_timeout bounds how long a recv blocks, so a client that connects but never sends (slowloris) can't hang the single-owner serve loop — recv times out, serve_conn returns false, the loop drops the stuck connection. matrix_serve_tcp sets it on every accepted connection.

In [ ]:
!clang++ -std=c++20 -O2 test_recv_timeout.cpp -o /tmp/trt 2>/dev/null || g++ -std=c++20 -O2 test_recv_timeout.cpp -o /tmp/trt; /tmp/trt

### Client driver (NW-4)
MatrixClient is the app-side of the wire protocol — wraps a connected socket with typed get/put/query/health/stats calls (frame -> send -> recv -> deframe), the inverse of matrix_serve_conn. Driven end-to-end over a socketpair (server in a thread); client results == direct engine calls. (-pthread)

In [ ]:
!clang++ -std=c++20 -O2 -pthread test_client.cpp -o /tmp/tcl 2>/dev/null || g++ -std=c++20 -O2 -pthread test_client.cpp -o /tmp/tcl; /tmp/tcl

### Build version (BP-3)
version.hpp carries the semver build version; the engine reports it (string + a packed major<<32|minor<<16|patch numeric form), and STATS exposes the packed version over the wire so a client can read/compare which build it's talking to.

In [ ]:
!clang++ -std=c++20 -O2 test_version.cpp -o /tmp/tver 2>/dev/null || g++ -std=c++20 -O2 test_version.cpp -o /tmp/tver; /tmp/tver

### Structured logging (OB-1)
A tiny leveled logger (DEBUG<INFO<WARN<ERROR) with a runtime-settable threshold, so operators can filter/route diagnostics instead of grepping unconditional cout. Default WARN; writes to stderr. The engine's data-loss (dropped-write) path now logs at ERROR; set_log_level tunes it.

In [ ]:
!clang++ -std=c++20 -O2 test_logging.cpp -o /tmp/tlog 2>/dev/null || g++ -std=c++20 -O2 test_logging.cpp -o /tmp/tlog; /tmp/tlog

## 4b. Migration GPU proof (needs T4 GPU)

A column migrated HOST->VRAM is byte-intact AND GPU-scannable in place: the u32x4 kernel run over the promoted column's device pointer matches a CPU scan of the same bytes. Closes the heat->decision->migration->faster-scan loop on real hardware.

In [ ]:
!nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_migration_gpu.cpp -o test_migration_gpu && ./test_migration_gpu

## 4c. GPU AGG-2 cross-backend proof (needs T4 GPU)

The GPU SUM/MIN/MAX/COUNT reduction kernels must equal `matrix_cpu_reduce` over the SAME bytes — the correctness anchor for the GPU aggregate phase. Checks a matching predicate and an empty-match predicate (the empty sentinels: SUM 0, MIN UINT64_MAX, MAX 0). This is the merge gate for GPU-1: green here means the GPU SUM/MIN/MAX can land on `main`.

In [ ]:
!nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_gpu_agg.cu -o test_gpu_agg && ./test_gpu_agg

## 4d. GPU-4 predicate cross-backend proof (needs T4 GPU)

The GPU predicate-filtered reductions (GE/LT/LE/EQ/NE/BETWEEN, not just `value > threshold`) must equal `matrix_cpu_reduce_pred` over the SAME bytes — for every MatrixCmp op x {COUNT,SUM,MIN,MAX} plus an empty-match case. Merge gate for GPU-4 (GPU WHERE matches the CPU's exactly).

In [ ]:
!nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_gpu_pred.cu -o test_gpu_pred && ./test_gpu_pred

## 4e. GPU-2 grouped cross-backend proof (needs T4 GPU)

The GPU per-group reduction (one atomic per row into its dense group slot) must equal `matrix_cpu_group_reduce` for {COUNT,SUM,MIN,MAX}; the dataset includes out-of-range keys to verify both backends ignore them. Merge gate for GPU-2 (GROUP BY on the GPU).

In [ ]:
!nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_gpu_grouped.cu -o test_gpu_grouped && ./test_gpu_grouped

## 4f. GPU-5 typed (int64 + double) cross-backend proof (needs T4 GPU)

The GPU int64 + double predicate reductions must equal `matrix_cpu_reduce_pred_i64`/`_f64` over the SAME bytes (incl. negatives, values > 2^32, and fractions), for several predicates x {COUNT,SUM,MIN,MAX}. int64 uses native signed atomics; double MIN/MAX use a CAS loop on the bit pattern (no native double atomicMin/Max). Merge gate for GPU-5. (double data is half-integers so the order-dependent float SUM is bit-exact across the GPU's nondeterministic accumulation order.)

In [ ]:
!nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_gpu_typed.cu -o test_gpu_typed && ./test_gpu_typed

## 4g. GPU-3 VRAM catalog proof (needs T4 GPU)

The payoff: the *analytical* `execute_query` path runs on the GPU. A catalog column is pinned to DEVICE/VRAM (`pin_device`), then `execute_query` reduces it **in place on the GPU** (the verified `matrix_gpu_reduce_dev_*` kernels) instead of borrowing it down to HOST. Must equal `matrix_cpu_reduce_*` over the same bytes for u32/i64/f64 x {COUNT,SUM,MIN,MAX} x {filtered,unfiltered}. Merge gate for GPU-3 — the live-engine version of the 24x scan thesis.

In [ ]:
!nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_gpu_catalog.cu -o test_gpu_catalog && ./test_gpu_catalog

## 4h. GPU-3g grouped VRAM proof (needs T4 GPU)

Completes the analytical-on-GPU surface: a `GROUP BY` over DEVICE/VRAM-resident key+value columns runs **in place on the GPU** (the `matrix_group_*_kernel[_pred]` kernels) instead of borrowing both down to HOST. Must equal `matrix_cpu_group_reduce(_pred)` per group for {COUNT,SUM,MIN,MAX} x {filtered,unfiltered}. Merge gate for grouped-on-device (the filtered grouped kernel is new this round — a predicate-gated variant of the 4e-verified unfiltered kernel).

In [ ]:
!nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA test_gpu_catalog_grouped.cu -o test_gpu_catalog_grouped && ./test_gpu_catalog_grouped

## 3b. Cost-model unit test (CPU, no GPU)

Pure-function check of the router's placement decisions — point ops -> HOST, small scans -> HOST, large scans -> DEVICE, monotonic crossover.

In [ ]:
!clang++ -std=c++20 -O2 test_cost_model.cpp -o /tmp/tcm 2>/dev/null || g++ -std=c++20 -O2 test_cost_model.cpp -o /tmp/tcm; /tmp/tcm

## 4. Build & run on the GPU

`-x cu` compiles `main.cpp` as CUDA so the `.cuh` kernels link in. `-D_GNU_SOURCE` exposes Linux thread-affinity APIs; `-Xcompiler -pthread` links std::thread.

In [ ]:
!nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA main.cpp -o matrixdb_proto

In [ ]:
!./matrixdb_proto

## 5. CPU fallback (run this if no GPU runtime is selected)

Same code, same asserts, no CUDA — proves the logic without a GPU.

In [ ]:
!g++ -std=c++17 -O3 -D_GNU_SOURCE -pthread main.cpp -o matrixdb_cpu && ./matrixdb_cpu